In [ ]:
!pip -q install datasets pandas pyarrow

In [ ]:
import pandas as pd
from datasets import load_dataset

In [ ]:
ds = load_dataset("loopholetape/pumpfun-launches", "launches", split="train")

In [ ]:
# 2) Turn it into a pandas table so we can inspect it easily
df = ds.to_pandas()

# 3) Basic facts
print("Rows, columns:", df.shape)
print()
print("Column names:")
print(list(df.columns))
print()

# 4) Time range (launched_at is text, so convert it to real dates first)
df["launched_dt"] = pd.to_datetime(df["launched_at"], errors="coerce", utc=True)
print("First launch:", df["launched_dt"].min())
print("Last launch: ", df["launched_dt"].max())
print()

# 5) Launches per day (should be ~25 days)
per_day = df.groupby(df["launched_dt"].dt.date).size()
print("Launches per day:")
print(per_day.to_string())
print()

# 6) Base rates: what share of launches graduated or got flagged as rugged?
for col in ["graduated", "rugged", "mayhem"]:
    if col in df.columns:
        print(f"{col}: mean = {df[col].mean():.4%}  (non-null rows: {df[col].notna().sum()})")

# 7) How many rows have early-life features? (the card says about half)
if "f10_rs" in df.columns:
    print()
    print("Rows with early features (f10_rs not null):",
          df["f10_rs"].notna().sum(), "of", len(df))

In [ ]:
from huggingface_hub import HfApi

# Record exactly which version of the dataset we are using
info = HfApi().dataset_info("loopholetape/pumpfun-launches")
print("Dataset version (commit):", info.sha)
print("Last modified:", info.last_modified)

# Save a frozen local copy (lives in Colab's temporary storage)
df.to_parquet("/content/launches_snapshot.parquet")
print("Saved to /content/launches_snapshot.parquet")

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/memecoin_rl', exist_ok=True)
df.to_parquet('/content/drive/MyDrive/memecoin_rl/launches_snapshot.parquet')
print("Saved to Google Drive")

In [ ]:
df["day"] = df["launched_dt"].dt.date

# 1) What share of each column is blank (null), day by day?
cols = ["graduated", "rugged", "curve_complete", "decoder_complete", "terminal_rs", "f10_rs"]
print("Share of blank values per day:")
print(df.groupby("day")[cols].agg(lambda s: s.isna().mean()).round(3))
print()

# 2) Rates per day (averages ignore blanks)
print("Rates per day:")
print(df.groupby("day")[["graduated", "rugged", "mayhem"]].mean().round(4))
print()

# 3) Do 'graduated' and 'rugged' overlap? (-1 means unknown)
print("Graduated (rows) vs rugged (columns); -1 = unknown:")
print(pd.crosstab(df["graduated"].fillna(-1), df["rugged"].fillna(-1)))
print()

# 4) The new columns
for c in ["features_basis", "quote_asset", "rug_branch_inferred"]:
    print(f"{c}:")
    print(df[c].value_counts(dropna=False).head(10))
    print()

print(df[["silence_3s", "silence_60s", "create_lag_s"]].describe().T)

In [ ]:

# Step 3: Check dataset integrity and group differences

# 1) Check whether each mint appears only once
print("=== Mint integrity ===")
print("Total rows:", len(df))
print("Unique mints:", df["mint"].nunique())
print("Missing mint values:", df["mint"].isna().sum())
print("Duplicate mint rows:", df["mint"].duplicated().sum())
print()

# 2) Compare outcomes and feature availability by quote asset
print("=== Results by quote_asset ===")

asset_summary = df.groupby("quote_asset", dropna=False).agg(
    launches=("mint", "size"),
    unique_mints=("mint", "nunique"),
    graduation_coverage=("graduated", lambda s: s.notna().mean()),
    graduation_rate=("graduated", "mean"),
    rug_coverage=("rugged", lambda s: s.notna().mean()),
    rug_rate=("rugged", "mean"),
    f10_coverage=("f10_rs", lambda s: s.notna().mean()),
)

print(asset_summary.round(4))
print()

# 3) Compare the recorded feature sources
print("=== Results by features_basis ===")

basis_summary = df.groupby("features_basis", dropna=False).agg(
    launches=("mint", "size"),
    graduation_coverage=("graduated", lambda s: s.notna().mean()),
    graduation_rate=("graduated", "mean"),
    rug_coverage=("rugged", lambda s: s.notna().mean()),
    rug_rate=("rugged", "mean"),
    f10_coverage=("f10_rs", lambda s: s.notna().mean()),
)

print(basis_summary.round(4))


In [ ]:
# Step 4: Investigate the harness_sample rows

# 1) Check which feature sources appear under each quote asset
print("=== Feature source vs quote asset ===")
print(pd.crosstab(
    df["features_basis"].fillna("<missing>"),
    df["quote_asset"].fillna("<missing>"),
    margins=True
))
print()

# 2) Compare the dates covered by the harness sample
h = df.loc[df["features_basis"].eq("harness_sample")].copy()

print("=== Harness sample overview ===")
print("Rows:", len(h))
print("Unique mints:", h["mint"].nunique())
print("First launch:", h["launched_dt"].min())
print("Last launch: ", h["launched_dt"].max())
print()

print("Harness launches per day:")
print(h.groupby(h["launched_dt"].dt.date).size().to_string())
print()

# 3) Inspect 10 representative rows
cols = [
    "mint",
    "launched_at",
    "quote_asset",
    "mayhem",
    "graduated",
    "rugged",
    "features_basis",
    "f10_rs",
    "f60_rs",
    "create_lag_s",
    "silence_3s",
    "silence_60s",
]

print("=== Example harness rows ===")
print(
    h[cols].sample(n=min(10, len(h)), random_state=42)
    .to_string(index=False)
)

In [ ]:

# Step 5: Audit the main SOL replay cohort

# Keep the original df unchanged.
mask = (
    df["features_basis"].eq("tape_replay")
    & df["quote_asset"].eq("sol")
)
sol_replay = df.loc[mask].copy()

print("=== Main cohort ===")
print("Rows:", len(sol_replay))
print("Unique mints:", sol_replay["mint"].nunique())
print("First launch:", sol_replay["launched_dt"].min())
print("Last launch: ", sol_replay["launched_dt"].max())
print()

# 1) Number of launches per day
sol_replay["day"] = sol_replay["launched_dt"].dt.date

print("=== Launches per day ===")
print(sol_replay.groupby("day").size().to_string())
print()

# 2) Check outcome label coverage
print("=== Outcome labels ===")

for c in ["graduated", "rugged", "curve_complete", "decoder_complete"]:
    if c in sol_replay.columns:
        print(f"\n{c}:")
        print(
            sol_replay[c]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

# 3) Audit missing values in potentially useful features
feature_cols = [
    "mayhem",
    "dev_buy_sol",
    "dev_share",
    "f10_dev_share",
    "f10_insider_share",
    "f10_bundle_wallets",
    "f10_rep_n",
    "f10_rep_rug",
    "f10_rep_grad",
    "f10_rs",
    "f10_n_buys",
    "f10_age",
    "f60_rs",
    "f60_insider_share",
    "f60_age",
    "create_lag_s",
    "silence_3s",
    "silence_60s",
]

feature_cols = [c for c in feature_cols if c in sol_replay.columns]

missing = sol_replay[feature_cols].isna().mean()

print("\n=== Feature availability ===")
print(
    pd.DataFrame({
        "missing_rows": sol_replay[feature_cols].isna().sum(),
        "missing_pct": (missing * 100).round(2),
        "available_pct": ((1 - missing) * 100).round(2),
    }).sort_values("missing_pct", ascending=False).to_string()
)


In [ ]:

# Step 6: Audit outcomes and missing representative features

# 1) Summarize outcome rates and label coverage
print("=== Overall outcome audit ===")

print("Launches:", len(sol_replay))
print(
    "Graduation rate:",
    f"{sol_replay['graduated'].mean():.3%}",
)
print(
    "Rug label coverage:",
    f"{sol_replay['rugged'].notna().mean():.3%}",
)
print(
    "Rug rate among known labels:",
    f"{sol_replay['rugged'].mean():.3%}",
)
print(
    "Curve completion rate:",
    f"{sol_replay['curve_complete'].mean():.3%}",
)
print()

# 2) Check whether graduation and rugged overlap
print("=== Graduated vs rugged ===")

print(
    pd.crosstab(
        sol_replay["graduated"].fillna(-1),
        sol_replay["rugged"].fillna(-1),
        rownames=["graduated (-1 = unknown)"],
        colnames=["rugged (-1 = unknown)"],
    )
)
print()

# 3) Compare outcomes and coverage by launch day
print("=== Daily outcome audit ===")

daily = sol_replay.groupby("day").agg(
    launches=("mint", "size"),
    graduation_rate=("graduated", "mean"),
    rug_label_coverage=("rugged", lambda s: s.notna().mean()),
    rug_rate_known=("rugged", "mean"),
    f10_coverage=("f10_rs", lambda s: s.notna().mean()),
)

print(daily.round(4).to_string())
print()

# 4) Check whether representative-label missingness relates to rep_n
print("=== Representative feature availability ===")

rep_n = sol_replay["f10_rep_n"]

rep_n_status = pd.Series(
    "positive_or_other", index=sol_replay.index, dtype="object"
)
rep_n_status.loc[rep_n.isna()] = "missing"
rep_n_status.loc[rep_n.eq(0)] = "zero"

print("f10_rep_n values:")
print(rep_n.describe().round(3))
print()

print("f10_rep_rug missingness by f10_rep_n status (row fractions):")
print(
    pd.crosstab(
        rep_n_status,
        sol_replay["f10_rep_rug"].isna(),
        normalize="index",
    )
    .rename(columns={False: "rep_rug_present", True: "rep_rug_missing"})
    .round(3)
)


In [ ]:

# Step 7: Inspect timestamp fields and event timing

# 1) Inspect the raw timestamp columns without modifying them
print("=== Timestamp columns ===")

for c in ["t0", "launched_at", "curve_read_at"]:
    if c in sol_replay.columns:
        s = sol_replay[c]
        print(f"\n{c}")
        print("dtype:", s.dtype)
        print("missing:", s.isna().sum())
        print("examples:")
        print(s.dropna().head(5).to_list())

print()

# 2) Inspect event-time and early-feature timing fields
print("=== Timing field distributions ===")

timing_cols = [
    "migrate_dt_s",
    "rug_dt_s",
    "dev_first_sell_dt_s",
    "create_lag_s",
    "silence_3s",
    "silence_60s",
    "f10_age",
    "f60_age",
]

for c in timing_cols:
    if c not in sol_replay.columns:
        continue

    s = pd.to_numeric(sol_replay[c], errors="coerce")

    print(f"\n{c}")
    print("Available:", s.notna().sum(), "/", len(s))
    print(
        s.describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ).round(3).to_string()
    )

print()

# 3) Inspect the fields that describe final outcomes
print("=== Example outcome records ===")

outcome_cols = [
    "mint",
    "launched_at",
    "migrate_dt_s",
    "rug_dt_s",
    "dev_first_sell_dt_s",
    "peak_rs",
    "terminal_rs",
    "graduated",
    "rugged",
    "curve_complete",
]

outcome_cols = [
    c for c in outcome_cols if c in sol_replay.columns
]

print(
    sol_replay[outcome_cols]
    .sample(n=min(10, len(sol_replay)), random_state=42)
    .to_string(index=False)
)


In [ ]:

# Step 8: Verify timestamp consistency

# 1) Convert Unix timestamps into UTC datetimes
t0_dt = pd.to_datetime(sol_replay["t0"], unit="s", utc=True)
curve_read_dt = pd.to_datetime(
    sol_replay["curve_read_at"], unit="s", utc=True
)
launch_dt = sol_replay["launched_dt"]

print("=== Launch timestamp consistency ===")

launch_diff_s = (t0_dt - launch_dt).dt.total_seconds()

print("Unique t0 timestamps:", t0_dt.nunique())
print("Rows differing by more than 1 second:",
      (launch_diff_s.abs() > 1).sum())
print("Maximum absolute difference (seconds):",
      launch_diff_s.abs().max())
print()

# 2) Check the curve-read timestamp
print("=== Curve-read timestamp audit ===")

print("Unique curve_read_at timestamps:", curve_read_dt.nunique())
print("Earliest curve_read_at:", curve_read_dt.min())
print("Latest curve_read_at:  ", curve_read_dt.max())
print()

read_before_launch = curve_read_dt < launch_dt

print("Curve read before launch:",
      int(read_before_launch.sum()))
print("Curve read at or after launch:",
      int((~read_before_launch).sum()))
print("Percentage read before launch:",
      f"{read_before_launch.mean():.2%}")
print()

read_age_s = (curve_read_dt - launch_dt).dt.total_seconds()

print("Curve-read time minus launch time (seconds):")
print(
    read_age_s.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
    ).round(2).to_string()
)
print()

# 3) Check whether rug timing coincides with creator first-sell timing
print("=== Rug and creator-sell timing ===")

paired = sol_replay.loc[
    sol_replay["rug_dt_s"].notna()
    & sol_replay["dev_first_sell_dt_s"].notna(),
    ["rug_dt_s", "dev_first_sell_dt_s"],
].copy()

delta_s = (
    paired["rug_dt_s"] - paired["dev_first_sell_dt_s"]
)

print("Rows with both timestamps:", len(paired))
print("Exactly equal:", int(delta_s.eq(0).sum()))
print("Within 1 millisecond:",
      int(delta_s.abs().le(0.001).sum()))
print("Median difference (seconds):",
      round(delta_s.median(), 4))
print("Rug time earlier than creator first sell:",
      int(delta_s.lt(-0.001).sum()))
print("Rug time later than creator first sell:",
      int(delta_s.gt(0.001).sum()))


In [ ]:

# Step 9: Compare outcomes by early-feature availability

# Keep all rows; this is a diagnostic, not a filtering step.
checks = {
    "f10_rs": sol_replay["f10_rs"].notna(),
    "f60_rs": sol_replay["f60_rs"].notna(),
    "f10_rep_rug": sol_replay["f10_rep_rug"].notna(),
}

for feature, available in checks.items():
    print(f"=== Outcomes by {feature} availability ===")

    group = sol_replay.assign(
        feature_status=available.map({
            True: "available",
            False: "missing",
        })
    )

    summary = group.groupby("feature_status").agg(
        launches=("mint", "size"),
        graduation_coverage=("graduated", lambda s: s.notna().mean()),
        graduation_rate=("graduated", "mean"),
        rug_coverage=("rugged", lambda s: s.notna().mean()),
        rug_rate_known=("rugged", "mean"),
    )

    print(summary.round(4).to_string())
    print()

# Check whether zero representative-wallet counts differ from
# genuinely missing representative-wallet counts.
print("=== Outcomes by f10_rep_n status ===")

rep_n_status = pd.Series(
    "positive",
    index=sol_replay.index,
    dtype="object",
)
rep_n_status.loc[sol_replay["f10_rep_n"].eq(0)] = "zero"
rep_n_status.loc[sol_replay["f10_rep_n"].isna()] = "missing"

rep_summary = sol_replay.assign(
    rep_n_status=rep_n_status
).groupby("rep_n_status").agg(
    launches=("mint", "size"),
    graduation_rate=("graduated", "mean"),
    rug_coverage=("rugged", lambda s: s.notna().mean()),
    rug_rate_known=("rugged", "mean"),
)

print(rep_summary.round(4).to_string())


In [ ]:

# Step 10: Inspect dataset documentation and known issues

from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path

repo_id = "loopholetape/pumpfun-launches"

# List files in the repository
repo_files = HfApi().list_repo_files(
    repo_id,
    repo_type="dataset",
)

# Find documentation files
docs = [
    f for f in repo_files
    if any(
        word in Path(f).name.lower()
        for word in [
            "readme",
            "issue",
            "schema",
            "datasheet",
            "manifest",
        ]
    )
    and Path(f).suffix.lower() in [".md", ".txt"]
]

print("=== Documentation files found ===")
for name in docs:
    print(name)
print()

# Read the dataset card and any separate issue documentation
selected = [
    f for f in docs
    if Path(f).name.lower() == "readme.md"
    or "issue" in Path(f).name.lower()
]

for name in selected:
    print("=" * 70)
    print("FILE:", name)
    print("=" * 70)

    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=name,
        repo_type="dataset",
    )

    with open(local_path, "r", encoding="utf-8") as file:
        content = file.read()

    # Keep notebook output manageable
    print(content[:15000])

    if len(content) > 15000:
        print("\n[Output truncated; remaining characters:", len(content) - 15000, "]")

    print()


In [ ]:

# Step 11: Read the remaining documented data issues

from pathlib import Path
from huggingface_hub import hf_hub_download

repo_id = "loopholetape/pumpfun-launches"

# Prefer the exact dataset commit recorded earlier in Cell 3.
revision = info.sha if "info" in globals() else None

print("Documentation revision:", revision or "current repository version")
print()

path = hf_hub_download(
    repo_id=repo_id,
    filename="KNOWN_ISSUES.md",
    repo_type="dataset",
    revision=revision,
)

with open(path, "r", encoding="utf-8") as f:
    issues_text = f.read()

# Print KI-11 onward instead of truncating the document at KI-10.
start = issues_text.find("## KI-11")

if start == -1:
    print("Could not find KI-11. Please check the downloaded file.")
else:
    print(issues_text[start:])


In [ ]:

# Step 12: Compare classic and mayhem launches

print("=== Cohort verification ===")
print("Rows:", len(sol_replay))
print("Decoder versions:")
print(sol_replay["decoder_complete"].value_counts(dropna=False).to_string())
print()

# Compare outcomes separately for classic and mayhem launches
summary = sol_replay.groupby("mayhem", dropna=False).agg(
    launches=("mint", "size"),
    graduations=("graduated", "sum"),
    graduation_rate=("graduated", "mean"),
    rug_label_coverage=("rugged", lambda s: s.notna().mean()),
    rug_rate_known=("rugged", "mean"),
    f10_coverage=("f10_rs", lambda s: s.notna().mean()),
)

summary.index = [
    "Classic (mayhem=0)" if x == 0
    else "Mayhem (mayhem=1)" if x == 1
    else f"Unknown ({x})"
    for x in summary.index
]

print("=== Outcome rates by launch type ===")
print(summary.round(4).to_string())
print()

# Graduated and rugged are not mutually exclusive.
print("=== Graduation vs rug label by launch type ===")
print(
    pd.crosstab(
        [sol_replay["mayhem"], sol_replay["graduated"]],
        sol_replay["rugged"].fillna(-1),
        rownames=["mayhem", "graduated"],
        colnames=["rugged (-1 = unknown)"],
    ).to_string()
)


In [ ]:

# Step 13: Create and inspect chronological splits
# No model training yet.

# Use classic launches only; keep the original sol_replay unchanged.
classic = sol_replay.loc[
    sol_replay["mayhem"].eq(0)
].copy()

classic["day"] = pd.to_datetime(
    classic["launched_dt"], utc=True
).dt.date

train_end = pd.Timestamp("2026-09-30").date()
validation_day = pd.Timestamp("2026-10-01").date()
test_day = pd.Timestamp("2026-10-02").date()

classic["split"] = "unassigned"

classic.loc[classic["day"] <= train_end, "split"] = "train"
classic.loc[classic["day"] == validation_day, "split"] = "validation"
classic.loc[classic["day"] == test_day, "split"] = "test"

# 1) Confirm row counts, label coverage, and graduation rates
print("=== Split summary ===")

split_summary = classic.groupby("split", sort=False).agg(
    launches=("mint", "size"),
    known_labels=("graduated", "count"),
    graduations=("graduated", "sum"),
    graduation_rate=("graduated", "mean"),
)

split_summary["label_coverage"] = (
    split_summary["known_labels"] / split_summary["launches"]
)

print(split_summary.round(4).to_string())
print()

# 2) Inspect graduation rate by individual launch day
print("=== Daily graduation rates ===")

daily = classic.groupby("day").agg(
    launches=("mint", "size"),
    graduations=("graduated", "sum"),
    graduation_rate=("graduated", "mean"),
)

print(daily.round(4).to_string())
print()

# 3) Check early-feature missingness across splits
feature_cols = [
    "dev_buy_sol",
    "dev_share",
    "f10_rs",
    "f10_n_buys",
    "f10_insider_share",
    "f10_bundle_wallets",
    "f10_rep_n",
    "f10_rep_rug",
    "f10_rep_grad",
    "f10_age",
]

feature_cols = [
    c for c in feature_cols if c in classic.columns
]

missing_pct = (
    classic.groupby("split", sort=False)[feature_cols]
    .agg(lambda s: s.isna().mean() * 100)
    .round(2)
)

print("=== Feature missingness by split (%) ===")
print(missing_pct.to_string())
print()

# 4) Confirm that every row belongs to exactly one intended split
print("Unassigned rows:", (classic["split"] == "unassigned").sum())
print("Unique mints:", classic["mint"].nunique())
print("Duplicate mints:", classic["mint"].duplicated().sum())


In [ ]:

# Step 14: First supervised baseline
# Predict graduation using launch-time and early-life features.

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

# 1) Select features that could be available near launch.
features = [
    "dev_buy_sol",
    "dev_share",
    "f10_rs",
    "f10_n_buys",
    "f10_insider_share",
    "f10_bundle_wallets",
    "f10_rep_n",
    "f10_age",
]

# Confirm all requested columns exist.
missing_columns = [c for c in features if c not in classic.columns]
if missing_columns:
    raise ValueError(f"Missing feature columns: {missing_columns}")

# 2) Separate the chronological datasets.
train_df = classic.loc[classic["split"] == "train"].copy()
val_df = classic.loc[classic["split"] == "validation"].copy()
test_df = classic.loc[classic["split"] == "test"].copy()

X_train = train_df[features]
y_train = train_df["graduated"].astype(int)

X_val = val_df[features]
y_val = val_df["graduated"].astype(int)

X_test = test_df[features]
y_test = test_df["graduated"].astype(int)

print("=== Training setup ===")
print("Features:", features)
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("Training graduation rate:", f"{y_train.mean():.3%}")
print()

# 3) Train one small, initial model.
model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)

model.fit(X_train, y_train)

print("Model training complete.")
print()

# 4) Evaluate probability ranking and calibration.
# No threshold or model setting is being tuned on the test set.

def evaluate_model(name, X, y):
    y_true = np.asarray(y, dtype=int)
    prob = model.predict_proba(X)[:, 1]

    prevalence = y_true.mean()
    ap = average_precision_score(y_true, prob)
    auc = roc_auc_score(y_true, prob)
    brier = brier_score_loss(y_true, prob)

    print(f"=== {name} ===")
    print("Graduation rate:", f"{prevalence:.3%}")
    print("Average precision (AP):", f"{ap:.4f}")
    print("ROC-AUC:", f"{auc:.4f}")
    print("Brier score (lower is better):", f"{brier:.5f}")
    print("No-skill AP reference:", f"{prevalence:.4f}")
    print("No-skill ROC-AUC reference: 0.5000")
    print()

    # Evaluate how well the model ranks the highest-scoring launches.
    for fraction in [0.01, 0.03, 0.05]:
        k = max(1, int(np.ceil(len(y_true) * fraction)))
        top_idx = np.argsort(prob)[-k:]

        top_precision = y_true[top_idx].mean()
        total_positives = y_true.sum()
        top_recall = (
            y_true[top_idx].sum() / total_positives
            if total_positives else 0.0
        )
        lift = (
            top_precision / prevalence
            if prevalence else float("nan")
        )

        print(
            f"Top {fraction:.0%}: "
            f"precision={top_precision:.3%}, "
            f"recall={top_recall:.3%}, "
            f"lift={lift:.2f}x"
        )

    print()

evaluate_model("Validation — October 1", X_val, y_val)
evaluate_model("Test — October 2", X_test, y_test)


In [ ]:

# Step 15: Model ablation and shuffled-label sanity check

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

# Use the existing chronological train/validation sets.
# October 2 has already been inspected, so use October 1 for debugging.

model_specs = {
    "Launch-only": [
        "dev_buy_sol",
        "dev_share",
    ],
    "Without f10_rs": [
        c for c in features if c != "f10_rs"
    ],
    "Full model": features,
}

def make_model():
    return HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.08,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42,
    )

def print_scores(name, y_true, probabilities):
    print(f"\n{name}")
    print("Average precision:",
          f"{average_precision_score(y_true, probabilities):.4f}")
    print("ROC-AUC:",
          f"{roc_auc_score(y_true, probabilities):.4f}")
    print("Brier score:",
          f"{brier_score_loss(y_true, probabilities):.5f}")

print("=== Validation baseline ===")

# Constant-probability reference model.
dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)

dummy_prob = dummy.predict_proba(X_val)[:, 1]
print_scores("Prior baseline", y_val, dummy_prob)

# 1) Compare the three feature sets.
for name, cols in model_specs.items():
    candidate = make_model()

    candidate.fit(
        train_df[cols],
        y_train,
    )

    probabilities = candidate.predict_proba(
        val_df[cols]
    )[:, 1]

    print_scores(name, y_val, probabilities)

# 2) Negative control: shuffle the training labels.
# Keep the feature rows unchanged, but destroy the relationship
# between features and graduation labels.
shuffled_y = y_train.sample(
    frac=1,
    random_state=2026,
).to_numpy()

control = make_model()
control.fit(X_train, shuffled_y)

control_prob = control.predict_proba(X_val)[:, 1]

print_scores(
    "Shuffled-label negative control",
    y_val,
    control_prob,
)

print("\nValidation graduation rate:", f"{y_val.mean():.4%}")
print("Expected no-skill ROC-AUC: approximately 0.5000")


In [ ]:

# Step 16: Repeat the shuffled-label control safely

import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

# Reset indexes so feature rows have an unambiguous positional order.
X_train_check = X_train.reset_index(drop=True).copy()
X_val_check = X_val.reset_index(drop=True).copy()

y_train_check = np.asarray(y_train, dtype=np.int64).copy()
y_val_check = np.asarray(y_val, dtype=np.int64).copy()

print("=== Data alignment ===")
print("Training feature rows:", len(X_train_check))
print("Training labels:", len(y_train_check))
print("Validation feature rows:", len(X_val_check))
print("Validation labels:", len(y_val_check))
print("Original training graduation rate:",
      f"{y_train_check.mean():.4%}")
print("Validation graduation rate:",
      f"{y_val_check.mean():.4%}")
print()

# The feature and label row counts must agree.
assert len(X_train_check) == len(y_train_check)
assert len(X_val_check) == len(y_val_check)

# Fit the same model on independently shuffled labels.
for seed in [11, 42, 2026]:
    rng = np.random.default_rng(seed)
    y_permuted = rng.permutation(y_train_check)

    correlation = np.corrcoef(
        y_train_check, y_permuted
    )[0, 1]

    control = make_model()
    control.fit(X_train_check, y_permuted)

    probabilities = control.predict_proba(
        X_val_check
    )[:, 1]

    ap = average_precision_score(
        y_val_check, probabilities
    )
    auc = roc_auc_score(
        y_val_check, probabilities
    )

    print(f"=== Shuffled-label control: seed {seed} ===")
    print("Shuffled training graduation rate:",
          f"{y_permuted.mean():.4%}")
    print("Correlation: original vs shuffled labels:",
          f"{correlation:.5f}")
    print("Validation average precision:", f"{ap:.4f}")
    print("Validation ROC-AUC:", f"{auc:.4f}")
    print()


In [ ]:

# Step 17: Diagnose the shuffled-label control

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

# Use aligned arrays from Step 16.
rng = np.random.default_rng(42)
y_permuted = rng.permutation(y_train_check)

control_diag = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)

control_diag.fit(X_train_check, y_permuted)

p = control_diag.predict_proba(X_val_check)[:, 1]

print("=== Model and probability checks ===")
print("Model classes:", control_diag.classes_)
print("Validation prevalence:", round(y_val_check.mean(), 5))
print("Mean predicted probability:", round(p.mean(), 5))
print("Min / median / max probability:",
      np.round([p.min(), np.median(p), p.max()], 6))
print("Distinct probabilities (rounded to 8 decimals):",
      np.unique(np.round(p, 8)).size)
print()

print("=== Scores against the real validation labels ===")
print("Average precision:",
      round(average_precision_score(y_val_check, p), 4))
print("ROC-AUC:",
      round(roc_auc_score(y_val_check, p), 4))
print("ROC-AUC with reversed score:",
      round(roc_auc_score(y_val_check, 1 - p), 4))
print()

print("Mean prediction for actual graduates:",
      round(p[y_val_check == 1].mean(), 6))
print("Mean prediction for actual non-graduates:",
      round(p[y_val_check == 0].mean(), 6))
print()

# Check the metric implementation using independent random scores.
print("=== Independent random-ranking reference ===")

random_aps = []
random_aucs = []

for seed in range(5):
    random_scores = np.random.default_rng(seed).random(
        len(y_val_check)
    )
    random_aps.append(
        average_precision_score(y_val_check, random_scores)
    )
    random_aucs.append(
        roc_auc_score(y_val_check, random_scores)
    )

print("Random-ranking AP values:",
      np.round(random_aps, 4))
print("Random-ranking ROC-AUC values:",
      np.round(random_aucs, 4))
print("Mean random-ranking AP:",
      round(float(np.mean(random_aps)), 4))
print("Mean random-ranking ROC-AUC:",
      round(float(np.mean(random_aucs)), 4))


In [ ]:

# Step 18: Compare real-label performance with a permutation baseline

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

results = []

# Original model's validation performance
original_prob = model.predict_proba(X_val)[:, 1]

results.append({
    "run": "real_labels",
    "AP": average_precision_score(y_val, original_prob),
    "ROC_AUC": roc_auc_score(y_val, original_prob),
})

# Ten independent label permutations
for seed in range(10):
    rng = np.random.default_rng(1000 + seed)
    y_shuffled = rng.permutation(y_train_check)

    perm_model = make_model()
    perm_model.fit(X_train_check, y_shuffled)

    perm_prob = perm_model.predict_proba(X_val_check)[:, 1]

    results.append({
        "run": f"shuffle_{seed + 1}",
        "AP": average_precision_score(y_val_check, perm_prob),
        "ROC_AUC": roc_auc_score(y_val_check, perm_prob),
    })

results_df = pd.DataFrame(results)

print("=== Individual results ===")
print(results_df.round(4).to_string(index=False))
print()

null = results_df.loc[results_df["run"] != "real_labels"]

print("=== Shuffled-label distribution ===")
print(null[["AP", "ROC_AUC"]].agg(
    ["mean", "std", "min", "median", "max"]
).round(4).to_string())
print()

print("=== Original model vs permutation controls ===")
print("Original AP:", round(results_df.iloc[0]["AP"], 4))
print("Original ROC-AUC:", round(results_df.iloc[0]["ROC_AUC"], 4))
print("Shuffled runs with AP >= original:",
      int((null["AP"] >= results_df.iloc[0]["AP"]).sum()), "/ 10")
print("Shuffled runs with ROC-AUC >= original:",
      int((null["ROC_AUC"] >= results_df.iloc[0]["ROC_AUC"]).sum()), "/ 10")


In [ ]:

# Step 19: Measure feature importance on validation data

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

# Use the existing model and October 1 validation set.
baseline_prob = model.predict_proba(X_val)[:, 1]

baseline_ap = average_precision_score(y_val, baseline_prob)
baseline_auc = roc_auc_score(y_val, baseline_prob)

print("Baseline AP:", round(baseline_ap, 4))
print("Baseline ROC-AUC:", round(baseline_auc, 4))
print()

rng = np.random.default_rng(2026)
results = []

# Shuffle each feature independently, five times.
for feature in features:
    ap_scores = []
    auc_scores = []

    for _ in range(5):
        X_shuffled = X_val.copy()

        # Preserve the feature's values and missingness rate,
        # but break its row-by-row relationship with the outcome.
        X_shuffled[feature] = rng.permutation(
            X_shuffled[feature].to_numpy()
        )

        probabilities = model.predict_proba(X_shuffled)[:, 1]

        ap_scores.append(
            average_precision_score(y_val, probabilities)
        )
        auc_scores.append(
            roc_auc_score(y_val, probabilities)
        )

    results.append({
        "feature": feature,
        "baseline_AP": baseline_ap,
        "AP_after_shuffle_mean": np.mean(ap_scores),
        "AP_drop_mean": baseline_ap - np.mean(ap_scores),
        "AP_drop_std": np.std(ap_scores),
        "AUC_drop_mean": baseline_auc - np.mean(auc_scores),
    })

importance = pd.DataFrame(results).sort_values(
    "AP_drop_mean", ascending=False
)

print("=== Feature permutation importance ===")
print(importance.round(4).to_string(index=False))
print()

print("Interpretation:")
print("- Larger positive AP_drop_mean means greater model reliance.")
print("- Near-zero importance can mean a feature adds little unique information.")
print("- Negative importance means shuffling improved AP in this experiment.")


In [ ]:

# Step 20: Validate f10_rs and early snapshot data quality

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

# 1) Inspect data-quality indicators across chronological splits.
print("=== Snapshot quality by split ===")

quality_rows = []

for split_name in ["train", "validation", "test"]:
    d = classic.loc[classic["split"].eq(split_name)]

    clean = (
        d["create_lag_s"].le(1)
        & d["silence_3s"].eq(0)
        & d["f10_rs"].notna()
    )

    quality_rows.append({
        "split": split_name,
        "launches": len(d),
        "f10_coverage": d["f10_rs"].notna().mean(),
        "create_lag_over_1s": d["create_lag_s"].gt(1).mean(),
        "silence_3s_positive": d["silence_3s"].gt(0).mean(),
        "clean_snapshot_share": clean.mean(),
        "graduation_rate": d["graduated"].mean(),
        "clean_graduation_rate": (
            d.loc[clean, "graduated"].mean()
            if clean.any() else np.nan
        ),
    })

print(
    pd.DataFrame(quality_rows)
    .set_index("split")
    .round(4)
    .to_string()
)
print()

# 2) Inspect graduation rates by f10_rs decile.
# These are descriptive outcome statistics, not causal effects.
observed = classic.loc[
    classic["f10_rs"].notna(),
    ["f10_rs", "graduated"],
].copy()

observed["reserve_decile"] = (
    pd.qcut(
        observed["f10_rs"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1
)

reserve_summary = observed.groupby("reserve_decile").agg(
    launches=("graduated", "size"),
    reserve_min=("f10_rs", "min"),
    reserve_median=("f10_rs", "median"),
    reserve_max=("f10_rs", "max"),
    graduations=("graduated", "sum"),
    graduation_rate=("graduated", "mean"),
)

print("=== Graduation rates by f10_rs decile ===")
print(reserve_summary.round(4).to_string())
print()

# 3) Compare the original validation result with a cleaner subset.
# The model is unchanged; we only restrict the validation rows.
clean_val = (
    val_df["create_lag_s"].le(1)
    & val_df["silence_3s"].eq(0)
    & val_df["f10_rs"].notna()
)

print("=== Validation performance by snapshot quality ===")

for name, mask in [
    ("All validation rows", pd.Series(True, index=val_df.index)),
    ("Clean early snapshots", clean_val),
]:
    X_part = X_val.loc[mask]
    y_part = y_val.loc[mask]

    probabilities = model.predict_proba(X_part)[:, 1]

    print(f"\n{name}")
    print("Rows:", len(y_part))
    print("Graduation rate:", f"{y_part.mean():.3%}")
    print(
        "Average precision:",
        f"{average_precision_score(y_part, probabilities):.4f}",
    )
    print(
        "ROC-AUC:",
        f"{roc_auc_score(y_part, probabilities):.4f}",
    )


In [ ]:

# Step 21: Check snapshot timing relative to graduation

import numpy as np
import pandas as pd

# The recorder restart changed the meaning of migrate_dt_s.
seam = pd.Timestamp("2026-09-29T10:56:23Z")

post_seam = classic.loc[
    classic["launched_dt"].ge(seam)
].copy()

# Focus on eventual graduates with a known snapshot age
# and recorded migration time.
graduated = post_seam.loc[
    post_seam["graduated"].eq(1)
    & post_seam["f10_rs"].notna()
    & post_seam["f10_age"].notna()
    & post_seam["migrate_dt_s"].notna()
].copy()

graduated["age_minus_migration_s"] = (
    graduated["f10_age"] - graduated["migrate_dt_s"]
)

print("=== Audit population ===")
print("Classic launches after recorder restart:", len(post_seam))
print("Graduates with usable snapshot and migration times:",
      len(graduated))
print()

# 1) Compare snapshot age with migration-event time.
delta = graduated["age_minus_migration_s"]

print("=== Snapshot timing relative to migration ===")
print("Snapshot age earlier than migration:",
      int(delta.lt(-0.001).sum()))
print("Times differ by at most 1 ms:",
      int(delta.abs().le(0.001).sum()))
print("Snapshot age later than migration:",
      int(delta.gt(0.001).sum()))
print()

print("Difference (snapshot age - migration time), seconds:")
print(
    delta.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
    ).round(4).to_string()
)
print()

# 2) Check how often extremely high reserves coincide with
# an observed graduation and a potentially late snapshot.
print("=== High early-reserve audit ===")

for threshold in [10, 50, 80]:
    subset = post_seam.loc[
        post_seam["f10_rs"].ge(threshold)
    ]

    print(f"\nf10_rs >= {threshold} SOL")
    print("Launches:", len(subset))
    print("Graduation rate:",
          f"{subset['graduated'].mean():.3%}"
          if len(subset) else "N/A")

    timing_known = subset.loc[
        subset["graduated"].eq(1)
        & subset["f10_age"].notna()
        & subset["migrate_dt_s"].notna()
    ]

    if len(timing_known):
        diff = (
            timing_known["f10_age"]
            - timing_known["migrate_dt_s"]
        )

        print("Graduates with both times:", len(timing_known))
        print("Snapshot age after migration time:",
              int(diff.gt(0.001).sum()))
        print("Snapshot age before migration time:",
              int(diff.lt(-0.001).sum()))
        print("Timing ambiguous within 1 ms:",
              int(diff.abs().le(0.001).sum()))


In [ ]:

# Step 22: Strict launch-time-only baseline

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

launch_features = ["dev_buy_sol", "dev_share"]

# Use the same chronological splits as before.
X_train_launch = train_df[launch_features]
X_val_launch = val_df[launch_features]
X_test_launch = test_df[launch_features]

y_train_launch = train_df["graduated"].astype(int)
y_val_launch = val_df["graduated"].astype(int)
y_test_launch = test_df["graduated"].astype(int)

print("=== Launch-time model setup ===")
print("Features:", launch_features)
print("Train rows:", len(X_train_launch))
print("Validation rows:", len(X_val_launch))
print("Test rows:", len(X_test_launch))
print()

# Train a fresh model using launch-time features only.
launch_model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)

launch_model.fit(X_train_launch, y_train_launch)

def evaluate_launch_model(name, X, y):
    y_true = np.asarray(y, dtype=int)
    probabilities = launch_model.predict_proba(X)[:, 1]
    prevalence = y_true.mean()

    print(f"=== {name} ===")
    print("Graduation rate:", f"{prevalence:.3%}")
    print(
        "Average precision:",
        f"{average_precision_score(y_true, probabilities):.4f}",
    )
    print(
        "ROC-AUC:",
        f"{roc_auc_score(y_true, probabilities):.4f}",
    )
    print(
        "Brier score:",
        f"{brier_score_loss(y_true, probabilities):.5f}",
    )

    for fraction in [0.01, 0.03, 0.05]:
        k = max(1, int(np.ceil(len(y_true) * fraction)))
        selected = np.argsort(probabilities)[-k:]

        precision = y_true[selected].mean()
        recall = y_true[selected].sum() / y_true.sum()
        lift = precision / prevalence

        print(
            f"Top {fraction:.0%}: "
            f"precision={precision:.3%}, "
            f"recall={recall:.3%}, "
            f"lift={lift:.2f}x"
        )

    print()

evaluate_launch_model(
    "Validation — October 1",
    X_val_launch,
    y_val_launch,
)

evaluate_launch_model(
    "Test — October 2 (exploratory)",
    X_test_launch,
    y_test_launch,
)


In [ ]:

# Step 23: Identify which launch-time feature drives prediction

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

# 1) Compare individual features with the two-feature model.
print("=== Single-feature models ===")

for feature in launch_features:
    candidate = HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.08,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42,
    )

    candidate.fit(train_df[[feature]], y_train_launch)

    print(f"\nFeature: {feature}")

    for split_name, X, y in [
        ("Validation", val_df[[feature]], y_val_launch),
        ("Test", test_df[[feature]], y_test_launch),
    ]:
        probabilities = candidate.predict_proba(X)[:, 1]

        k = max(1, int(np.ceil(len(y) * 0.01)))
        top_idx = np.argsort(probabilities)[-k:]
        top_precision = y.iloc[top_idx].mean()

        print(
            f"{split_name}: "
            f"AP={average_precision_score(y, probabilities):.4f}, "
            f"ROC-AUC={roc_auc_score(y, probabilities):.4f}, "
            f"top-1% precision={top_precision:.3%}"
        )

print()

# 2) Inspect feature distributions by actual outcome.
print("=== Feature distributions by graduation outcome ===")

for feature in launch_features:
    print(f"\n{feature}")

    stats = train_df.groupby("graduated")[feature].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )

    print(stats.round(5).to_string())

print()

# 3) Measure how much each feature matters to the combined model.
# Shuffle one validation feature at a time without retraining.
print("=== Two-feature model permutation check ===")

baseline_prob = launch_model.predict_proba(X_val_launch)[:, 1]
baseline_ap = average_precision_score(y_val_launch, baseline_prob)

print("Baseline validation AP:", round(baseline_ap, 4))

rng = np.random.default_rng(2026)

for feature in launch_features:
    shuffled_aps = []

    for _ in range(5):
        X_shuffled = X_val_launch.copy()
        X_shuffled[feature] = rng.permutation(
            X_shuffled[feature].to_numpy()
        )

        probabilities = launch_model.predict_proba(X_shuffled)[:, 1]
        shuffled_aps.append(
            average_precision_score(y_val_launch, probabilities)
        )

    print(
        f"{feature}: "
        f"mean AP after shuffle={np.mean(shuffled_aps):.4f}, "
        f"mean AP drop={baseline_ap - np.mean(shuffled_aps):.4f}"
    )


In [ ]:

# Step 24: Investigate suspicious feature values

import numpy as np
import pandas as pd

# 1) Count the suspicious values by graduation outcome.
print("=== Exact-value frequency by outcome ===")

for feature, suspicious_value in [
    ("dev_buy_sol", 85.00536),
    ("dev_share", 0.79310),
]:
    values = pd.to_numeric(classic[feature], errors="coerce")

    # Allow a very small floating-point tolerance.
    matches = np.isclose(
        values.to_numpy(dtype=float),
        suspicious_value,
        atol=1e-5,
        rtol=0,
        equal_nan=False,
    )

    print(f"\n{feature} ~= {suspicious_value}")

    result = classic.assign(
        suspicious_value=matches
    ).groupby("graduated").agg(
        launches=("mint", "size"),
        matching_rows=("suspicious_value", "sum"),
        graduation_rate=("graduated", "mean"),
    )

    result["matching_share"] = (
        result["matching_rows"] / result["launches"]
    )

    print(result.round(5).to_string())

# 2) See which exact values are most frequent among graduates.
print("\n=== Most frequent dev_buy_sol values among graduates ===")

graduates = classic.loc[classic["graduated"].eq(1)]

print(
    graduates["dev_buy_sol"]
    .round(5)
    .value_counts()
    .head(15)
    .to_string()
)

# 3) Check how often high dev_buy_sol values occur by outcome.
print("\n=== dev_buy_sol thresholds by outcome ===")

for threshold in [10, 50, 80, 85]:
    subset = classic.loc[
        classic["dev_buy_sol"].ge(threshold)
    ]

    print(
        f">= {threshold} SOL: "
        f"{len(subset)} launches; "
        f"graduation rate={subset['graduated'].mean():.3%}"
        if len(subset)
        else f">= {threshold} SOL: no rows"
    )

# 4) Inspect rows with the suspicious value.
print("\n=== Example rows with dev_buy_sol near 85.00536 ===")

example = classic.loc[
    np.isclose(
        classic["dev_buy_sol"].to_numpy(dtype=float),
        85.00536,
        atol=1e-5,
        rtol=0,
    ),
    [
        "mint",
        "launched_at",
        "dev_buy_sol",
        "dev_share",
        "graduated",
        "rugged",
        "features_basis",
        "decoder_complete",
    ],
]

print(example.head(15).to_string(index=False))


In [ ]:

# Step 25: Check creator-buy size against migration timing

import numpy as np
import pandas as pd

# Use classic launches after the documented recorder restart.
seam = pd.Timestamp("2026-09-29T10:56:23Z")

post_seam = classic.loc[
    classic["launched_dt"].ge(seam)
].copy()

# Group launches by creator's first-buy size.
bins = [-np.inf, 1, 10, 50, 80, np.inf]
labels = [
    "< 1 SOL",
    "1–10 SOL",
    "10–50 SOL",
    "50–80 SOL",
    ">= 80 SOL",
]

post_seam["buy_band"] = pd.cut(
    post_seam["dev_buy_sol"],
    bins=bins,
    labels=labels,
    right=False,
)

print("=== Graduation and migration by creator-buy size ===")

def summarize_group(g):
    migration_known = g["migrate_dt_s"].notna()
    migration = g.loc[migration_known, "migrate_dt_s"]

    return pd.Series({
        "launches": len(g),
        "graduations": g["graduated"].sum(),
        "graduation_rate": g["graduated"].mean(),
        "migration_coverage": migration_known.mean(),
        "median_migrate_s": migration.median(),
        "migration_within_3s": (
            migration.le(3).mean() if len(migration) else np.nan
        ),
        "migration_within_10s": (
            migration.le(10).mean() if len(migration) else np.nan
        ),
    })

print(
    post_seam.groupby("buy_band", observed=False)
    .apply(summarize_group)
    .round(4)
    .to_string()
)
print()

# Check whether the suspicious cap-like values cluster by launch day.
print("=== Exact high-buy values by launch day ===")

high = post_seam.loc[
    post_seam["dev_buy_sol"].ge(80)
].copy()

high["day"] = high["launched_dt"].dt.date

high_daily = high.groupby("day").agg(
    high_buy_launches=("mint", "size"),
    graduations=("graduated", "sum"),
    exact_8500536=(
        "dev_buy_sol",
        lambda s: np.isclose(
            s, 85.00536, atol=1e-5, rtol=0
        ).sum(),
    ),
)

print(high_daily.to_string())
print()

# Inspect individual high-buy records and their event timings.
print("=== Example high-buy launches ===")

example_cols = [
    "mint",
    "launched_at",
    "dev_buy_sol",
    "dev_share",
    "graduated",
    "migrate_dt_s",
    "f10_age",
    "f10_rs",
]

example_cols = [
    c for c in example_cols if c in high.columns
]

print(
    high[example_cols]
    .sort_values("migrate_dt_s")
    .head(15)
    .to_string(index=False)
)


In [ ]:

# Step 26: Sensitivity test excluding the >= 80 SOL group

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

features_launch = ["dev_buy_sol", "dev_share"]

# 1) Quantify the high-buy group in each chronological split.
print("=== Outcome rates by dev_buy_sol threshold ===")

for split_name in ["train", "validation", "test"]:
    d = classic.loc[classic["split"].eq(split_name)]

    print(f"\n{split_name}:")

    for label, mask in [
        ("Below 80 SOL", d["dev_buy_sol"] < 80),
        ("At least 80 SOL", d["dev_buy_sol"] >= 80),
    ]:
        group = d.loc[mask]

        print(
            f"{label}: "
            f"n={len(group)}, "
            f"graduations={int(group['graduated'].sum())}, "
            f"graduation_rate={group['graduated'].mean():.3%}"
        )

# 2) Build filtered training, validation, and test subsets.
# This is strictly a sensitivity analysis.
train_cut = train_df.loc[
    train_df["dev_buy_sol"] < 80
].copy()

val_cut = val_df.loc[
    val_df["dev_buy_sol"] < 80
].copy()

test_cut = test_df.loc[
    test_df["dev_buy_sol"] < 80
].copy()

X_train_cut = train_cut[features_launch]
y_train_cut = train_cut["graduated"].astype(int)

X_val_cut = val_cut[features_launch]
y_val_cut = val_cut["graduated"].astype(int)

X_test_cut = test_cut[features_launch]
y_test_cut = test_cut["graduated"].astype(int)

print("\n=== Filtered sample sizes ===")
print("Train:", len(train_cut))
print("Validation:", len(val_cut))
print("Test:", len(test_cut))
print()

# 3) Train a fresh model using only the below-80-SOL group.
cut_model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)

cut_model.fit(X_train_cut, y_train_cut)

# 4) Evaluate ranking performance and precision among the
# highest-scoring candidates.
def evaluate_filtered(name, X, y):
    y_true = np.asarray(y, dtype=int)
    probabilities = cut_model.predict_proba(X)[:, 1]
    prevalence = y_true.mean()

    print(f"=== {name} ===")
    print("Rows:", len(y_true))
    print("Graduation rate:", f"{prevalence:.3%}")
    print(
        "Average precision:",
        f"{average_precision_score(y_true, probabilities):.4f}",
    )
    print(
        "ROC-AUC:",
        f"{roc_auc_score(y_true, probabilities):.4f}",
    )
    print(
        "Brier score:",
        f"{brier_score_loss(y_true, probabilities):.5f}",
    )

    for fraction in [0.01, 0.03, 0.05]:
        k = max(1, int(np.ceil(len(y_true) * fraction)))
        selected = np.argsort(probabilities)[-k:]

        precision = y_true[selected].mean()
        recall = y_true[selected].sum() / y_true.sum()
        lift = precision / prevalence

        print(
            f"Top {fraction:.0%}: "
            f"precision={precision:.3%}, "
            f"recall={recall:.3%}, "
            f"lift={lift:.2f}x"
        )

    print()

evaluate_filtered(
    "Validation — below 80 SOL",
    X_val_cut,
    y_val_cut,
)

evaluate_filtered(
    "Test — below 80 SOL (exploratory)",
    X_test_cut,
    y_test_cut,
)


In [ ]:

# Step 27: Threshold sensitivity for launch-time features

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

launch_features = ["dev_buy_sol", "dev_share"]
thresholds = [10, 50, 80]

results = []

for threshold in thresholds:
    print(f"\n{'=' * 55}")
    print(f"Creator-buy threshold: < {threshold} SOL")
    print("=" * 55)

    # Filter each chronological split independently.
    tr = train_df.loc[
        train_df["dev_buy_sol"] < threshold
    ]
    va = val_df.loc[
        val_df["dev_buy_sol"] < threshold
    ]
    te = test_df.loc[
        test_df["dev_buy_sol"] < threshold
    ]

    Xtr, ytr = tr[launch_features], tr["graduated"].astype(int)
    Xva, yva = va[launch_features], va["graduated"].astype(int)
    Xte, yte = te[launch_features], te["graduated"].astype(int)

    print("Rows: train / validation / test:",
          len(tr), "/", len(va), "/", len(te))
    print("Graduation rates:",
          f"train={ytr.mean():.3%},",
          f"validation={yva.mean():.3%},",
          f"test={yte.mean():.3%}")

    # Fit a fresh model for this cohort.
    candidate = HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.08,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42,
    )

    candidate.fit(Xtr, ytr)

    for split_name, X, y in [
        ("validation", Xva, yva),
        ("test", Xte, yte),
    ]:
        probabilities = candidate.predict_proba(X)[:, 1]
        prevalence = y.mean()

        ap = average_precision_score(y, probabilities)
        auc = roc_auc_score(y, probabilities)

        k = max(1, int(np.ceil(len(y) * 0.01)))
        top_idx = np.argsort(probabilities)[-k:]
        top_precision = y.iloc[top_idx].mean()

        results.append({
            "threshold": threshold,
            "split": split_name,
            "rows": len(y),
            "graduation_rate": prevalence,
            "AP": ap,
            "ROC_AUC": auc,
            "top_1pct_precision": top_precision,
            "AP_lift_over_prevalence": ap / prevalence,
        })

        print(
            f"{split_name}: "
            f"AP={ap:.4f}, "
            f"ROC-AUC={auc:.4f}, "
            f"top-1% precision={top_precision:.3%}, "
            f"AP/prevalence={ap / prevalence:.2f}x"
        )

print("\n=== Combined comparison ===")
results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))


In [ ]:

# Step 28: Check for fresh, out-of-time evaluation data

import pandas as pd
from huggingface_hub import HfApi
from datasets import load_dataset
from sklearn.metrics import average_precision_score, roc_auc_score

repo_id = "loopholetape/pumpfun-launches"

# The commit used for our original snapshot.
original_revision = "21be6aeb5d66721e3d40e34b09141f408e1e0ca0"

# 1) Check the latest published version.
api = HfApi()

latest_info = api.dataset_info(repo_id)

print("=== Dataset version ===")
print("Original commit:", original_revision)
print("Latest commit:  ", latest_info.sha)
print("Last modified:  ", latest_info.last_modified)
print("New commit available:", latest_info.sha != original_revision)
print()

# 2) Load the latest published launches table separately.
# The original df and all existing splits remain unchanged.
latest_ds = load_dataset(
    repo_id,
    "launches",
    split="train",
    revision=latest_info.sha,
)

latest_df = latest_ds.to_pandas()

latest_df["launched_dt"] = pd.to_datetime(
    latest_df["launched_at"],
    errors="coerce",
    utc=True,
)

print("=== Latest data coverage ===")
print("Rows:", len(latest_df))
print("Unique mints:", latest_df["mint"].nunique())
print("First launch:", latest_df["launched_dt"].min())
print("Last launch: ", latest_df["launched_dt"].max())
print()

# 3) Keep only launches after the previously inspected test day.
fresh = latest_df.loc[
    latest_df["launched_dt"] >= pd.Timestamp("2026-10-03", tz="UTC")
].copy()

print("=== Fresh launch dates ===")
print("Rows after October 2:", len(fresh))

if len(fresh):
    fresh["day"] = fresh["launched_dt"].dt.date
    print(fresh.groupby("day").size().to_string())

# 4) Restrict to the same reliably labeled population.
required = [
    "quote_asset",
    "mayhem",
    "decoder_complete",
    "graduated",
    "dev_buy_sol",
    "dev_share",
    "mint",
]

missing_cols = [c for c in required if c not in fresh.columns]

if missing_cols:
    print("\nRequired columns missing:", missing_cols)

else:
    fresh_classic = fresh.loc[
        fresh["quote_asset"].eq("sol")
        & fresh["mayhem"].eq(0)
        & fresh["decoder_complete"].eq(1)
        & fresh["graduated"].notna()
    ].copy()

    print("\n=== Fresh classic cohort ===")
    print("Rows with known graduation:", len(fresh_classic))
    print("Graduations:", int(fresh_classic["graduated"].sum()))

    if len(fresh_classic) == 0:
        print("No eligible fresh cohort is available yet.")

    else:
        launch_features = ["dev_buy_sol", "dev_share"]

        # Existing models only: do not retrain or tune on fresh data.
        def report_fresh(name, fitted_model, data):
            data = data.dropna(subset=launch_features + ["graduated"])

            X = data[launch_features]
            y = data["graduated"].astype(int)
            p = fitted_model.predict_proba(X)[:, 1]

            print(f"\n{name}")
            print("Rows:", len(data))
            print("Graduation rate:", f"{y.mean():.3%}")
            print(
                "Average precision:",
                f"{average_precision_score(y, p):.4f}",
            )

            if y.nunique() == 2:
                print(
                    "ROC-AUC:",
                    f"{roc_auc_score(y, p):.4f}",
                )

            k = max(1, int(len(data) * 0.01))
            selected = p.argsort()[-k:]

            print(
                "Top-1% precision:",
                f"{y.iloc[selected].mean():.3%}",
            )

        if "launch_model" not in globals():
            print("Run Step 22 first to restore launch_model.")

        else:
            report_fresh(
                "Existing launch-time model — all fresh classic launches",
                launch_model,
                fresh_classic,
            )

            below_80 = fresh_classic.loc[
                fresh_classic["dev_buy_sol"] < 80
            ]

            if len(below_80) and "cut_model" in globals():
                report_fresh(
                    "Existing below-80-SOL model — fresh launches",
                    cut_model,
                    below_80,
                )


In [ ]:

# Step 29: Inspect the available historical event archive

import requests
import pandas as pd
from datetime import datetime, timezone

index_url = "https://replay.shrine.trade/pump/index.json"

response = requests.get(index_url, timeout=30)
response.raise_for_status()

archive = response.json()
hours = sorted(
    datetime.strptime(h, "%Y/%m/%d/%H").replace(
        tzinfo=timezone.utc
    )
    for h in archive["hours"]
)

print("=== Archive information ===")
print("Base URL:", archive["base"])
print("Reported archive hours:", archive["count"])
print("Parsed archive hours:", len(hours))
print("First available hour:", hours[0])
print("Last available hour: ", hours[-1])
print("Last index update:", archive.get("updatedAt", "not provided"))
print()

# 1) Count missing hours between the first and last archive entries.
hour_set = set(hours)
expected_count = int(
    (hours[-1] - hours[0]).total_seconds() // 3600
) + 1

missing_hours = sorted(
    hour for hour in pd.date_range(
        start=hours[0],
        end=hours[-1],
        freq="h",
    ).to_pydatetime()
    if hour not in hour_set
)

print("=== Coverage ===")
print("Expected hourly slots:", expected_count)
print("Missing hourly slots:", len(missing_hours))
print(
    "Coverage percentage:",
    f"{100 * len(hours) / expected_count:.2f}%",
)
print()

# 2) Show missing periods, combining consecutive missing hours.
if missing_hours:
    gaps = []
    start = previous = missing_hours[0]

    for hour in missing_hours[1:]:
        if (hour - previous).total_seconds() > 3600:
            gaps.append((start, previous))
            start = hour
        previous = hour

    gaps.append((start, previous))

    print("=== First 15 missing periods (UTC) ===")
    for start, end in gaps[:15]:
        print(
            f"{start:%Y-%m-%d %H}:00 "
            f"to {end:%Y-%m-%d %H}:59"
        )
else:
    print("No missing hourly slots.")


In [ ]:

# Step 30: Inspect a small sample from one historical archive hour

!pip -q install zstandard requests

import io
import json
import requests
import zstandard as zstd
from collections import Counter

# Select one hour from the archive index we inspected in Step 29.
hour_key = "2026/10/04/12"

if hour_key not in archive["hours"]:
    raise ValueError(f"Archive hour not found in index: {hour_key}")

url = f"{archive['base']}/{hour_key}.jsonl.zst"

print("=== Archive file ===")
print("Hour (UTC):", hour_key)
print("URL:", url)
print()

# Stream only the beginning of the file.
# Do not download or decompress the entire hour.
action_counts = Counter()
protocol_counts = Counter()
events_sample = []
first_event = None
blocks = []
lines_read = 0

with requests.get(
    url,
    stream=True,
    timeout=(15, 60),
) as response:

    print("HTTP status:", response.status_code)
    print(
        "Compressed size:",
        (
            f"{int(response.headers['Content-Length']) / (1024**2):.2f} MiB"
            if response.headers.get("Content-Length")
            else "Not provided by server"
        ),
    )

    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=16384,
    ) as reader:

        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_read += 1

                if first_event is None:
                    first_event = event

                action_counts[str(event.get("action", "<missing>"))] += 1
                protocol_counts[str(event.get("protocol", "<missing>"))] += 1

                if isinstance(event.get("block"), (int, float)):
                    blocks.append(event["block"])

                if len(events_sample) < 3:
                    events_sample.append(event)

                # Stop after 100 parsed events.
                if lines_read >= 100:
                    break
        finally:
            # Leave closing the decompressor to its context manager.
            text_stream.detach()

print()
print("=== Sample size ===")
print("Events inspected:", lines_read)
if blocks:
    print("Block range in sample:", min(blocks), "to", max(blocks))
print()

print("=== Actions in the first 100 events ===")
print(dict(action_counts))
print()

print("=== Protocols in the first 100 events ===")
print(dict(protocol_counts))
print()

print("=== First event: available field names ===")
print(sorted(first_event.keys()) if first_event else "No event parsed")
print()

if first_event:
    # Print selected values without dumping potentially large nested fields.
    scalar_fields = [
        "signature",
        "block",
        "timestamp",
        "localTimestamp",
        "action",
        "protocol",
        "mint",
    ]

    print("=== First event: selected values ===")
    print(json.dumps(
        {k: first_event.get(k) for k in scalar_fields},
        indent=2,
        default=str,
    ))
    print()

    print("=== Nested-field structure ===")
    for field in ["breakdown", "tradersInvolved", "reserves"]:
        value = first_event.get(field)

        if isinstance(value, dict):
            description = {
                "type": "dict",
                "keys": list(value.keys()),
            }
        elif isinstance(value, list):
            description = {
                "type": "list",
                "length": len(value),
                "first_item_keys": (
                    list(value[0].keys())
                    if value and isinstance(value[0], dict)
                    else None
                ),
            }
        else:
            description = {"type": type(value).__name__}

        print(f"{field}:", description)


In [ ]:

# Step 31: Inspect actual event values and timestamp units

from decimal import Decimal, InvalidOperation
import pandas as pd
import json

if "events_sample" not in globals() or not events_sample:
    raise ValueError(
        "events_sample is missing. Rerun Step 30 first."
    )

def to_decimal(value):
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError):
        return None

fields = [
    "price",
    "quoteAmount",
    "tokenAmount",
    "marketCapQuote",
    "quoteInPool",
    "tokensInPool",
    "quoteMint",
    "pool",
    "pub",
    "recv",
    "txSigner",
]

for i, event in enumerate(events_sample, start=1):
    print("=" * 60)
    print(f"EVENT {i}")
    print("=" * 60)

    print("Mint:", event.get("mint"))
    print("Action:", event.get("action"))
    print("Protocol:", event.get("protocol"))
    print("Block:", event.get("block"))

    # Compare the two timestamp representations.
    ts = event.get("timestamp")
    local_ts = event.get("localTimestamp")

    print(
        "timestamp as UTC:",
        pd.to_datetime(ts, unit="s", utc=True)
        if ts is not None else None,
    )
    print(
        "localTimestamp as UTC:",
        pd.to_datetime(local_ts, unit="ms", utc=True)
        if local_ts is not None else None,
    )

    print("\nMain field values:")
    for field in fields:
        value = event.get(field)
        print(f"{field}: {value!r}")

    # Check whether price approximately equals quoteAmount / tokenAmount.
    quote = to_decimal(event.get("quoteAmount"))
    tokens = to_decimal(event.get("tokenAmount"))
    price = to_decimal(event.get("price"))

    if (
        quote is not None
        and tokens is not None
        and price is not None
        and tokens != 0
    ):
        ratio = quote / tokens
        relative_difference = (
            abs(price - ratio) / abs(price)
            if price != 0 else None
        )

        print("\nPrice consistency check:")
        print("quoteAmount / tokenAmount:", ratio)
        print("Recorded price:", price)
        print("Relative difference:", relative_difference)

    print("\nBreakdown:")
    print(json.dumps(
        event.get("breakdown"),
        indent=2,
        ensure_ascii=False,
        default=str,
    )[:2500])

    traders = event.get("tradersInvolved")
    if isinstance(traders, list):
        print("\nTrader identifiers:")
        print(traders[:5])

    print()


In [ ]:

# Step 32: Find active classic Pump.fun launches

import io
import json
import requests
import zstandard as zstd
import pandas as pd
from collections import defaultdict

hour_key = "2026/10/04/12"
url = f"{archive['base']}/{hour_key}.jsonl.zst"

SOL_MINT = "So11111111111111111111111111111111111111112"

# Store launch metadata for classic Pump.fun SOL launches.
creates = {}

# Store compact trading statistics per mint.
stats = defaultdict(lambda: {
    "buy_events": 0,
    "sell_events": 0,
    "pumpfun_events": 0,
    "pumpswap_events": 0,
    "first_local_ms": None,
    "last_local_ms": None,
    "min_price": None,
    "max_price": None,
    "last_price": None,
    "last_price_time_ms": None,
})

lines_read = 0
relevant_events = 0
create_events = 0

print("Scanning archive hour:", hour_key)
print("This reads the full compressed hour as a stream.")
print()

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:

    response.raise_for_status()

    size = response.headers.get("Content-Length")
    print(
        "Compressed file size:",
        f"{int(size) / (1024**2):.2f} MiB" if size else "not provided",
    )

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:

        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_read += 1

                if lines_read % 500_000 == 0:
                    print("Events scanned:", lines_read)

                mint = event.get("mint")
                if not mint:
                    continue

                action = event.get("action")
                protocol = event.get("protocol")
                quote_mint = event.get("quoteMint")

                # Identify classic Pump.fun launches quoted in SOL.
                if (
                    action == "create"
                    and protocol == "PUMPFUN"
                    and quote_mint == SOL_MINT
                ):
                    create_events += 1

                    local_ms = event.get("localTimestamp")
                    creates[mint] = {
                        "mint": mint,
                        "name": event.get("name"),
                        "symbol": event.get("symbol"),
                        "create_block": event.get("block"),
                        "launch_time": (
                            pd.to_datetime(
                                local_ms, unit="ms", utc=True
                            )
                            if local_ms is not None
                            else pd.to_datetime(
                                event.get("timestamp"),
                                unit="s",
                                utc=True,
                            )
                        ),
                        "pool_at_create": event.get("pool"),
                    }

                # Count SOL-quoted trades on the initial curve
                # and the PumpSwap pool after migration.
                if (
                    action in ("buy", "sell")
                    and protocol in ("PUMPFUN", "PUMPSWAP")
                    and quote_mint == SOL_MINT
                ):
                    relevant_events += 1
                    s = stats[mint]

                    if action == "buy":
                        s["buy_events"] += 1
                    else:
                        s["sell_events"] += 1

                    if protocol == "PUMPFUN":
                        s["pumpfun_events"] += 1
                    else:
                        s["pumpswap_events"] += 1

                    local_ms = event.get("localTimestamp")

                    if local_ms is not None:
                        if s["first_local_ms"] is None:
                            s["first_local_ms"] = local_ms
                            s["last_local_ms"] = local_ms
                        else:
                            s["first_local_ms"] = min(
                                s["first_local_ms"], local_ms
                            )
                            s["last_local_ms"] = max(
                                s["last_local_ms"], local_ms
                            )

                    price = event.get("price")

                    if isinstance(price, (int, float)) and price > 0:
                        s["min_price"] = (
                            price if s["min_price"] is None
                            else min(s["min_price"], price)
                        )
                        s["max_price"] = (
                            price if s["max_price"] is None
                            else max(s["max_price"], price)
                        )

                        if (
                            local_ms is not None
                            and (
                                s["last_price_time_ms"] is None
                                or local_ms >= s["last_price_time_ms"]
                            )
                        ):
                            s["last_price"] = price
                            s["last_price_time_ms"] = local_ms

        finally:
            text_stream.detach()

print()
print("=== Scan summary ===")
print("Total events scanned:", lines_read)
print("Classic SOL Pump.fun creates:", create_events)
print("Relevant SOL-quoted buy/sell events:", relevant_events)
print("Unique classic launches:", len(creates))
print()

# Rank launches by their number of observed trade events.
rows = []

for mint, launch in creates.items():
    s = stats[mint]

    rows.append({
        **launch,
        "buy_events": s["buy_events"],
        "sell_events": s["sell_events"],
        "pumpfun_events": s["pumpfun_events"],
        "pumpswap_events": s["pumpswap_events"],
        "total_trades": s["buy_events"] + s["sell_events"],
        "first_trade_time": (
            pd.to_datetime(
                s["first_local_ms"], unit="ms", utc=True
            ) if s["first_local_ms"] is not None else pd.NaT
        ),
        "last_trade_time": (
            pd.to_datetime(
                s["last_local_ms"], unit="ms", utc=True
            ) if s["last_local_ms"] is not None else pd.NaT
        ),
        "min_price": s["min_price"],
        "max_price": s["max_price"],
        "last_price": s["last_price"],
    })

candidates = pd.DataFrame(rows)

if candidates.empty:
    print("No classic SOL Pump.fun launches were found in this hour.")

else:
    candidates = candidates.sort_values(
        ["total_trades", "buy_events"],
        ascending=False,
    )

    print("=== Most active newly launched tokens ===")

    display_cols = [
        "mint",
        "symbol",
        "launch_time",
        "total_trades",
        "buy_events",
        "sell_events",
        "pumpfun_events",
        "pumpswap_events",
        "min_price",
        "max_price",
    ]

    print(
        candidates[display_cols]
        .head(20)
        .to_string(index=False)
    )

    eligible = candidates.loc[
        (candidates["total_trades"] >= 20)
        & (candidates["buy_events"] > 0)
        & (candidates["sell_events"] > 0)
    ]

    print()
    print("Tokens with at least 20 observed trades, including buys and sells:",
          len(eligible))

    if len(eligible):
        print("\nBest candidate for our first sequence inspection:")
        print(eligible.iloc[0][display_cols].to_string())


In [ ]:

# Step 33: Extract one token's chronological event sequence

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

TARGET_MINT = "DoVAVzViX8Bjy3r15nwikSaSbzE6dV4ovd28aWpJpump"
hour_key = "2026/10/04/12"
url = f"{archive['base']}/{hour_key}.jsonl.zst"

# Preserve the original archive order and key fields.
keep_fields = [
    "signature",
    "block",
    "timestamp",
    "localTimestamp",
    "action",
    "protocol",
    "mint",
    "pool",
    "quoteMint",
    "price",
    "marketCapQuote",
    "quoteAmount",
    "tokenAmount",
    "quoteInPool",
    "tokensInPool",
    "txSigner",
    "breakdown",
]

records = []
lines_scanned = 0

print("Extracting mint:", TARGET_MINT)
print("Archive hour:", hour_key)
print()

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:

        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_scanned += 1

                if event.get("mint") != TARGET_MINT:
                    continue

                row = {
                    "archive_seq": lines_scanned,
                    **{
                        field: event.get(field)
                        for field in keep_fields
                    },
                }

                records.append(row)

        finally:
            text_stream.detach()

higgs_events = pd.DataFrame(records)

if higgs_events.empty:
    raise ValueError(
        "No events found for the selected mint in this archive hour."
    )

# Convert timestamps without overwriting the original values.
higgs_events["event_time_utc"] = pd.to_datetime(
    higgs_events["timestamp"],
    unit="s",
    utc=True,
    errors="coerce",
)

higgs_events["observed_time_utc"] = pd.to_datetime(
    higgs_events["localTimestamp"],
    unit="ms",
    utc=True,
    errors="coerce",
)

higgs_events["price"] = pd.to_numeric(
    higgs_events["price"],
    errors="coerce",
)

print("=== Extraction summary ===")
print("Archive lines scanned:", lines_scanned)
print("HIGGS events extracted:", len(higgs_events))
print("Unique signatures:", higgs_events["signature"].nunique())
print("Unique blocks:", higgs_events["block"].nunique())
print("First event time:", higgs_events["event_time_utc"].min())
print("Last event time: ", higgs_events["event_time_utc"].max())
print()

print("=== Actions and protocols ===")
print(pd.crosstab(
    higgs_events["protocol"],
    higgs_events["action"],
    margins=True,
).to_string())
print()

# Check whether archive order has backward block/time movement.
block_diff = pd.to_numeric(
    higgs_events["block"], errors="coerce"
).diff()

time_diff = higgs_events["observed_time_utc"].diff().dt.total_seconds()

print("=== Ordering diagnostics ===")
print("Block decreases in archive order:", int(block_diff.lt(0).sum()))
print("Observed timestamp decreases (seconds):",
      int(time_diff.lt(0).sum()))
print("Zero or missing top-level prices:",
      int(higgs_events["price"].isna().sum()
          + higgs_events["price"].eq(0).sum()))
print()

positive_prices = higgs_events.loc[
    higgs_events["price"].gt(0), "price"
]

if len(positive_prices):
    print("Positive price range:",
          positive_prices.min(), "to", positive_prices.max())
    print("Positive price observations:", len(positive_prices))
print()

# Show the opening and closing parts of the observed sequence.
view_cols = [
    "archive_seq",
    "event_time_utc",
    "observed_time_utc",
    "block",
    "action",
    "protocol",
    "price",
    "quoteAmount",
    "tokenAmount",
]

print("=== First 10 HIGGS events in archive order ===")
print(
    higgs_events[view_cols]
    .head(10)
    .to_string(index=False)
)
print()

print("=== Last 10 HIGGS events in archive order ===")
print(
    higgs_events[view_cols]
    .tail(10)
    .to_string(index=False)
)
print()

# Inspect the transition into PumpSwap.
pumpswap_idx = np.flatnonzero(
    higgs_events["protocol"].eq("PUMPSWAP").to_numpy()
)

if len(pumpswap_idx):
    idx = int(pumpswap_idx[0])
    lo = max(0, idx - 5)
    hi = min(len(higgs_events), idx + 6)

    print("=== First PumpSwap event and surrounding records ===")
    print(
        higgs_events[view_cols]
        .iloc[lo:hi]
        .to_string(index=False)
    )
else:
    print("No PumpSwap events found.")


In [ ]:

# Step 34: Audit quote assets, price extremes, and duplicate signatures

import numpy as np
import pandas as pd

h = higgs_events.copy()

SOL_MINT = "So11111111111111111111111111111111111111112"

# Extract the first breakdown price for inspection.
# We retain the full breakdown column unchanged.
def first_breakdown_price(value):
    if not isinstance(value, list):
        return np.nan

    for item in value:
        if isinstance(item, dict):
            price = pd.to_numeric(
                item.get("price"), errors="coerce"
            )
            if pd.notna(price):
                return float(price)

    return np.nan

h["breakdown_price_first"] = h["breakdown"].apply(
    first_breakdown_price
)

h["price"] = pd.to_numeric(h["price"], errors="coerce")
h["quoteAmount"] = pd.to_numeric(
    h["quoteAmount"], errors="coerce"
)
h["tokenAmount"] = pd.to_numeric(
    h["tokenAmount"], errors="coerce"
)

# 1) See which quote assets occur in this token's events.
print("=== Events by protocol and quote asset ===")

quote_summary = h.groupby(
    ["protocol", "quoteMint"],
    dropna=False,
).agg(
    events=("mint", "size"),
    positive_prices=("price", lambda s: s.gt(0).sum()),
    median_price=("price", "median"),
    maximum_price=("price", "max"),
)

print(quote_summary.round(10).to_string())
print()

# 2) Compare price distributions for each quote asset.
print("=== Positive price ranges by quote asset ===")

for quote_mint, group in h.groupby("quoteMint", dropna=False):
    positive = group.loc[group["price"].gt(0), "price"]

    print(f"\nQuote mint: {quote_mint}")
    print("Events:", len(group))
    print("Positive prices:", len(positive))

    if len(positive):
        print(
            positive.describe(
                percentiles=[0.50, 0.90, 0.99, 0.999]
            ).to_string()
        )

# 3) Inspect extreme prices, including their breakdown prices.
print("\n=== Ten largest top-level prices ===")

extreme_cols = [
    "archive_seq",
    "event_time_utc",
    "observed_time_utc",
    "signature",
    "block",
    "action",
    "protocol",
    "pool",
    "quoteMint",
    "price",
    "breakdown_price_first",
    "quoteAmount",
    "tokenAmount",
]

extreme_cols = [
    c for c in extreme_cols if c in h.columns
]

print(
    h.nlargest(10, "price")[extreme_cols]
    .to_string(index=False)
)

# 4) Inspect the SOL-quoted sequence separately.
sol = h.loc[h["quoteMint"].eq(SOL_MINT)].copy()

print("\n=== SOL-quoted events only ===")
print("Rows:", len(sol))
print("Positive prices:", int(sol["price"].gt(0).sum()))
print("Missing or zero prices:",
      int((sol["price"].isna() | sol["price"].eq(0)).sum()))

sol_positive = sol.loc[sol["price"].gt(0), "price"]

if len(sol_positive):
    print("Minimum positive price:", sol_positive.min())
    print("Median positive price:", sol_positive.median())
    print("99th percentile:", sol_positive.quantile(0.99))
    print("Maximum positive price:", sol_positive.max())

print("\nLargest SOL-quoted prices:")
print(
    sol.nlargest(10, "price")[extreme_cols]
    .to_string(index=False)
)

# 5) Check repeated signatures.
print("\n=== Repeated signatures ===")

signature_counts = h["signature"].value_counts()

print("Total events:", len(h))
print("Unique signatures:", h["signature"].nunique())
print("Signatures repeated:", int(signature_counts.gt(1).sum()))
print("Extra rows beyond one per signature:",
      int((signature_counts - 1).clip(lower=0).sum()))

# Multiple records per signature can be legitimate.
# Check repeated full event keys before considering any deduplication.
key_cols = ["signature", "mint", "action", "protocol", "pool"]
key_counts = h.groupby(
    key_cols, dropna=False
).size()

print("Repeated signature/mint/action/protocol/pool groups:",
      int(key_counts.gt(1).sum()))
print()

repeated_signatures = h.loc[
    h["signature"].isin(signature_counts[signature_counts.gt(1)].index)
].sort_values(["block", "signature", "archive_seq"])

print("Examples of repeated-signature events:")
print(
    repeated_signatures[
        [
            "archive_seq",
            "signature",
            "block",
            "action",
            "protocol",
            "pool",
            "quoteMint",
            "price",
        ]
    ]
    .head(15)
    .to_string(index=False)
)


In [ ]:

# Step 35: Build a clean first HIGGS price path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SOL_MINT = "So11111111111111111111111111111111111111112"

# 1) Keep only trades from the two protocols in our initial scope.
candidate = higgs_events.loc[
    higgs_events["action"].isin(["buy", "sell"])
    & higgs_events["protocol"].isin(["PUMPFUN", "PUMPSWAP"])
    & higgs_events["quoteMint"].eq(SOL_MINT)
].copy()

candidate["price"] = pd.to_numeric(
    candidate["price"], errors="coerce"
)
candidate["quoteAmount"] = pd.to_numeric(
    candidate["quoteAmount"], errors="coerce"
)
candidate["tokenAmount"] = pd.to_numeric(
    candidate["tokenAmount"], errors="coerce"
)

# 2) Report data quality before filtering invalid observations.
bad_price = (
    ~np.isfinite(candidate["price"])
    | candidate["price"].le(0)
)
bad_amount = (
    ~np.isfinite(candidate["quoteAmount"])
    | candidate["quoteAmount"].le(0)
    | ~np.isfinite(candidate["tokenAmount"])
    | candidate["tokenAmount"].le(0)
)

print("=== Initial trade selection ===")
print("Candidate trades:", len(candidate))
print("Invalid prices:", int(bad_price.sum()))
print("Invalid trade amounts:", int(bad_amount.sum()))
print()

# Require positive price and trade amounts for this first price path.
# Keep candidate unchanged so we can revisit excluded records.
higgs_trade_path = candidate.loc[
    ~bad_price & ~bad_amount
].copy()

# 3) Order by when the archive says the event was observed.
higgs_trade_path["block"] = pd.to_numeric(
    higgs_trade_path["block"], errors="coerce"
)

higgs_trade_path = higgs_trade_path.sort_values(
    ["observed_time_utc", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

print("=== Clean price path ===")
print("Rows retained:", len(higgs_trade_path))
print("Rows excluded:", len(candidate) - len(higgs_trade_path))
print("Unique signatures:",
      higgs_trade_path["signature"].nunique())
print("Unique pools:", higgs_trade_path["pool"].nunique())
print("First observed trade:",
      higgs_trade_path["observed_time_utc"].min())
print("Last observed trade:",
      higgs_trade_path["observed_time_utc"].max())
print()

# 4) Summarize each protocol separately.
print("=== Summary by protocol ===")

summary = higgs_trade_path.groupby("protocol").agg(
    trades=("mint", "size"),
    buys=("action", lambda s: s.eq("buy").sum()),
    sells=("action", lambda s: s.eq("sell").sum()),
    pools=("pool", "nunique"),
    first_observed=("observed_time_utc", "min"),
    last_observed=("observed_time_utc", "max"),
    first_price=("price", "first"),
    median_price=("price", "median"),
    last_price=("price", "last"),
    min_price=("price", "min"),
    max_price=("price", "max"),
)

summary["first_to_last_price_change_pct"] = (
    (summary["last_price"] / summary["first_price"] - 1) * 100
)

print(summary.round(8).to_string())
print()

# 5) Inspect the transition between the bonding curve and PumpSwap.
pumpfun = higgs_trade_path.loc[
    higgs_trade_path["protocol"].eq("PUMPFUN")
]
pumpswap = higgs_trade_path.loc[
    higgs_trade_path["protocol"].eq("PUMPSWAP")
]

if len(pumpfun) and len(pumpswap):
    last_curve = pumpfun.iloc[-1]
    first_pool = pumpswap.iloc[0]

    print("=== Protocol transition ===")
    print("Last Pump.fun observation:",
          last_curve["observed_time_utc"])
    print("Last Pump.fun price:", last_curve["price"])
    print("First PumpSwap observation:",
          first_pool["observed_time_utc"])
    print("First PumpSwap price:", first_pool["price"])
    print("Pump.fun pool:", last_curve["pool"])
    print("PumpSwap pool:", first_pool["pool"])
    print()

# 6) Plot the observed prices. Log scale helps display early prices.
fig, ax = plt.subplots(figsize=(12, 5))

for protocol, group in higgs_trade_path.groupby("protocol"):
    ax.scatter(
        group["observed_time_utc"],
        group["price"],
        s=3,
        alpha=0.3,
        label=protocol,
    )

if len(pumpswap):
    ax.axvline(
        pumpswap["observed_time_utc"].min(),
        linestyle="--",
        linewidth=1,
        label="First PumpSwap observation",
    )

ax.set_yscale("log")
ax.set_xlabel("Observation time (UTC)")
ax.set_ylabel("Recorded price (SOL per token)")
ax.set_title("HIGGS — observed trade prices, October 4, 2026")
ax.legend()
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()


In [ ]:

# Step 36: Forward price-movement analysis

import numpy as np
import pandas as pd

# Analyze each pool independently. Do not stitch the migration
# transition into one continuous price series.
path = higgs_trade_path.loc[
    higgs_trade_path["price"].gt(0)
    & higgs_trade_path["observed_time_utc"].notna()
].copy()

path = path.sort_values(
    ["protocol", "pool", "observed_time_utc", "block", "archive_seq"],
    kind="stable",
)

horizons_seconds = [5, 15, 30, 60, 180]
results = []

for (protocol, pool), group in path.groupby(
    ["protocol", "pool"], dropna=False
):
    group = group.sort_values(
        ["observed_time_utc", "block", "archive_seq"],
        kind="stable",
    ).reset_index(drop=True)

    prices = group[[
        "observed_time_utc",
        "price",
    ]].copy()

    print(f"\nProcessing {protocol} pool")
    print("Events:", len(prices))
    print("First observation:", prices["observed_time_utc"].min())
    print("Last observation: ", prices["observed_time_utc"].max())

    # Create future observations independently for each horizon.
    future = prices.rename(columns={
        "observed_time_utc": "future_time",
        "price": "future_price",
    }).sort_values("future_time")

    for seconds in horizons_seconds:
        entries = prices.rename(columns={
            "observed_time_utc": "entry_time",
            "price": "entry_price",
        }).copy()

        entries["target_time"] = (
            entries["entry_time"]
            + pd.Timedelta(seconds=seconds)
        )

        # Find the first recorded price at or after the target time.
        matched = pd.merge_asof(
            entries.sort_values("target_time"),
            future,
            left_on="target_time",
            right_on="future_time",
            direction="forward",
            allow_exact_matches=True,
        )

        matched["actual_wait_seconds"] = (
            matched["future_time"] - matched["entry_time"]
        ).dt.total_seconds()

        matched["forward_return_pct"] = (
            matched["future_price"] / matched["entry_price"] - 1
        ) * 100

        valid = matched.loc[
            np.isfinite(matched["forward_return_pct"])
            & matched["future_price"].gt(0)
        ].copy()

        if len(valid):
            r = valid["forward_return_pct"]

            results.append({
                "protocol": protocol,
                "pool": pool,
                "horizon_seconds": seconds,
                "entry_observations": len(entries),
                "valid_forward_observations": len(valid),
                "coverage_pct": 100 * len(valid) / len(entries),
                "median_return_pct": r.median(),
                "mean_return_pct": r.mean(),
                "positive_return_pct": 100 * r.gt(0).mean(),
                "p10_return_pct": r.quantile(0.10),
                "p90_return_pct": r.quantile(0.90),
                "median_actual_wait_s": (
                    valid["actual_wait_seconds"].median()
                ),
            })

forward_summary = pd.DataFrame(results)

print("\n=== Forward price-movement summary ===")

if forward_summary.empty:
    print("No valid forward observations were found.")
else:
    print(
        forward_summary.drop(columns="pool")
        .round(3)
        .to_string(index=False)
    )

    print("\nNote:")
    print("- Returns are measured within each pool, not across migration.")
    print("- Coverage falls at longer horizons near the end of each pool's data.")
    print("- These are price changes, not realized trading profits.")
    print("- Fees, entry delay, trade impact, and exit costs are not yet modeled.")


In [ ]:

# Step 37: Forward returns after simulated entry delays

import numpy as np
import pandas as pd

path = higgs_trade_path.loc[
    higgs_trade_path["protocol"].isin(["PUMPFUN", "PUMPSWAP"])
    & higgs_trade_path["price"].gt(0)
    & higgs_trade_path["observed_time_utc"].notna()
].copy()

path["block"] = pd.to_numeric(path["block"], errors="coerce")

latencies = [0, 1, 3, 5]
holding_periods = [5, 15, 30, 60, 180]

results = []

for (protocol, pool), group in path.groupby(
    ["protocol", "pool"], dropna=False
):
    group = group.sort_values(
        ["observed_time_utc", "block", "archive_seq"],
        kind="stable",
    )

    # Use the last recorded price at each observation timestamp
    # so tied timestamps do not create artificial zero-time moves.
    market = (
        group[
            ["observed_time_utc", "price"]
        ]
        .drop_duplicates(
            subset=["observed_time_utc"],
            keep="last",
        )
        .rename(columns={
            "observed_time_utc": "market_time",
            "price": "market_price",
        })
        .sort_values("market_time")
        .reset_index(drop=True)
    )

    # Every original event is a hypothetical observation time.
    signals = group[["observed_time_utc"]].rename(
        columns={"observed_time_utc": "signal_time"}
    ).reset_index(drop=True)

    for latency in latencies:
        entries = signals.copy()
        entries["entry_target_time"] = (
            entries["signal_time"]
            + pd.Timedelta(seconds=latency)
        )

        # Reference entry: first observed market price at or after
        # the signal time plus the simulated delay.
        entered = pd.merge_asof(
            entries.sort_values("entry_target_time"),
            market.rename(columns={
                "market_time": "entry_time",
                "market_price": "entry_price",
            }).sort_values("entry_time"),
            left_on="entry_target_time",
            right_on="entry_time",
            direction="forward",
            allow_exact_matches=True,
        )

        entered = entered.loc[
            entered["entry_price"].gt(0)
            & entered["entry_time"].notna()
        ].copy()

        for horizon in holding_periods:
            exits = entered.copy()

            exits["exit_target_time"] = (
                exits["entry_time"]
                + pd.Timedelta(seconds=horizon)
            )

            # Reference exit: first observation at or after the
            # intended holding period following the reference entry.
            matched = pd.merge_asof(
                exits.sort_values("exit_target_time"),
                market.rename(columns={
                    "market_time": "exit_time",
                    "market_price": "exit_price",
                }).sort_values("exit_time"),
                left_on="exit_target_time",
                right_on="exit_time",
                direction="forward",
                allow_exact_matches=True,
            )

            valid = matched.loc[
                matched["exit_price"].gt(0)
                & matched["exit_time"].notna()
            ].copy()

            if valid.empty:
                continue

            returns = (
                valid["exit_price"] / valid["entry_price"] - 1
            ) * 100

            results.append({
                "protocol": protocol,
                "latency_s": latency,
                "holding_s": horizon,
                "signals": len(signals),
                "valid_pairs": len(valid),
                "coverage_pct": 100 * len(valid) / len(signals),
                "median_return_pct": returns.median(),
                "mean_return_pct": returns.mean(),
                "positive_pct": 100 * returns.gt(0).mean(),
                "p10_pct": returns.quantile(0.10),
                "p90_pct": returns.quantile(0.90),
                "median_actual_entry_delay_s": (
                    valid["entry_time"] - valid["signal_time"]
                ).dt.total_seconds().median(),
            })

latency_summary = pd.DataFrame(results)

print("=== Forward returns after simulated entry delay ===")
print(
    latency_summary.round(3).to_string(index=False)
)

print("\nImportant limitations:")
print("1. Every observed event is treated as a hypothetical signal.")
print("2. These observations overlap heavily.")
print("3. Reference prices are not executable trade quotes.")
print("4. Fees, price impact, failed transactions and exit latency are not included.")
print("5. HIGGS was selected for high activity, so this is not a representative sample.")


In [ ]:

# Step 38: Cross-token baseline using a random launch sample

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

SOL_MINT = "So11111111111111111111111111111111111111112"
hour_key = "2026/10/04/12"
url = f"{archive['base']}/{hour_key}.jsonl.zst"

# Sample from all classic SOL launch mints, not from the most active tokens.
universe = candidates.drop_duplicates("mint").copy()

sample_size = min(100, len(universe))
sample = universe.sample(
    n=sample_size,
    random_state=20261004,
).copy()

selected_mints = set(sample["mint"])

print("=== Random sample ===")
print("Available launches:", len(universe))
print("Sampled launches:", len(sample))
print("Sampling seed: 20261004")
print()

# Stream only the selected tokens' Pump.fun trades.
records = []
lines_scanned = 0

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:

        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_scanned += 1

                if (
                    event.get("mint") not in selected_mints
                    or event.get("protocol") != "PUMPFUN"
                    or event.get("action") not in ("buy", "sell")
                    or event.get("quoteMint") != SOL_MINT
                ):
                    continue

                local_ms = event.get("localTimestamp")
                price = event.get("price")

                if local_ms is None:
                    continue

                records.append({
                    "mint": event["mint"],
                    "block": event.get("block"),
                    "archive_seq": lines_scanned,
                    "observed_time": pd.to_datetime(
                        local_ms,
                        unit="ms",
                        utc=True,
                    ),
                    "action": event.get("action"),
                    "price": price,
                    "signature": event.get("signature"),
                    "pool": event.get("pool"),
                })

        finally:
            text_stream.detach()

trades = pd.DataFrame(records)

if trades.empty:
    raise ValueError("No matching Pump.fun trades found.")

trades["price"] = pd.to_numeric(
    trades["price"], errors="coerce"
)
trades["block"] = pd.to_numeric(
    trades["block"], errors="coerce"
)

trades = trades.loc[
    np.isfinite(trades["price"]) & trades["price"].gt(0)
].copy()

trades = trades.sort_values(
    ["mint", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

print("=== Observed sample activity ===")
print("Sampled launches:", len(sample))
print("Sampled launches with Pump.fun trades:",
      trades["mint"].nunique())
print("Matching Pump.fun trades:", len(trades))
print()

# These are descriptive reference prices, not executable quotes.
# Entry is the first observation >= 1 second after the first observed trade.
# Exit is the first observation near the requested holding horizon.
entry_delay_s = 1
max_match_lag_s = 2
horizons_s = [5, 15, 30, 60]

results = []
launch_results = []

for mint in sample["mint"]:
    g = trades.loc[trades["mint"].eq(mint)].copy()

    if g.empty:
        launch_results.append({
            "mint": mint,
            "has_curve_trades": False,
            "entry_found": False,
        })
        continue

    g = g.sort_values(
        ["observed_time", "block", "archive_seq"],
        kind="stable",
    ).reset_index(drop=True)

    # Treat the first observed curve trade as the signal reference.
    signal_time = g.iloc[0]["observed_time"]
    entry_target = signal_time + pd.Timedelta(
        seconds=entry_delay_s
    )

    after_target = g.loc[
        g["observed_time"].ge(entry_target)
    ]

    if after_target.empty:
        launch_results.append({
            "mint": mint,
            "has_curve_trades": True,
            "entry_found": False,
        })
        continue

    entry = after_target.iloc[0]
    actual_entry_delay = (
        entry["observed_time"] - signal_time
    ).total_seconds()

    if actual_entry_delay > entry_delay_s + max_match_lag_s:
        launch_results.append({
            "mint": mint,
            "has_curve_trades": True,
            "entry_found": False,
        })
        continue

    launch_results.append({
        "mint": mint,
        "has_curve_trades": True,
        "entry_found": True,
    })

    for horizon_s in horizons_s:
        exit_target = entry["observed_time"] + pd.Timedelta(
            seconds=horizon_s
        )

        later = g.loc[
            g["observed_time"].ge(exit_target)
        ]

        if later.empty:
            continue

        exit_row = later.iloc[0]
        actual_hold_s = (
            exit_row["observed_time"] - entry["observed_time"]
        ).total_seconds()

        if actual_hold_s > horizon_s + max_match_lag_s:
            continue

        results.append({
            "mint": mint,
            "horizon_s": horizon_s,
            "entry_price": entry["price"],
            "exit_price": exit_row["price"],
            "return_pct": (
                exit_row["price"] / entry["price"] - 1
            ) * 100,
            "actual_entry_delay_s": actual_entry_delay,
            "actual_hold_s": actual_hold_s,
        })

coverage = pd.DataFrame(launch_results)

print("=== Sample coverage ===")
print("Launches with observed curve trades:",
      int(coverage["has_curve_trades"].sum()))
print("Launches with a reference entry:",
      int(coverage["entry_found"].sum()))
print("Launches with no curve trades:",
      int((~coverage["has_curve_trades"]).sum()))
print()

if results:
    returns = pd.DataFrame(results)

    summary = returns.groupby("horizon_s").agg(
        valid_tokens=("mint", "nunique"),
        median_return_pct=("return_pct", "median"),
        mean_return_pct=("return_pct", "mean"),
        positive_return_pct=(
            "return_pct", lambda s: 100 * s.gt(0).mean()
        ),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
        median_actual_hold_s=("actual_hold_s", "median"),
    )

    summary["coverage_of_sample_pct"] = (
        summary["valid_tokens"] / len(sample) * 100
    )

    print("=== Cross-token forward price changes ===")
    print(summary.round(3).to_string())
else:
    print("No valid forward-price observations within the allowed time tolerance.")

print()
print("Limitations:")
print("- The sample covers one archive hour only.")
print("- Price changes are conditional on a usable curve entry and exit reference.")
print("- Pool migration can end the Pump.fun-only price path.")
print("- Returns exclude fees, price impact, failed transactions, and exit latency.")


In [ ]:

# Step 39: Extend the random sample across Pump.fun and PumpSwap

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

SOL_MINT = "So11111111111111111111111111111111111111112"
hour_key = "2026/10/04/12"
url = f"{archive['base']}/{hour_key}.jsonl.zst"

selected_mints = set(sample["mint"])

records = []
lines_scanned = 0

print("Scanning the same archive hour for both protocols...")

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:

        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_scanned += 1

                if event.get("mint") not in selected_mints:
                    continue

                if (
                    event.get("action") not in ("buy", "sell")
                    or event.get("protocol") not in ("PUMPFUN", "PUMPSWAP")
                    or event.get("quoteMint") != SOL_MINT
                ):
                    continue

                local_ms = event.get("localTimestamp")
                price = event.get("price")

                if local_ms is None:
                    continue

                records.append({
                    "mint": event["mint"],
                    "signature": event.get("signature"),
                    "block": event.get("block"),
                    "archive_seq": lines_scanned,
                    "observed_time": pd.to_datetime(
                        local_ms, unit="ms", utc=True
                    ),
                    "action": event.get("action"),
                    "protocol": event.get("protocol"),
                    "pool": event.get("pool"),
                    "price": price,
                })

        finally:
            text_stream.detach()

all_sample_trades = pd.DataFrame(records)

if all_sample_trades.empty:
    raise ValueError("No matching trades found.")

all_sample_trades["price"] = pd.to_numeric(
    all_sample_trades["price"], errors="coerce"
)
all_sample_trades["block"] = pd.to_numeric(
    all_sample_trades["block"], errors="coerce"
)

all_sample_trades = all_sample_trades.loc[
    np.isfinite(all_sample_trades["price"])
    & all_sample_trades["price"].gt(0)
].copy()

all_sample_trades = all_sample_trades.sort_values(
    ["mint", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

print("\n=== Expanded event coverage ===")
print("Selected launches:", len(sample))
print("Mints with trade events:", all_sample_trades["mint"].nunique())
print("Matching trade events:", len(all_sample_trades))
print("\nTrades by protocol:")
print(
    all_sample_trades.groupby("protocol")
    .agg(
        events=("mint", "size"),
        tokens=("mint", "nunique"),
        buys=("action", lambda s: s.eq("buy").sum()),
        sells=("action", lambda s: s.eq("sell").sum()),
    )
    .to_string()
)

# Entry assumptions:
# - The launch time is the signal reference.
# - Simulated observation/execution delay is 1 second.
# - An entry must have an observed trade within 5 additional seconds.
entry_delay_s = 1
max_entry_wait_s = 5
max_exit_wait_s = 2
horizons_s = [5, 15, 30, 60]

launch_lookup = sample.set_index("mint")["launch_time"]
trade_groups = {
    mint: group.sort_values(
        ["observed_time", "block", "archive_seq"],
        kind="stable",
    ).reset_index(drop=True)
    for mint, group in all_sample_trades.groupby("mint")
}

entries = []
for mint in sample["mint"]:
    launch_time = pd.Timestamp(launch_lookup.loc[mint])
    if launch_time.tzinfo is None:
        launch_time = launch_time.tz_localize("UTC")

    target = launch_time + pd.Timedelta(seconds=entry_delay_s)
    group = trade_groups.get(mint)

    if group is None:
        continue

    candidates_after_target = group.loc[
        group["observed_time"].ge(target)
    ]

    if candidates_after_target.empty:
        continue

    entry = candidates_after_target.iloc[0]
    entry_wait = (
        entry["observed_time"] - target
    ).total_seconds()

    if entry_wait > max_entry_wait_s:
        continue

    entries.append({
        "mint": mint,
        "launch_time": launch_time,
        "signal_target_time": target,
        "entry_time": entry["observed_time"],
        "entry_price": entry["price"],
        "entry_protocol": entry["protocol"],
        "entry_pool": entry["pool"],
        "entry_action": entry["action"],
        "entry_delay_from_launch_s": (
            entry["observed_time"] - launch_time
        ).total_seconds(),
    })

entries_df = pd.DataFrame(entries)

print("\n=== Entry coverage ===")
print("Selected launches:", len(sample))
print("Eligible entries:", len(entries_df))

if not entries_df.empty:
    print(
        "Entry protocol counts:",
        entries_df["entry_protocol"].value_counts().to_dict(),
    )
    print(
        "Median observed entry delay from launch (s):",
        round(entries_df["entry_delay_from_launch_s"].median(), 3),
    )

# Measure returns using the first observed trade at or after each target.
# Keep same-pool and cross-pool exits separate.
results = []

for _, entry in entries_df.iterrows():
    group = trade_groups[entry["mint"]]

    for horizon_s in horizons_s:
        target = entry["entry_time"] + pd.Timedelta(
            seconds=horizon_s
        )

        later = group.loc[group["observed_time"].ge(target)]
        if later.empty:
            continue

        exit_event = later.iloc[0]
        actual_wait = (
            exit_event["observed_time"] - target
        ).total_seconds()

        if actual_wait > max_exit_wait_s:
            continue

        cross_pool = exit_event["pool"] != entry["entry_pool"]

        results.append({
            "mint": entry["mint"],
            "horizon_s": horizon_s,
            "entry_price": entry["entry_price"],
            "exit_price": exit_event["price"],
            "return_pct": (
                exit_event["price"] / entry["entry_price"] - 1
            ) * 100,
            "entry_protocol": entry["entry_protocol"],
            "exit_protocol": exit_event["protocol"],
            "same_pool": not cross_pool,
            "cross_pool": cross_pool,
            "actual_hold_s": (
                exit_event["observed_time"] - entry["entry_time"]
            ).total_seconds(),
        })

returns_df = pd.DataFrame(results)

print("\n=== Forward return coverage by pool relationship ===")

if returns_df.empty:
    print("No valid forward observations under these assumptions.")
else:
    summary = returns_df.groupby(
        ["horizon_s", "same_pool"],
        dropna=False,
    ).agg(
        tokens=("mint", "nunique"),
        observations=("mint", "size"),
        median_return_pct=("return_pct", "median"),
        mean_return_pct=("return_pct", "mean"),
        positive_return_pct=(
            "return_pct", lambda s: 100 * s.gt(0).mean()
        ),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
        median_actual_hold_s=("actual_hold_s", "median"),
    ).reset_index()

    summary["sample_coverage_pct"] = (
        summary["tokens"] / len(sample) * 100
    )

    print(summary.round(3).to_string(index=False))

print("\nAssumptions and limitations:")
print("- A price observation is used as a reference, not a guaranteed fill.")
print("- Pool migrations are explicitly flagged, not silently treated as continuous.")
print("- Tokens that fail entry or exit timing checks remain out of coverage.")
print("- Fees, price impact, transaction failures, and exit latency are not modeled.")


In [ ]:

# Step 40: Diagnose PumpSwap coverage in the random sample

import pandas as pd

print("=== Sample metadata from the original archive scan ===")

check_cols = [
    "mint",
    "symbol",
    "total_trades",
    "pumpfun_events",
    "pumpswap_events",
]

available_cols = [
    c for c in check_cols if c in sample.columns
]

print(sample[available_cols].head(15).to_string(index=False))
print()

if "pumpswap_events" in sample.columns:
    expected = sample["pumpswap_events"].fillna(0)

    print("=== Expected PumpSwap coverage ===")
    print("Sampled mints:", len(sample))
    print("Mints with PumpSwap events in Step 32:",
          int(expected.gt(0).sum()))
    print("Total expected PumpSwap events:",
          int(expected.sum()))
    print()

# Check what Step 39 actually retained.
print("=== Actual extracted protocol coverage ===")

if "all_sample_trades" not in globals() or all_sample_trades.empty:
    print("all_sample_trades is missing or empty.")
else:
    print(
        all_sample_trades.groupby("protocol").agg(
            events=("mint", "size"),
            mints=("mint", "nunique"),
        ).to_string()
    )
    print()

    # Compare the extracted records with the original scan.
    extracted_counts = (
        all_sample_trades.groupby(["mint", "protocol"])
        .size()
        .unstack(fill_value=0)
    )

    if "pumpswap_events" in sample.columns:
        expected_by_mint = sample.set_index("mint")["pumpswap_events"]

        comparison = pd.DataFrame({
            "expected_pumpswap": expected_by_mint,
            "extracted_pumpswap": extracted_counts.get(
                "PUMPSWAP",
                pd.Series(dtype=int),
            ),
        }).fillna(0)

        comparison["missing_from_extraction"] = (
            comparison["expected_pumpswap"]
            - comparison["extracted_pumpswap"]
        )

        print("=== Largest PumpSwap discrepancies ===")
        print(
            comparison.sort_values(
                "missing_from_extraction",
                ascending=False,
            ).head(15).to_string()
        )


In [ ]:

# Step 41: Build a stratified sample based on observed PumpSwap activity

import pandas as pd

universe_step41 = candidates.drop_duplicates("mint").copy()

# Ensure activity counts are numeric.
for col in ["pumpfun_events", "pumpswap_events", "total_trades"]:
    universe_step41[col] = pd.to_numeric(
        universe_step41[col], errors="coerce"
    ).fillna(0)

# These are archive-hour observations, not lifetime migration labels.
universe_step41["pumpswap_seen"] = (
    universe_step41["pumpswap_events"] > 0
)

print("=== Activity strata in the launch universe ===")
print("Unique launches:", len(universe_step41))
print(
    universe_step41.groupby("pumpswap_seen").agg(
        launches=("mint", "size"),
        median_total_trades=("total_trades", "median"),
        median_pumpswap_events=("pumpswap_events", "median"),
        total_pumpswap_events=("pumpswap_events", "sum"),
    ).to_string()
)
print()

# Select up to 50 launches from each stratum.
with_pumpswap = universe_step41.loc[
    universe_step41["pumpswap_seen"]
]

without_pumpswap = universe_step41.loc[
    ~universe_step41["pumpswap_seen"]
]

sample_a = with_pumpswap.sample(
    n=min(50, len(with_pumpswap)),
    random_state=4101,
).copy()

sample_b = without_pumpswap.sample(
    n=min(50, len(without_pumpswap)),
    random_state=4102,
).copy()

sample_stratified = pd.concat(
    [sample_a, sample_b],
    ignore_index=True,
)

print("=== Stratified sample ===")
print("Total selected:", len(sample_stratified))
print(
    "With PumpSwap observed:",
    int(sample_stratified["pumpswap_seen"].sum()),
)
print(
    "Without PumpSwap observed:",
    int((~sample_stratified["pumpswap_seen"]).sum()),
)
print()

print("=== Selected launches ===")
cols = [
    "mint",
    "symbol",
    "launch_time",
    "pumpfun_events",
    "pumpswap_events",
    "total_trades",
    "pumpswap_seen",
]

print(
    sample_stratified[cols]
    .sort_values(
        ["pumpswap_seen", "total_trades"],
        ascending=[False, False],
    )
    .head(20)
    .to_string(index=False)
)


In [ ]:

# Step 42: Analyze the stratified sample across both protocols

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

SOL_MINT = "So11111111111111111111111111111111111111112"
hour_key = "2026/10/04/12"
url = f"{archive['base']}/{hour_key}.jsonl.zst"

# IMPORTANT: use the 66-token stratified sample from Step 41.
selected_mints = set(sample_stratified["mint"])

records = []
lines_scanned = 0

print("Scanning the archive for the 66 selected launches...")

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:

        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_scanned += 1

                if (
                    event.get("mint") not in selected_mints
                    or event.get("action") not in ("buy", "sell")
                    or event.get("protocol") not in ("PUMPFUN", "PUMPSWAP")
                    or event.get("quoteMint") != SOL_MINT
                ):
                    continue

                local_ms = event.get("localTimestamp")
                price = event.get("price")

                if local_ms is None:
                    continue

                records.append({
                    "mint": event["mint"],
                    "signature": event.get("signature"),
                    "block": event.get("block"),
                    "archive_seq": lines_scanned,
                    "observed_time": pd.to_datetime(
                        local_ms, unit="ms", utc=True
                    ),
                    "action": event.get("action"),
                    "protocol": event.get("protocol"),
                    "pool": event.get("pool"),
                    "price": price,
                })

        finally:
            text_stream.detach()

stratified_trades = pd.DataFrame(records)

if stratified_trades.empty:
    raise ValueError("No matching trades found.")

stratified_trades["price"] = pd.to_numeric(
    stratified_trades["price"], errors="coerce"
)
stratified_trades["block"] = pd.to_numeric(
    stratified_trades["block"], errors="coerce"
)

stratified_trades = stratified_trades.loc[
    np.isfinite(stratified_trades["price"])
    & stratified_trades["price"].gt(0)
    & stratified_trades["pool"].notna()
].copy()

stratified_trades = stratified_trades.sort_values(
    ["mint", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

print("\n=== Extracted event coverage ===")
print("Selected launches:", len(sample_stratified))
print("Tokens with valid trades:",
      stratified_trades["mint"].nunique())
print("Valid trades:", len(stratified_trades))
print()

print("=== Trades by protocol ===")
print(
    stratified_trades.groupby("protocol").agg(
        events=("mint", "size"),
        tokens=("mint", "nunique"),
        buys=("action", lambda s: s.eq("buy").sum()),
        sells=("action", lambda s: s.eq("sell").sum()),
    ).to_string()
)
print()

# Confirm which sample stratum each token belongs to.
stratum_lookup = sample_stratified.set_index("mint")[
    "pumpswap_seen"
].to_dict()

launch_lookup = sample_stratified.set_index("mint")[
    "launch_time"
].to_dict()

groups = {
    mint: g.sort_values(
        ["observed_time", "block", "archive_seq"],
        kind="stable",
    ).reset_index(drop=True)
    for mint, g in stratified_trades.groupby("mint")
}

# Find the first observed trade at least one second after launch,
# but no later than six seconds after launch.
entry_delay_s = 1
max_entry_wait_s = 5
max_exit_wait_s = 2
horizons_s = [5, 15, 30, 60]

entries = []

for mint in sample_stratified["mint"]:
    group = groups.get(mint)

    if group is None or group.empty:
        continue

    launch_time = pd.Timestamp(launch_lookup[mint])
    target = launch_time + pd.Timedelta(seconds=entry_delay_s)

    eligible = group.loc[group["observed_time"].ge(target)]

    if eligible.empty:
        continue

    entry = eligible.iloc[0]
    entry_wait = (entry["observed_time"] - target).total_seconds()

    if entry_wait > max_entry_wait_s:
        continue

    entries.append({
        "mint": mint,
        "pumpswap_seen_in_hour": bool(stratum_lookup[mint]),
        "launch_time": launch_time,
        "entry_time": entry["observed_time"],
        "entry_price": entry["price"],
        "entry_pool": entry["pool"],
        "entry_protocol": entry["protocol"],
        "entry_action": entry["action"],
        "entry_delay_from_launch_s": (
            entry["observed_time"] - launch_time
        ).total_seconds(),
    })

entries_df = pd.DataFrame(entries)

print("=== Reference entry coverage ===")
print("Selected launches:", len(sample_stratified))
print("Reference entries found:", len(entries_df))
print()

if not entries_df.empty:
    print("Entry protocol:")
    print(entries_df["entry_protocol"].value_counts().to_string())
    print()

    print("Entry coverage by sample stratum:")
    print(
        entries_df.groupby("pumpswap_seen_in_hour").agg(
            entries=("mint", "size"),
            median_entry_delay_s=("entry_delay_from_launch_s", "median"),
        ).to_string()
    )
    print()

# Measure only same-pool forward prices.
# If the token migrates to a different pool, do not stitch its
# new pool price onto the old pool's series.
results = []

for _, entry in entries_df.iterrows():
    group = groups[entry["mint"]]

    same_pool = group.loc[
        group["pool"].eq(entry["entry_pool"])
        & group["observed_time"].gt(entry["entry_time"])
    ].copy()

    for horizon_s in horizons_s:
        target = entry["entry_time"] + pd.Timedelta(
            seconds=horizon_s
        )

        later = same_pool.loc[same_pool["observed_time"].ge(target)]

        if later.empty:
            continue

        exit_event = later.iloc[0]
        actual_wait = (
            exit_event["observed_time"] - target
        ).total_seconds()

        # Reject observations too far beyond the target horizon.
        if actual_wait > max_exit_wait_s:
            continue

        results.append({
            "mint": entry["mint"],
            "pumpswap_seen_in_hour": entry["pumpswap_seen_in_hour"],
            "horizon_s": horizon_s,
            "entry_protocol": entry["entry_protocol"],
            "exit_protocol": exit_event["protocol"],
            "entry_pool": entry["entry_pool"],
            "entry_price": entry["entry_price"],
            "exit_price": exit_event["price"],
            "return_pct": (
                exit_event["price"] / entry["entry_price"] - 1
            ) * 100,
            "actual_hold_s": (
                exit_event["observed_time"] - entry["entry_time"]
            ).total_seconds(),
        })

returns_df = pd.DataFrame(results)

print("=== Same-pool forward-return coverage ===")

if returns_df.empty:
    print("No valid same-pool forward returns under these assumptions.")

else:
    summary = returns_df.groupby(
        ["horizon_s", "pumpswap_seen_in_hour"],
        dropna=False,
    ).agg(
        tokens=("mint", "nunique"),
        observations=("mint", "size"),
        median_return_pct=("return_pct", "median"),
        mean_return_pct=("return_pct", "mean"),
        positive_return_pct=(
            "return_pct", lambda s: 100 * s.gt(0).mean()
        ),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
        median_actual_hold_s=("actual_hold_s", "median"),
    ).reset_index()

    summary["sample_coverage_pct"] = (
        summary["tokens"] / len(sample_stratified) * 100
    )

    print(summary.round(3).to_string(index=False))

print("\nLimitations:")
print("- This is a deliberately stratified, non-representative sample.")
print("- Entries use observed trades as reference prices, not guaranteed fills.")
print("- Exit observations must be from the same pool as entry.")
print("- Fees, price impact, failed transactions and realistic execution are not modeled.")


In [ ]:

# Step 43: Inspect Pump.fun-to-PumpSwap price transitions

import numpy as np
import pandas as pd

# This is a transition diagnostic for the 16 PumpSwap-active
# launches in the stratified sample, not a representative sample.

rows = []

for mint, group in stratified_trades.groupby("mint"):
    group = group.sort_values(
        ["observed_time", "block", "archive_seq"],
        kind="stable",
    )

    curve = group.loc[group["protocol"].eq("PUMPFUN")]
    pool = group.loc[group["protocol"].eq("PUMPSWAP")]

    if curve.empty or pool.empty:
        continue

    last_curve = curve.iloc[-1]
    first_pool = pool.iloc[0]

    gap_s = (
        first_pool["observed_time"]
        - last_curve["observed_time"]
    ).total_seconds()

    curve_price = float(last_curve["price"])
    pool_price = float(first_pool["price"])

    rows.append({
        "mint": mint,
        "pumpfun_events": len(curve),
        "pumpswap_events": len(pool),
        "last_curve_time": last_curve["observed_time"],
        "first_pool_time": first_pool["observed_time"],
        "gap_seconds": gap_s,
        "last_curve_price": curve_price,
        "first_pool_price": pool_price,
        "price_change_pct": (
            (pool_price / curve_price) - 1
        ) * 100,
        "chronological_transition": gap_s >= 0,
    })

transitions = pd.DataFrame(rows)

if transitions.empty:
    print("No tokens with both Pump.fun and PumpSwap events.")
else:
    print("=== Transition sample ===")
    print("Tokens with both protocols:", len(transitions))
    print(
        "Chronologically ordered transitions:",
        int(transitions["chronological_transition"].sum()),
    )
    print()

    print("=== Transition price differences ===")

    valid = transitions.loc[
        transitions["chronological_transition"]
        & transitions["last_curve_price"].gt(0)
        & transitions["first_pool_price"].gt(0)
    ]

    if valid.empty:
        print("No chronologically valid transitions to summarize.")
    else:
        print("Valid transitions:", len(valid))
        print(
            valid["price_change_pct"]
            .describe(
                percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
            )
            .round(3)
            .to_string()
        )
        print()

        print("=== Individual transitions ===")
        display_cols = [
            "mint",
            "pumpfun_events",
            "pumpswap_events",
            "gap_seconds",
            "last_curve_price",
            "first_pool_price",
            "price_change_pct",
        ]

        print(
            valid[display_cols]
            .sort_values("gap_seconds")
            .round(8)
            .to_string(index=False)
        )


In [ ]:

# Step 44: Inspect underlying records at the pool transition

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

SOL_MINT = "So11111111111111111111111111111111111111112"
hour_key = "2026/10/04/12"
url = f"{archive['base']}/{hour_key}.jsonl.zst"

transition_mints = set(transitions["mint"])

records = []
lines_scanned = 0

print("Selected transition mints:", len(transition_mints))
print("Scanning the archive hour again...")

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                event = json.loads(line)
                lines_scanned += 1

                if (
                    event.get("mint") not in transition_mints
                    or event.get("protocol") not in ("PUMPFUN", "PUMPSWAP")
                    or event.get("action") not in ("buy", "sell")
                    or event.get("quoteMint") != SOL_MINT
                ):
                    continue

                local_ms = event.get("localTimestamp")
                if local_ms is None:
                    continue

                breakdown = event.get("breakdown")
                leg_prices = []

                if isinstance(breakdown, list):
                    for item in breakdown:
                        if isinstance(item, dict):
                            p = pd.to_numeric(
                                item.get("price"), errors="coerce"
                            )
                            if pd.notna(p) and p > 0:
                                leg_prices.append(float(p))

                records.append({
                    "mint": event["mint"],
                    "signature": event.get("signature"),
                    "block": event.get("block"),
                    "archive_seq": lines_scanned,
                    "observed_time": pd.to_datetime(
                        local_ms, unit="ms", utc=True
                    ),
                    "action": event.get("action"),
                    "protocol": event.get("protocol"),
                    "pool": event.get("pool"),
                    "price": event.get("price"),
                    "breakdown_price": (
                        leg_prices[0] if leg_prices else np.nan
                    ),
                    "quoteAmount": event.get("quoteAmount"),
                    "tokenAmount": event.get("tokenAmount"),
                    "marketCapQuote": event.get("marketCapQuote"),
                    "quoteInPool": event.get("quoteInPool"),
                    "tokensInPool": event.get("tokensInPool"),
                })

        finally:
            text_stream.detach()

detail = pd.DataFrame(records)

if detail.empty:
    raise ValueError("No underlying transition records found.")

for col in [
    "price", "breakdown_price", "quoteAmount", "tokenAmount",
    "marketCapQuote", "quoteInPool", "tokensInPool", "block",
]:
    detail[col] = pd.to_numeric(detail[col], errors="coerce")

detail = detail.sort_values(
    ["mint", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

transition_details = []

for mint, group in detail.groupby("mint"):
    curve = group.loc[group["protocol"].eq("PUMPFUN")]
    pool = group.loc[group["protocol"].eq("PUMPSWAP")]

    if curve.empty or pool.empty:
        continue

    last_curve = curve.iloc[-1]
    first_pool = pool.iloc[0]

    time_gap = (
        first_pool["observed_time"] - last_curve["observed_time"]
    ).total_seconds()

    if time_gap < 0:
        continue

    transition_details.append({
        "mint": mint,
        "gap_s": time_gap,
        "curve_time": last_curve["observed_time"],
        "curve_action": last_curve["action"],
        "curve_price": last_curve["price"],
        "curve_leg_price": last_curve["breakdown_price"],
        "curve_quote_amount": last_curve["quoteAmount"],
        "curve_token_amount": last_curve["tokenAmount"],
        "curve_quote_in_pool": last_curve["quoteInPool"],
        "curve_tokens_in_pool": last_curve["tokensInPool"],
        "pool_time": first_pool["observed_time"],
        "pool_action": first_pool["action"],
        "pool_price": first_pool["price"],
        "pool_leg_price": first_pool["breakdown_price"],
        "pool_quote_amount": first_pool["quoteAmount"],
        "pool_token_amount": first_pool["tokenAmount"],
        "pool_quote_in_pool": first_pool["quoteInPool"],
        "pool_tokens_in_pool": first_pool["tokensInPool"],
        "price_change_pct": (
            first_pool["price"] / last_curve["price"] - 1
        ) * 100,
    })

transition_detail_df = pd.DataFrame(transition_details)

print("\n=== Transition records with pool-state details ===")
print("Chronologically valid transitions:", len(transition_detail_df))
print()

if not transition_detail_df.empty:
    show_cols = [
        "mint",
        "gap_s",
        "curve_price",
        "curve_leg_price",
        "curve_quote_amount",
        "curve_token_amount",
        "curve_quote_in_pool",
        "curve_tokens_in_pool",
        "pool_price",
        "pool_leg_price",
        "pool_quote_amount",
        "pool_token_amount",
        "pool_quote_in_pool",
        "pool_tokens_in_pool",
        "price_change_pct",
    ]

    print(
        transition_detail_df[show_cols]
        .round(10)
        .to_string(index=False)
    )
    print()

    print("=== Recorded-price versus trade-leg-price comparison ===")
    transition_detail_df["curve_price_vs_leg_pct"] = (
        transition_detail_df["curve_price"]
        / transition_detail_df["curve_leg_price"] - 1
    ) * 100

    transition_detail_df["pool_price_vs_leg_pct"] = (
        transition_detail_df["pool_price"]
        / transition_detail_df["pool_leg_price"] - 1
    ) * 100

    print(
        transition_detail_df[
            [
                "mint",
                "curve_price_vs_leg_pct",
                "pool_price_vs_leg_pct",
                "price_change_pct",
            ]
        ].round(4).to_string(index=False)
    )


In [ ]:

# Step 45: Validate price fields and inspect original transition events

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

# Reconstruct the same endpoints from the events already loaded in Step 44.
endpoint_rows = []

for mint, group in detail.groupby("mint"):
    group = group.sort_values(
        ["observed_time", "block", "archive_seq"],
        kind="stable",
    )

    curve = group[group["protocol"].eq("PUMPFUN")]
    pool = group[group["protocol"].eq("PUMPSWAP")]

    if curve.empty or pool.empty:
        continue

    endpoints = [
        ("last_PUMPFUN", curve.iloc[-1]),
        ("first_PUMPSWAP", pool.iloc[0]),
    ]

    for role, row in endpoints:
        reserve_price = np.nan
        if (
            pd.notna(row["quoteInPool"])
            and pd.notna(row["tokensInPool"])
            and row["quoteInPool"] > 0
            and row["tokensInPool"] > 0
        ):
            reserve_price = row["quoteInPool"] / row["tokensInPool"]

        endpoint_rows.append({
            "mint": mint,
            "role": role,
            "protocol": row["protocol"],
            "archive_seq": int(row["archive_seq"]),
            "observed_time": row["observed_time"],
            "signature": row["signature"],
            "pool": row["pool"],
            "price": row["price"],
            "breakdown_price": row["breakdown_price"],
            "quoteAmount": row["quoteAmount"],
            "tokenAmount": row["tokenAmount"],
            "quoteInPool": row["quoteInPool"],
            "tokensInPool": row["tokensInPool"],
            "reserve_price": reserve_price,
        })

endpoints_df = pd.DataFrame(endpoint_rows)

valid = endpoints_df[
    endpoints_df["price"].gt(0)
    & endpoints_df["reserve_price"].gt(0)
].copy()

valid["price_vs_reserve_pct"] = (
    valid["price"] / valid["reserve_price"] - 1
) * 100

valid["breakdown_vs_top_pct"] = (
    valid["breakdown_price"] / valid["price"] - 1
) * 100

print("=== Price / reserve consistency ===")
print("Valid endpoints:", len(valid))
print(
    valid.groupby("role")["price_vs_reserve_pct"]
    .agg(["count", "median", "min", "max"])
    .round(6)
    .to_string()
)

print("\n=== Repeated reserve states ===")
for protocol in ["PUMPFUN", "PUMPSWAP"]:
    subset = endpoints_df[endpoints_df["protocol"].eq(protocol)]
    state_cols = ["quoteInPool", "tokensInPool", "price"]
    print(
        protocol,
        "| endpoints:", len(subset),
        "| distinct reserve/price tuples:",
        subset[state_cols].drop_duplicates().shape[0],
    )

# Select one Pump.fun endpoint whose breakdown price differs most,
# and another whose breakdown price is closest to the top-level price.
curve_endpoints = valid[valid["role"].eq("last_PUMPFUN")].copy()
curve_endpoints["breakdown_ratio"] = (
    curve_endpoints["breakdown_price"] / curve_endpoints["price"]
)

odd_curve = curve_endpoints.loc[
    (curve_endpoints["breakdown_ratio"] - 1).abs().idxmax()
]
normal_curve = curve_endpoints.loc[
    (curve_endpoints["breakdown_ratio"] - 1).abs().idxmin()
]

chosen_mints = {odd_curve["mint"], normal_curve["mint"]}
chosen = endpoints_df[endpoints_df["mint"].isin(chosen_mints)].copy()

print("\n=== Selected endpoint summary ===")
print(
    chosen[
        [
            "mint", "role", "archive_seq", "signature",
            "price", "breakdown_price", "quoteAmount",
            "tokenAmount", "quoteInPool", "tokensInPool",
            "reserve_price",
        ]
    ].round(12).to_string(index=False)
)

# Re-read only the selected original archive records.
target_by_seq = {
    int(row["archive_seq"]): row
    for _, row in chosen.iterrows()
}
found = {}

url = f"{archive['base']}/{hour_key}.jsonl.zst"
line_number = 0

print("\nFetching original event records for the selected endpoints...")

with requests.get(url, stream=True, timeout=(20, 120)) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                line_number += 1

                if line_number in target_by_seq:
                    event = json.loads(line)
                    expected = target_by_seq[line_number]

                    if (
                        event.get("mint") == expected["mint"]
                        and event.get("protocol") == expected["protocol"]
                    ):
                        found[line_number] = event

                if len(found) == len(target_by_seq):
                    break
        finally:
            text_stream.detach()

print("Original records recovered:", len(found), "/", len(target_by_seq))

for seq, event in sorted(found.items()):
    print(f"\n--- {event.get('protocol')} | {event.get('mint')} ---")

    fields = [
        "signature", "block", "timestamp", "localTimestamp",
        "action", "protocol", "mint", "pool", "price",
        "quoteAmount", "tokenAmount", "quoteMint",
        "quoteInPool", "tokensInPool", "tradersInvolved",
        "txSigner", "breakdown",
    ]
    compact = {key: event.get(key) for key in fields if key in event}
    print(json.dumps(compact, indent=2, default=str)[:4500])


In [ ]:

# Step 46: Compare reserve-price changes with amount-implied trade prices

tx = transition_detail_df.copy()

numeric_cols = [
    "curve_price", "pool_price",
    "curve_quote_amount", "curve_token_amount",
    "pool_quote_amount", "pool_token_amount",
]

for col in numeric_cols:
    tx[col] = pd.to_numeric(tx[col], errors="coerce")

tx["curve_avg_trade_price"] = (
    tx["curve_quote_amount"] / tx["curve_token_amount"]
)
tx["pool_avg_trade_price"] = (
    tx["pool_quote_amount"] / tx["pool_token_amount"]
)

tx["spot_change_pct"] = (
    tx["pool_price"] / tx["curve_price"] - 1
) * 100

tx["avg_trade_change_pct"] = (
    tx["pool_avg_trade_price"] / tx["curve_avg_trade_price"] - 1
) * 100

valid_tx = tx[
    tx["curve_avg_trade_price"].gt(0)
    & tx["pool_avg_trade_price"].gt(0)
    & np.isfinite(tx["avg_trade_change_pct"])
].copy()

print("=== Transition price comparison ===")
print("Valid transitions:", len(valid_tx))

for label, col in [
    ("Reserve-implied spot change", "spot_change_pct"),
    ("Amount-implied average trade-price change", "avg_trade_change_pct"),
]:
    print(f"\n{label}:")
    print(
        valid_tx[col]
        .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
        .round(3)
        .to_string()
    )
    print("Positive:", int((valid_tx[col] > 0).sum()))
    print("Negative:", int((valid_tx[col] < 0).sum()))
    print("Approximately unchanged (within +/-1%):",
          int(valid_tx[col].abs().le(1).sum()))

print("\n=== Per-token comparison ===")

display_cols = [
    "mint", "gap_s",
    "curve_quote_amount", "curve_token_amount",
    "pool_quote_amount", "pool_token_amount",
    "spot_change_pct", "avg_trade_change_pct",
]

print(
    valid_tx[display_cols]
    .sort_values("avg_trade_change_pct")
    .to_string(
        index=False,
        formatters={
            "gap_s": "{:.3f}".format,
            "curve_quote_amount": "{:.6f}".format,
            "curve_token_amount": "{:.3f}".format,
            "pool_quote_amount": "{:.6f}".format,
            "pool_token_amount": "{:.3f}".format,
            "spot_change_pct": "{:+.2f}%".format,
            "avg_trade_change_pct": "{:+.2f}%".format,
        },
    )
)


In [ ]:

# Step 47: Check whether raw breakdown amounts and prices reconcile

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

# endpoints_df contains both endpoints for each transition.
wanted = {
    int(row["archive_seq"]): row
    for _, row in endpoints_df.iterrows()
}

raw_events = {}
url = f"{archive['base']}/{hour_key}.jsonl.zst"
line_number = 0

print("Target endpoint records:", len(wanted))
print("Scanning the archive hour...")

with requests.get(url, stream=True, timeout=(20, 120)) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                line_number += 1

                if line_number not in wanted:
                    continue

                event = json.loads(line)
                expected = wanted[line_number]

                if (
                    event.get("mint") == expected["mint"]
                    and event.get("protocol") == expected["protocol"]
                ):
                    raw_events[line_number] = event

                if len(raw_events) == len(wanted):
                    break
        finally:
            text_stream.detach()

print("Recovered:", len(raw_events), "/", len(wanted))

audit_rows = []

for seq, event in sorted(raw_events.items()):
    breakdown = event.get("breakdown") or []
    if not isinstance(breakdown, list):
        breakdown = []

    def num(value):
        value = pd.to_numeric(value, errors="coerce")
        return float(value) if pd.notna(value) else np.nan

    top_q = num(event.get("quoteAmount"))
    top_t = num(event.get("tokenAmount"))
    top_p = num(event.get("price"))
    reserve_q = num(event.get("quoteInPool"))
    reserve_t = num(event.get("tokensInPool"))

    leg_q = [
        num(item.get("quoteAmount"))
        for item in breakdown if isinstance(item, dict)
    ]
    leg_t = [
        num(item.get("tokenAmount"))
        for item in breakdown if isinstance(item, dict)
    ]
    leg_p = [
        num(item.get("price"))
        for item in breakdown if isinstance(item, dict)
    ]

    sum_q = sum(x for x in leg_q if np.isfinite(x))
    sum_t = sum(x for x in leg_t if np.isfinite(x))

    amount_price = (
        top_q / top_t
        if np.isfinite(top_q) and np.isfinite(top_t) and top_t > 0
        else np.nan
    )

    reserve_price = (
        reserve_q / reserve_t
        if np.isfinite(reserve_q)
        and np.isfinite(reserve_t)
        and reserve_t > 0
        else np.nan
    )

    breakdown_amount_price = (
        sum_q / sum_t if sum_t > 0 else np.nan
    )

    valid_prices = [x for x in leg_p if np.isfinite(x) and x > 0]

    audit_rows.append({
        "mint": event.get("mint"),
        "role": wanted[seq]["role"],
        "protocol": event.get("protocol"),
        "action": event.get("action"),
        "breakdown_count": len(breakdown),
        "breakdown_actions": ",".join(
            str(item.get("action", "?"))
            for item in breakdown if isinstance(item, dict)
        ),
        "top_price": top_p,
        "reserve_price": reserve_price,
        "amount_price": amount_price,
        "breakdown_amount_price": breakdown_amount_price,
        "breakdown_min_price": min(valid_prices) if valid_prices else np.nan,
        "breakdown_max_price": max(valid_prices) if valid_prices else np.nan,
        "top_quote": top_q,
        "breakdown_quote_sum": sum_q,
        "top_token": top_t,
        "breakdown_token_sum": sum_t,
        "quote_sum_diff_pct": (
            (sum_q / top_q - 1) * 100
            if np.isfinite(top_q) and top_q != 0 else np.nan
        ),
        "token_sum_diff_pct": (
            (sum_t / top_t - 1) * 100
            if np.isfinite(top_t) and top_t != 0 else np.nan
        ),
        "top_vs_reserve_pct": (
            (top_p / reserve_price - 1) * 100
            if np.isfinite(top_p) and reserve_price > 0 else np.nan
        ),
        "top_vs_amount_price_pct": (
            (top_p / amount_price - 1) * 100
            if np.isfinite(top_p) and amount_price > 0 else np.nan
        ),
    })

audit = pd.DataFrame(audit_rows)

print("\n=== Reconciliation by protocol ===")
for protocol, group in audit.groupby("protocol"):
    print(f"\n{protocol}: {len(group)} events")
    print(
        "Events with multiple breakdown legs:",
        int(group["breakdown_count"].gt(1).sum()),
    )
    print(
        "Quote sums matching top-level within 1%:",
        int(group["quote_sum_diff_pct"].abs().le(1).sum()),
        "/", len(group),
    )
    print(
        "Token sums matching top-level within 1%:",
        int(group["token_sum_diff_pct"].abs().le(1).sum()),
        "/", len(group),
    )
    print(
        "Median top-price vs reserve-price difference:",
        round(group["top_vs_reserve_pct"].median(), 4), "%",
    )
    print(
        "Median top-price vs amount-price difference:",
        round(group["top_vs_amount_price_pct"].median(), 4), "%",
    )

print("\n=== All transition endpoints ===")
cols = [
    "mint", "role", "protocol", "action", "breakdown_count",
    "top_price", "reserve_price", "amount_price",
    "breakdown_min_price", "breakdown_max_price",
    "quote_sum_diff_pct", "token_sum_diff_pct",
    "top_vs_reserve_pct", "top_vs_amount_price_pct",
]
print(
    audit[cols].sort_values(["mint", "role"]).to_string(
        index=False,
        formatters={
            col: (lambda x: f"{x:.3g}" if pd.notna(x) else "NaN")
            for col in cols
            if col not in ["mint", "role", "protocol", "action"]
        },
    )
)


In [ ]:

# Step 48: Is the recorded price pre-trade-like or post-trade-like?

probe = detail[
    detail["protocol"].isin(["PUMPFUN", "PUMPSWAP"])
    & detail["action"].isin(["buy", "sell"])
    & detail["price"].gt(0)
].copy()

probe = probe.sort_values(
    ["mint", "pool", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

pool_groups = probe.groupby(["mint", "pool"], sort=False)

probe["prev_price"] = pool_groups["price"].shift(1)
probe["prev_action"] = pool_groups["action"].shift(1)
probe["prev_time"] = pool_groups["observed_time"].shift(1)

probe["gap_s"] = (
    probe["observed_time"] - probe["prev_time"]
).dt.total_seconds()

probe["move_pct"] = (
    probe["price"] / probe["prev_price"] - 1
) * 100

probe["current_action_sign"] = probe["action"].map({
    "buy": 1,
    "sell": -1,
})

probe["previous_action_sign"] = probe["prev_action"].map({
    "buy": 1,
    "sell": -1,
})

# Keep short-gap observations with a measurable price change.
# This is a diagnostic filter, not a trading-entry rule.
check = probe[
    probe["prev_price"].gt(0)
    & probe["gap_s"].between(0, 2)
    & probe["move_pct"].abs().ge(0.01)
].copy()

check["move_sign"] = np.sign(check["move_pct"])
check["matches_current_action"] = (
    check["move_sign"] == check["current_action_sign"]
)
check["matches_previous_action"] = (
    check["move_sign"] == check["previous_action_sign"]
)

print("=== Price movement / trade direction ===")
print("All eligible consecutive observations:", len(check))

for protocol in ["PUMPFUN", "PUMPSWAP"]:
    x = check[check["protocol"].eq(protocol)]

    print(f"\n--- {protocol} ---")
    print("Observations:", len(x))

    if x.empty:
        continue

    print(
        "Price-change median:",
        round(x["move_pct"].median(), 4), "%",
    )
    print(
        "Price rises:",
        round((x["move_pct"] > 0).mean() * 100, 2), "%",
    )
    print(
        "Price falls:",
        round((x["move_pct"] < 0).mean() * 100, 2), "%",
    )
    print(
        "Move direction matches CURRENT trade:",
        round(x["matches_current_action"].mean() * 100, 2), "%",
    )

    previous_valid = x["previous_action_sign"].notna()
    print(
        "Move direction matches PREVIOUS trade:",
        round(
            x.loc[previous_valid, "matches_previous_action"].mean() * 100,
            2,
        ),
        "%",
        "| observations:", int(previous_valid.sum()),
    )

    print("\nBy current trade action:")
    print(
        x.groupby("action")["move_pct"]
        .agg(["count", "median", "mean"])
        .round(4)
        .to_string()
    )

print("\nInterpretation caution:")
print(
    "Directional alignment is diagnostic only; it does not establish "
    "that a price was executable or that a simulated fill is realistic."
)


In [ ]:

# Step 49: Test price-direction alignment under alternative event orderings

import numpy as np
import pandas as pd

base = detail[
    detail["protocol"].isin(["PUMPFUN", "PUMPSWAP"])
    & detail["action"].isin(["buy", "sell"])
    & detail["price"].gt(0)
    & detail["pool"].notna()
].copy()

base["action_sign"] = base["action"].map({"buy": 1, "sell": -1})

orderings = {
    "observed_time": ["mint", "pool", "observed_time", "block", "archive_seq"],
    "block_order": ["mint", "pool", "block", "archive_seq"],
    "archive_order": ["mint", "pool", "archive_seq"],
}

results = []

for ordering_name, sort_cols in orderings.items():
    x = base.sort_values(sort_cols, kind="stable").copy()

    grouped = x.groupby(["mint", "pool"], sort=False)
    x["prev_price"] = grouped["price"].shift(1)
    x["prev_action_sign"] = grouped["action_sign"].shift(1)
    x["prev_time"] = grouped["observed_time"].shift(1)

    x["gap_s"] = (x["observed_time"] - x["prev_time"]).dt.total_seconds()
    x["move_pct"] = (x["price"] / x["prev_price"] - 1) * 100
    x["move_sign"] = np.sign(x["move_pct"])

    adjacent = x[x["prev_price"].gt(0)].copy()
    short_gap = adjacent[adjacent["gap_s"].between(0, 2)].copy()
    check = short_gap[short_gap["move_pct"].abs().ge(0.01)].copy()

    for protocol in ["PUMPFUN", "PUMPSWAP"]:
        p = check[check["protocol"].eq(protocol)]
        negative_gaps = int((adjacent["gap_s"] < 0).sum())
        eligible_p = short_gap[short_gap["protocol"].eq(protocol)]

        results.append({
            "ordering": ordering_name,
            "protocol": protocol,
            "adjacent_pairs": int(
                adjacent["protocol"].eq(protocol).sum()
            ),
            "negative_time_gaps_all_protocols": negative_gaps,
            "pairs_with_0_to_2s_gap": len(eligible_p),
            "pairs_after_move_filter": len(p),
            "move_matches_current_trade_pct": (
                (p["move_sign"] == p["action_sign"]).mean() * 100
                if len(p) else np.nan
            ),
            "move_matches_previous_trade_pct": (
                (p["move_sign"] == p["prev_action_sign"]).mean() * 100
                if len(p) else np.nan
            ),
        })

result = pd.DataFrame(results)

print("=== Ordering sensitivity ===")
print(
    result.to_string(
        index=False,
        formatters={
            "move_matches_current_trade_pct": lambda v: (
                f"{v:.2f}%" if pd.notna(v) else "NaN"
            ),
            "move_matches_previous_trade_pct": lambda v: (
                f"{v:.2f}%" if pd.notna(v) else "NaN"
            ),
        },
    )
)
print(
    "\nReminder: this is the selected 16-token transition sample, "
    "not a representative market-wide sample."
)


In [ ]:

# Step 50: Cross-hour validation of recorded-price behavior

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

SOL_MINT = "So11111111111111111111111111111111111111112"

# Find the next chronologically available hour.
hours_raw = archive["hours"]
if isinstance(hours_raw, dict):
    hours_raw = list(hours_raw.keys())

available_hours = sorted(
    h for h in hours_raw
    if isinstance(h, str) and h > hour_key
)

if not available_hours:
    raise ValueError("No archive hour is available after " + hour_key)

validation_hour = available_hours[0]
validation_url = f"{archive['base']}/{validation_hour}.jsonl.zst"

print("Reference hour:", hour_key)
print("Validation hour:", validation_hour)
print("Streaming archive; this may take a little while...")

rows = []
line_count = 0
missing_local_time = 0

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                line_count += 1
                event = json.loads(line)

                if (
                    event.get("protocol") not in ("PUMPFUN", "PUMPSWAP")
                    or event.get("action") not in ("buy", "sell")
                    or event.get("quoteMint") != SOL_MINT
                    or not event.get("pool")
                ):
                    continue

                price = pd.to_numeric(event.get("price"), errors="coerce")
                local_ms = event.get("localTimestamp")

                if pd.isna(price) or price <= 0:
                    continue

                if local_ms is None:
                    missing_local_time += 1
                    continue

                rows.append((
                    event.get("mint"),
                    event.get("pool"),
                    event.get("protocol"),
                    event.get("action"),
                    float(price),
                    int(local_ms),
                    event.get("block"),
                    line_count,
                ))
        finally:
            text_stream.detach()

validation = pd.DataFrame(
    rows,
    columns=[
        "mint", "pool", "protocol", "action", "price",
        "local_ms", "block", "archive_seq",
    ],
)

if validation.empty:
    raise ValueError("No eligible trade events found in " + validation_hour)

validation["observed_time"] = pd.to_datetime(
    validation["local_ms"], unit="ms", utc=True
)
validation["block"] = pd.to_numeric(
    validation["block"], errors="coerce"
)
validation["action_sign"] = validation["action"].map({
    "buy": 1,
    "sell": -1,
})

validation = validation.sort_values(
    ["mint", "pool", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

groups = validation.groupby(["mint", "pool"], sort=False)
validation["prev_price"] = groups["price"].shift(1)
validation["prev_action_sign"] = groups["action_sign"].shift(1)
validation["prev_time"] = groups["observed_time"].shift(1)

validation["gap_s"] = (
    validation["observed_time"] - validation["prev_time"]
).dt.total_seconds()

validation["move_pct"] = (
    validation["price"] / validation["prev_price"] - 1
) * 100

eligible = validation[
    validation["prev_price"].gt(0)
    & validation["gap_s"].between(0, 2)
    & validation["move_pct"].abs().ge(0.01)
].copy()

eligible["move_sign"] = np.sign(eligible["move_pct"])
eligible["matches_current"] = (
    eligible["move_sign"] == eligible["action_sign"]
)
eligible["matches_previous"] = (
    eligible["move_sign"] == eligible["prev_action_sign"]
)

print("\n=== Archive coverage ===")
print("Raw lines scanned:", f"{line_count:,}")
print("Eligible trade events:", f"{len(validation):,}")
print("Missing observed timestamps:", f"{missing_local_time:,}")

for protocol in ["PUMPFUN", "PUMPSWAP"]:
    subset = validation[validation["protocol"].eq(protocol)]
    check = eligible[eligible["protocol"].eq(protocol)]

    print(f"\n=== {protocol} ===")
    print("Events:", f"{len(subset):,}")
    print("Unique mints:", f"{subset['mint'].nunique():,}")
    print("Unique pools:", f"{subset['pool'].nunique():,}")
    print("Eligible consecutive price changes:", f"{len(check):,}")

    if check.empty:
        continue

    print("Price changes matching current trade:",
          f"{check['matches_current'].mean() * 100:.2f}%")

    prior = check["prev_action_sign"].notna()
    print("Price changes matching previous trade:",
          f"{check.loc[prior, 'matches_previous'].mean() * 100:.2f}%")

    print("Median price change:",
          f"{check['move_pct'].median():.4f}%")

    print("Price rises / falls:",
          f"{(check['move_pct'] > 0).mean() * 100:.2f}% / "
          f"{(check['move_pct'] < 0).mean() * 100:.2f}%")

# Show how widely represented the two protocols are in this hour.
mint_protocols = validation.groupby("mint")["protocol"].nunique()

print("\nMints with both protocols in this hour:",
      int((mint_protocols >= 2).sum()))

print(
    "\nThis tests recorded-price behavior only. "
    "It does not estimate trade profitability or execution slippage."
)


In [ ]:

# Step 51: Diagnose the PumpSwap price/event alignment

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

SOL_MINT = "So11111111111111111111111111111111111111112"

# Use the full PumpSwap sample already loaded in Step 50.
ps = validation.loc[
    validation["protocol"].eq("PUMPSWAP")
].copy()

pool_counts = (
    ps.groupby(["mint", "pool"])
    .size()
    .sort_values(ascending=False)
)

if pool_counts.empty:
    raise ValueError("No PumpSwap pool events in Step 50 data.")

target_mint, target_pool = pool_counts.index[0]

print("=== Selected pool ===")
print("Validation hour:", validation_hour)
print("Mint:", target_mint)
print("Pool:", target_pool)
print("Events in Step 50:", int(pool_counts.iloc[0]))
print()

# Re-read only records for this pool, retaining original breakdowns.
url = f"{archive['base']}/{validation_hour}.jsonl.zst"

records = []
line_count = 0

with requests.get(
    url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                line_count += 1
                event = json.loads(line)

                if (
                    event.get("mint") != target_mint
                    or event.get("pool") != target_pool
                    or event.get("protocol") != "PUMPSWAP"
                    or event.get("quoteMint") != SOL_MINT
                    or event.get("action") not in ("buy", "sell")
                ):
                    continue

                if event.get("localTimestamp") is None:
                    continue

                breakdown = event.get("breakdown") or []
                leg_prices = []
                leg_quotes = []
                leg_tokens = []

                if isinstance(breakdown, list):
                    for leg in breakdown:
                        if not isinstance(leg, dict):
                            continue

                        for field, output in [
                            ("price", leg_prices),
                            ("quoteAmount", leg_quotes),
                            ("tokenAmount", leg_tokens),
                        ]:
                            value = pd.to_numeric(
                                leg.get(field), errors="coerce"
                            )
                            output.append(
                                float(value) if pd.notna(value) else np.nan
                            )

                records.append({
                    "archive_seq": line_count,
                    "signature": event.get("signature"),
                    "block": event.get("block"),
                    "observed_time": pd.to_datetime(
                        event["localTimestamp"],
                        unit="ms",
                        utc=True,
                    ),
                    "action": event.get("action"),
                    "price": pd.to_numeric(
                        event.get("price"), errors="coerce"
                    ),
                    "quote_amount": pd.to_numeric(
                        event.get("quoteAmount"), errors="coerce"
                    ),
                    "token_amount": pd.to_numeric(
                        event.get("tokenAmount"), errors="coerce"
                    ),
                    "quote_in_pool": pd.to_numeric(
                        event.get("quoteInPool"), errors="coerce"
                    ),
                    "tokens_in_pool": pd.to_numeric(
                        event.get("tokensInPool"), errors="coerce"
                    ),
                    "breakdown_count": len(breakdown),
                    "leg_prices": leg_prices,
                    "leg_quotes": leg_quotes,
                    "leg_tokens": leg_tokens,
                })

        finally:
            text_stream.detach()

pool_events = pd.DataFrame(records)

if pool_events.empty:
    raise ValueError("No raw events found for selected pool.")

pool_events["block"] = pd.to_numeric(
    pool_events["block"], errors="coerce"
)

pool_events = pool_events.sort_values(
    ["observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

# Price implied by the pool reserves.
pool_events["reserve_price"] = (
    pool_events["quote_in_pool"]
    / pool_events["tokens_in_pool"]
)

# Average price implied by top-level amounts.
pool_events["amount_price"] = (
    pool_events["quote_amount"]
    / pool_events["token_amount"]
)

# First-leg price, for single-leg transactions and as a diagnostic.
pool_events["leg_price"] = pool_events["leg_prices"].apply(
    lambda values: (
        values[0]
        if isinstance(values, list) and len(values) == 1
        else np.nan
    )
)

# Compute consecutive recorded-price movements within the same pool.
pool_events["prev_price"] = pool_events["price"].shift(1)
pool_events["prev_action"] = pool_events["action"].shift(1)
pool_events["move_pct"] = (
    pool_events["price"] / pool_events["prev_price"] - 1
) * 100

pool_events["action_sign"] = pool_events["action"].map({
    "buy": 1,
    "sell": -1,
})
pool_events["prev_action_sign"] = pool_events["prev_action"].map({
    "buy": 1,
    "sell": -1,
})

valid_moves = pool_events.loc[
    pool_events["prev_price"].gt(0)
    & pool_events["move_pct"].abs().ge(0.01)
].copy()

if len(valid_moves):
    print("=== Directional alignment ===")
    print("Eligible changes:", len(valid_moves))
    print(
        "Move matches current trade:",
        f"{(np.sign(valid_moves['move_pct']) == valid_moves['action_sign']).mean():.2%}",
    )
    print(
        "Move matches previous trade:",
        f"{(np.sign(valid_moves['move_pct']) == valid_moves['prev_action_sign']).mean():.2%}",
    )
    print()

# Compare recorded post-trade price with the single swap-leg execution price.
single_leg = pool_events.loc[
    pool_events["breakdown_count"].eq(1)
    & pool_events["leg_price"].gt(0)
    & pool_events["price"].gt(0)
].copy()

if len(single_leg):
    # Under the documented post-trade interpretation:
    # buys should usually finish above their average fill price;
    # sells should usually finish below their average fill price.
    single_leg["post_price_matches_action"] = (
        (single_leg["action"].eq("buy")
         & single_leg["price"].gt(single_leg["leg_price"]))
        |
        (single_leg["action"].eq("sell")
         & single_leg["price"].lt(single_leg["leg_price"]))
    )

    print("=== Post-trade price vs single-leg execution price ===")
    print("Single-leg records:", len(single_leg))
    print(
        "Directionally consistent records:",
        f"{single_leg['post_price_matches_action'].mean():.2%}",
    )
    print()

# Show a small, ordered window without dumping the entire pool.
view_cols = [
    "observed_time",
    "block",
    "action",
    "breakdown_count",
    "price",
    "prev_price",
    "move_pct",
    "leg_price",
    "amount_price",
    "reserve_price",
    "quote_amount",
    "token_amount",
]

print("=== First 25 events in pool order ===")
print(
    pool_events[view_cols]
    .head(25)
    .round(10)
    .to_string(index=False)
)


In [ ]:
# Step 52: Check PumpSwap price alignment across multiple pools

import pandas as pd

# Reuse the eligible price movements calculated in Step 50.
ps_moves = eligible.loc[
    eligible["protocol"].eq("PUMPSWAP")
].copy()

if ps_moves.empty:
    raise ValueError("No eligible PumpSwap price movements found.")

# Summarize price behavior separately for every mint and pool.
pool_stats = (
    ps_moves.groupby(["mint", "pool"])
    .agg(
        eligible_changes=("move_pct", "size"),
        current_match_pct=(
            "matches_current",
            lambda s: 100 * s.mean()
        ),
        previous_match_pct=(
            "matches_previous",
            lambda s: 100 * s.mean()
        ),
        median_abs_move_pct=(
            "move_pct",
            lambda s: s.abs().median()
        ),
    )
    .reset_index()
)

# Add the number of recorded events for each pool.
event_counts = (
    validation.groupby(["mint", "pool"])
    .size()
    .rename("pool_events")
    .reset_index()
)

pool_stats = pool_stats.merge(
    event_counts,
    on=["mint", "pool"],
    how="left",
)

pool_stats["previous_minus_current_pp"] = (
    pool_stats["previous_match_pct"]
    - pool_stats["current_match_pct"]
)

# Exclude small samples from the pool-level comparison.
# This threshold is for analysis, not a trading rule.
qualified = pool_stats.loc[
    pool_stats["eligible_changes"] >= 100
].copy()

print("=== PumpSwap multi-pool audit ===")
print("PumpSwap pools with eligible changes:", len(pool_stats))
print("Pools with at least 100 eligible changes:", len(qualified))
print("Total eligible price movements:", len(ps_moves))
print()

# Overall percentages weight every eligible movement equally.
print("=== Overall alignment ===")
print(
    "Matches current trade:",
    f"{100 * ps_moves['matches_current'].mean():.2f}%"
)
print(
    "Matches previous trade:",
    f"{100 * ps_moves['matches_previous'].mean():.2f}%"
)
print()

if not qualified.empty:
    print("=== Pool-level comparison ===")
    print(
        "Pools where previous alignment is higher:",
        f"{100 * (qualified['previous_minus_current_pp'] > 0).mean():.2f}%"
    )
    print(
        "Median previous-minus-current difference:",
        f"{qualified['previous_minus_current_pp'].median():.2f} percentage points"
    )
    print()

    print("=== 15 pools with the most eligible price changes ===")
    columns = [
        "mint",
        "pool",
        "pool_events",
        "eligible_changes",
        "current_match_pct",
        "previous_match_pct",
        "previous_minus_current_pp",
    ]

    print(
        qualified.sort_values(
            "eligible_changes",
            ascending=False
        )[columns]
        .head(15)
        .round(2)
        .to_string(index=False)
    )

    print()

    # Compare the specific pool examined in Step 51.
    selected = qualified.loc[
        qualified["mint"].eq(target_mint)
        & qualified["pool"].eq(target_pool)
    ]

    print("=== Step 51 selected pool ===")
    if selected.empty:
        print("Selected pool has fewer than 100 eligible changes.")
    else:
        print(
            selected[columns]
            .round(2)
            .to_string(index=False)
        )

print()
print(
    "Diagnostic only: alignment percentages do not prove "
    "executable prices or profitable trades."
)

In [ ]:
# Step 53: Compare price movement against nearby trade events
# Uses the existing validation DataFrame from Step 50.

import numpy as np
import pandas as pd

probe = validation.copy()

# Preserve event order within each mint and pool.
probe = probe.sort_values(
    ["mint", "pool", "observed_time", "block", "archive_seq"],
    kind="stable",
).reset_index(drop=True)

groups = probe.groupby(["mint", "pool"], sort=False)

# Calculate the price movement ending at each event.
probe["prev_price"] = groups["price"].shift(1)
probe["prev_time"] = groups["observed_time"].shift(1)

probe["gap_s"] = (
    probe["observed_time"] - probe["prev_time"]
).dt.total_seconds()

probe["move_pct"] = (
    probe["price"] / probe["prev_price"] - 1
) * 100

probe["move_sign"] = np.sign(probe["move_pct"])

# Compare the price movement with trades around the event.
# Offset -1 = previous trade; 0 = current trade;
# +1 = next trade. Future offsets are diagnostic only.
for offset in [-2, -1, 0, 1, 2]:
    probe[f"action_offset_{offset}"] = (
        groups["action_sign"].shift(-offset)
    )

# Keep the same basic eligibility criteria as Step 50.
eligible_offsets = probe.loc[
    probe["prev_price"].gt(0)
    & probe["gap_s"].between(0, 2)
    & probe["move_pct"].abs().ge(0.01)
].copy()

print("=== Step 53: Event-offset alignment ===")
print("Eligible movements:", f"{len(eligible_offsets):,}")
print()

results = []

for protocol in ["PUMPFUN", "PUMPSWAP"]:
    subset = eligible_offsets.loc[
        eligible_offsets["protocol"].eq(protocol)
    ]

    print(f"--- {protocol} ---")

    for offset in [-2, -1, 0, 1, 2]:
        col = f"action_offset_{offset}"
        valid = subset.loc[subset[col].notna()]

        alignment = (
            (valid["move_sign"] == valid[col]).mean() * 100
            if len(valid) else np.nan
        )

        labels = {
            -2: "Two events before",
            -1: "Previous event",
             0: "Current event",
             1: "Next event",
             2: "Two events after",
        }

        results.append({
            "protocol": protocol,
            "offset": offset,
            "comparison": labels[offset],
            "observations": len(valid),
            "alignment_pct": alignment,
        })

    display(
        pd.DataFrame([
            r for r in results if r["protocol"] == protocol
        ])[["comparison", "observations", "alignment_pct"]]
        .assign(
            alignment_pct=lambda x: x["alignment_pct"].round(2)
        )
        .reset_index(drop=True)
    )

print(
    "Diagnostic only: alignment does not establish price timing, "
    "executable fills, or profitability."
)

In [ ]:
# Step 54: Compare recorded, reserve-implied, and trade prices

import io
import json
import numpy as np
import pandas as pd
import requests
import zstandard as zstd

SOL_MINT = "So11111111111111111111111111111111111111112"

# Select 10 most active pools and 10 additional pools
# with at least 100 recorded events, per protocol.
pool_counts = (
    validation.groupby(["protocol", "mint", "pool"])
    .size()
    .reset_index(name="validation_events")
)

selected = []

for protocol, seed in [("PUMPFUN", 5401), ("PUMPSWAP", 5402)]:
    p = pool_counts.loc[
        pool_counts["protocol"].eq(protocol)
    ].copy()

    top = p.nlargest(10, "validation_events")
    top_keys = set(
        zip(top["protocol"], top["mint"], top["pool"])
    )

    remaining = p.loc[p["validation_events"].ge(100)].copy()

    remaining = remaining.loc[
        [
            (proto, mint, pool) not in top_keys
            for proto, mint, pool in zip(
                remaining["protocol"],
                remaining["mint"],
                remaining["pool"],
            )
        ]
    ]

    random_pools = remaining.sample(
        n=min(10, len(remaining)),
        random_state=seed,
    )

    selected.extend([top, random_pools])

pool_sample = (
    pd.concat(selected, ignore_index=True)
    .drop_duplicates(["protocol", "mint", "pool"])
)

target_keys = set(
    zip(
        pool_sample["protocol"],
        pool_sample["mint"],
        pool_sample["pool"],
    )
)

print("=== Selected pools ===")
print(
    pool_sample.groupby("protocol")
    .agg(
        pools=("pool", "size"),
        expected_events=("validation_events", "sum"),
    )
    .to_string()
)
print()

# Read the same archive hour and retain only selected pools.
records = []
line_count = 0

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in text_stream:
                if not line.strip():
                    continue

                line_count += 1
                event = json.loads(line)

                key = (
                    event.get("protocol"),
                    event.get("mint"),
                    event.get("pool"),
                )

                if (
                    key not in target_keys
                    or event.get("action") not in ("buy", "sell")
                    or event.get("quoteMint") != SOL_MINT
                ):
                    continue

                breakdown = event.get("breakdown") or []

                records.append({
                    "protocol": event.get("protocol"),
                    "mint": event.get("mint"),
                    "pool": event.get("pool"),
                    "action": event.get("action"),
                    "price": event.get("price"),
                    "quote_amount": event.get("quoteAmount"),
                    "token_amount": event.get("tokenAmount"),
                    "quote_in_pool": event.get("quoteInPool"),
                    "tokens_in_pool": event.get("tokensInPool"),
                    "breakdown_count": (
                        len(breakdown)
                        if isinstance(breakdown, list)
                        else np.nan
                    ),
                })
        finally:
            text_stream.detach()

audit = pd.DataFrame(records)

if audit.empty:
    raise ValueError("No matching trade records recovered.")

for col in [
    "price",
    "quote_amount",
    "token_amount",
    "quote_in_pool",
    "tokens_in_pool",
]:
    audit[col] = pd.to_numeric(audit[col], errors="coerce")

audit["reserve_price"] = (
    audit["quote_in_pool"] / audit["tokens_in_pool"]
)

audit["amount_price"] = (
    audit["quote_amount"] / audit["token_amount"]
)

# Difference between recorded price and reserve-implied price.
audit["reserve_diff_bps"] = (
    audit["price"] / audit["reserve_price"] - 1
) * 10_000

print("=== Archive scan ===")
print("Raw lines scanned:", f"{line_count:,}")
print("Recovered events:", f"{len(audit):,}")
print()

# First check whether the recorded price matches the reserve ratio.
reserve_valid = audit.loc[
    audit["price"].gt(0)
    & audit["reserve_price"].gt(0)
    & audit["reserve_diff_bps"].notna()
].copy()

print("=== Recorded price vs reserve-implied price ===")

print(
    reserve_valid.groupby("protocol").agg(
        events=("price", "size"),
        median_abs_difference_bps=(
            "reserve_diff_bps",
            lambda s: s.abs().median(),
        ),
        within_0_1_bps_pct=(
            "reserve_diff_bps",
            lambda s: 100 * s.abs().le(0.1).mean(),
        ),
    ).round(3).to_string()
)
print()

# Next compare recorded and average trade price for single-leg trades.
single = audit.loc[
    audit["breakdown_count"].eq(1)
    & audit["price"].gt(0)
    & audit["amount_price"].gt(0)
].copy()

single["amount_diff_bps"] = (
    single["amount_price"] / single["price"] - 1
) * 10_000

# Exclude nearly equal prices, where precision makes the
# direction difficult to interpret.
single = single.loc[
    single["amount_diff_bps"].abs().ge(0.5)
].copy()

# Pre-trade hypothesis:
# buys execute above the pre-trade price; sells below it.
single["pretrade_consistent"] = (
    (single["action"].eq("buy")
     & single["amount_diff_bps"].gt(0))
    |
    (single["action"].eq("sell")
     & single["amount_diff_bps"].lt(0))
)

# Post-trade hypothesis: the recorded spot price moves
# beyond the average execution price in the trade direction.
single["posttrade_consistent"] = (
    (single["action"].eq("buy")
     & single["amount_diff_bps"].lt(0))
    |
    (single["action"].eq("sell")
     & single["amount_diff_bps"].gt(0))
)

print("=== Recorded price vs average trade price ===")
print("Single-leg records with a difference of at least 0.5 bps:",
      f"{len(single):,}")
print()

if single.empty:
    print("No sufficiently different single-leg records to compare.")
else:
    print(
        single.groupby("protocol").agg(
            observations=("price", "size"),
            pretrade_consistent_pct=(
                "pretrade_consistent",
                lambda s: 100 * s.mean(),
            ),
            posttrade_consistent_pct=(
                "posttrade_consistent",
                lambda s: 100 * s.mean(),
            ),
            median_amount_difference_bps=(
                "amount_diff_bps", "median"
            ),
        ).round(2).to_string()
    )

print()
print(
    "These comparisons are diagnostic. They do not establish "
    "executable fills, transaction ordering, or profitability."
)

In [ ]:
# Step 55: Compare recorded-price differences by trade direction

# Reuse "single" from Step 54.
if "single" not in globals():
    raise ValueError("Run Step 54 first; its 'single' DataFrame is missing.")

# Summarize each protocol separately for buys and sells.
summary = (
    single.groupby(["protocol", "action"])
    .agg(
        observations=("price", "size"),
        median_difference_bps=("amount_diff_bps", "median"),
        amount_price_below_recorded_pct=(
            "amount_diff_bps",
            lambda s: 100 * (s < 0).mean(),
        ),
        amount_price_above_recorded_pct=(
            "amount_diff_bps",
            lambda s: 100 * (s > 0).mean(),
        ),
        pretrade_consistent_pct=(
            "pretrade_consistent",
            lambda s: 100 * s.mean(),
        ),
        posttrade_consistent_pct=(
            "posttrade_consistent",
            lambda s: 100 * s.mean(),
        ),
    )
    .reset_index()
)

print("=== Price relationship by protocol and action ===")
print(summary.round(2).to_string(index=False))
print()

# Inspect the distribution of differences for PumpSwap.
pump_swap = single.loc[
    single["protocol"].eq("PUMPSWAP")
]

print("=== PumpSwap difference percentiles (basis points) ===")

print(
    pump_swap.groupby("action")["amount_diff_bps"]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    .unstack()
    .rename(columns={
        0.10: "p10",
        0.25: "p25",
        0.50: "p50",
        0.75: "p75",
        0.90: "p90",
    })
    .round(2)
    .to_string()
)

print()
print(
    "Diagnostic only: these price relationships do not establish "
    "realistic fills or profitability."
)

In [ ]:
# Step 56: Inspect raw individual trade breakdowns

import json
import random
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

# Reuse the selected pools from Step 54.
if "target_keys" not in globals() or "validation_url" not in globals():
    raise ValueError("Run Step 54 first; required variables are missing.")

rng = random.Random(5601)
sample_limit = 20

groups = [
    (protocol, action)
    for protocol in ["PUMPFUN", "PUMPSWAP"]
    for action in ["buy", "sell"]
]

reservoirs = {key: [] for key in groups}
seen = {key: 0 for key in groups}

# Stream the archive without loading the full hour into memory.
with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        for line in reader.read().splitlines():
            if not line.strip():
                continue

            event = json.loads(line)

            key = (
                event.get("protocol"),
                event.get("mint"),
                event.get("pool"),
            )

            if key not in target_keys:
                continue

            protocol = event.get("protocol")
            action = event.get("action")
            breakdown = event.get("breakdown") or []

            if (
                protocol not in ("PUMPFUN", "PUMPSWAP")
                or action not in ("buy", "sell")
                or len(breakdown) != 1
            ):
                continue

            leg = breakdown[0]

            # Require usable amounts for both top-level
            # and individual-trade price calculations.
            try:
                top_quote = float(event["quoteAmount"])
                top_token = float(event["tokenAmount"])
                top_price = float(event["price"])

                leg_quote = float(leg["quoteAmount"])
                leg_token = float(leg["tokenAmount"])
            except (KeyError, TypeError, ValueError):
                continue

            if min(
                top_quote, top_token, top_price,
                leg_quote, leg_token
            ) <= 0:
                continue

            group = (protocol, action)
            seen[group] += 1

            record = {
                "protocol": protocol,
                "top_action": action,
                "leg_action": leg.get("action"),
                "mint": event.get("mint"),
                "pool": event.get("pool"),
                "signature": event.get("signature"),
                "price": top_price,
                "quote_amount": top_quote,
                "token_amount": top_token,
                "amount_price": top_quote / top_token,
                "leg_quote": leg_quote,
                "leg_token": leg_token,
                "leg_amount_price": leg_quote / leg_token,
                "leg_price": leg.get("price"),
                "quote_reconciliation_pct": (
                    100 * (leg_quote / top_quote - 1)
                ),
                "token_reconciliation_pct": (
                    100 * (leg_token / top_token - 1)
                ),
            }

            # Reservoir sampling gives a reproducible
            # sample across the entire selected archive.
            bucket = reservoirs[group]

            if len(bucket) < sample_limit:
                bucket.append(record)
            else:
                j = rng.randrange(seen[group])
                if j < sample_limit:
                    bucket[j] = record

# Build one compact table.
sample_rows = [
    record
    for group in groups
    for record in reservoirs[group]
]

raw_sample = pd.DataFrame(sample_rows)

if raw_sample.empty:
    raise ValueError("No suitable individual trades were recovered.")

raw_sample["leg_action_matches_top"] = (
    raw_sample["top_action"] == raw_sample["leg_action"]
)

raw_sample["leg_price_diff_pct"] = (
    pd.to_numeric(raw_sample["leg_price"], errors="coerce")
    / raw_sample["price"] - 1
) * 100

print("=== Sample coverage ===")
print(
    pd.DataFrame([
        {
            "protocol": protocol,
            "top_action": action,
            "available_records": seen[(protocol, action)],
            "sampled_records": len(reservoirs[(protocol, action)]),
        }
        for protocol, action in groups
    ]).to_string(index=False)
)
print()

print("=== Top-level vs individual trade reconciliation ===")
print(
    raw_sample.groupby(["protocol", "top_action"]).agg(
        samples=("price", "size"),
        leg_action_matches_pct=(
            "leg_action_matches_top",
            lambda s: 100 * s.mean(),
        ),
        median_quote_diff_pct=(
            "quote_reconciliation_pct", "median"
        ),
        median_token_diff_pct=(
            "token_reconciliation_pct", "median"
        ),
        median_top_amount_price_bps=(
            "amount_price",
            lambda s: np.nan,
        ),
    ).round(3).to_string()
)
print()

# Show the actual direction and amounts for individual records.
columns = [
    "protocol",
    "top_action",
    "leg_action",
    "price",
    "amount_price",
    "leg_amount_price",
    "leg_price",
    "quote_reconciliation_pct",
    "token_reconciliation_pct",
]

print("=== Raw trade examples ===")
print(
    raw_sample[columns]
    .sort_values(["protocol", "top_action"])
    .round(10)
    .to_string(index=False)
)

print()
print(
    "Diagnostic only: a sampled record does not establish "
    "executable fills or strategy profitability."
)

In [ ]:
# Step 57: Inspect PumpSwap price differences and fee metadata
import io
import json
import random
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

rng = random.Random(5701)
sample_limit = 100

groups = [
    (protocol, action)
    for protocol in ["PUMPFUN", "PUMPSWAP"]
    for action in ["buy", "sell"]
]

samples = {key: [] for key in groups}
seen = {key: 0 for key in groups}

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
      response.raw,
      read_size=1 << 20,
    ) as reader:
      # Wrap the decompression stream to support line-by-line reading.
      text_stream = io.TextIOWrapper(reader, encoding="utf-8")

      try:
          for line in text_stream:
              if not line.strip():
                  continue

              event = json.loads(line)

              key = (
                  event.get("protocol"),
                  event.get("mint"),
                  event.get("pool"),
              )

              if key not in target_keys:
                  continue

              protocol = event.get("protocol")
              action = event.get("action")
              breakdown = event.get("breakdown") or []

              if (
                  protocol not in ("PUMPFUN", "PUMPSWAP")
                  or action not in ("buy", "sell")
                  or len(breakdown) != 1
              ):
                  continue

              try:
                  price = float(event["price"])
                  quote_amount = float(event["quoteAmount"])
                  token_amount = float(event["tokenAmount"])
              except (KeyError, TypeError, ValueError):
                  continue

              if min(price, quote_amount, token_amount) <= 0:
                  continue

              fee_info = event.get("tokenExtensions") or {}
              transfer_info = fee_info.get("transferFeeConfig") or {}

              record = {
                  "protocol": protocol,
                  "action": action,
                  "price": price,
                  "amount_price": quote_amount / token_amount,
                  "difference_pct": (
                      (quote_amount / token_amount / price) - 1
                  ) * 100,
                  "pool": event.get("pool"),
                  "pool_fee_rate": event.get("poolFeeRate"),
                  "pool_fee_after_migration": event.get(
                      "poolFeeRateAfterMigration"
                  ),
                  "holder_rewards_rate": event.get("holderRewardsRate"),
                  "transfer_fee_rate": transfer_info.get("feeRate"),
                  "cashback_enabled": event.get("cashbackEnabled"),
              }

              group = (protocol, action)
              seen[group] += 1
              bucket = samples[group]

              if len(bucket) < sample_limit:
                  bucket.append(record)
              else:
                  index = rng.randrange(seen[group])
                  if index < sample_limit:
                      bucket[index] = record

      finally:
          # Leave the underlying reader for its context manager to close.
          text_stream.detach()

data = pd.DataFrame([
    record
    for group in groups
    for record in samples[group]
])

if data.empty:
    raise ValueError("No suitable trade records were recovered.")

print("=== Sample counts ===")
print(
    pd.DataFrame([
        {
            "protocol": protocol,
            "action": action,
            "available_records": seen[(protocol, action)],
            "sampled_records": len(samples[(protocol, action)]),
        }
        for protocol, action in groups
    ]).to_string(index=False)
)
print()

# Convert known fractional fee rates into percentages.
fee_columns = [
    "pool_fee_rate",
    "pool_fee_after_migration",
    "holder_rewards_rate",
    "transfer_fee_rate",
]

for col in fee_columns:
    data[col + "_pct"] = pd.to_numeric(
        data[col], errors="coerce"
    ) * 100

print("=== Price difference and available fee metadata ===")
summary = (
    data.groupby(["protocol", "action"])
    .agg(
        samples=("difference_pct", "size"),
        median_price_difference_pct=("difference_pct", "median"),
        amount_price_above_recorded_pct=(
            "difference_pct",
            lambda s: 100 * s.gt(0).mean(),
        ),
        pool_fee_median_pct=("pool_fee_rate_pct", "median"),
        holder_rewards_median_pct=("holder_rewards_rate_pct", "median"),
        transfer_fee_median_pct=("transfer_fee_rate_pct", "median"),
        pool_fee_available_pct=(
            "pool_fee_rate",
            lambda s: 100 * s.notna().mean(),
        ),
    )
    .reset_index()
)

print(summary.round(3).to_string(index=False))
print()

print("=== PumpSwap examples ===")
print(
    data.loc[data["protocol"].eq("PUMPSWAP")][
        [
            "action",
            "pool",
            "difference_pct",
            "pool_fee_rate",
            "pool_fee_after_migration",
            "holder_rewards_rate",
            "transfer_fee_rate",
        ]
    ]
    .head(20)
    .round(5)
    .to_string(index=False)
)

In [ ]:
# Step 58: Inspect raw event sequence for one PumpSwap pool

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

# Select the most active PumpSwap pool in the validation hour.
counts = (
    validation.loc[validation["protocol"].eq("PUMPSWAP")]
    .groupby(["mint", "pool"])
    .size()
    .sort_values(ascending=False)
)

if counts.empty:
    raise ValueError("No PumpSwap pools found in validation.")

target_mint, target_pool = counts.index[0]

print("=== Selected pool ===")
print("Mint:", target_mint)
print("Pool:", target_pool)
print("Events in validation:", int(counts.iloc[0]))
print()

# Read only this pool's events from the existing archive.
rows = []
root_keys = set()
breakdown_keys = set()
line_number = 0

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in stream:
                line_number += 1

                if not line.strip():
                    continue

                event = json.loads(line)

                if (
                    event.get("protocol") != "PUMPSWAP"
                    or event.get("mint") != target_mint
                    or event.get("pool") != target_pool
                    or event.get("action") not in ("buy", "sell")
                ):
                    continue

                root_keys.update(event.keys())
                breakdown = event.get("breakdown") or []

                for leg in breakdown:
                    if isinstance(leg, dict):
                        breakdown_keys.update(leg.keys())

                try:
                    price = float(event["price"])
                    quote = float(event["quoteAmount"])
                    token = float(event["tokenAmount"])
                    quote_reserve = float(event["quoteInPool"])
                    token_reserve = float(event["tokensInPool"])
                except (KeyError, TypeError, ValueError):
                    continue

                if min(
                    price, quote, token, quote_reserve, token_reserve
                ) <= 0:
                    continue

                leg = breakdown[0] if len(breakdown) == 1 else {}

                rows.append({
                    "archive_seq": line_number,
                    "signature": event.get("signature"),
                    "block": event.get("block"),
                    "timestamp": event.get("timestamp"),
                    "localTimestamp": event.get("localTimestamp"),
                    "action": event.get("action"),
                    "price": price,
                    "quote_reserve": quote_reserve,
                    "token_reserve": token_reserve,
                    "reserve_price": quote_reserve / token_reserve,
                    "amount_price": quote / token,
                    "breakdown_count": len(breakdown),
                    "leg_action": leg.get("action"),
                    "leg_price": leg.get("price"),
                    "poolFeeRate": event.get("poolFeeRate"),
                    "holderRewardsRate": event.get("holderRewardsRate"),
                })

                if len(rows) >= 150:
                    break
        finally:
            stream.detach()

raw_pool = pd.DataFrame(rows)

if raw_pool.empty:
    raise ValueError("No valid events recovered for the selected pool.")

# Keep the original archive order for this inspection.
raw_pool["price_move_pct"] = raw_pool["price"].pct_change() * 100
raw_pool["amount_vs_recorded_pct"] = (
    raw_pool["amount_price"] / raw_pool["price"] - 1
) * 100
raw_pool["previous_action"] = raw_pool["action"].shift(1)

print("=== Raw schema ===")
print("Top-level keys:", sorted(root_keys))
print("Breakdown keys:", sorted(breakdown_keys))
print("Events recovered:", len(raw_pool))
print()

# Check whether the suspected fee fields exist and have values.
for field in ["poolFeeRate", "holderRewardsRate"]:
    print(
        f"{field}:",
        int(raw_pool[field].notna().sum()),
        "non-null values out of",
        len(raw_pool),
    )
print()

print("=== First 30 events in archive order ===")
cols = [
    "archive_seq",
    "block",
    "timestamp",
    "localTimestamp",
    "action",
    "previous_action",
    "price",
    "reserve_price",
    "amount_price",
    "price_move_pct",
    "amount_vs_recorded_pct",
    "breakdown_count",
    "leg_action",
    "leg_price",
]

print(
    raw_pool[cols].head(30).to_string(
        index=False,
        float_format=lambda x: f"{x:.8g}",
    )
)

In [ ]:
# Step 59: Test whether a missing virtual quote reserve
# could explain the PumpSwap price discrepancy.

import numpy as np
import pandas as pd

required = {
    "action",
    "price",
    "reserve_price",
    "amount_price",
    "quote_reserve",
    "token_reserve",
}

missing = required.difference(raw_pool.columns)

if missing:
    raise ValueError(f"Missing Step 58 columns: {sorted(missing)}")

estimate = raw_pool.copy()

# The recorded price matches quote_reserve / token_reserve.
# Estimate the extra quote reserve that would make the
# average trade price equal to the adjusted reserve ratio.
estimate["inferred_extra_quote_sol"] = (
    estimate["amount_price"] * estimate["token_reserve"]
    - estimate["quote_reserve"]
)

estimate["amount_vs_recorded_pct"] = (
    estimate["amount_price"] / estimate["price"] - 1
) * 100

valid = estimate.loc[
    np.isfinite(estimate["inferred_extra_quote_sol"])
    & estimate["inferred_extra_quote_sol"].gt(0)
].copy()

print("=== Inferred extra quote reserve ===")
print("Valid records:", len(valid))
print()

if not valid.empty:
    print("By trade action:")

    summary = (
        valid.groupby("action")["inferred_extra_quote_sol"]
        .agg(
            observations="size",
            median="median",
            p10=lambda s: s.quantile(0.10),
            p25=lambda s: s.quantile(0.25),
            p75=lambda s: s.quantile(0.75),
            p90=lambda s: s.quantile(0.90),
        )
    )

    print(summary.round(4).to_string())
    print()

    print("Share of estimates between 15 and 19 SOL:")
    print(
        f"{100 * valid['inferred_extra_quote_sol'].between(15, 19).mean():.2f}%"
    )
    print()

    print("=== First 15 records ===")

    columns = [
        "action",
        "price",
        "reserve_price",
        "amount_price",
        "amount_vs_recorded_pct",
        "inferred_extra_quote_sol",
    ]

    print(
        estimate[columns]
        .head(15)
        .round(6)
        .to_string(index=False)
    )

In [ ]:
# Step 60: Test the documented virtual quote reserve

import numpy as np
import pandas as pd

# Reference value from the PumpAPI pump-amm documentation.
virtual_quote_reference = 17.584505288

test = raw_pool.copy()

# Estimate an adjusted reserve price using the reference value.
test["adjusted_reserve_price"] = (
    test["quote_reserve"] + virtual_quote_reference
) / test["token_reserve"]

# Compare the average execution price with the adjusted reserve price.
test["trade_vs_adjusted_pct"] = (
    test["amount_price"] / test["adjusted_reserve_price"] - 1
) * 100

# Retain the inferred-reserve calculation for comparison.
test["inferred_extra_quote_sol"] = (
    test["amount_price"] * test["token_reserve"]
    - test["quote_reserve"]
)

print("=== Virtual-reserve comparison ===")
print("Reference virtual quote reserve:", virtual_quote_reference, "SOL")
print()

summary = test.groupby("action").agg(
    observations=("price", "size"),
    median_trade_vs_adjusted_pct=(
        "trade_vs_adjusted_pct", "median"
    ),
    p10_trade_vs_adjusted_pct=(
        "trade_vs_adjusted_pct",
        lambda s: s.quantile(0.10),
    ),
    p90_trade_vs_adjusted_pct=(
        "trade_vs_adjusted_pct",
        lambda s: s.quantile(0.90),
    ),
    within_1pct_of_adjusted_price=(
        "trade_vs_adjusted_pct",
        lambda s: 100 * s.abs().le(1).mean(),
    ),
    median_inferred_extra_quote_sol=(
        "inferred_extra_quote_sol", "median"
    ),
)

print(summary.round(3).to_string())
print()

print("=== First 10 events ===")
print(
    test[
        [
            "action",
            "price",
            "amount_price",
            "adjusted_reserve_price",
            "trade_vs_adjusted_pct",
            "inferred_extra_quote_sol",
        ]
    ]
    .head(10)
    .round(8)
    .to_string(index=False)
)

In [ ]:
# Step 61: Check inferred reserve throughout the full archive hour

import io
import json
import requests
import zstandard as zstd
import numpy as np
import pandas as pd

if not all(
    name in globals()
    for name in ["validation_url", "target_mint", "target_pool"]
):
    raise ValueError(
        "Run the earlier cells defining validation_url, "
        "target_mint and target_pool."
    )

rows = []
matched_events = 0
breakdown_counts = {}

# Stream the original compressed archive.
with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in stream:
                if not line.strip():
                    continue

                event = json.loads(line)

                if (
                    event.get("protocol") != "PUMPSWAP"
                    or event.get("mint") != target_mint
                    or event.get("pool") != target_pool
                    or event.get("action") not in ("buy", "sell")
                ):
                    continue

                matched_events += 1
                breakdown = event.get("breakdown") or []
                count = len(breakdown) if isinstance(breakdown, list) else 0
                breakdown_counts[count] = (
                    breakdown_counts.get(count, 0) + 1
                )

                # Keep single-leg events so that average trade price
                # is not distorted by an aggregate of multiple swaps.
                if count != 1:
                    continue

                try:
                    price = float(event["price"])
                    quote_amount = float(event["quoteAmount"])
                    token_amount = float(event["tokenAmount"])
                    quote_reserve = float(event["quoteInPool"])
                    token_reserve = float(event["tokensInPool"])
                    local_ms = float(event["localTimestamp"])
                except (KeyError, TypeError, ValueError):
                    continue

                if min(
                    price,
                    quote_amount,
                    token_amount,
                    quote_reserve,
                    token_reserve,
                ) <= 0:
                    continue

                rows.append({
                    "action": event["action"],
                    "price": price,
                    "quote_amount": quote_amount,
                    "token_amount": token_amount,
                    "quote_reserve": quote_reserve,
                    "token_reserve": token_reserve,
                    "local_ms": local_ms,
                })

        finally:
            stream.detach()

pool_all = pd.DataFrame(rows)

if pool_all.empty:
    raise ValueError("No valid single-leg events recovered.")

# Reconstruct prices and the implied additional quote reserve.
pool_all["observed_time"] = pd.to_datetime(
    pool_all["local_ms"],
    unit="ms",
    utc=True,
)

pool_all = pool_all.sort_values("observed_time").reset_index(drop=True)

pool_all["amount_price"] = (
    pool_all["quote_amount"] / pool_all["token_amount"]
)

pool_all["inferred_extra_quote_sol"] = (
    pool_all["amount_price"] * pool_all["token_reserve"]
    - pool_all["quote_reserve"]
)

pool_all["amount_vs_recorded_pct"] = (
    pool_all["amount_price"] / pool_all["price"] - 1
) * 100

virtual_quote_reference = 17.584505288

pool_all["trade_vs_adjusted_pct"] = (
    pool_all["amount_price"]
    / (
        (pool_all["quote_reserve"] + virtual_quote_reference)
        / pool_all["token_reserve"]
    )
    - 1
) * 100

pool_all = pool_all.loc[
    np.isfinite(pool_all["inferred_extra_quote_sol"])
].copy()

pool_all["minute"] = (
    (pool_all["observed_time"] - pool_all["observed_time"].min())
    .dt.total_seconds()
    .floordiv(60)
    .astype(int)
)

print("=== Pool coverage ===")
print("Mint:", target_mint)
print("Pool:", target_pool)
print("Total matching archive events:", f"{matched_events:,}")
print("Breakdown counts:", dict(sorted(breakdown_counts.items())))
print("Valid single-leg events:", f"{len(pool_all):,}")
print("First event:", pool_all["observed_time"].min())
print("Last event:", pool_all["observed_time"].max())
print("Minutes covered:", pool_all["minute"].nunique())
print()

print("=== Full-hour inferred reserve by action ===")
overall = pool_all.groupby("action").agg(
    events=("inferred_extra_quote_sol", "size"),
    median_inferred_extra=("inferred_extra_quote_sol", "median"),
    p10_inferred_extra=(
        "inferred_extra_quote_sol",
        lambda s: s.quantile(0.10),
    ),
    p90_inferred_extra=(
        "inferred_extra_quote_sol",
        lambda s: s.quantile(0.90),
    ),
    median_amount_vs_recorded_pct=(
        "amount_vs_recorded_pct", "median"
    ),
    median_trade_vs_adjusted_pct=(
        "trade_vs_adjusted_pct", "median"
    ),
    within_1pct_adjusted_pct=(
        "trade_vs_adjusted_pct",
        lambda s: 100 * s.abs().le(1).mean(),
    ),
)

print(overall.round(4).to_string())
print()

minute_summary = (
    pool_all.groupby(["minute", "action"])
    .agg(
        events=("inferred_extra_quote_sol", "size"),
        median_inferred_extra=(
            "inferred_extra_quote_sol", "median"
        ),
        median_adjusted_diff_pct=(
            "trade_vs_adjusted_pct", "median"
        ),
    )
    .reset_index()
)

print("=== First 10 minute/action groups ===")
print(minute_summary.head(10).round(4).to_string(index=False))
print()

print("=== Last 10 minute/action groups ===")
print(minute_summary.tail(10).round(4).to_string(index=False))

In [ ]:
# Step 62: Compare inferred offsets across PumpSwap pools

import numpy as np
import pandas as pd

if "audit" not in globals():
    raise ValueError("Run Step 54 first; the 'audit' DataFrame is missing.")

required = {
    "protocol",
    "mint",
    "pool",
    "action",
    "price",
    "amount_price",
    "quote_in_pool",
    "tokens_in_pool",
    "breakdown_count",
}

missing = required.difference(audit.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

reference_offset = 17.584505288

ps = audit.loc[
    audit["protocol"].eq("PUMPSWAP")
    & audit["breakdown_count"].eq(1)
].copy()

for col in [
    "price",
    "amount_price",
    "quote_in_pool",
    "tokens_in_pool",
]:
    ps[col] = pd.to_numeric(ps[col], errors="coerce")

ps = ps.replace([np.inf, -np.inf], np.nan).dropna(
    subset=[
        "price",
        "amount_price",
        "quote_in_pool",
        "tokens_in_pool",
    ]
)

ps = ps.loc[
    (ps["price"] > 0)
    & (ps["tokens_in_pool"] > 0)
].copy()

# Same inferred-offset calculation used in Step 59.
ps["inferred_offset_sol"] = (
    ps["amount_price"] * ps["tokens_in_pool"]
    - ps["quote_in_pool"]
)

ps["amount_vs_recorded_pct"] = (
    ps["amount_price"] / ps["price"] - 1
) * 100

print("=== Overall inferred offset ===")
print("Single-leg PumpSwap records:", f"{len(ps):,}")
print()

print(
    ps.groupby("action").agg(
        events=("inferred_offset_sol", "size"),
        median_offset_sol=("inferred_offset_sol", "median"),
        p10_offset_sol=(
            "inferred_offset_sol",
            lambda s: s.quantile(0.10),
        ),
        p90_offset_sol=(
            "inferred_offset_sol",
            lambda s: s.quantile(0.90),
        ),
        median_price_difference_pct=(
            "amount_vs_recorded_pct", "median"
        ),
    ).round(4).to_string()
)
print()

# Summarize each pool and action separately.
per_pool = (
    ps.groupby(["mint", "pool", "action"])
    .agg(
        events=("inferred_offset_sol", "size"),
        median_offset_sol=("inferred_offset_sol", "median"),
        median_price_difference_pct=(
            "amount_vs_recorded_pct", "median"
        ),
    )
    .reset_index()
)

pool_counts = (
    ps.groupby(["mint", "pool"])
    .size()
    .sort_values(ascending=False)
)

pool_medians = per_pool.pivot_table(
    index=["mint", "pool"],
    columns="action",
    values="median_offset_sol",
)

print("=== Pool-level comparison with reference ===")
print("Reference offset:", reference_offset, "SOL")
print("Pools with both buy and sell samples:", int(
    pool_medians[["buy", "sell"]].dropna().shape[0]
))
print()

for action in ["buy", "sell"]:
    if action not in pool_medians.columns:
        continue

    values = pool_medians[action].dropna()
    if values.empty:
        continue

    print(
        f"{action.capitalize()} pool medians within +/-1 SOL:",
        f"{100 * (values - reference_offset).abs().le(1).mean():.2f}%",
        f"({int((values - reference_offset).abs().le(1).sum())}"
        f" of {len(values)} pools)"
    )

print()

# Display the 12 most active sampled pools.
top_keys = (
    pool_counts.head(12)
    .rename("total_events")
    .reset_index()
)

top = per_pool.merge(
    top_keys[["mint", "pool", "total_events"]],
    on=["mint", "pool"],
    how="inner",
)

print("=== 12 most active sampled pools ===")
print(
    top.sort_values(
        ["total_events", "action"],
        ascending=[False, True],
    )[
        [
            "mint",
            "pool",
            "total_events",
            "action",
            "events",
            "median_offset_sol",
            "median_price_difference_pct",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

In [ ]:
# Step 63: Decode actual PumpSwap virtual quote reserves
# from a small sample of on-chain transaction logs.

import base64
import io
import json
import time

import pandas as pd
import requests
import zstandard as zstd

if not all(
    name in globals()
    for name in ["validation_url", "target_mint", "target_pool"]
):
    raise ValueError(
        "Run the earlier cells defining validation_url, "
        "target_mint and target_pool."
    )

# RPC_URL = "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}"

# 1. Read the Helius API key from Colab Secrets.
HELIUS_API_KEY = userdata.get("HELIUS_API_KEY")

if not HELIUS_API_KEY:
    raise RuntimeError(
        "HELIUS_API_KEY not found. Add it in Google Colab Secrets."
    )

RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"

IDL_URL = (
    "https://raw.githubusercontent.com/pump-fun/"
    "pump-public-docs/main/idl/pump_amm.json"
)

# 1. Load the official PumpSwap IDL.
idl_response = requests.get(IDL_URL, timeout=30)
idl_response.raise_for_status()
idl = idl_response.json()

event_discriminators = {
    event["name"]: bytes(event["discriminator"])
    for event in idl.get("events", [])
    if event["name"] in ("BuyEvent", "SellEvent")
}

event_fields = {
    item["name"]: item["type"]["fields"]
    for item in idl.get("types", [])
    if item["name"] in ("BuyEvent", "SellEvent")
}

for event_name in ("BuyEvent", "SellEvent"):
    if event_name not in event_discriminators or event_name not in event_fields:
        raise ValueError(
            f"The current IDL is missing {event_name}."
        )

    names = [field["name"] for field in event_fields[event_name]]
    if "virtual_quote_reserves" not in names or "pool" not in names:
        raise ValueError(
            f"The expected fields are missing from {event_name}."
        )

print("=== IDL check ===")
for event_name in ("BuyEvent", "SellEvent"):
    print(
        event_name,
        "| virtual_quote_reserves present:",
        "virtual_quote_reserves"
        in [f["name"] for f in event_fields[event_name]],
    )
print()

# 2. Collect a small sample of signatures at different points
# in the hour, with both buy and sell actions when available.
pool_validation = validation.loc[
    validation["protocol"].eq("PUMPSWAP")
    & validation["mint"].eq(target_mint)
    & validation["pool"].eq(target_pool)
].copy()

if pool_validation.empty:
    raise ValueError("The selected pool is absent from validation.")

pool_validation["observed_time"] = pd.to_datetime(
    pool_validation["observed_time"], utc=True
)

start_epoch = pool_validation["observed_time"].min().timestamp()

# Five elapsed-time bins; collect one event per action per bin.
time_bins = [0, 10, 20, 30, 40]
signatures = {}

line_count = 0

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in stream:
                if not line.strip():
                    continue

                line_count += 1
                event = json.loads(line)

                if (
                    event.get("protocol") != "PUMPSWAP"
                    or event.get("mint") != target_mint
                    or event.get("pool") != target_pool
                    or event.get("action") not in ("buy", "sell")
                ):
                    continue

                if len(event.get("breakdown") or []) != 1:
                    continue

                signature = event.get("signature")
                local_ms = event.get("localTimestamp")

                if not signature or local_ms is None:
                    continue

                try:
                    elapsed_min = int(
                        (float(local_ms) / 1000 - start_epoch) // 60
                    )
                except (TypeError, ValueError):
                    continue

                if elapsed_min not in time_bins:
                    continue

                key = (event["action"], elapsed_min)

                if key not in signatures:
                    signatures[key] = {
                        "signature": signature,
                        "action": event["action"],
                        "minute": elapsed_min,
                    }
        finally:
            stream.detach()

sample_transactions = list(signatures.values())

if not sample_transactions:
    raise ValueError(
        "No suitable transaction signatures were recovered."
    )

print("=== Transaction sample ===")
print("Archive lines scanned:", f"{line_count:,}")
print("Transactions selected:", len(sample_transactions))
print(
    pd.DataFrame(sample_transactions)
    .sort_values(["minute", "action"])
    .to_string(index=False)
)
print()

# 3. Helper functions for decoding Borsh event data.

B58_ALPHABET = (
    "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
)

def base58_encode(raw):
    """Convert a 32-byte public key to Base58."""
    zero_count = len(raw) - len(raw.lstrip(b"\x00"))
    number = int.from_bytes(raw, "big")
    chars = []

    while number:
        number, remainder = divmod(number, 58)
        chars.append(B58_ALPHABET[remainder])

    encoded = "".join(reversed(chars))
    return "1" * zero_count + encoded


def decode_event_fields(data, fields):
    """
    Decode the supported Borsh field types in BuyEvent/SellEvent.
    The first 8 bytes are the Anchor event discriminator.
    """
    offset = 8
    result = {}

    widths = {
        "u8": (1, False),
        "i8": (1, True),
        "u16": (2, False),
        "i16": (2, True),
        "u32": (4, False),
        "i32": (4, True),
        "u64": (8, False),
        "i64": (8, True),
        "u128": (16, False),
        "i128": (16, True),
    }

    for field in fields:
        name = field["name"]
        kind = field["type"]

        if kind == "pubkey":
            raw = data[offset:offset + 32]
            if len(raw) != 32:
                raise ValueError("Incomplete public key.")
            value = base58_encode(raw)
            offset += 32

        elif kind == "bool":
            raw = data[offset:offset + 1]
            if len(raw) != 1:
                raise ValueError("Incomplete bool.")
            value = bool(raw[0])
            offset += 1

        elif kind == "string":
            raw_len = data[offset:offset + 4]
            if len(raw_len) != 4:
                raise ValueError("Incomplete string length.")

            length = int.from_bytes(raw_len, "little")
            offset += 4
            raw = data[offset:offset + length]

            if len(raw) != length:
                raise ValueError("Incomplete string.")

            value = raw.decode("utf-8")
            offset += length

        elif kind in widths:
            width, signed = widths[kind]
            raw = data[offset:offset + width]

            if len(raw) != width:
                raise ValueError(f"Incomplete field: {name}")

            value = int.from_bytes(
                raw, "little", signed=signed
            )
            offset += width

        else:
            raise ValueError(
                f"Unsupported IDL type {kind!r} for field {name!r}."
            )

        # Retain only the fields needed for this audit.
        if name in {
            "timestamp",
            "pool",
            "virtual_quote_reserves",
            "pool_quote_token_reserves",
            "pool_base_token_reserves",
            "quote_amount_in",
            "quote_amount_out",
            "lp_fee",
            "protocol_fee",
            "coin_creator_fee",
            "creator_fee_unclaimed",
        }:
            result[name] = value

    return result


# 4. Query recent transactions and decode their PumpSwap events.
results = []

for sample in sample_transactions:
    signature = sample["signature"]

    try:
        rpc_response = requests.post(
            RPC_URL,
            json={
                "jsonrpc": "2.0",
                "id": 1,
                "method": "getTransaction",
                "params": [
                    signature,
                    {
                        "encoding": "json",
                        "maxSupportedTransactionVersion": 0,
                        "commitment": "confirmed",
                    },
                ],
            },
            timeout=30,
        )
        rpc_response.raise_for_status()
        rpc_json = rpc_response.json()

    except Exception as exc:
        results.append({
            **sample,
            "status": f"RPC request error: {type(exc).__name__}",
        })
        time.sleep(0.2)
        continue

    transaction = rpc_json.get("result")

    if transaction is None:
        results.append({
            **sample,
            "status": "Transaction unavailable from this RPC",
        })
        time.sleep(0.2)
        continue

    meta = transaction.get("meta") or {}

    if meta.get("err") is not None:
        results.append({
            **sample,
            "status": "Transaction failed",
        })
        time.sleep(0.2)
        continue

    logs = meta.get("logMessages") or []
    decoded_count = 0

    for log in logs:
        prefix = "Program data: "

        if not log.startswith(prefix):
            continue

        try:
            payload = base64.b64decode(
                log[len(prefix):].strip(),
                validate=True,
            )
        except Exception:
            continue

        for event_name in ("BuyEvent", "SellEvent"):
            discriminator = event_discriminators[event_name]

            if not payload.startswith(discriminator):
                continue

            try:
                fields = decode_event_fields(
                    payload,
                    event_fields[event_name],
                )
            except (ValueError, UnicodeDecodeError):
                continue

            # A transaction can contain several pool events.
            if fields.get("pool") != target_pool:
                continue

            virtual_raw = fields.get("virtual_quote_reserves")

            results.append({
                **sample,
                "status": "Decoded",
                "event_name": event_name,
                "decoded_action": (
                    "buy" if event_name == "BuyEvent" else "sell"
                ),
                "event_timestamp": fields.get("timestamp"),
                "virtual_quote_reserves_raw": virtual_raw,
                "virtual_quote_reserves_SOL": (
                    virtual_raw / 1_000_000_000
                    if virtual_raw is not None
                    else None
                ),
                "pool_quote_reserves_raw": fields.get(
                    "pool_quote_token_reserves"
                ),
                "pool_base_reserves_raw": fields.get(
                    "pool_base_token_reserves"
                ),
                "lp_fee_raw": fields.get("lp_fee"),
                "protocol_fee_raw": fields.get("protocol_fee"),
            })

            decoded_count += 1

    if decoded_count == 0:
        results.append({
            **sample,
            "status": "No matching pool event decoded",
        })

    time.sleep(0.2)

decoded_df = pd.DataFrame(results)

print("=== On-chain event results ===")
if decoded_df.empty:
    print("No results returned.")
else:
    print(decoded_df.to_string(index=False))

In [ ]:
# Step 64: Compare archive prices with on-chain effective reserves

import io
import json
import requests
import pandas as pd
import zstandard as zstd

required_globals = [
    "validation_url",
    "target_mint",
    "target_pool",
    "sample_transactions",
    "decoded_df",
]

missing_globals = [
    name for name in required_globals if name not in globals()
]

if missing_globals:
    raise ValueError(
        f"Run Step 63 first. Missing variables: {missing_globals}"
    )

# RPC_URL = "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}"

# 1. Read the Helius API key from Colab Secrets.
HELIUS_API_KEY = userdata.get("HELIUS_API_KEY")

if not HELIUS_API_KEY:
    raise RuntimeError(
        "HELIUS_API_KEY not found. Add it in Google Colab Secrets."
    )

RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"


# 1. Query the token's actual decimal precision.
rpc_response = requests.post(
    RPC_URL,
    json={
        "jsonrpc": "2.0",
        "id": 1,
        "method": "getTokenSupply",
        "params": [target_mint],
    },
    timeout=30,
)

rpc_response.raise_for_status()
rpc_result = rpc_response.json()

if "error" in rpc_result or not rpc_result.get("result"):
    raise ValueError(f"Could not retrieve token decimals: {rpc_result}")

token_decimals = int(
    rpc_result["result"]["value"]["decimals"]
)

print("=== Token units ===")
print("Token:", target_mint)
print("Token decimals:", token_decimals)
print("Quote asset: SOL (9 decimals)")
print()

# 2. Recover the exact archive records for the sampled signatures.
wanted_signatures = {
    item["signature"] for item in sample_transactions
}

archive_rows = []

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in stream:
                if not line.strip():
                    continue

                event = json.loads(line)

                if event.get("signature") not in wanted_signatures:
                    continue

                if (
                    event.get("protocol") != "PUMPSWAP"
                    or event.get("mint") != target_mint
                    or event.get("pool") != target_pool
                ):
                    continue

                archive_rows.append({
                    "signature": event["signature"],
                    "action": event.get("action"),
                    "archive_price": event.get("price"),
                    "quoteAmount": event.get("quoteAmount"),
                    "tokenAmount": event.get("tokenAmount"),
                    "quoteInPool": event.get("quoteInPool"),
                    "tokensInPool": event.get("tokensInPool"),
                    "localTimestamp": event.get("localTimestamp"),
                })
        finally:
            stream.detach()

archive_sample = pd.DataFrame(archive_rows)

if archive_sample.empty:
    raise ValueError("None of the sampled archive events were recovered.")

# 3. Join with the decoded on-chain events from Step 63.
chain_sample = decoded_df.loc[
    decoded_df["status"].eq("Decoded")
].copy()

chain_sample = chain_sample.drop_duplicates(
    subset=["signature"]
)

columns_needed = [
    "signature",
    "event_name",
    "decoded_action",
    "virtual_quote_reserves_raw",
    "pool_quote_reserves_raw",
    "pool_base_reserves_raw",
]

missing_columns = set(columns_needed).difference(chain_sample.columns)

if missing_columns:
    raise ValueError(
        f"Decoded events lack required fields: {sorted(missing_columns)}"
    )

comparison = archive_sample.merge(
    chain_sample[columns_needed],
    on="signature",
    how="inner",
    validate="one_to_one",
)

if comparison.empty:
    raise ValueError("No matching archive/on-chain events.")

# 4. Convert human-readable archive values and raw on-chain units.
numeric_cols = [
    "archive_price",
    "quoteAmount",
    "tokenAmount",
    "quoteInPool",
    "tokensInPool",
    "virtual_quote_reserves_raw",
    "pool_quote_reserves_raw",
    "pool_base_reserves_raw",
]

for col in numeric_cols:
    comparison[col] = pd.to_numeric(
        comparison[col], errors="coerce"
    )

comparison["onchain_quote_sol"] = (
    comparison["pool_quote_reserves_raw"] / 1_000_000_000
)

comparison["onchain_virtual_quote_sol"] = (
    comparison["virtual_quote_reserves_raw"] / 1_000_000_000
)

comparison["onchain_base_tokens"] = (
    comparison["pool_base_reserves_raw"]
    / (10 ** token_decimals)
)

comparison["archive_amount_price"] = (
    comparison["quoteAmount"] / comparison["tokenAmount"]
)

comparison["onchain_raw_price"] = (
    comparison["onchain_quote_sol"]
    / comparison["onchain_base_tokens"]
)

comparison["onchain_effective_price"] = (
    (
        comparison["onchain_quote_sol"]
        + comparison["onchain_virtual_quote_sol"]
    )
    / comparison["onchain_base_tokens"]
)

# Differences relative to the corresponding reference prices.
comparison["archive_vs_raw_pct"] = (
    comparison["archive_price"]
    / comparison["onchain_raw_price"] - 1
) * 100

comparison["archive_reserve_quote_diff_pct"] = (
    comparison["quoteInPool"]
    / comparison["onchain_quote_sol"] - 1
) * 100

comparison["archive_reserve_token_diff_pct"] = (
    comparison["tokensInPool"]
    / comparison["onchain_base_tokens"] - 1
) * 100

comparison["amount_vs_effective_pct"] = (
    comparison["archive_amount_price"]
    / comparison["onchain_effective_price"] - 1
) * 100

# 5. Print a compact comparison table.
print("=== Matching records ===")
print("Archive records:", len(archive_sample))
print("Matched on-chain records:", len(comparison))
print()

print("=== Per-transaction price comparison ===")
display_columns = [
    "action",
    "archive_price",
    "onchain_raw_price",
    "onchain_effective_price",
    "archive_amount_price",
    "onchain_virtual_quote_sol",
    "archive_reserve_quote_diff_pct",
    "archive_reserve_token_diff_pct",
    "amount_vs_effective_pct",
]

print(
    comparison[display_columns]
    .sort_values("action")
    .round(8)
    .to_string(index=False)
)
print()

print("=== Summary by trade direction ===")
print(
    comparison.groupby("action").agg(
        transactions=("signature", "size"),
        median_virtual_quote_sol=(
            "onchain_virtual_quote_sol", "median"
        ),
        median_archive_vs_raw_pct=(
            "archive_vs_raw_pct", "median"
        ),
        median_archive_quote_reserve_diff_pct=(
            "archive_reserve_quote_diff_pct", "median"
        ),
        median_archive_token_reserve_diff_pct=(
            "archive_reserve_token_diff_pct", "median"
        ),
        median_amount_vs_effective_pct=(
            "amount_vs_effective_pct", "median"
        ),
    ).round(5).to_string()
)

In [ ]:
# Step 65: Decode all available PumpSwap trade fee fields

import base64
import time
import requests
import pandas as pd

if not all(
    name in globals()
    for name in [
        "sample_transactions",
        "archive_sample",
        "event_fields",
        "event_discriminators",
        "target_pool",
    ]
):
    raise ValueError("Run Steps 63 and 64 first.")

# Use the public Solana RPC defined in Step 63.
rpc_url = globals().get(
    "RPC_URL",
    "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}",
)

def decode_all_fields(data, fields):
    """Decode primitive Borsh fields from a PumpSwap event."""
    offset = 8
    result = {}

    widths = {
        "u8": (1, False),
        "i8": (1, True),
        "u16": (2, False),
        "i16": (2, True),
        "u32": (4, False),
        "i32": (4, True),
        "u64": (8, False),
        "i64": (8, True),
        "u128": (16, False),
        "i128": (16, True),
    }

    alphabet = (
        "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
    )

    def encode_pubkey(raw):
        zero_count = len(raw) - len(raw.lstrip(b"\x00"))
        number = int.from_bytes(raw, "big")
        chars = []

        while number:
            number, rem = divmod(number, 58)
            chars.append(alphabet[rem])

        return "1" * zero_count + "".join(reversed(chars))

    for field in fields:
        name = field["name"]
        kind = field["type"]

        if kind == "pubkey":
            raw = data[offset:offset + 32]
            if len(raw) != 32:
                raise ValueError("Incomplete public key")
            value = encode_pubkey(raw)
            offset += 32

        elif kind == "bool":
            raw = data[offset:offset + 1]
            if len(raw) != 1:
                raise ValueError("Incomplete boolean")
            value = bool(raw[0])
            offset += 1

        elif kind == "string":
            raw = data[offset:offset + 4]
            if len(raw) != 4:
                raise ValueError("Incomplete string length")

            length = int.from_bytes(raw, "little")
            offset += 4
            raw = data[offset:offset + length]

            if len(raw) != length:
                raise ValueError("Incomplete string")

            value = raw.decode("utf-8")
            offset += length

        elif kind in widths:
            width, signed = widths[kind]
            raw = data[offset:offset + width]

            if len(raw) != width:
                raise ValueError(f"Incomplete field: {name}")

            value = int.from_bytes(
                raw, "little", signed=signed
            )
            offset += width

        else:
            raise ValueError(
                f"Unsupported type {kind!r} in field {name!r}"
            )

        result[name] = value

    return result


# Retain only fields useful for explaining fees and prices.
wanted_fields = [
    "timestamp",
    "pool",
    "base_amount_out",
    "base_amount_in",
    "quote_amount_in",
    "quote_amount_out",
    "quote_amount_in_with_lp_fee",
    "quote_amount_out_without_lp_fee",
    "user_quote_amount_in",
    "user_quote_amount_out",
    "lp_fee_basis_points",
    "lp_fee",
    "protocol_fee_basis_points",
    "protocol_fee",
    "coin_creator_fee_basis_points",
    "coin_creator_fee",
    "cashback_fee_basis_points",
    "cashback",
    "buyback_fee_basis_points",
    "buyback_fee",
    "holder_rewards_bps",
    "holder_rewards",
    "virtual_quote_reserves",
    "ix_name",
]

records = []

for sample in sample_transactions:
    signature = sample["signature"]

    try:
        response = requests.post(
            rpc_url,
            json={
                "jsonrpc": "2.0",
                "id": 1,
                "method": "getTransaction",
                "params": [
                    signature,
                    {
                        "encoding": "json",
                        "maxSupportedTransactionVersion": 0,
                        "commitment": "confirmed",
                    },
                ],
            },
            timeout=30,
        )

        response.raise_for_status()
        payload = response.json()

    except Exception as exc:
        records.append({
            "signature": signature,
            "status": f"RPC error: {type(exc).__name__}",
        })
        continue

    tx = payload.get("result")

    if not tx:
        records.append({
            "signature": signature,
            "status": "Transaction unavailable",
        })
        continue

    meta = tx.get("meta") or {}

    if meta.get("err") is not None:
        records.append({
            "signature": signature,
            "status": "Transaction failed",
        })
        continue

    found = False

    for log in meta.get("logMessages") or []:
        prefix = "Program data: "

        if not log.startswith(prefix):
            continue

        try:
            raw = base64.b64decode(
                log[len(prefix):].strip(),
                validate=True,
            )
        except Exception:
            continue

        event_name = None

        for candidate in ("BuyEvent", "SellEvent"):
            if raw.startswith(event_discriminators[candidate]):
                event_name = candidate
                break

        if event_name is None:
            continue

        try:
            decoded = decode_all_fields(
                raw,
                event_fields[event_name],
            )
        except (ValueError, UnicodeDecodeError):
            continue

        if decoded.get("pool") != target_pool:
            continue

        row = {
            "signature": signature,
            "archive_action": sample["action"],
            "event_name": event_name,
            "decoded_action": (
                "buy" if event_name == "BuyEvent" else "sell"
            ),
            "status": "Decoded",
        }

        for field in wanted_fields:
            if field in decoded:
                row[field] = decoded[field]

        records.append(row)
        found = True
        break

    if not found:
        records.append({
            "signature": signature,
            "status": "No matching event decoded",
        })

    time.sleep(0.2)

fees_df = pd.DataFrame(records)

print("=== Decoding results ===")
print(
    fees_df["status"]
    .value_counts(dropna=False)
    .to_string()
)
print()

valid = fees_df.loc[
    fees_df["status"].eq("Decoded")
].copy()

if valid.empty:
    print("No matching events decoded. Inspect the RPC errors above.")
else:
    # Convert fee amounts from lamports to SOL.
    fee_amount_fields = [
        "lp_fee",
        "protocol_fee",
        "coin_creator_fee",
        "cashback",
        "buyback_fee",
    ]

    for field in fee_amount_fields:
        if field in valid.columns:
            valid[field + "_SOL"] = (
                pd.to_numeric(valid[field], errors="coerce")
                / 1_000_000_000
            )

    # Normalize the decoded field names into a single table.
    columns = [
        "archive_action",
        "decoded_action",
        "ix_name",
        "quote_amount_in",
        "quote_amount_out",
        "user_quote_amount_in",
        "user_quote_amount_out",
        "lp_fee_basis_points",
        "lp_fee_SOL",
        "protocol_fee_basis_points",
        "protocol_fee_SOL",
        "coin_creator_fee_basis_points",
        "coin_creator_fee_SOL",
        "cashback_fee_basis_points",
        "cashback_SOL",
        "buyback_fee_basis_points",
        "buyback_fee_SOL",
        "virtual_quote_reserves",
    ]

    columns = [c for c in columns if c in valid.columns]

    print("=== Transaction fee details ===")
    print(
        valid[columns].to_string(
            index=False,
            float_format=lambda x: f"{x:.9g}",
        )
    )
    print()

    # Summarize the actual fee rates reported by the events.
    bps_fields = [
        "lp_fee_basis_points",
        "protocol_fee_basis_points",
        "coin_creator_fee_basis_points",
        "cashback_fee_basis_points",
        "buyback_fee_basis_points",
    ]

    available_bps = [
        field for field in bps_fields
        if field in valid.columns
    ]

    print("=== Fee-rate summary (basis points) ===")
    if available_bps:
        print(
            valid[available_bps]
            .apply(pd.to_numeric, errors="coerce")
            .describe()
            .round(3)
            .to_string()
        )

    print()
    print("=== Archive vs decoded event actions ===")
    print(
        valid[
            [
                "archive_action",
                "decoded_action",
                "ix_name",
            ]
        ].to_string(index=False)
    )

In [ ]:
# Step 66: Reconcile actual trade fees and prices

import numpy as np
import pandas as pd

if "fees_df" not in globals() or "archive_sample" not in globals():
    raise ValueError("Run Steps 64 and 65 first.")

fees = fees_df.loc[
    fees_df["status"].eq("Decoded")
].copy()

archive = archive_sample.copy()

# Join the decoded on-chain fee information to the original
# archive records using the transaction signature.
df66 = archive.merge(
    fees,
    on="signature",
    how="inner",
    validate="one_to_one",
    suffixes=("_archive", "_event"),
)

if df66.empty:
    raise ValueError("No matching transactions found.")

# Convert monetary and reserve fields to numeric values.
numeric_cols = [
    "quoteAmount",
    "tokenAmount",
    "quoteInPool",
    "tokensInPool",
    "price",
    "quote_amount_in",
    "quote_amount_out",
    "user_quote_amount_in",
    "user_quote_amount_out",
    "lp_fee",
    "protocol_fee",
    "coin_creator_fee",
    "virtual_quote_reserves",
]

for col in numeric_cols:
    if col in df66.columns:
        df66[col] = pd.to_numeric(
            df66[col], errors="coerce"
        )

# Gross and user-side quote amounts are recorded in lamports
# in the decoded event fields.
is_buy = df66["action"].eq("buy")

df66["gross_quote_lamports"] = np.where(
    is_buy,
    df66["quote_amount_in"],
    df66["quote_amount_out"],
)

df66["user_quote_lamports"] = np.where(
    is_buy,
    df66["user_quote_amount_in"],
    df66["user_quote_amount_out"],
)

# Core fees are the LP, protocol and creator fees.
# Do not add the buyback fee separately here.
df66["core_fees_lamports"] = (
    df66["lp_fee"]
    + df66["protocol_fee"]
    + df66["coin_creator_fee"]
)

df66["gross_minus_user_lamports"] = (
    df66["gross_quote_lamports"]
    - df66["user_quote_lamports"]
)

df66["fee_reconciliation_error_lamports"] = (
    df66["gross_minus_user_lamports"]
    - df66["core_fees_lamports"]
)

df66["reported_fee_bps"] = (
    df66["lp_fee_basis_points"]
    + df66["protocol_fee_basis_points"]
    + df66["coin_creator_fee_basis_points"]
)

# Convert the quote amounts to SOL and calculate unit prices.
df66["gross_quote_sol"] = (
    df66["gross_quote_lamports"] / 1_000_000_000
)

df66["user_quote_sol"] = (
    df66["user_quote_lamports"] / 1_000_000_000
)

df66["fee_free_curve_price"] = np.where(
    is_buy,
    df66["user_quote_sol"] / df66["tokenAmount"],
    (
        df66["quote_amount_out"] / 1_000_000_000
    ) / df66["tokenAmount"],
)

# Actual cost per token for buys; actual proceeds per token for sells.
df66["actual_trader_price"] = np.where(
    is_buy,
    df66["gross_quote_sol"] / df66["tokenAmount"],
    df66["user_quote_sol"] / df66["tokenAmount"],
)

# Calculate effective reserve price using the archived reserve
# and the signed virtual quote reserve decoded from the event.
df66["effective_reserve_price"] = (
    df66["quoteInPool"]
    + df66["virtual_quote_reserves"] / 1_000_000_000
) / df66["tokensInPool"]

df66["curve_vs_reserve_pct"] = (
    df66["fee_free_curve_price"]
    / df66["effective_reserve_price"] - 1
) * 100

df66["trader_vs_reserve_pct"] = (
    df66["actual_trader_price"]
    / df66["effective_reserve_price"] - 1
) * 100

print("=== Fee accounting ===")
print("Matched transactions:", len(df66))
print(
    "Transactions where gross minus user amount equals "
    "LP + protocol + creator fees:",
    int(
        df66["fee_reconciliation_error_lamports"]
        .eq(0)
        .sum()
    ),
    "of",
    len(df66),
)
print()

print("=== Price comparison by trade direction ===")
summary = (
    df66.groupby("action")
    .agg(
        transactions=("signature", "size"),
        median_total_fee_bps=("reported_fee_bps", "median"),
        median_fee_reconciliation_error_lamports=(
            "fee_reconciliation_error_lamports", "median"
        ),
        median_curve_vs_reserve_pct=(
            "curve_vs_reserve_pct", "median"
        ),
        median_actual_trader_vs_reserve_pct=(
            "trader_vs_reserve_pct", "median"
        ),
    )
)

print(summary.round(5).to_string())
print()

print("=== Individual transaction results ===")
columns = [
    "action",
    "gross_quote_lamports",
    "user_quote_lamports",
    "core_fees_lamports",
    "reported_fee_bps",
    "fee_reconciliation_error_lamports",
    "curve_vs_reserve_pct",
    "trader_vs_reserve_pct",
]

print(
    df66[columns]
    .sort_values("action")
    .round(5)
    .to_string(index=False)
)

In [ ]:
# Step 67: Verify quote and base token transfers from RPC balances

import time
import requests
import pandas as pd

if "sample_transactions" not in globals() or "fees_df" not in globals():
    raise ValueError("Run Steps 63 and 65 first.")

# RPC_URL = "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}"

# 1. Read the Helius API key from Colab Secrets.
HELIUS_API_KEY = userdata.get("HELIUS_API_KEY")

if not HELIUS_API_KEY:
    raise RuntimeError(
        "HELIUS_API_KEY not found. Add it in Google Colab Secrets."
    )

RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"


WSOL_MINT = "So11111111111111111111111111111111111111112"

balance_rows = []
event_rows = []
errors = []

for sample in sample_transactions:
    signature = sample["signature"]

    response = requests.post(
        RPC_URL,
        json={
            "jsonrpc": "2.0",
            "id": 1,
            "method": "getTransaction",
            "params": [
                signature,
                {
                    "encoding": "jsonParsed",
                    "maxSupportedTransactionVersion": 0,
                    "commitment": "confirmed",
                },
            ],
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    if payload.get("error"):
        errors.append({
            "signature": signature,
            "error": str(payload["error"]),
        })
        continue

    tx = payload.get("result")

    if tx is None:
        errors.append({
            "signature": signature,
            "error": "Transaction unavailable",
        })
        continue

    meta = tx.get("meta") or {}
    message = tx.get("transaction", {}).get("message", {})
    account_keys = message.get("accountKeys", [])

    def account_address(index):
        """Return a public key for a transaction account index."""
        item = account_keys[index]

        if isinstance(item, dict):
            return item.get("pubkey", str(item))

        return str(item)

    # Map pre/post balances by token account index and mint.
    balance_maps = {}

    for side, field in [
        ("pre", "preTokenBalances"),
        ("post", "postTokenBalances"),
    ]:
        mapping = {}

        for entry in meta.get(field) or []:
            mint = entry.get("mint")

            if mint not in (target_mint, WSOL_MINT):
                continue

            amount_info = entry.get("uiTokenAmount") or {}

            try:
                raw_amount = int(amount_info["amount"])
            except (KeyError, TypeError, ValueError):
                continue

            key = (int(entry["accountIndex"]), mint)

            mapping[key] = {
                "amount": raw_amount,
                "owner": entry.get("owner"),
                "decimals": amount_info.get("decimals"),
            }

        balance_maps[side] = mapping

    pre = balance_maps["pre"]
    post = balance_maps["post"]
    all_keys = set(pre) | set(post)

    for account_index, mint in sorted(all_keys):
        before = pre.get((account_index, mint), {})
        after = post.get((account_index, mint), {})

        pre_amount = before.get("amount", 0)
        post_amount = after.get("amount", 0)
        delta = post_amount - pre_amount

        # Ignore unchanged token accounts.
        if delta == 0:
            continue

        balance_rows.append({
            "signature": signature,
            "action": sample["action"],
            "minute": sample["minute"],
            "mint_type": "WSOL" if mint == WSOL_MINT else "TOKEN",
            "token_account": account_address(account_index),
            "token_owner": after.get("owner") or before.get("owner"),
            "pre_amount_raw": pre_amount,
            "post_amount_raw": post_amount,
            "delta_raw": delta,
            "decimals": (
                after.get("decimals") or before.get("decimals")
            ),
        })

    # Preserve the relevant decoded event fields.
    event = fees_df.loc[
        fees_df["signature"].eq(signature)
        & fees_df["status"].eq("Decoded")
    ]

    if len(event):
        e = event.iloc[0]

        event_rows.append({
            "signature": signature,
            "action": sample["action"],
            "minute": sample["minute"],
            "event_name": e.get("event_name"),
            "ix_name": e.get("ix_name"),
            "base_amount_in": e.get("base_amount_in"),
            "base_amount_out": e.get("base_amount_out"),
            "quote_amount_in": e.get("quote_amount_in"),
            "quote_amount_out": e.get("quote_amount_out"),
            "user_quote_amount_in": e.get("user_quote_amount_in"),
            "user_quote_amount_out": e.get("user_quote_amount_out"),
            "lp_fee": e.get("lp_fee"),
            "protocol_fee": e.get("protocol_fee"),
            "coin_creator_fee": e.get("coin_creator_fee"),
        })

    time.sleep(0.2)

balances = pd.DataFrame(balance_rows)
events = pd.DataFrame(event_rows)

print("=== RPC retrieval ===")
print("Transactions inspected:", len(sample_transactions))
print("Transactions with decoded event:", len(events))
print("Balance changes recovered:", len(balances))
print("RPC errors:", len(errors))
print()

if errors:
    print("=== Retrieval errors ===")
    print(pd.DataFrame(errors).to_string(index=False))
    print()

print("=== Decoded event amounts (raw token units) ===")
if events.empty:
    print("No decoded event rows recovered.")
else:
    print(
        events.drop(
            columns=["signature"],
            errors="ignore",
        ).to_string(index=False)
    )
    print()

print("=== Actual token-account balance changes ===")
if balances.empty:
    print("No token-account balance changes recovered.")
else:
    print(
        balances[
            [
                "action",
                "minute",
                "mint_type",
                "token_account",
                "token_owner",
                "delta_raw",
                "decimals",
            ]
        ].to_string(index=False)
    )

In [ ]:
# Step 68: Compare native SOL balance changes with trade amounts

import time
import requests
import pandas as pd

if "sample_transactions" not in globals() or "fees_df" not in globals():
    raise ValueError("Run Steps 63 and 65 first.")

RPC_URL = globals().get(
    "RPC_URL",
    "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}",
)

results = []

for sample in sample_transactions:
    signature = sample["signature"]

    response = requests.post(
        RPC_URL,
        json={
            "jsonrpc": "2.0",
            "id": 1,
            "method": "getTransaction",
            "params": [
                signature,
                {
                    "encoding": "jsonParsed",
                    "maxSupportedTransactionVersion": 0,
                    "commitment": "confirmed",
                },
            ],
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    tx = payload.get("result")

    if tx is None:
        results.append({
            "signature": signature,
            "action": sample["action"],
            "status": "Transaction unavailable",
        })
        continue

    meta = tx.get("meta") or {}
    message = tx.get("transaction", {}).get("message", {})
    keys = message.get("accountKeys", [])

    if not keys or not meta.get("preBalances") or not meta.get("postBalances"):
        results.append({
            "signature": signature,
            "action": sample["action"],
            "status": "Balance information unavailable",
        })
        continue

    fee_payer = (
        keys[0].get("pubkey")
        if isinstance(keys[0], dict)
        else str(keys[0])
    )

    # The fee payer is account index 0 in the message.
    pre = int(meta["preBalances"][0])
    post = int(meta["postBalances"][0])
    tx_fee = int(meta.get("fee", 0))

    # Native SOL change after adding back the network transaction fee.
    # Rent, tips and other SOL transfers can still affect this number.
    delta_excluding_network_fee = post - pre + tx_fee

    event = fees_df.loc[
        fees_df["signature"].eq(signature)
        & fees_df["status"].eq("Decoded")
    ]

    if event.empty:
        results.append({
            "signature": signature,
            "action": sample["action"],
            "fee_payer": fee_payer,
            "status": "Decoded event unavailable",
        })
        continue

    e = event.iloc[0]
    action = sample["action"]

    if action == "buy":
        trade_amount_lamports = int(e["quote_amount_in"])
        expected_trade_delta = -trade_amount_lamports
    else:
        trade_amount_lamports = int(e["user_quote_amount_out"])
        expected_trade_delta = trade_amount_lamports

    residual = (
        delta_excluding_network_fee - expected_trade_delta
    )

    results.append({
        "signature": signature,
        "action": action,
        "minute": sample["minute"],
        "fee_payer": fee_payer,
        "transaction_fee_SOL": tx_fee / 1_000_000_000,
        "native_SOL_delta": (post - pre) / 1_000_000_000,
        "delta_ex_network_fee_SOL": (
            delta_excluding_network_fee / 1_000_000_000
        ),
        "expected_trade_delta_SOL": (
            expected_trade_delta / 1_000_000_000
        ),
        "unexplained_residual_SOL": (
            residual / 1_000_000_000
        ),
        "status": "OK",
    })

    time.sleep(0.2)

native_df = pd.DataFrame(results)

print("=== Native SOL balance reconciliation ===")
print("Transactions inspected:", len(sample_transactions))
print(
    native_df["status"].value_counts(dropna=False).to_string()
)
print()

columns = [
    "action",
    "minute",
    "transaction_fee_SOL",
    "native_SOL_delta",
    "delta_ex_network_fee_SOL",
    "expected_trade_delta_SOL",
    "unexplained_residual_SOL",
]

columns = [c for c in columns if c in native_df.columns]

print(
    native_df[columns]
    .round(9)
    .to_string(index=False)
)

In [ ]:
# Step 69: Determine whether archived reserves are
# pre-trade or post-trade values.

import time
import numpy as np
import pandas as pd
import requests

required_vars = [
    "sample_transactions",
    "archive_sample",
    "target_pool",
    "target_mint",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise ValueError(
        f"Missing variables: {missing}. "
        "Run Steps 63, 64 and 68 first."
    )

RPC_URL = globals().get(
    "RPC_URL",
    "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}",
)

WSOL_MINT = "So11111111111111111111111111111111111111112"

required_cols = {
    "signature",
    "quoteInPool",
    "tokensInPool",
}
missing_cols = required_cols.difference(archive_sample.columns)

if missing_cols:
    raise ValueError(
        f"Missing archive columns: {sorted(missing_cols)}"
    )

rows = []

# Look up each sampled transaction.
for sample in sample_transactions:
    signature = sample["signature"]

    response = requests.post(
        RPC_URL,
        json={
            "jsonrpc": "2.0",
            "id": 1,
            "method": "getTransaction",
            "params": [
                signature,
                {
                    "encoding": "jsonParsed",
                    "maxSupportedTransactionVersion": 0,
                    "commitment": "confirmed",
                },
            ],
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    tx = payload.get("result")

    if tx is None:
        rows.append({
            "action": sample["action"],
            "minute": sample["minute"],
            "status": "Transaction unavailable",
        })
        time.sleep(0.2)
        continue

    meta = tx.get("meta") or {}

    # Each token balance records its mint, amount and owner.
    pre_entries = meta.get("preTokenBalances") or []
    post_entries = meta.get("postTokenBalances") or []
    all_entries = pre_entries + post_entries

    decimals_by_mint = {}
    for entry in all_entries:
        mint = entry.get("mint")
        amount_info = entry.get("uiTokenAmount") or {}

        if mint and amount_info.get("decimals") is not None:
            decimals_by_mint[mint] = int(
                amount_info["decimals"]
            )

    def pool_balance(entries, mint):
        """Get the pool-owned balance for one mint."""
        total_raw = 0
        found = False

        for entry in entries:
            if (
                entry.get("mint") == mint
                and entry.get("owner") == target_pool
            ):
                info = entry.get("uiTokenAmount") or {}
                total_raw += int(info.get("amount", 0))
                found = True

        decimals = decimals_by_mint.get(mint)

        if decimals is None:
            return np.nan if not found else float(total_raw)

        return total_raw / (10 ** decimals)

    pre_quote = pool_balance(pre_entries, WSOL_MINT)
    post_quote = pool_balance(post_entries, WSOL_MINT)

    pre_token = pool_balance(pre_entries, target_mint)
    post_token = pool_balance(post_entries, target_mint)

    archived = archive_sample.loc[
        archive_sample["signature"].eq(signature)
    ]

    if archived.empty:
        rows.append({
            "action": sample["action"],
            "minute": sample["minute"],
            "status": "Archive record unavailable",
        })
        time.sleep(0.2)
        continue

    a = archived.iloc[0]
    archive_quote = float(a["quoteInPool"])
    archive_token = float(a["tokensInPool"])

    def relative_error_pct(reference, candidate):
        if (
            pd.isna(reference)
            or pd.isna(candidate)
            or reference == 0
        ):
            return np.nan

        return abs(candidate - reference) / abs(reference) * 100

    rows.append({
        "action": sample["action"],
        "minute": sample["minute"],
        "archive_quote_SOL": archive_quote,
        "pre_quote_SOL": pre_quote,
        "post_quote_SOL": post_quote,
        "quote_pre_error_pct": relative_error_pct(
            archive_quote, pre_quote
        ),
        "quote_post_error_pct": relative_error_pct(
            archive_quote, post_quote
        ),
        "archive_token_reserve": archive_token,
        "pre_token_reserve": pre_token,
        "post_token_reserve": post_token,
        "token_pre_error_pct": relative_error_pct(
            archive_token, pre_token
        ),
        "token_post_error_pct": relative_error_pct(
            archive_token, post_token
        ),
        "status": "OK",
    })

    time.sleep(0.2)

reserve_check = pd.DataFrame(rows)

print("=== Reserve-state comparison ===")
print("Transactions checked:", len(reserve_check))
print(reserve_check["status"].value_counts().to_string())
print()

ok = reserve_check.loc[
    reserve_check["status"].eq("OK")
].copy()

if ok.empty:
    print("No successful reserve comparisons.")
else:
    columns = [
        "action",
        "minute",
        "archive_quote_SOL",
        "pre_quote_SOL",
        "post_quote_SOL",
        "quote_pre_error_pct",
        "quote_post_error_pct",
        "token_pre_error_pct",
        "token_post_error_pct",
    ]

    print(
        ok[columns]
        .round(7)
        .to_string(index=False)
    )
    print()

    print("=== Median absolute relative error ===")
    print(
        ok[
            [
                "quote_pre_error_pct",
                "quote_post_error_pct",
                "token_pre_error_pct",
                "token_post_error_pct",
            ]
        ].median().round(7).to_string()
    )


In [ ]:
# Step 70 (corrected): Audit archive prices against reserve-derived prices.

import numpy as np
import pandas as pd

if "df66" not in globals():
    raise ValueError("Run Step 66 first.")

# Step 64 names the archive's price column archive_price.
required = {
    "signature",
    "action",
    "archive_price",
    "quoteInPool",
    "tokensInPool",
    "virtual_quote_reserves",
    "fee_free_curve_price",
    "actual_trader_price",
}

missing = required.difference(df66.columns)
if missing:
    raise ValueError(
        f"Missing columns in df66: {sorted(missing)}"
    )

audit70 = df66.copy()

numeric_cols = [
    "archive_price",
    "quoteInPool",
    "tokensInPool",
    "virtual_quote_reserves",
    "fee_free_curve_price",
    "actual_trader_price",
]

for col in numeric_cols:
    audit70[col] = pd.to_numeric(
        audit70[col], errors="coerce"
    )

# Raw reserve price: real quote reserves / real token reserves.
audit70["raw_reserve_price"] = (
    audit70["quoteInPool"] / audit70["tokensInPool"]
)

# Effective reserve price: include the signed virtual quote reserve.
audit70["effective_reserve_price"] = (
    audit70["quoteInPool"]
    + audit70["virtual_quote_reserves"] / 1_000_000_000
) / audit70["tokensInPool"]

# Compare the archive's recorded price with both reserve prices.
audit70["archive_vs_raw_pct"] = (
    audit70["archive_price"]
    / audit70["raw_reserve_price"] - 1
) * 100

audit70["archive_vs_effective_pct"] = (
    audit70["archive_price"]
    / audit70["effective_reserve_price"] - 1
) * 100

audit70["effective_uplift_vs_raw_pct"] = (
    audit70["effective_reserve_price"]
    / audit70["raw_reserve_price"] - 1
) * 100

audit70["curve_vs_effective_pct"] = (
    audit70["fee_free_curve_price"]
    / audit70["effective_reserve_price"] - 1
) * 100

audit70["trader_vs_effective_pct"] = (
    audit70["actual_trader_price"]
    / audit70["effective_reserve_price"] - 1
) * 100

audit70["signature_short"] = (
    audit70["signature"].astype(str).str[:8]
)

print("=== Per-transaction price audit ===")

columns = [
    "signature_short",
    "action",
    "archive_price",
    "raw_reserve_price",
    "effective_reserve_price",
    "archive_vs_raw_pct",
    "archive_vs_effective_pct",
    "curve_vs_effective_pct",
    "trader_vs_effective_pct",
]

print(
    audit70[columns].round(8).to_string(index=False)
)

print("\n=== Median percentage differences by direction ===")

summary70 = audit70.groupby("action").agg(
    transactions=("signature", "count"),
    archive_vs_raw_pct=("archive_vs_raw_pct", "median"),
    archive_vs_effective_pct=(
        "archive_vs_effective_pct", "median"
    ),
    effective_uplift_vs_raw_pct=(
        "effective_uplift_vs_raw_pct", "median"
    ),
    curve_vs_effective_pct=(
        "curve_vs_effective_pct", "median"
    ),
    trader_vs_effective_pct=(
        "trader_vs_effective_pct", "median"
    ),
)

print(summary70.round(6).to_string())


In [ ]:
# Step 71: Separate AMM price impact from trader fees.

import numpy as np
import pandas as pd

if "df66" not in globals():
    raise ValueError("Run Steps 64–66 first.")

required = [
    "signature",
    "action",
    "archive_price",
    "quoteInPool",
    "tokensInPool",
    "virtual_quote_reserves",
    "tokenAmount",
    "quote_amount_in",
    "user_quote_amount_in",
    "quote_amount_out",
    "user_quote_amount_out",
    "lp_fee_basis_points",
    "protocol_fee_basis_points",
    "coin_creator_fee_basis_points",
]

missing = [c for c in required if c not in df66.columns]
if missing:
    raise ValueError(f"Missing columns in df66: {missing}")

a71 = df66.copy()

# Convert source columns to numeric values.
numeric_cols = [c for c in required if c not in ("signature", "action")]
for col in numeric_cols:
    a71[col] = pd.to_numeric(a71[col], errors="coerce")

is_buy = a71["action"].eq("buy")
scale = 1_000_000_000  # quote amounts and virtual reserves use lamports

# 1. Starting spot price, before the transaction.
a71["raw_spot_price"] = (
    a71["quoteInPool"] / a71["tokensInPool"]
)

a71["effective_quote_reserve_SOL"] = (
    a71["quoteInPool"]
    + a71["virtual_quote_reserves"] / scale
)

a71["effective_spot_price"] = (
    a71["effective_quote_reserve_SOL"] / a71["tokensInPool"]
)

# 2. AMM-side quote amount, excluding the trader-facing fee.
# Buy: user_quote_amount_in is the curve-side amount.
# Sell: quote_amount_out is the gross amount removed from the pool.
a71["amm_quote_SOL"] = np.where(
    is_buy,
    a71["user_quote_amount_in"] / scale,
    a71["quote_amount_out"] / scale,
)

a71["amm_execution_price"] = (
    a71["amm_quote_SOL"] / a71["tokenAmount"]
)

# 3. Trader-side quote amount.
# Buy: quote_amount_in is the buyer's total spend.
# Sell: user_quote_amount_out is the seller's net proceeds.
a71["trader_quote_SOL"] = np.where(
    is_buy,
    a71["quote_amount_in"] / scale,
    a71["user_quote_amount_out"] / scale,
)

a71["trader_execution_price"] = (
    a71["trader_quote_SOL"] / a71["tokenAmount"]
)

# 4. Decompose the differences.
a71["archive_vs_raw_pct"] = (
    a71["archive_price"] / a71["raw_spot_price"] - 1
) * 100

a71["amm_vs_spot_pct"] = (
    a71["amm_execution_price"] / a71["effective_spot_price"] - 1
) * 100

a71["trader_vs_amm_pct"] = (
    a71["trader_execution_price"] / a71["amm_execution_price"] - 1
) * 100

a71["trader_vs_spot_pct"] = (
    a71["trader_execution_price"] / a71["effective_spot_price"] - 1
) * 100

# 5. Compare implied fees with fee rates reported on-chain.
fee_fields = [
    "lp_fee_basis_points",
    "protocol_fee_basis_points",
    "coin_creator_fee_basis_points",
]

a71["reported_fee_bps"] = a71[fee_fields].sum(axis=1)

a71["implied_fee_bps_abs"] = np.where(
    is_buy,
    (
        a71["quote_amount_in"]
        / a71["user_quote_amount_in"] - 1
    ) * 10_000,
    (
        1
        - a71["user_quote_amount_out"]
        / a71["quote_amount_out"]
    ) * 10_000,
)

a71["implied_fee_bps_error"] = (
    a71["implied_fee_bps_abs"] - a71["reported_fee_bps"]
)

# 6. Display each transaction.
a71["signature_short"] = a71["signature"].astype(str).str[:8]

columns = [
    "signature_short",
    "action",
    "archive_vs_raw_pct",
    "amm_vs_spot_pct",
    "trader_vs_amm_pct",
    "trader_vs_spot_pct",
    "reported_fee_bps",
    "implied_fee_bps_abs",
    "implied_fee_bps_error",
]

print("=== Transaction-level price and fee decomposition ===")
print(
    a71[columns].round(6).to_string(index=False)
)

print("\n=== Median results by trade direction ===")
summary71 = a71.groupby("action").agg(
    transactions=("signature", "size"),
    median_archive_vs_raw_pct=("archive_vs_raw_pct", "median"),
    median_amm_vs_spot_pct=("amm_vs_spot_pct", "median"),
    median_trader_vs_amm_pct=("trader_vs_amm_pct", "median"),
    median_trader_vs_spot_pct=("trader_vs_spot_pct", "median"),
    median_reported_fee_bps=("reported_fee_bps", "median"),
    median_implied_fee_bps=("implied_fee_bps_abs", "median"),
    max_abs_fee_bps_error=(
        "implied_fee_bps_error",
        lambda x: x.abs().max(),
    ),
)

print(summary71.round(6).to_string())

print("\n=== Largest absolute AMM price impact ===")
largest = a71.loc[a71["amm_vs_spot_pct"].abs().idxmax()]

print("Signature:", largest["signature"])
print("Action:", largest["action"])
print("AMM execution vs spot (%):", round(
    largest["amm_vs_spot_pct"], 6
))
print("Trader execution vs spot (%):", round(
    largest["trader_vs_spot_pct"], 6
))
print("Reported fees (bps):", round(
    largest["reported_fee_bps"], 6
))


In [ ]:
# Step 72: Test constant-product output against actual swaps.

import numpy as np
import pandas as pd

if "df66" not in globals():
    raise ValueError("Run Step 66 first.")

required = [
    "signature",
    "action",
    "quoteInPool",
    "tokensInPool",
    "virtual_quote_reserves",
    "tokenAmount",
    "user_quote_amount_in",
    "quote_amount_out",
]

missing = [c for c in required if c not in df66.columns]

if missing:
    raise ValueError(f"Missing columns in df66: {missing}")

if "lp_fee" not in df66.columns:
    raise ValueError(
        "df66 is missing lp_fee. Check the Step 65 fee merge."
    )

a72 = df66.copy()

numeric_cols = [
    "quoteInPool",
    "tokensInPool",
    "virtual_quote_reserves",
    "tokenAmount",
    "user_quote_amount_in",
    "quote_amount_out",
    "lp_fee",
]

for col in numeric_cols:
    a72[col] = pd.to_numeric(a72[col], errors="coerce")

LAMPORTS_PER_SOL = 1_000_000_000

# Pre-trade effective reserves.
# virtual_quote_reserves is signed and expressed in lamports.
a72["quote_reserve_effective_SOL"] = (
    a72["quoteInPool"]
    + a72["virtual_quote_reserves"] / LAMPORTS_PER_SOL
)

a72["base_reserve_tokens"] = a72["tokensInPool"]

valid_reserves = (
    a72["quote_reserve_effective_SOL"].gt(0)
    & a72["base_reserve_tokens"].gt(0)
)

a72 = a72.loc[valid_reserves].copy()

is_buy = a72["action"].eq("buy")
is_sell = a72["action"].eq("sell")

# --- BUY: predict token output from quote input ---

Q = a72["quote_reserve_effective_SOL"]
B = a72["base_reserve_tokens"]

buy_input_user_SOL = (
    a72["user_quote_amount_in"] / LAMPORTS_PER_SOL
)

buy_input_user_plus_lp_SOL = (
    a72["user_quote_amount_in"] + a72["lp_fee"]
) / LAMPORTS_PER_SOL

# Constant-product formula:
# token_out = B * quote_in / (Q + quote_in)
a72["pred_buy_tokens_user_only"] = (
    B * buy_input_user_SOL
    / (Q + buy_input_user_SOL)
)

a72["pred_buy_tokens_user_plus_lp"] = (
    B * buy_input_user_plus_lp_SOL
    / (Q + buy_input_user_plus_lp_SOL)
)

a72["observed_buy_tokens"] = a72["tokenAmount"]

a72["buy_error_user_only_pct"] = (
    a72["pred_buy_tokens_user_only"]
    / a72["observed_buy_tokens"] - 1
) * 100

a72["buy_error_user_plus_lp_pct"] = (
    a72["pred_buy_tokens_user_plus_lp"]
    / a72["observed_buy_tokens"] - 1
) * 100

# --- SELL: predict gross quote output from token input ---

# quote_out = Q * token_in / (B + token_in)
sell_token_input = a72["tokenAmount"]

a72["pred_sell_quote_SOL"] = (
    Q * sell_token_input
    / (B + sell_token_input)
)

a72["observed_sell_quote_SOL"] = (
    a72["quote_amount_out"] / LAMPORTS_PER_SOL
)

a72["sell_error_pct"] = (
    a72["pred_sell_quote_SOL"]
    / a72["observed_sell_quote_SOL"] - 1
) * 100

# Print each transaction's prediction error.
a72["signature_short"] = (
    a72["signature"].astype(str).str[:8]
)

columns = [
    "signature_short",
    "action",
    "buy_error_user_only_pct",
    "buy_error_user_plus_lp_pct",
    "sell_error_pct",
]

print("=== Per-transaction formula errors (%) ===")
print(
    a72[columns]
    .round(7)
    .to_string(index=False)
)

# Summarize the absolute prediction errors separately by action.
print("\n=== Absolute prediction error summary (%) ===")

buy_rows = a72.loc[is_buy.reindex(a72.index, fill_value=False)]
sell_rows = a72.loc[is_sell.reindex(a72.index, fill_value=False)]

def error_summary(series):
    return pd.Series({
        "n": int(series.notna().sum()),
        "median_abs_error_pct": series.abs().median(),
        "max_abs_error_pct": series.abs().max(),
    })

summary_rows = {}

if not buy_rows.empty:
    summary_rows["buy_user_only"] = error_summary(
        buy_rows["buy_error_user_only_pct"]
    )
    summary_rows["buy_user_plus_lp"] = error_summary(
        buy_rows["buy_error_user_plus_lp_pct"]
    )

if not sell_rows.empty:
    summary_rows["sell_gross_quote_out"] = error_summary(
        sell_rows["sell_error_pct"]
    )

print(
    pd.DataFrame(summary_rows).T.round(7).to_string()
)


In [ ]:
# Step 73: Broader validation of the PumpSwap AMM formula.

import base64
import io
import json
import random
import time

import numpy as np
import pandas as pd
import requests
import zstandard as zstd

required_globals = [
    "validation_url",
    "target_mint",
    "target_pool",
    "event_fields",
    "event_discriminators",
    "decode_all_fields",
]

missing = [v for v in required_globals if v not in globals()]
if missing:
    raise ValueError(
        f"Missing variables: {missing}. Run Steps 63–65 first."
    )

RPC_URL = globals().get(
    "RPC_URL",
    "https://mainnet.helius-rpc.com/?api-key=${HELIUS_API_KEY}",
)

LAMPORTS = 1_000_000_000
SAMPLE_PER_ACTION = 25
rng = random.Random(7301)

# ---------------------------------------------------------
# 1. Reservoir-sample transactions uniformly from the hour.
# ---------------------------------------------------------

reservoirs = {"buy": [], "sell": []}
seen_counts = {"buy": 0, "sell": 0}
seen_signatures = set()

with requests.get(
    validation_url,
    stream=True,
    timeout=(20, 120),
) as response:
    response.raise_for_status()

    with zstd.ZstdDecompressor().stream_reader(
        response.raw,
        read_size=1 << 20,
    ) as reader:
        stream = io.TextIOWrapper(reader, encoding="utf-8")

        try:
            for line in stream:
                if not line.strip():
                    continue

                event = json.loads(line)

                action = event.get("action")
                signature = event.get("signature")

                if action not in reservoirs or not signature:
                    continue

                if (
                    event.get("protocol") != "PUMPSWAP"
                    or event.get("mint") != target_mint
                    or event.get("pool") != target_pool
                    or len(event.get("breakdown") or []) != 1
                ):
                    continue

                if signature in seen_signatures:
                    continue

                seen_signatures.add(signature)
                seen_counts[action] += 1

                row = {
                    "signature": signature,
                    "action": action,
                    "archive_price": event.get("price"),
                    "tokenAmount": event.get("tokenAmount"),
                    "quoteInPool": event.get("quoteInPool"),
                    "tokensInPool": event.get("tokensInPool"),
                    "localTimestamp": event.get("localTimestamp"),
                    "archive_ix_name": (
                        (event.get("breakdown") or [{}])[0].get(
                            "ix_name"
                        )
                    ),
                }

                bucket = reservoirs[action]
                n = seen_counts[action]

                if len(bucket) < SAMPLE_PER_ACTION:
                    bucket.append(row)
                else:
                    j = rng.randrange(n)
                    if j < SAMPLE_PER_ACTION:
                        bucket[j] = row
        finally:
            stream.detach()

sample73 = reservoirs["buy"] + reservoirs["sell"]

print("=== Sample construction ===")
print("Eligible buys in archive:", seen_counts["buy"])
print("Eligible sells in archive:", seen_counts["sell"])
print("Buys sampled:", len(reservoirs["buy"]))
print("Sells sampled:", len(reservoirs["sell"]))
print()

if not sample73:
    raise ValueError("No eligible transactions found for this pool.")

# Get token decimals if Step 64 did not leave them available.
if "token_decimals" not in globals():
    supply_response = requests.post(
        RPC_URL,
        json={
            "jsonrpc": "2.0",
            "id": 1,
            "method": "getTokenSupply",
            "params": [target_mint],
        },
        timeout=30,
    )
    supply_response.raise_for_status()
    token_decimals = int(
        supply_response.json()["result"]["value"]["decimals"]
    )

token_decimals = int(token_decimals)
token_scale = 10 ** token_decimals

# ---------------------------------------------------------
# 2. Decode each sampled transaction's on-chain event.
# ---------------------------------------------------------

records = []
session = requests.Session()

for i, sample in enumerate(sample73, start=1):
    signature = sample["signature"]

    try:
        response = session.post(
            RPC_URL,
            json={
                "jsonrpc": "2.0",
                "id": 1,
                "method": "getTransaction",
                "params": [
                    signature,
                    {
                        "encoding": "json",
                        "maxSupportedTransactionVersion": 0,
                        "commitment": "confirmed",
                    },
                ],
            },
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

    except Exception as exc:
        records.append({
            **sample,
            "status": f"RPC error: {type(exc).__name__}",
        })
        time.sleep(0.15)
        continue

    tx = payload.get("result")

    if not tx:
        records.append({
            **sample,
            "status": "Transaction unavailable",
        })
        time.sleep(0.15)
        continue

    meta = tx.get("meta") or {}

    if meta.get("err") is not None:
        records.append({
            **sample,
            "status": "Transaction failed",
        })
        time.sleep(0.15)
        continue

    expected_event = (
        "BuyEvent" if sample["action"] == "buy"
        else "SellEvent"
    )

    decoded = None

    for log in meta.get("logMessages") or []:
        prefix = "Program data: "

        if not log.startswith(prefix):
            continue

        try:
            raw = base64.b64decode(
                log[len(prefix):].strip(),
                validate=True,
            )
        except Exception:
            continue

        if not raw.startswith(event_discriminators[expected_event]):
            continue

        try:
            candidate = decode_all_fields(
                raw,
                event_fields[expected_event],
            )
        except (ValueError, UnicodeDecodeError):
            continue

        if candidate.get("pool") == target_pool:
            decoded = candidate
            break

    if decoded is None:
        records.append({
            **sample,
            "status": "No matching pool event",
        })
        time.sleep(0.15)
        continue

    try:
        quote_pool = float(sample["quoteInPool"])
        token_pool = float(sample["tokensInPool"])
        archived_price = float(sample["archive_price"])
        archived_token_amount = float(sample["tokenAmount"])

        virtual_quote_sol = (
            float(decoded["virtual_quote_reserves"]) / LAMPORTS
        )

        effective_quote_pool = quote_pool + virtual_quote_sol

        if (
            quote_pool <= 0
            or token_pool <= 0
            or effective_quote_pool <= 0
        ):
            raise ValueError("Invalid pool reserves")

        # Check that the recorded archive price matches
        # the raw reserve ratio.
        raw_spot_price = quote_pool / token_pool
        archive_vs_raw_pct = (
            archived_price / raw_spot_price - 1
        ) * 100

        if sample["action"] == "buy":
            # Buy input to the curve excludes trade fees.
            quote_input_sol = (
                float(decoded["user_quote_amount_in"]) / LAMPORTS
            )

            observed_tokens = (
                float(decoded["base_amount_out"]) / token_scale
            )

            predicted_tokens = (
                token_pool * quote_input_sol
                / (effective_quote_pool + quote_input_sol)
            )

            formula_error_pct = (
                predicted_tokens / observed_tokens - 1
            ) * 100

            event_token_amount = observed_tokens

        else:
            token_input = (
                float(decoded["base_amount_in"]) / token_scale
            )

            observed_quote_sol = (
                float(decoded["quote_amount_out"]) / LAMPORTS
            )

            predicted_quote_sol = (
                effective_quote_pool * token_input
                / (token_pool + token_input)
            )

            formula_error_pct = (
                predicted_quote_sol / observed_quote_sol - 1
            ) * 100

            event_token_amount = token_input

        archive_token_vs_event_pct = (
            archived_token_amount / event_token_amount - 1
        ) * 100

        records.append({
            **sample,
            "status": "OK",
            "ix_name": decoded.get("ix_name"),
            "archive_vs_raw_pct": archive_vs_raw_pct,
            "archive_token_vs_event_pct": (
                archive_token_vs_event_pct
            ),
            "formula_error_pct": formula_error_pct,
        })

    except (KeyError, TypeError, ValueError, ZeroDivisionError) as exc:
        records.append({
            **sample,
            "status": f"Calculation error: {type(exc).__name__}",
            "ix_name": decoded.get("ix_name"),
        })

    time.sleep(0.15)

audit73 = pd.DataFrame(records)

# ---------------------------------------------------------
# 3. Review results and separate instruction variants.
# ---------------------------------------------------------

print("=== On-chain decoding status ===")
print(audit73["status"].value_counts().to_string())
print()

valid73 = audit73.loc[
    audit73["status"].eq("OK")
].copy()

if valid73.empty:
    print("No successful formula comparisons. Inspect decoding statuses.")
else:
    valid73["signature_short"] = (
        valid73["signature"].astype(str).str[:8]
    )

    detail_cols = [
        "signature_short",
        "action",
        "ix_name",
        "formula_error_pct",
        "archive_vs_raw_pct",
        "archive_token_vs_event_pct",
    ]

    print("=== Individual formula errors (%) ===")
    print(
        valid73[detail_cols]
        .round(7)
        .to_string(index=False)
    )
    print()

    print("=== Summary by trade direction and instruction ===")
    summary73 = (
        valid73.groupby(
            ["action", "ix_name"],
            dropna=False,
        )
        .agg(
            transactions=("signature", "size"),
            median_abs_formula_error_pct=(
                "formula_error_pct",
                lambda s: s.abs().median(),
            ),
            max_abs_formula_error_pct=(
                "formula_error_pct",
                lambda s: s.abs().max(),
            ),
            median_abs_archive_price_error_pct=(
                "archive_vs_raw_pct",
                lambda s: s.abs().median(),
            ),
            max_abs_archive_price_error_pct=(
                "archive_vs_raw_pct",
                lambda s: s.abs().max(),
            ),
            max_abs_archive_token_error_pct=(
                "archive_token_vs_event_pct",
                lambda s: s.abs().max(),
            ),
        )
    )

    print(summary73.round(7).to_string())

    print("\nSuccessful comparisons:", len(valid73))
    print("Requested sample:", len(sample73))


In [ ]:
# Step 74: Validate PumpSwap fee accounting across the 50 transactions.
# Corrected: BuyEvent and SellEvent have different amount fields.

import base64
import time
import requests
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Obtain the RPC key securely from Colab Secrets.
# ---------------------------------------------------------

try:
    from google.colab import userdata
    HELIUS_API_KEY = userdata.get("HELIUS_API_KEY")
except Exception as exc:
    raise ValueError(
        "Set HELIUS_API_KEY in Colab Secrets and grant notebook access."
    ) from exc

if not HELIUS_API_KEY:
    raise ValueError(
        "HELIUS_API_KEY is empty. Configure it in Colab Secrets."
    )

RPC_URL = (
    "https://mainnet.helius-rpc.com/?api-key="
    + HELIUS_API_KEY
)

# ---------------------------------------------------------
# 2. Check that the required data and decoder exist.
# ---------------------------------------------------------

required_globals = [
    "audit73",
    "event_fields",
    "event_discriminators",
    "decode_all_fields",
    "target_pool",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise ValueError(
        f"Missing notebook variables: {missing_globals}. "
        "Run Steps 63–65 and Step 73 first."
    )

sample74 = (
    audit73.loc[
        audit73["status"].eq("OK"),
        ["signature", "action"],
    ]
    .drop_duplicates("signature")
    .reset_index(drop=True)
)

if sample74.empty:
    raise ValueError("No successful transactions are available in audit73.")

# Common fee fields exist in both event schemas.
common_fee_fields = [
    "pool",
    "lp_fee",
    "protocol_fee",
    "coin_creator_fee",
    "lp_fee_basis_points",
    "protocol_fee_basis_points",
    "coin_creator_fee_basis_points",
]

# IMPORTANT: quote input fields belong to BuyEvent;
# quote output fields belong to SellEvent.
amount_fields_by_event = {
    "BuyEvent": [
        "quote_amount_in",
        "user_quote_amount_in",
    ],
    "SellEvent": [
        "quote_amount_out",
        "user_quote_amount_out",
    ],
}

for event_name, amount_fields in amount_fields_by_event.items():
    if event_name not in event_fields:
        raise ValueError(f"{event_name} is absent from the IDL.")

    available_fields = {
        field["name"] for field in event_fields[event_name]
    }

    expected_fields = set(common_fee_fields + amount_fields)
    missing_fields = expected_fields - available_fields

    if missing_fields:
        raise ValueError(
            f"{event_name} is missing required fields: "
            f"{sorted(missing_fields)}"
        )

    if event_name not in event_discriminators:
        raise ValueError(
            f"Missing discriminator for {event_name}."
        )

print("=== Schema validation ===")
print("BuyEvent: buy-specific amounts and common fees verified.")
print("SellEvent: sell-specific amounts and common fees verified.")
print("Transactions selected:", len(sample74))
print()

# ---------------------------------------------------------
# 3. Fetch transactions, with retries for transient errors.
# ---------------------------------------------------------

session74 = requests.Session()
rows74 = []

def fetch_tx74(signature, attempts=4):
    """Fetch a transaction without printing or exposing the API key."""

    last_error = "Unknown RPC error"

    for attempt in range(attempts):
        try:
            response = session74.post(
                RPC_URL,
                json={
                    "jsonrpc": "2.0",
                    "id": 1,
                    "method": "getTransaction",
                    "params": [
                        signature,
                        {
                            "encoding": "json",
                            "maxSupportedTransactionVersion": 0,
                            "commitment": "confirmed",
                        },
                    ],
                },
                timeout=30,
            )

            if (
                response.status_code == 429
                or response.status_code >= 500
            ):
                last_error = f"HTTP {response.status_code}"

                if attempt < attempts - 1:
                    retry_after = response.headers.get("Retry-After")

                    try:
                        delay = float(retry_after)
                    except (TypeError, ValueError):
                        delay = 2 ** (attempt + 1)

                    time.sleep(min(delay, 20))
                    continue

                return None, last_error

            if response.status_code >= 400:
                return None, f"HTTP {response.status_code}"

            payload = response.json()

            if payload.get("error"):
                error = payload["error"]
                message = str(error).lower()

                retryable = any(
                    phrase in message
                    for phrase in [
                        "rate limit",
                        "too many requests",
                        "temporarily unavailable",
                        "node is unhealthy",
                    ]
                )

                last_error = "JSON-RPC error"

                if retryable and attempt < attempts - 1:
                    time.sleep(2 ** (attempt + 1))
                    continue

                return None, last_error

            tx = payload.get("result")

            if tx is None:
                return None, "Transaction unavailable"

            return tx, None

        except (requests.RequestException, ValueError) as exc:
            last_error = type(exc).__name__

            if attempt < attempts - 1:
                time.sleep(2 ** (attempt + 1))
                continue

            return None, last_error

    return None, last_error


# ---------------------------------------------------------
# 4. Decode the correct trade event and reconcile amounts.
# ---------------------------------------------------------

optional_fee_fields = [
    "cashback",
    "buyback_fee",
    "holder_rewards",
]

for i, sample in sample74.iterrows():
    signature = sample["signature"]
    action = sample["action"]

    if action not in ("buy", "sell"):
        rows74.append({
            "signature": signature,
            "action": action,
            "status": "Unsupported action",
        })
        continue

    tx, error = fetch_tx74(signature)

    if tx is None:
        rows74.append({
            "signature": signature,
            "action": action,
            "status": error,
        })
        continue

    meta = tx.get("meta") or {}

    if meta.get("err") is not None:
        rows74.append({
            "signature": signature,
            "action": action,
            "status": "On-chain transaction failed",
        })
        continue

    expected_event = (
        "BuyEvent" if action == "buy" else "SellEvent"
    )
    expected_discriminator = event_discriminators[expected_event]

    decoded = None

    for log in meta.get("logMessages") or []:
        prefix = "Program data: "

        if not log.startswith(prefix):
            continue

        try:
            raw = base64.b64decode(
                log[len(prefix):].strip(),
                validate=True,
            )
        except Exception:
            continue

        if not raw.startswith(expected_discriminator):
            continue

        try:
            candidate = decode_all_fields(
                raw,
                event_fields[expected_event],
            )
        except (ValueError, UnicodeDecodeError):
            continue

        if candidate.get("pool") == target_pool:
            decoded = candidate
            break

    if decoded is None:
        rows74.append({
            "signature": signature,
            "action": action,
            "status": "Matching pool event not found",
        })
        time.sleep(0.25)
        continue

    # Correct field pair for each trade direction.
    gross_field = (
        "quote_amount_in" if action == "buy"
        else "quote_amount_out"
    )

    user_field = (
        "user_quote_amount_in" if action == "buy"
        else "user_quote_amount_out"
    )

    try:
        gross = int(decoded[gross_field])
        user_amount = int(decoded[user_field])

        lp_fee = int(decoded["lp_fee"])
        protocol_fee = int(decoded["protocol_fee"])
        creator_fee = int(decoded["coin_creator_fee"])

        if gross <= 0 or user_amount <= 0:
            raise ValueError("Non-positive quote amount")

        core_fees = lp_fee + protocol_fee + creator_fee

        # Check whether gross - user amount equals decoded fees.
        residual = gross - user_amount - core_fees

        reported_bps = (
            int(decoded["lp_fee_basis_points"])
            + int(decoded["protocol_fee_basis_points"])
            + int(decoded["coin_creator_fee_basis_points"])
        )

        if action == "buy":
            implied_bps = (
                gross / user_amount - 1
            ) * 10_000
        else:
            implied_bps = (
                1 - user_amount / gross
            ) * 10_000

        row = {
            "signature": signature,
            "action": action,
            "event_name": expected_event,
            "status": "OK",
            "gross_lamports": gross,
            "user_lamports": user_amount,
            "lp_fee_lamports": lp_fee,
            "protocol_fee_lamports": protocol_fee,
            "creator_fee_lamports": creator_fee,
            "core_fees_lamports": core_fees,
            "fee_residual_lamports": residual,
            "reported_fee_bps": reported_bps,
            "implied_fee_bps": implied_bps,
            "fee_rate_error_bps": implied_bps - reported_bps,
        }

        # Keep additional fee-like fields separate from core fees.
        for field in optional_fee_fields:
            row[f"{field}_amount"] = int(
                decoded.get(field, 0) or 0
            )

        rows74.append(row)

    except (KeyError, TypeError, ValueError, OverflowError) as exc:
        rows74.append({
            "signature": signature,
            "action": action,
            "status": f"Calculation error: {type(exc).__name__}",
        })

    time.sleep(0.25)


# ---------------------------------------------------------
# 5. Report results.
# ---------------------------------------------------------

fee_audit74 = pd.DataFrame(rows74)

print("=== Step 74: Request and decoding status ===")
print("Transactions selected:", len(sample74))
print(fee_audit74["status"].value_counts().to_string())

valid74 = fee_audit74.loc[
    fee_audit74["status"].eq("OK")
].copy()

if valid74.empty:
    print("\nNo transactions were successfully reconciled.")

else:
    # Exact amount reconciliation summary.
    summary74 = valid74.groupby("action").agg(
        transactions=("signature", "size"),
        exact_fee_matches=(
            "fee_residual_lamports",
            lambda s: int(s.eq(0).sum()),
        ),
        median_abs_residual_lamports=(
            "fee_residual_lamports",
            lambda s: s.abs().median(),
        ),
        max_abs_residual_lamports=(
            "fee_residual_lamports",
            lambda s: s.abs().max(),
        ),
        median_reported_fee_bps=("reported_fee_bps", "median"),
        median_implied_fee_bps=("implied_fee_bps", "median"),
        max_abs_fee_rate_error_bps=(
            "fee_rate_error_bps",
            lambda s: s.abs().max(),
        ),
    )

    print("\n=== Fee reconciliation by trade direction ===")
    print(summary74.round(6).to_string())

    # Separate diagnostic for optional fee-like amounts.
    print("\n=== Optional fee-field diagnostic ===")

    for field in optional_fee_fields:
        col = f"{field}_amount"

        if col not in valid74.columns:
            print(f"{field}: not available")
            continue

        amounts = pd.to_numeric(
            valid74[col], errors="coerce"
        ).fillna(0)

        print(
            f"{field}: "
            f"{int(amounts.ne(0).sum())} nonzero transactions; "
            f"total raw units = {int(amounts.sum())}"
        )

    # Show residuals using shortened signatures only.
    residuals74 = valid74.copy()
    residuals74["signature_short"] = (
        residuals74["signature"].astype(str).str[:8]
    )
    residuals74["abs_residual"] = (
        residuals74["fee_residual_lamports"].abs()
    )

    print("\n=== Largest absolute fee residuals ===")

    print(
        residuals74.nlargest(5, "abs_residual")[
            [
                "signature_short",
                "action",
                "fee_residual_lamports",
                "reported_fee_bps",
                "implied_fee_bps",
            ]
        ].to_string(index=False)
    )

    print("\nSuccessful fee reconciliations:", len(valid74))
    print(
        "Exact gross = user amount + core fees:",
        int(valid74["fee_residual_lamports"].eq(0).sum()),
        "of",
        len(valid74),
    )


In [ ]:
# Step 75: Validate buyback fee accounting and test round trips.
# Uses existing notebook data only; no RPC calls.

import numpy as np
import pandas as pd

LAMPORTS_PER_SOL = 1_000_000_000

# ---------------------------------------------------------
# 1. Validate prerequisites.
# ---------------------------------------------------------

for name in ["fee_audit74", "valid74", "df66", "a71", "token_decimals"]:
    if name not in globals():
        raise ValueError(
            f"Missing {name}. Run Steps 64–74 first."
        )

fee_data = valid74.loc[
    valid74["status"].eq("OK")
].copy()

if fee_data.empty:
    raise ValueError("No successfully reconciled fee records.")

rate_columns = {
    "lp": "lp_fee_basis_points",
    "protocol": "protocol_fee_basis_points",
    "creator": "coin_creator_fee_basis_points",
}

missing = [
    col for col in rate_columns.values()
    if col not in a71.columns
]

if missing:
    raise ValueError(
        f"Missing fee-rate columns in a71: {missing}"
    )

# ---------------------------------------------------------
# 2. Verify buyback_fee is not an additional core fee.
# ---------------------------------------------------------

print("=== Buyback fee accounting ===")

if "buyback_fee_amount" not in fee_data.columns:
    print("Buyback fee field is not available in valid74.")
else:
    buyback = pd.to_numeric(
        fee_data["buyback_fee_amount"], errors="coerce"
    )

    protocol = pd.to_numeric(
        fee_data["protocol_fee_lamports"], errors="coerce"
    )

    comparable = buyback.notna() & protocol.notna()

    subset_ok = buyback[comparable] <= protocol[comparable]

    print("Comparable transactions:", int(comparable.sum()))
    print(
        "Buyback fee <= protocol fee:",
        int(subset_ok.sum()),
        "of",
        int(comparable.sum()),
    )

    if (~subset_ok).any():
        print(
            "WARNING: Some buyback amounts exceed protocol fees. "
            "Inspect those event records before proceeding."
        )
    else:
        print(
            "No buyback fee is larger than the protocol fee. "
            "Do not add buyback_fee as a fourth fee component."
        )

print()

# ---------------------------------------------------------
# 3. Set a starting pool state using a recorded pre-trade state.
# ---------------------------------------------------------

seed_candidates = df66.loc[
    df66["action"].eq("buy")
].copy()

if seed_candidates.empty:
    raise ValueError("No buy event is available to seed the simulator.")

seed = seed_candidates.iloc[0]

token_scale = 10 ** int(token_decimals)

# The archive reserves are displayed in SOL and human token units.
# Convert them to raw units before doing integer AMM calculations.
q0 = (
    int(round(float(seed["quoteInPool"]) * LAMPORTS_PER_SOL))
    + int(round(float(seed["virtual_quote_reserves"]))
)
)

b0 = int(
    round(float(seed["tokensInPool"]) * token_scale)
)

if q0 <= 0 or b0 <= 0:
    raise ValueError(
        f"Invalid initial reserves: quote={q0}, base={b0}"
    )

print("=== Initial effective pool reserves ===")
print("Effective quote reserve (SOL):", q0 / LAMPORTS_PER_SOL)
print("Base token reserve:", b0 / token_scale)
print("Token decimals:", token_decimals)
print()

# ---------------------------------------------------------
# 4. Estimate a fixed fee schedule from the existing sample.
# These rates are used only for the synthetic sanity test.
# ---------------------------------------------------------

fee_rates = {}

for action in ["buy", "sell"]:
    action_rows = a71.loc[
        a71["action"].eq(action)
    ]

    if action_rows.empty:
        raise ValueError(f"No observed fee rates for {action}.")

    fee_rates[action] = {}

    for component, column in rate_columns.items():
        values = pd.to_numeric(
            action_rows[column], errors="coerce"
        ).dropna()

        if values.empty:
            fee_rates[action][component] = 0
        else:
            fee_rates[action][component] = max(
                0, int(round(float(values.median())))
            )

print("=== Fixed sample-median rates (basis points) ===")
for action in ["buy", "sell"]:
    print(action, fee_rates[action])

print(
    "Note: the buyback fee is included in the protocol component, "
    "not added separately."
)
print()

# ---------------------------------------------------------
# 5. Define integer-rounded fee and AMM functions.
# ---------------------------------------------------------

def calculate_fee_components(amount_raw, rates):
    """Apply each component's basis-point rate with ceiling rounding."""
    amount_raw = int(amount_raw)

    return {
        name: (
            (amount_raw * int(bps) + 9_999) // 10_000
            if int(bps) > 0 else 0
        )
        for name, bps in rates.items()
    }


def simulate_round_trip(start_quote, start_base, net_buy_input):
    """
    Buy using a pre-fee curve input, then sell all acquired tokens.

    Model conventions:
      - Buy token output uses the net quote input.
      - Buy LP fee remains in the effective quote reserves.
      - Sell quote output is gross AMM output before fees.
      - Sell LP fee remains in the effective quote reserves.
      - Protocol and creator fees are excluded from effective
        reserves (or offset through virtual reserves).
      - buyback_fee is not added as an extra fee.
    """
    start_quote = int(start_quote)
    start_base = int(start_base)
    net_buy_input = int(net_buy_input)

    if min(start_quote, start_base, net_buy_input) <= 0:
        raise ValueError("Reserves and trade input must be positive.")

    # BUY: calculate token output from the net curve input.
    buy_fees = calculate_fee_components(
        net_buy_input,
        fee_rates["buy"],
    )

    buy_total_fees = sum(buy_fees.values())
    buy_gross_spend = net_buy_input + buy_total_fees

    token_out = (
        start_base * net_buy_input
    ) // (start_quote + net_buy_input)

    if token_out <= 0 or token_out >= start_base:
        raise ValueError("Buy output is outside valid pool bounds.")

    # The LP share remains in the pool. The protocol and creator
    # shares are not added to effective reserves.
    quote_after_buy = (
        start_quote + net_buy_input + buy_fees["lp"]
    )
    base_after_buy = start_base - token_out

    # SELL: compute gross quote output before fees.
    gross_sell_output = (
        quote_after_buy * token_out
    ) // (base_after_buy + token_out)

    sell_fees = calculate_fee_components(
        gross_sell_output,
        fee_rates["sell"],
    )

    sell_total_fees = sum(sell_fees.values())
    net_sell_proceeds = gross_sell_output - sell_total_fees

    if net_sell_proceeds < 0:
        raise ValueError("Fees exceed gross sell proceeds.")

    quote_after_sell = (
        quote_after_buy - gross_sell_output + sell_fees["lp"]
    )
    base_after_sell = base_after_buy + token_out

    pnl_raw = net_sell_proceeds - buy_gross_spend

    # Check that the effective constant-product invariant did not
    # decrease during either swap, allowing only rounding tolerance.
    k0 = start_quote * start_base
    k1 = quote_after_buy * base_after_buy
    k2 = quote_after_sell * base_after_sell

    return {
        "net_buy_input_lamports": net_buy_input,
        "buy_fees_lamports": buy_total_fees,
        "total_buy_cost_SOL": buy_gross_spend / LAMPORTS_PER_SOL,
        "tokens_bought": token_out / token_scale,
        "gross_sell_output_SOL": (
            gross_sell_output / LAMPORTS_PER_SOL
        ),
        "sell_fees_SOL": sell_total_fees / LAMPORTS_PER_SOL,
        "net_sell_proceeds_SOL": (
            net_sell_proceeds / LAMPORTS_PER_SOL
        ),
        "round_trip_pnl_SOL": pnl_raw / LAMPORTS_PER_SOL,
        "round_trip_return_pct": (
            pnl_raw / buy_gross_spend * 100
        ),
        "k_change_buy_pct": (k1 / k0 - 1) * 100,
        "k_change_sell_pct": (k2 / k1 - 1) * 100,
        "final_quote_reserve_SOL": (
            quote_after_sell / LAMPORTS_PER_SOL
        ),
        "final_base_reserve": base_after_sell / token_scale,
    }


# ---------------------------------------------------------
# 6. Run several synthetic trade sizes.
# ---------------------------------------------------------

scenarios = [0.01, 0.1, 0.5, 1.0]
results75 = []

for net_input_sol in scenarios:
    result = simulate_round_trip(
        q0,
        b0,
        int(round(net_input_sol * LAMPORTS_PER_SOL)),
    )

    result["net_curve_input_SOL"] = net_input_sol
    results75.append(result)

results75 = pd.DataFrame(results75)

columns = [
    "net_curve_input_SOL",
    "total_buy_cost_SOL",
    "tokens_bought",
    "gross_sell_output_SOL",
    "sell_fees_SOL",
    "net_sell_proceeds_SOL",
    "round_trip_pnl_SOL",
    "round_trip_return_pct",
    "k_change_buy_pct",
    "k_change_sell_pct",
]

print("=== Synthetic round-trip results ===")
print(results75[columns].round(9).to_string(index=False))

# ---------------------------------------------------------
# 7. Basic sanity checks.
# ---------------------------------------------------------

print("\n=== Sanity checks ===")

print(
    "All round trips are loss-making after fees:",
    bool((results75["round_trip_pnl_SOL"] < 0).all()),
)

print(
    "Effective invariant did not decrease on buys:",
    bool((results75["k_change_buy_pct"] >= -1e-9).all()),
)

print(
    "Effective invariant did not decrease on sells:",
    bool((results75["k_change_sell_pct"] >= -1e-9).all()),
)


In [ ]:

import pandas as pd

found = 0

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame) or obj.empty:
        continue

    sig_cols = [
        c for c in obj.columns
        if any(term in str(c).lower()
               for term in ["signature", "tx_sig", "txsig"])
        or str(c).lower() in ["sig", "transaction_id"]
    ]

    if not sig_cols:
        continue

    found += 1
    print(f"\nDataFrame: {name} | rows: {len(obj)}")
    print("Signature columns:", sig_cols)
    print("Other columns:", [
        c for c in obj.columns
        if any(term in str(c).lower()
               for term in ["side", "type", "buy", "sell", "instruction"])
    ])

    cols = list(dict.fromkeys(
        sig_cols + [
            c for c in obj.columns
            if any(term in str(c).lower()
                   for term in ["side", "type", "buy", "sell", "instruction"])
        ]
    ))

    display(obj[cols].head(5))

if found == 0:
    print(
        "No dataframe with a recognizable signature column was found. "
        "Check the dataframe containing your successful 50-transaction audit."
    )


In [ ]:
# Step 76: Independent constant-product and round-trip check.

import pandas as pd

def independent_round_trip(q0, b0, quote_in, fee_bps=100):
    """
    q0: effective quote reserve in lamports
    b0: base token reserve in raw token units
    quote_in: net quote entering the curve, in lamports
    fee_bps: total fee in basis points per trade
    """

    q0 = int(q0)
    b0 = int(b0)
    quote_in = int(quote_in)

    # Buy output from the constant-product curve.
    tokens_out = (b0 * quote_in) // (q0 + quote_in)

    # Total buy cost assuming the fee is added to curve input.
    buy_fee = (quote_in * fee_bps + 9999) // 10000
    buy_cost = quote_in + buy_fee

    # The pool receives curve input, plus any LP fee retained.
    lp_fee_bps = 20
    lp_fee = (quote_in * lp_fee_bps + 9999) // 10000

    q1 = q0 + quote_in + lp_fee
    b1 = b0 - tokens_out

    # Sell the same number of tokens back.
    gross_out = (q1 * tokens_out) // (b1 + tokens_out)

    sell_fee = (gross_out * fee_bps + 9999) // 10000
    net_out = gross_out - sell_fee

    return {
        "quote_input_SOL": quote_in / 1e9,
        "buy_cost_SOL": buy_cost / 1e9,
        "gross_sell_SOL": gross_out / 1e9,
        "net_sell_SOL": net_out / 1e9,
        "round_trip_pnl_SOL": (net_out - buy_cost) / 1e9,
        "gross_sell_vs_curve_input_pct": (
            gross_out / quote_in - 1
        ) * 100,
        "buy_invariant_change_pct": (
            q1 * b1 / (q0 * b0) - 1
        ) * 100,
    }


independent_results = pd.DataFrame([
    independent_round_trip(
        q0,
        b0,
        int(round(sol * 1e9)),
    )
    for sol in [0.01, 0.1, 0.5, 1.0]
])

print(independent_results.round(9).to_string(index=False))


In [ ]:
import re
import requests
import pandas as pd
from decimal import Decimal
from google.colab import userdata
from IPython.display import display

# ============================================================
# STEP 77 — AUDIT REAL PUMPSWAP BUY AND SELL TRANSACTIONS
# Automatically select signatures from the existing audit.
# ============================================================

# 1. Read the Helius API key from Colab Secrets.
HELIUS_API_KEY = userdata.get("HELIUS_API_KEY")

if not HELIUS_API_KEY:
    raise RuntimeError(
        "HELIUS_API_KEY not found. Add it in Google Colab Secrets."
    )

RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"


# 2. Find valid buy/sell signatures in the existing audit tables.
def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def select_audit_signatures():
    candidates = []

    for table_name in ("audit73", "valid74"):
        table = globals().get(table_name)

        if isinstance(table, pd.DataFrame) and not table.empty:
            candidates.append((table_name, table))

    if not candidates:
        raise RuntimeError(
            "Neither audit73 nor valid74 exists as a non-empty DataFrame. "
            "Run the transaction audit cells first."
        )

    errors = []

    for table_name, df in candidates:
        column_lookup = {
            normalize_name(column): column
            for column in df.columns
        }

        signature_col = next(
            (
                column_lookup[name]
                for name in (
                    "signature",
                    "transactionsignature",
                    "txsignature",
                    "sig",
                )
                if name in column_lookup
            ),
            None,
        )

        action_col = next(
            (
                column_lookup[name]
                for name in (
                    "action",
                    "side",
                    "txtype",
                    "tradetype",
                    "transactiontype",
                    "sw ap type".replace(" ", ""),
                )
                if name in column_lookup
            ),
            None,
        )

        status_col = next(
            (
                column_lookup[name]
                for name in (
                    "status",
                    "auditstatus",
                    "decodestatus",
                )
                if name in column_lookup
            ),
            None,
        )

        if signature_col is None or action_col is None:
            errors.append(
                f"{table_name}: columns={list(df.columns)}"
            )
            continue

        work = df.copy()

        # Keep only valid-looking, complete Solana signatures.
        work["_signature"] = (
            work[signature_col].astype("string").str.strip()
        )

        signature_mask = work["_signature"].str.fullmatch(
            r"[1-9A-HJ-NP-Za-km-z]{80,100}",
            na=False,
        )
        work = work.loc[signature_mask].copy()

        # Restrict to successful/decoded records when status is available.
        if status_col is not None:
            accepted_statuses = {
                "ok",
                "success",
                "successful",
                "confirmed",
                "decoded",
                "pass",
                "passed",
            }

            status = (
                work[status_col]
                .astype("string")
                .str.strip()
                .str.lower()
            )

            successful = status.isin(accepted_statuses)

            if not successful.any():
                errors.append(
                    f"{table_name}: no rows with an accepted status "
                    f"in column {status_col!r}"
                )
                continue

            work = work.loc[successful].copy()

        # Normalize action labels, including values such as BUY_EXACT_IN.
        action = (
            work[action_col]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(r"[^a-z0-9]+", " ", regex=True)
            .str.strip()
        )

        work["_action"] = action

        buy_rows = work.loc[
            work["_action"].str.contains(r"\bbuy\b", regex=True, na=False)
        ]

        sell_rows = work.loc[
            work["_action"].str.contains(r"\bsell\b", regex=True, na=False)
        ]

        if buy_rows.empty or sell_rows.empty:
            errors.append(
                f"{table_name}: no valid buy/sell pair; "
                f"action counts={action.value_counts().to_dict()}"
            )
            continue

        buy_sig = buy_rows.iloc[0]["_signature"]
        sell_sig = sell_rows.iloc[0]["_signature"]

        if buy_sig == sell_sig:
            alternative_sells = sell_rows.loc[
                sell_rows["_signature"] != buy_sig
            ]

            if alternative_sells.empty:
                errors.append(
                    f"{table_name}: buy and sell resolved to the same signature."
                )
                continue

            sell_sig = alternative_sells.iloc[0]["_signature"]

        print(f"Selected audit table: {table_name}")
        print(f"Validated buy rows available: {len(buy_rows)}")
        print(f"Validated sell rows available: {len(sell_rows)}")

        return str(buy_sig), str(sell_sig)

    raise RuntimeError(
        "Could not automatically select a valid buy/sell pair.\n"
        + "\n".join(errors)
        + "\nInspect the audit DataFrame columns and action/status values."
    )


BUY_SIG, SELL_SIG = select_audit_signatures()

print("\nSelected transactions:")
print("BUY :", BUY_SIG)
print("SELL:", SELL_SIG)


# 3. Fetch the transaction from Solana through Helius.
def fetch_transaction(signature):
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "getTransaction",
        "params": [
            signature,
            {
                "encoding": "jsonParsed",
                "commitment": "finalized",
                "maxSupportedTransactionVersion": 0,
            },
        ],
    }

    response = requests.post(
        RPC_URL,
        json=payload,
        timeout=40,
    )
    response.raise_for_status()

    data = response.json()

    if data.get("error"):
        raise RuntimeError(
            f"Helius RPC error for {signature}: {data['error']}"
        )

    result = data.get("result")

    if result is None:
        raise RuntimeError(
            f"No transaction result for {signature}. "
            "The transaction may be unavailable or the signature may be wrong."
        )

    return result


# 4. Print parsed and compiled instructions, including inner instructions.
def print_instruction(instruction, account_keys, level, outer_index):
    parsed = instruction.get("parsed")
    program = instruction.get("program", "unknown")

    program_id = instruction.get("programId")

    if program_id is None:
        program_id_index = instruction.get("programIdIndex")
        if (
            isinstance(program_id_index, int)
            and 0 <= program_id_index < len(account_keys)
        ):
            program_id = account_keys[program_id_index]

    if parsed:
        print({
            "level": level,
            "outer_instruction_index": outer_index,
            "program": program,
            "programId": program_id,
            "type": parsed.get("type"),
            "info": parsed.get("info", {}),
        })
    else:
        print({
            "level": level,
            "outer_instruction_index": outer_index,
            "program": program,
            "programId": program_id,
            "programIdIndex": instruction.get("programIdIndex"),
            "accounts": instruction.get("accounts"),
            "data": instruction.get("data"),
        })


# 5. Audit one transaction's native SOL, token balances and instructions.
def audit_transaction(signature, label):
    tx = fetch_transaction(signature)
    meta = tx.get("meta") or {}
    message = tx["transaction"]["message"]

    account_keys = []

    for key in message.get("accountKeys", []):
        if isinstance(key, dict):
            account_keys.append(key.get("pubkey", str(key)))
        else:
            account_keys.append(str(key))

    print("\n" + "=" * 90)
    print(f"{label}: {signature}")
    print("Slot:", tx.get("slot"))
    print("Transaction error:", meta.get("err"))
    print("Network fee (SOL):", float(meta.get("fee", 0)) / 1e9)

    if meta.get("err") is not None:
        print(
            "WARNING: Transaction failed on-chain. "
            "Do not use it as a successful swap."
        )

    # --------------------------------------------------------
    # Native SOL balance changes.
    # Account index 0 is normally the fee payer.
    # Fee-adjusted delta adds back the network fee for index 0.
    # --------------------------------------------------------
    pre_lamports = meta.get("preBalances", [])
    post_lamports = meta.get("postBalances", [])
    fee_lamports = int(meta.get("fee", 0))

    native_rows = []
    count = min(
        len(account_keys),
        len(pre_lamports),
        len(post_lamports),
    )

    for i in range(count):
        before = int(pre_lamports[i])
        after = int(post_lamports[i])
        delta = after - before

        if delta == 0:
            continue

        adjusted_delta = (
            delta + fee_lamports if i == 0 else delta
        )

        native_rows.append({
            "account_index": i,
            "account": account_keys[i],
            "pre_SOL": before / 1e9,
            "post_SOL": after / 1e9,
            "delta_SOL": delta / 1e9,
            "delta_SOL_fee_adjusted": adjusted_delta / 1e9,
            "is_fee_payer": i == 0,
        })

    print("\nNative SOL balance changes:")

    if native_rows:
        native_df = pd.DataFrame(native_rows)
        native_df = native_df.sort_values(
            "delta_SOL",
            key=lambda s: s.abs(),
            ascending=False,
        ).reset_index(drop=True)
        display(native_df)
    else:
        native_df = pd.DataFrame()
        print("No native SOL balance changes.")

    # --------------------------------------------------------
    # SPL-token changes, keyed by token-account index and mint.
    # Includes wrapped SOL when represented as an SPL token.
    # --------------------------------------------------------
    token_map = {}

    for phase, entries in (
        ("pre", meta.get("preTokenBalances") or []),
        ("post", meta.get("postTokenBalances") or []),
    ):
        for entry in entries:
            idx = int(entry["accountIndex"])
            mint = entry.get("mint", "unknown")
            amount_info = entry.get("uiTokenAmount") or {}

            decimals = int(amount_info.get("decimals", 0))
            raw_amount = int(amount_info.get("amount", "0"))

            ui_amount = (
                Decimal(raw_amount) / (Decimal(10) ** decimals)
            )

            key = (idx, mint)

            if key not in token_map:
                token_map[key] = {
                    "account_index": idx,
                    "token_account": (
                        account_keys[idx]
                        if idx < len(account_keys)
                        else "unknown"
                    ),
                    "mint": mint,
                    "owner": entry.get("owner"),
                    "decimals": decimals,
                    "pre_raw": 0,
                    "post_raw": 0,
                    "pre_amount": Decimal(0),
                    "post_amount": Decimal(0),
                }

            record = token_map[key]

            if entry.get("owner"):
                record["owner"] = entry["owner"]

            record[f"{phase}_raw"] = raw_amount
            record[f"{phase}_amount"] = ui_amount

    token_rows = []

    for record in token_map.values():
        delta_raw = record["post_raw"] - record["pre_raw"]
        delta_amount = record["post_amount"] - record["pre_amount"]

        if delta_raw == 0:
            continue

        token_rows.append({
            "account_index": record["account_index"],
            "token_account": record["token_account"],
            "owner": record["owner"],
            "mint": record["mint"],
            "decimals": record["decimals"],
            "pre_amount": float(record["pre_amount"]),
            "post_amount": float(record["post_amount"]),
            "delta_amount": float(delta_amount),
            "delta_raw_units": delta_raw,
        })

    print("\nSPL token balance changes:")

    if token_rows:
        token_df = pd.DataFrame(token_rows)
        token_df = token_df.sort_values(
            "delta_raw_units",
            key=lambda s: s.abs(),
            ascending=False,
        ).reset_index(drop=True)
        display(token_df)
    else:
        token_df = pd.DataFrame()
        print("No SPL token balance changes.")

    # --------------------------------------------------------
    # Top-level instructions.
    # --------------------------------------------------------
    print("\nTop-level transaction instructions:")

    outer_instructions = message.get("instructions", []) or []

    for index, instruction in enumerate(outer_instructions):
        print_instruction(
            instruction,
            account_keys,
            level="top-level",
            outer_index=index,
        )

    # --------------------------------------------------------
    # Inner instructions invoked by other programs.
    # --------------------------------------------------------
    print("\nInner instructions:")

    inner_groups = meta.get("innerInstructions") or []
    inner_count = 0

    for group in inner_groups:
        outer_index = group.get("index", -1)

        for instruction in group.get("instructions", []) or []:
            inner_count += 1

            print_instruction(
                instruction,
                account_keys,
                level="inner",
                outer_index=outer_index,
            )

    if inner_count == 0:
        print(
            "No inner instructions were returned in transaction metadata."
        )


    return {
        "signature": signature,
        "slot": tx.get("slot"),
        "transaction_error": meta.get("err"),
        "meta": meta,
        "account_keys": account_keys,
        "native_changes": native_df,
        "token_changes": token_df,
        "transaction": tx,
    }


# 6. Run both chain audits.
buy_chain_audit = audit_transaction(BUY_SIG, "REAL BUY")
sell_chain_audit = audit_transaction(SELL_SIG, "REAL SELL")

print("\n" + "=" * 90)
print("STEP 77 COMPLETE")
print("Buy slot :", buy_chain_audit["slot"])
print("Sell slot:", sell_chain_audit["slot"])
print(
    "Buy succeeded on-chain:",
    buy_chain_audit["transaction_error"] is None,
)
print(
    "Sell succeeded on-chain:",
    sell_chain_audit["transaction_error"] is None,
)

In [ ]:
# Step 78: Reconcile actual pool vault deltas with PumpSwap events.

import base64
import pandas as pd
from IPython.display import display

LAMPORTS_PER_SOL = 1_000_000_000
WSOL_MINT = "So11111111111111111111111111111111111111112"

# ------------------------------------------------------------
# 1. Check prerequisites.
# ------------------------------------------------------------
required_globals = [
    "buy_chain_audit",
    "sell_chain_audit",
    "event_fields",
    "event_discriminators",
    "decode_all_fields",
]

missing = [name for name in required_globals if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing {missing}. Run Steps 63-65 and Step 77 first."
    )


# ------------------------------------------------------------
# 2. Decode the PumpSwap event directly from transaction logs.
# ------------------------------------------------------------
def decode_step78_event(audit, action):
    tx = audit["transaction"]
    meta = tx.get("meta") or {}

    expected_name = "BuyEvent" if action == "buy" else "SellEvent"

    for log in meta.get("logMessages") or []:
        prefix = "Program data: "

        if not log.startswith(prefix):
            continue

        try:
            raw = base64.b64decode(
                log[len(prefix):].strip(),
                validate=True,
            )
        except Exception:
            continue

        discriminator = event_discriminators.get(expected_name)

        if discriminator is None:
            raise RuntimeError(
                f"Missing discriminator for {expected_name}."
            )

        if not raw.startswith(bytes(discriminator)):
            continue

        decoded = decode_all_fields(
            raw,
            event_fields[expected_name],
        )

        decoded["_event_name"] = expected_name
        return decoded

    raise RuntimeError(
        f"No {expected_name} found in transaction "
        f"{audit['signature']}."
    )


# ------------------------------------------------------------
# 3. Find the pool's token and wrapped-SOL vault balance changes.
# ------------------------------------------------------------
buy_changes = buy_chain_audit["token_changes"].copy()
sell_changes = sell_chain_audit["token_changes"].copy()

for frame in (buy_changes, sell_changes):
    if frame.empty:
        raise RuntimeError("Step 77 returned an empty token-change table.")

    frame["delta_raw_units"] = pd.to_numeric(
        frame["delta_raw_units"], errors="raise"
    )


# For a buy, the pool's token vault should send tokens out.
buy_token_debits = buy_changes.loc[
    buy_changes["mint"].ne(WSOL_MINT)
    & buy_changes["delta_raw_units"].lt(0)
].copy()

if buy_token_debits.empty:
    raise RuntimeError(
        "Could not identify the pool token vault from the buy."
    )

pool_token_row = buy_token_debits.loc[
    buy_token_debits["delta_raw_units"].abs().idxmax()
]

TOKEN_MINT = str(pool_token_row["mint"])
POOL_AUTHORITY = pool_token_row["owner"]
TOKEN_DECIMALS = int(pool_token_row["decimals"])

if pd.isna(POOL_AUTHORITY) or not str(POOL_AUTHORITY).strip():
    raise RuntimeError(
        "The pool token vault owner is missing from Step 77."
    )

POOL_AUTHORITY = str(POOL_AUTHORITY)


def select_pool_vault(changes, mint, authority, expected_sign):
    rows = changes.loc[
        changes["mint"].eq(mint)
        & changes["owner"].astype(str).eq(authority)
    ].copy()

    if expected_sign < 0:
        rows = rows.loc[rows["delta_raw_units"].lt(0)]
    else:
        rows = rows.loc[rows["delta_raw_units"].gt(0)]

    if rows.empty:
        raise RuntimeError(
            f"Could not identify pool-owned vault for mint {mint}, "
            f"expected sign {expected_sign}."
        )

    return rows.loc[rows["delta_raw_units"].abs().idxmax()]


buy_token_vault = select_pool_vault(
    buy_changes, TOKEN_MINT, POOL_AUTHORITY, -1
)
buy_quote_vault = select_pool_vault(
    buy_changes, WSOL_MINT, POOL_AUTHORITY, +1
)

sell_token_vault = select_pool_vault(
    sell_changes, TOKEN_MINT, POOL_AUTHORITY, +1
)
sell_quote_vault = select_pool_vault(
    sell_changes, WSOL_MINT, POOL_AUTHORITY, -1
)

print("=== Identified pool accounts ===")
print("Token mint:", TOKEN_MINT)
print("Pool authority:", POOL_AUTHORITY)
print("Token vault:", buy_token_vault["token_account"])
print("WSOL vault:", buy_quote_vault["token_account"])
print(
    "Same token vault in both transactions:",
    buy_token_vault["token_account"]
    == sell_token_vault["token_account"],
)
print(
    "Same WSOL vault in both transactions:",
    buy_quote_vault["token_account"]
    == sell_quote_vault["token_account"],
)


# ------------------------------------------------------------
# 4. Decode the trade events and compare exact raw-unit amounts.
# ------------------------------------------------------------
buy_event = decode_step78_event(buy_chain_audit, "buy")
sell_event = decode_step78_event(sell_chain_audit, "sell")


def event_int(event, field):
    value = event.get(field)

    if value is None:
        raise RuntimeError(
            f"Event {event.get('_event_name')} lacks {field}."
        )

    return int(value)


def get_fee_adjusted_native_delta(audit):
    """
    Fee-adjusted native delta for account index 0.
    Adds back the network fee, but does not alter token balances.
    """
    meta = audit["meta"]
    pre = meta.get("preBalances") or []
    post = meta.get("postBalances") or []

    if not pre or not post:
        raise RuntimeError("Native SOL balance arrays are unavailable.")

    return int(post[0]) - int(pre[0]) + int(meta.get("fee", 0))


def make_reconciliation(action, audit, event, token_row, quote_row):
    actual_token_delta = int(token_row["delta_raw_units"])
    actual_quote_delta = int(quote_row["delta_raw_units"])

    if action == "buy":
        # Buy: pool receives quote and sends base tokens.
        expected_token_delta = -event_int(event, "base_amount_out")
        expected_quote_delta = event_int(event, "user_quote_amount_in")

        gross_trader_quote = event_int(event, "quote_amount_in")
        net_trader_quote = gross_trader_quote
        fee_gap = (
            gross_trader_quote
            - event_int(event, "user_quote_amount_in")
        )

        expected_native_delta = -net_trader_quote

    else:
        # Sell: pool receives base tokens and sends gross quote.
        expected_token_delta = event_int(event, "base_amount_in")
        expected_quote_delta = -event_int(event, "quote_amount_out")

        gross_trader_quote = event_int(event, "quote_amount_out")
        net_trader_quote = event_int(event, "user_quote_amount_out")
        fee_gap = gross_trader_quote - net_trader_quote

        expected_native_delta = net_trader_quote

    actual_native_delta = get_fee_adjusted_native_delta(audit)

    return {
        "action": action,
        "signature": audit["signature"],
        "slot": audit["slot"],
        "event_pool": event.get("pool"),
        "pool_token_vault": token_row["token_account"],
        "actual_token_delta_raw": actual_token_delta,
        "event_token_delta_raw": expected_token_delta,
        "token_delta_matches": actual_token_delta == expected_token_delta,
        "actual_token_delta": actual_token_delta / (10 ** TOKEN_DECIMALS),
        "pool_wsol_vault": quote_row["token_account"],
        "actual_quote_delta_lamports": actual_quote_delta,
        "event_quote_delta_lamports": expected_quote_delta,
        "quote_delta_matches": actual_quote_delta == expected_quote_delta,
        "actual_quote_delta_SOL": actual_quote_delta / LAMPORTS_PER_SOL,
        "event_fee_gap_lamports": fee_gap,
        "actual_trader_native_delta_lamports": actual_native_delta,
        "event_trader_native_delta_lamports": expected_native_delta,
        "trader_flow_matches": actual_native_delta == expected_native_delta,
        "network_fee_SOL": int(audit["meta"].get("fee", 0))
            / LAMPORTS_PER_SOL,
    }


reconciliation78 = pd.DataFrame([
    make_reconciliation(
        "buy", buy_chain_audit, buy_event,
        buy_token_vault, buy_quote_vault,
    ),
    make_reconciliation(
        "sell", sell_chain_audit, sell_event,
        sell_token_vault, sell_quote_vault,
    ),
])

print("\n=== Step 78: Exact balance and event reconciliation ===")
display(reconciliation78.T)

print("\n=== Match checks ===")
checks = [
    "token_delta_matches",
    "quote_delta_matches",
    "trader_flow_matches",
]

for column in checks:
    print(
        f"{column}:",
        bool(reconciliation78[column].all()),
    )

print("\n=== Event fee fields (raw lamports) ===")
fee_fields = [
    "lp_fee",
    "protocol_fee",
    "coin_creator_fee",
    "cashback",
    "buyback_fee",
]

fee_rows = []

for action, event in [("buy", buy_event), ("sell", sell_event)]:
    row = {"action": action, "pool": event.get("pool")}

    for field in fee_fields:
        row[field] = event.get(field)

    fee_rows.append(row)

display(pd.DataFrame(fee_rows))

if reconciliation78[checks].all().all():
    print(
        "\nPASS: Actual token-vault changes, quote-vault changes, "
        "and fee-adjusted trader SOL flows match the decoded events."
    )
else:
    print(
        "\nREVIEW REQUIRED: At least one reconciliation failed. "
        "Inspect the mismatched rows before changing the simulator."
    )

In [ ]:
# Step 78 FIX: Correct the pool WSOL reserve accounting.

def make_reconciliation(action, audit, event, token_row, quote_row):
    actual_token_delta = int(token_row["delta_raw_units"])
    actual_quote_delta = int(quote_row["delta_raw_units"])

    lp_fee = event_int(event, "lp_fee")
    protocol_fee = event_int(event, "protocol_fee")
    creator_fee = event_int(event, "coin_creator_fee")
    buyback_fee = int(event.get("buyback_fee") or 0)

    if action == "buy":
        expected_token_delta = -event_int(event, "base_amount_out")

        # The LP fee remains in the pool's WSOL reserve.
        expected_quote_delta = (
            event_int(event, "user_quote_amount_in") + lp_fee
        )

        gross_trader_quote = event_int(event, "quote_amount_in")
        net_trader_quote = gross_trader_quote

        fee_gap = (
            gross_trader_quote
            - event_int(event, "user_quote_amount_in")
        )
        expected_native_delta = -net_trader_quote

    else:
        expected_token_delta = event_int(event, "base_amount_in")

        # The LP fee stays in the pool, so it is not an
        # additional amount leaving the pool vault.
        expected_quote_delta = -(
            event_int(event, "quote_amount_out") - lp_fee
        )

        gross_trader_quote = event_int(event, "quote_amount_out")
        net_trader_quote = event_int(event, "user_quote_amount_out")

        fee_gap = gross_trader_quote - net_trader_quote
        expected_native_delta = net_trader_quote

    actual_native_delta = get_fee_adjusted_native_delta(audit)

    return {
        "action": action,
        "signature": audit["signature"],
        "slot": audit["slot"],
        "pool": event.get("pool"),

        "actual_token_delta_raw": actual_token_delta,
        "event_token_delta_raw": expected_token_delta,
        "token_delta_matches": (
            actual_token_delta == expected_token_delta
        ),

        "actual_quote_delta_lamports": actual_quote_delta,
        "expected_quote_delta_lamports": expected_quote_delta,
        "quote_delta_matches": (
            actual_quote_delta == expected_quote_delta
        ),

        "lp_fee_lamports": lp_fee,
        "protocol_fee_lamports": protocol_fee,
        "creator_fee_lamports": creator_fee,
        "buyback_fee_reported_lamports": buyback_fee,

        "event_fee_gap_lamports": fee_gap,

        "actual_trader_native_delta_lamports": actual_native_delta,
        "event_trader_native_delta_lamports": expected_native_delta,
        "trader_flow_matches": (
            actual_native_delta == expected_native_delta
        ),
    }


# Rebuild the reconciliation using the corrected formula.
reconciliation78 = pd.DataFrame([
    make_reconciliation(
        "buy", buy_chain_audit, buy_event,
        buy_token_vault, buy_quote_vault,
    ),
    make_reconciliation(
        "sell", sell_chain_audit, sell_event,
        sell_token_vault, sell_quote_vault,
    ),
])

display(reconciliation78.T)

checks = [
    "token_delta_matches",
    "quote_delta_matches",
    "trader_flow_matches",
]

print("\n=== Corrected validation ===")
for col in checks:
    print(f"{col}: {reconciliation78[col].all()}")

if reconciliation78[checks].all().all():
    print(
        "\nPASS: Both transactions reconcile at the token vault, "
        "WSOL vault, and trader native-balance level."
    )
else:
    print(
        "\nREVIEW REQUIRED: At least one reconciliation still fails."
    )

In [ ]:
# ============================================================
# STEP 79: Validate constant-product predictions against
# the actual PumpSwap buy and sell events.
#
# Uses existing Step 77/78 results; no RPC calls.
# ============================================================

from decimal import Decimal, ROUND_HALF_UP
import pandas as pd
from IPython.display import display

LAMPORTS_PER_SOL = 1_000_000_000

# ------------------------------------------------------------
# 1. Validate the required variables.
# ------------------------------------------------------------
required_globals = [
    "audit73",
    "buy_event",
    "sell_event",
    "buy_chain_audit",
    "sell_chain_audit",
]

missing = [name for name in required_globals if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run Steps 73, 77 and 78 first."
    )

required_columns = {
    "signature",
    "action",
    "quoteInPool",
    "tokensInPool",
    "status",
}

missing_columns = required_columns - set(audit73.columns)

if missing_columns:
    raise ValueError(
        f"audit73 is missing columns: {sorted(missing_columns)}"
    )

if "token_decimals" in globals():
    TOKEN_DECIMALS = int(token_decimals)
elif "TOKEN_DECIMALS" in globals():
    TOKEN_DECIMALS = int(globals()["TOKEN_DECIMALS"])
else:
    raise RuntimeError("Token decimals are unavailable.")

TOKEN_SCALE = 10 ** TOKEN_DECIMALS


# ------------------------------------------------------------
# 2. Get the archive reserve snapshot for each exact signature.
# These are the pre-trade reserve estimates used by the earlier
# notebook formula tests.
# ------------------------------------------------------------
def get_archive_snapshot(signature, action):
    rows = audit73.loc[
        audit73["signature"].astype(str).eq(str(signature))
        & audit73["action"].astype(str).str.lower().eq(action)
    ].copy()

    if "status" in rows.columns:
        successful = rows.loc[rows["status"].eq("OK")]
        if not successful.empty:
            rows = successful

    if rows.empty:
        raise ValueError(
            f"No archive reserve snapshot for {action}: {signature}"
        )

    if len(rows) > 1:
        print(
            f"WARNING: {len(rows)} snapshots found for {action}; "
            "using the first matching row."
        )

    return rows.iloc[0]


def to_raw_units(value, decimals):
    value = Decimal(str(value))

    if not value.is_finite():
        raise ValueError(f"Invalid reserve value: {value}")

    return int(
        (value * (Decimal(10) ** decimals)).to_integral_value(
            rounding=ROUND_HALF_UP
        )
    )


# ------------------------------------------------------------
# 3. Reconstruct effective quote and base reserves.
# Q = recorded real quote reserve + signed virtual quote reserve.
# B = recorded real token reserve.
#
# All computations below use raw integer units.
# ------------------------------------------------------------
def get_effective_reserves(snapshot, event):
    required_event_fields = [
        "virtual_quote_reserves",
    ]

    missing = [
        field for field in required_event_fields
        if event.get(field) is None
    ]

    if missing:
        raise ValueError(
            f"Event is missing required fields: {missing}"
        )

    real_quote_lamports = to_raw_units(
        snapshot["quoteInPool"],
        9,
    )

    virtual_quote_lamports = int(
        event["virtual_quote_reserves"]
    )

    base_atoms = to_raw_units(
        snapshot["tokensInPool"],
        TOKEN_DECIMALS,
    )

    effective_quote_lamports = (
        real_quote_lamports + virtual_quote_lamports
    )

    if effective_quote_lamports <= 0 or base_atoms <= 0:
        raise ValueError(
            "Invalid effective reserves: "
            f"Q={effective_quote_lamports}, B={base_atoms}"
        )

    return {
        "real_quote_lamports": real_quote_lamports,
        "virtual_quote_lamports": virtual_quote_lamports,
        "effective_quote_lamports": effective_quote_lamports,
        "base_atoms": base_atoms,
    }


buy_snapshot = get_archive_snapshot(
    buy_chain_audit["signature"],
    "buy",
)

sell_snapshot = get_archive_snapshot(
    sell_chain_audit["signature"],
    "sell",
)

buy_reserves = get_effective_reserves(
    buy_snapshot,
    buy_event,
)

sell_reserves = get_effective_reserves(
    sell_snapshot,
    sell_event,
)


# ------------------------------------------------------------
# 4. Exact integer constant-product formulas.
# ------------------------------------------------------------
def cp_buy_output(Q, B, quote_input):
    """Token atoms out = floor(B * quote_input / (Q + quote_input))."""
    Q = int(Q)
    B = int(B)
    quote_input = int(quote_input)

    if Q <= 0 or B <= 0 or quote_input <= 0:
        raise ValueError("Buy reserves and input must be positive.")

    return (B * quote_input) // (Q + quote_input)


def cp_sell_output(Q, B, token_input):
    """Gross quote lamports out = floor(Q * token_input / (B + token_input))."""
    Q = int(Q)
    B = int(B)
    token_input = int(token_input)

    if Q <= 0 or B <= 0 or token_input <= 0:
        raise ValueError("Sell reserves and input must be positive.")

    return (Q * token_input) // (B + token_input)


def pct_error(predicted, actual):
    if actual == 0:
        return float("nan")

    return (predicted / actual - 1) * 100


# ------------------------------------------------------------
# 5. Test the buy formula.
# Compare:
#   A. user_quote_amount_in
#   B. user_quote_amount_in + lp_fee
# Against actual base_amount_out.
# ------------------------------------------------------------
buy_input = int(buy_event["user_quote_amount_in"])
buy_lp_fee = int(buy_event.get("lp_fee") or 0)
actual_buy_tokens = int(buy_event["base_amount_out"])

buy_pred_user = cp_buy_output(
    buy_reserves["effective_quote_lamports"],
    buy_reserves["base_atoms"],
    buy_input,
)

buy_pred_user_plus_lp = cp_buy_output(
    buy_reserves["effective_quote_lamports"],
    buy_reserves["base_atoms"],
    buy_input + buy_lp_fee,
)

buy_rows = [
    {
        "action": "buy",
        "model": "user quote input only",
        "predicted_raw": buy_pred_user,
        "actual_raw": actual_buy_tokens,
    },
    {
        "action": "buy",
        "model": "user quote input + LP fee",
        "predicted_raw": buy_pred_user_plus_lp,
        "actual_raw": actual_buy_tokens,
    },
]


# ------------------------------------------------------------
# 6. Test the sell formula.
# Compare predicted gross quote output with quote_amount_out.
# ------------------------------------------------------------
sell_input_tokens = int(sell_event["base_amount_in"])
actual_sell_quote = int(sell_event["quote_amount_out"])

sell_pred_gross = cp_sell_output(
    sell_reserves["effective_quote_lamports"],
    sell_reserves["base_atoms"],
    sell_input_tokens,
)

sell_rows = [
    {
        "action": "sell",
        "model": "constant-product gross quote output",
        "predicted_raw": sell_pred_gross,
        "actual_raw": actual_sell_quote,
    },
]


# ------------------------------------------------------------
# 7. Present errors in raw units and human-readable amounts.
# ------------------------------------------------------------
result_rows = []

for row in buy_rows + sell_rows:
    action = row["action"]
    predicted = int(row["predicted_raw"])
    actual = int(row["actual_raw"])

    scale = TOKEN_SCALE if action == "buy" else LAMPORTS_PER_SOL

    result_rows.append({
        "action": action,
        "model": row["model"],
        "predicted_raw": predicted,
        "actual_raw": actual,
        "raw_error": predicted - actual,
        "predicted_amount": predicted / scale,
        "actual_amount": actual / scale,
        "absolute_error_pct": abs(pct_error(predicted, actual)),
        "signed_error_pct": pct_error(predicted, actual),
    })

step79_results = pd.DataFrame(result_rows)

print("=== Step 79: Constant-product formula validation ===")
display(step79_results)

print("\n=== Effective reserve inputs ===")

for action, reserves in [
    ("buy", buy_reserves),
    ("sell", sell_reserves),
]:
    print(f"\n{action.upper()}")
    print(
        "Archive real quote reserve (SOL):",
        reserves["real_quote_lamports"] / LAMPORTS_PER_SOL,
    )
    print(
        "Signed virtual quote reserve (SOL):",
        reserves["virtual_quote_lamports"] / LAMPORTS_PER_SOL,
    )
    print(
        "Effective quote reserve (SOL):",
        reserves["effective_quote_lamports"] / LAMPORTS_PER_SOL,
    )
    print(
        "Base reserve (tokens):",
        reserves["base_atoms"] / TOKEN_SCALE,
    )

print("\n=== Interpretation ===")

buy_errors = step79_results.loc[
    step79_results["action"].eq("buy")
].set_index("model")["absolute_error_pct"]

sell_error = step79_results.loc[
    step79_results["action"].eq("sell"),
    "absolute_error_pct",
].iloc[0]

print("Buy error, user input only (%):", buy_errors.iloc[0])
print("Buy error, user input + LP fee (%):", buy_errors.iloc[1])
print("Sell gross-output error (%):", sell_error)

if buy_errors.iloc[0] < buy_errors.iloc[1]:
    print("Buy: user input only is the closer of the two tested formulas.")
elif buy_errors.iloc[1] < buy_errors.iloc[0]:
    print("Buy: user input + LP fee is the closer of the two tested formulas.")
else:
    print("Buy: both tested formulas have equal absolute error.")

print(
    "\nThese comparisons test only two buys/sells. "
    "Do not finalize the simulator from two transactions alone."
)

In [ ]:
# ============================================================
# STEP 80: Validate AMM formulas across the 50-transaction sample.
#
# Compare:
#   1. Buy formula using user_quote_amount_in only
#   2. Buy formula using user_quote_amount_in + lp_fee
#   3. Sell constant-product gross quote output
#
# Uses integer raw units throughout.
# ============================================================

import base64
import time
import pandas as pd
from IPython.display import display

LAMPORTS = 1_000_000_000
TOKEN_DECIMALS = int(token_decimals)
TOKEN_SCALE = 10 ** TOKEN_DECIMALS

# ------------------------------------------------------------
# 1. Check prerequisites.
# ------------------------------------------------------------
required = [
    "audit73",
    "event_fields",
    "event_discriminators",
    "decode_all_fields",
    "fetch_transaction",
    "target_pool",
    "token_decimals",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run Steps 63-65, 73 and 77."
    )

required_columns = {
    "signature",
    "action",
    "status",
    "quoteInPool",
    "tokensInPool",
}

missing_columns = required_columns - set(audit73.columns)

if missing_columns:
    raise ValueError(
        f"audit73 is missing columns: {sorted(missing_columns)}"
    )

sample80 = audit73.loc[
    audit73["status"].eq("OK")
].drop_duplicates("signature").copy()

if sample80.empty:
    raise RuntimeError("No valid transactions found in audit73.")

print("Transactions queued:", len(sample80))
print("Actions:")
print(sample80["action"].value_counts().to_string())


# ------------------------------------------------------------
# 2. Decode the appropriate PumpSwap event from transaction logs.
# ------------------------------------------------------------
def decode_step80_event(tx, action):
    expected = "BuyEvent" if action == "buy" else "SellEvent"
    discriminator = event_discriminators[expected]
    meta = tx.get("meta") or {}

    if meta.get("err") is not None:
        raise ValueError("Transaction failed on-chain.")

    for log in meta.get("logMessages") or []:
        prefix = "Program data: "

        if not log.startswith(prefix):
            continue

        try:
            raw = base64.b64decode(
                log[len(prefix):].strip(),
                validate=True,
            )
        except Exception:
            continue

        if not raw.startswith(discriminator):
            continue

        try:
            event = decode_all_fields(
                raw,
                event_fields[expected],
            )
        except (ValueError, UnicodeDecodeError):
            continue

        if event.get("pool") == target_pool:
            return event

    raise ValueError(f"No matching {expected} for the target pool.")


# ------------------------------------------------------------
# 3. Integer reserve and constant-product utilities.
# ------------------------------------------------------------
def archive_to_raw(value, decimals):
    return int(round(float(value) * (10 ** decimals)))


def cp_buy_raw(quote_reserve, token_reserve, quote_input):
    q = int(quote_reserve)
    b = int(token_reserve)
    x = int(quote_input)

    if min(q, b, x) <= 0:
        raise ValueError("Non-positive buy reserve/input.")

    return (b * x) // (q + x)


def cp_sell_raw(quote_reserve, token_reserve, token_input):
    q = int(quote_reserve)
    b = int(token_reserve)
    x = int(token_input)

    if min(q, b, x) <= 0:
        raise ValueError("Non-positive sell reserve/input.")

    return (q * x) // (b + x)


# ------------------------------------------------------------
# 4. Re-evaluate every valid sample.
# ------------------------------------------------------------
results80 = []
failures80 = []

for number, (_, sample) in enumerate(sample80.iterrows(), start=1):
    signature = str(sample["signature"])
    action = str(sample["action"]).strip().lower()

    if action not in ("buy", "sell"):
        failures80.append({
            "signature": signature,
            "action": action,
            "error": "Unsupported action",
        })
        continue

    try:
        tx = fetch_transaction(signature)
        event = decode_step80_event(tx, action)

        # Archive snapshots represent the recorded pre-trade reserves.
        real_quote = archive_to_raw(sample["quoteInPool"], 9)
        base_reserve = archive_to_raw(
            sample["tokensInPool"],
            TOKEN_DECIMALS,
        )

        virtual_quote = int(event["virtual_quote_reserves"])
        quote_reserve = real_quote + virtual_quote

        if quote_reserve <= 0 or base_reserve <= 0:
            raise ValueError("Invalid effective reserves.")

        if action == "buy":
            actual_output = int(event["base_amount_out"])
            user_input = int(event["user_quote_amount_in"])
            lp_fee = int(event.get("lp_fee") or 0)

            predicted_user_only = cp_buy_raw(
                quote_reserve,
                base_reserve,
                user_input,
            )

            predicted_plus_lp = cp_buy_raw(
                quote_reserve,
                base_reserve,
                user_input + lp_fee,
            )

            candidates = [
                ("buy_user_input_only", predicted_user_only),
                ("buy_input_plus_lp_fee", predicted_plus_lp),
            ]

        else:
            actual_output = int(event["quote_amount_out"])
            token_input = int(event["base_amount_in"])

            predicted_gross = cp_sell_raw(
                quote_reserve,
                base_reserve,
                token_input,
            )

            candidates = [
                ("sell_constant_product_gross", predicted_gross),
            ]

        if actual_output <= 0:
            raise ValueError("Non-positive actual event output.")

        for model, predicted in candidates:
            error_raw = int(predicted) - actual_output
            signed_pct = (
                (int(predicted) / actual_output - 1) * 100
            )

            output_scale = (
                TOKEN_SCALE if action == "buy" else LAMPORTS
            )

            results80.append({
                "signature": signature,
                "action": action,
                "model": model,
                "slot": tx.get("slot"),
                "instruction": event.get("ix_name"),
                "actual_output_raw": actual_output,
                "predicted_output_raw": int(predicted),
                "error_raw": error_raw,
                "actual_output": actual_output / output_scale,
                "predicted_output": int(predicted) / output_scale,
                "signed_error_pct": signed_pct,
                "absolute_error_pct": abs(signed_pct),
            })

    except Exception as exc:
        failures80.append({
            "signature": signature,
            "action": action,
            "error": f"{type(exc).__name__}: {str(exc)[:180]}",
        })

    if number % 10 == 0:
        print(f"Processed {number}/{len(sample80)} transactions.")

    # A small pause between RPC requests.
    time.sleep(0.15)


# ------------------------------------------------------------
# 5. Report detailed results and error summaries.
# ------------------------------------------------------------
if not results80:
    raise RuntimeError(
        "No formula results were produced. Inspect failures80."
    )

formula_results80 = pd.DataFrame(results80)
failures80_df = pd.DataFrame(failures80)

summary80 = (
    formula_results80.groupby(["action", "model"])
    .agg(
        transactions=("signature", "nunique"),
        median_abs_error_pct=("absolute_error_pct", "median"),
        p90_abs_error_pct=(
            "absolute_error_pct",
            lambda s: s.quantile(0.90),
        ),
        p95_abs_error_pct=(
            "absolute_error_pct",
            lambda s: s.quantile(0.95),
        ),
        max_abs_error_pct=("absolute_error_pct", "max"),
        median_signed_error_pct=("signed_error_pct", "median"),
        within_0_001_pct=(
            "absolute_error_pct",
            lambda s: int(s.lt(0.001).sum()),
        ),
        within_0_01_pct=(
            "absolute_error_pct",
            lambda s: int(s.lt(0.01).sum()),
        ),
    )
    .reset_index()
)

print("\n=== Step 80: Formula error summary ===")
display(summary80.round(8))

print("\n=== Transaction processing ===")
print("Input sample:", len(sample80))
print("Successful transaction formula evaluations:",
      formula_results80["signature"].nunique())
print("Failed or skipped transactions:", len(failures80_df))

if not failures80_df.empty:
    print("\nFirst 10 processing failures:")
    display(failures80_df.head(10))

print("\n=== Largest prediction errors ===")
display(
    formula_results80.sort_values(
        "absolute_error_pct",
        ascending=False,
    )
    .groupby("model", group_keys=False)
    .head(5)[
        [
            "action",
            "model",
            "signature",
            "instruction",
            "actual_output",
            "predicted_output",
            "error_raw",
            "absolute_error_pct",
        ]
    ].reset_index(drop=True)
)

print(
    "\nInterpretation: compare the median, p90, p95, and maximum "
    "errors across models. Review outliers before selecting the "
    "simulator formula. Archive reserve snapshots and virtual "
    "reserve timing can also contribute to prediction errors."
)

In [ ]:
# ============================================================
# STEP 81: Diagnose residual formula errors from Step 80.
# No RPC calls. Does not modify the simulator.
# ============================================================

import pandas as pd
from IPython.display import display

if "formula_results80" not in globals():
    raise RuntimeError("Run Step 80 first.")

df81 = formula_results80.copy()

# Separate buy and sell models.
buy81 = df81.loc[
    df81["model"].eq("buy_user_input_only")
].copy()

sell81 = df81.loc[
    df81["model"].eq("sell_constant_product_gross")
].copy()

if buy81.empty or sell81.empty:
    raise RuntimeError(
        "Expected buy_user_input_only and "
        "sell_constant_product_gross results."
    )

# Positive signed error means the model overpredicts output.
buy81["overprediction_raw"] = (
    buy81["predicted_output_raw"] - buy81["actual_output_raw"]
)

buy81["overprediction_pct"] = (
    buy81["signed_error_pct"]
)

print("=== Buy residual diagnostics ===")

display(
    buy81["overprediction_pct"]
    .describe(percentiles=[0.50, 0.90, 0.95])
    .to_frame("signed_error_pct")
)

print("\n=== Buy prediction errors in raw token units ===")

display(
    buy81[
        [
            "signature",
            "instruction",
            "actual_output_raw",
            "predicted_output_raw",
            "overprediction_raw",
            "overprediction_pct",
        ]
    ]
    .sort_values("overprediction_raw", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("\n=== Sell residual diagnostics ===")

print("Sell transactions:", len(sell81))
print(
    "Exact integer matches:",
    int(
        (
            sell81["predicted_output_raw"]
            == sell81["actual_output_raw"]
        ).sum()
    ),
    "/",
    len(sell81),
)

print(
    "Maximum absolute sell error in raw units:",
    int(
        (
            sell81["predicted_output_raw"]
            - sell81["actual_output_raw"]
        ).abs().max()
    ),
)

# Confirm whether every buy prediction overestimates output.
print("\n=== Directional bias ===")
print(
    "Buy predictions above actual:",
    int((buy81["overprediction_raw"] > 0).sum()),
    "/",
    len(buy81),
)
print(
    "Buy predictions exactly equal to actual:",
    int((buy81["overprediction_raw"] == 0).sum()),
    "/",
    len(buy81),
)
print(
    "Buy predictions below actual:",
    int((buy81["overprediction_raw"] < 0).sum()),
    "/",
    len(buy81),
)

print(
    "\nInterpretation: a consistent positive buy residual suggests "
    "a small systematic difference remains. These summaries alone "
    "cannot distinguish fee rounding from reserve timing or another "
    "instruction-specific adjustment."
)

In [ ]:
import pandas as pd
from IPython.display import display

if "formula_results80" not in globals():
    raise NameError("Run Step 80 first; formula_results80 is missing.")

df82 = formula_results80.copy()

print("formula_results80 columns:")
print(sorted(df82.columns.tolist()))

print("\naudit73 columns:")
print(sorted(audit73.columns.tolist()))

# Keep the buy-only model and its observed residuals.
buy82 = df82[
    df82["model"].eq("buy_user_input_only")
].copy()

# Join the original archive snapshot fields for those signatures.
snapshot_cols82 = [
    c for c in [
        "signature", "action", "status",
        "quoteInPool", "tokensInPool"
    ]
    if c in audit73.columns
]

if "signature" in buy82.columns and "signature" in snapshot_cols82:
    snapshots82 = audit73[snapshot_cols82].drop_duplicates("signature")
    buy82 = buy82.merge(
        snapshots82,
        on="signature",
        how="left",
        suffixes=("", "_snapshot")
    )

# Show identifiers, outputs, residuals, and any retained input/reserve fields.
keywords82 = (
    "signature", "instruction", "actual_output", "predicted_output",
    "overprediction", "error", "quote", "token", "reserve",
    "input", "virtual", "pool"
)

cols82 = [
    c for c in buy82.columns
    if any(k in c.lower() for k in keywords82)
]

display(buy82[cols82].head(10))

In [ ]:
import inspect

for name in ["fetch_transaction", "decode_all_fields"]:
    fn = globals().get(name)
    print(f"\n{name}:")
    if fn is None:
        print("Not found")
    else:
        try:
            print("Signature:", inspect.signature(fn))
        except (TypeError, ValueError):
            print("Signature unavailable")
        print(inspect.getdoc(fn) or "No docstring")

print("\nevent_fields:")
if "event_fields" in globals():
    print(type(event_fields).__name__)
    print(repr(event_fields)[:6000])
else:
    print("event_fields not found")

print("\nevent_discriminators:")
if "event_discriminators" in globals():
    print(repr(event_discriminators)[:3000])
else:
    print("event_discriminators not found")

In [ ]:
import base64
import pandas as pd
from IPython.display import display

# Use the buy predictions from Step 80.
buy84 = formula_results80[
    formula_results80["model"].eq("buy_user_input_only")
].copy()

def extract_event84(signature):
    tx = fetch_transaction(signature)

    # Support either a direct transaction result or an RPC response wrapper.
    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    meta = (tx or {}).get("meta") or {}
    logs = meta.get("logMessages") or []

    # Some helper implementations wrap the transaction differently.
    if not logs and isinstance(tx, dict):
        inner = tx.get("transaction")
        if isinstance(inner, dict):
            logs = (inner.get("meta") or {}).get("logMessages") or []

    for line in logs:
        marker = "Program data: "
        if marker not in line:
            continue

        encoded = line.split(marker, 1)[1].strip()
        try:
            raw = base64.b64decode(encoded)
        except Exception:
            continue

        event_name = next(
            (
                name for name, discriminator in event_discriminators.items()
                if raw.startswith(discriminator)
            ),
            None,
        )

        if event_name == "BuyEvent":
            return decode_all_fields(raw[8:], event_fields["BuyEvent"])

    return None


rows84 = []
errors84 = []

for _, row in buy84.iterrows():
    sig = row["signature"]

    try:
        ev = extract_event84(sig)
        if ev is None:
            errors84.append({"signature": sig, "error": "BuyEvent not found in logs"})
            continue

        # These are integer raw-unit values decoded from the event.
        B = int(ev["pool_base_token_reserves"])
        Q = int(ev["pool_quote_token_reserves"])
        V = int(ev["virtual_quote_reserves"])
        x = int(ev["user_quote_amount_in"])
        actual = int(ev["base_amount_out"])

        # Candidate A: pool quote reserve plus virtual quote reserve.
        denominator_a = Q + V + x
        pred_a = (B * x) // denominator_a if denominator_a > 0 else None

        # Candidate B: use pool quote reserve without adding virtual reserves.
        denominator_b = Q + x
        pred_b = (B * x) // denominator_b if denominator_b > 0 else None

        rows84.append({
            "signature": sig,
            "actual_output_raw_event": actual,
            "actual_output_raw_step80": int(row["actual_output_raw"]),
            "user_quote_amount_in_raw": x,
            "quote_amount_in_raw": int(ev["quote_amount_in"]),
            "quote_amount_in_with_lp_fee_raw": int(
                ev["quote_amount_in_with_lp_fee"]
            ),
            "lp_fee_raw": int(ev["lp_fee"]),
            "pool_base_reserves_raw": B,
            "pool_quote_reserves_raw": Q,
            "virtual_quote_reserves_raw": V,
            "candidate_A_pred_raw": pred_a,
            "candidate_A_error_raw": (
                pred_a - actual if pred_a is not None else None
            ),
            "candidate_B_pred_raw": pred_b,
            "candidate_B_error_raw": (
                pred_b - actual if pred_b is not None else None
            ),
            "step80_error_raw": int(row["predicted_output_raw"]) - actual,
        })

    except Exception as exc:
        errors84.append({"signature": sig, "error": repr(exc)})

events84 = pd.DataFrame(rows84)

if not events84.empty:
    print(f"Successfully decoded {len(events84)} / {len(buy84)} buy events.")

    display(events84[[
        "signature",
        "actual_output_raw_event",
        "actual_output_raw_step80",
        "user_quote_amount_in_raw",
        "quote_amount_in_raw",
        "quote_amount_in_with_lp_fee_raw",
        "lp_fee_raw",
        "pool_base_reserves_raw",
        "pool_quote_reserves_raw",
        "virtual_quote_reserves_raw",
        "candidate_A_error_raw",
        "candidate_B_error_raw",
        "step80_error_raw",
    ]].head(10))

    summary84 = pd.DataFrame([
        {
            "candidate": "A: pool quote + virtual quote",
            "exact_matches": int((events84["candidate_A_error_raw"] == 0).sum()),
            "median_abs_error_raw": events84["candidate_A_error_raw"].abs().median(),
            "max_abs_error_raw": events84["candidate_A_error_raw"].abs().max(),
        },
        {
            "candidate": "B: pool quote only",
            "exact_matches": int((events84["candidate_B_error_raw"] == 0).sum()),
            "median_abs_error_raw": events84["candidate_B_error_raw"].abs().median(),
            "max_abs_error_raw": events84["candidate_B_error_raw"].abs().max(),
        },
    ])
    print("\nCandidate comparison:")
    display(summary84)

if errors84:
    print(f"\nTransactions with decode/fetch issues: {len(errors84)}")
    display(pd.DataFrame(errors84).head(10))

In [ ]:
import base64
import inspect

# Inspect the existing decoder implementation.
print("=== decode_all_fields implementation ===")
try:
    print(inspect.getsource(decode_all_fields))
except (OSError, TypeError):
    print("Source unavailable")

# Use a known buy transaction from the existing results.
sig84a = buy84.iloc[0]["signature"]
tx84a = fetch_transaction(sig84a)

if isinstance(tx84a, dict) and tx84a.get("result") is not None:
    tx84a = tx84a["result"]

meta84a = (tx84a or {}).get("meta") or {}
logs84a = meta84a.get("logMessages") or []

print("\nTransaction response keys:", list(tx84a.keys()) if isinstance(tx84a, dict) else type(tx84a))
print("Log messages found:", len(logs84a))

buy_disc84a = event_discriminators["BuyEvent"]
buy_fields84a = event_fields["BuyEvent"]
found84a = False

for line in logs84a:
    marker = "Program data: "
    if marker not in line:
        continue

    encoded = line.split(marker, 1)[1].strip()

    try:
        raw = base64.b64decode(encoded, validate=True)
    except Exception:
        continue

    if not raw.startswith(buy_disc84a):
        continue

    found84a = True
    print("\n=== Matching BuyEvent payload ===")
    print("Payload length including discriminator:", len(raw))
    print("First 16 bytes:", raw[:16].hex())
    print("Expected discriminator:", buy_disc84a.hex())

    for label, payload in [
        ("Discriminator removed", raw[8:]),
        ("Discriminator included", raw),
    ]:
        try:
            decoded = decode_all_fields(payload, buy_fields84a)
            print(f"\n{label}: SUCCESS")
            print({
                k: decoded.get(k)
                for k in [
                    "base_amount_out",
                    "user_quote_amount_in",
                    "pool_base_token_reserves",
                    "pool_quote_token_reserves",
                    "virtual_quote_reserves",
                    "ix_name",
                ]
            })
        except Exception as exc:
            print(f"\n{label}: FAILED — {type(exc).__name__}: {exc}")

    break

if not found84a:
    print("No BuyEvent discriminator found in this transaction's log messages.")

In [ ]:
import base64
import pandas as pd
from IPython.display import display

buy84b = formula_results80[
    formula_results80["model"].eq("buy_user_input_only")
].copy()

def extract_buy_event84b(signature):
    tx = fetch_transaction(signature)

    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    logs = ((tx or {}).get("meta") or {}).get("logMessages") or []
    discriminator = event_discriminators["BuyEvent"]

    for line in logs:
        marker = "Program data: "
        if marker not in line:
            continue

        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(),
                validate=True
            )
        except Exception:
            continue

        if raw.startswith(discriminator):
            # Important: decoder itself skips the 8-byte discriminator.
            return decode_all_fields(raw, event_fields["BuyEvent"])

    raise ValueError("BuyEvent not found in transaction logs")


rows84b = []
errors84b = []

for _, row in buy84b.iterrows():
    try:
        ev = extract_buy_event84b(row["signature"])

        B = int(ev["pool_base_token_reserves"])
        Q = int(ev["pool_quote_token_reserves"])
        V = int(ev["virtual_quote_reserves"])
        actual = int(ev["base_amount_out"])

        # Compare plausible input fields from the decoded event.
        inputs = {
            "user_quote_amount_in": int(ev["user_quote_amount_in"]),
            "quote_amount_in": int(ev["quote_amount_in"]),
            "quote_amount_in_with_lp_fee": int(
                ev["quote_amount_in_with_lp_fee"]
            ),
        }

        # Compare effective quote reserve and real quote reserve only.
        quote_reserves = {
            "pool_quote_plus_virtual": Q + V,
            "pool_quote_only": Q,
        }

        out = {
            "signature": row["signature"],
            "actual_event_raw": actual,
            "actual_step80_raw": int(row["actual_output_raw"]),
            "pool_base_raw": B,
            "pool_quote_raw": Q,
            "virtual_quote_raw": V,
            "lp_fee_raw": int(ev["lp_fee"]),
            **{f"{k}_raw": v for k, v in inputs.items()},
        }

        for reserve_name, reserve in quote_reserves.items():
            for input_name, amount_in in inputs.items():
                key = f"{reserve_name}__{input_name}"
                predicted = (B * amount_in) // (reserve + amount_in)
                out[f"{key}__pred_raw"] = predicted
                out[f"{key}__error_raw"] = predicted - actual

        rows84b.append(out)

    except Exception as exc:
        errors84b.append({
            "signature": row["signature"],
            "error": repr(exc),
        })

events84b = pd.DataFrame(rows84b)

print(f"Successfully decoded: {len(events84b)} / {len(buy84b)} buy events")

if not events84b.empty:
    print("\nEvent output consistency check:")
    print(
        "Event output equals Step 80 actual output:",
        int((events84b["actual_event_raw"] ==
             events84b["actual_step80_raw"]).sum()),
        "/",
        len(events84b),
    )

    candidate_cols84b = [
        c for c in events84b.columns if c.endswith("__error_raw")
    ]

    summaries84b = []
    for col in candidate_cols84b:
        errors = events84b[col]
        summaries84b.append({
            "candidate": col.replace("__error_raw", ""),
            "exact_matches": int((errors == 0).sum()),
            "median_signed_error_raw": errors.median(),
            "median_abs_error_raw": errors.abs().median(),
            "max_abs_error_raw": errors.abs().max(),
        })

    summary84b = pd.DataFrame(summaries84b).sort_values(
        ["exact_matches", "median_abs_error_raw"],
        ascending=[False, True],
    )

    print("\nFormula candidate ranking:")
    display(summary84b)

    print("\nDecoded example transactions:")
    display(events84b[[
        "signature",
        "actual_event_raw",
        "pool_base_raw",
        "pool_quote_raw",
        "virtual_quote_raw",
        "user_quote_amount_in_raw",
        "quote_amount_in_raw",
        "quote_amount_in_with_lp_fee_raw",
        "lp_fee_raw",
    ]].head(5))

if errors84b:
    print("\nDecode/fetch failures:")
    display(pd.DataFrame(errors84b).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

if "events84b" not in globals() or events84b.empty:
    raise NameError("Run Step 84B first; events84b is missing or empty.")

rows85 = []

for _, r in events84b.iterrows():
    B = int(r["pool_base_raw"])
    Q_effective = (
        int(r["pool_quote_raw"]) +
        int(r["virtual_quote_raw"])
    )
    x = int(r["user_quote_amount_in_raw"])
    actual = int(r["actual_event_raw"])

    for offset in range(-3, 2):
        adjusted_input = x + offset
        predicted = (
            B * adjusted_input // (Q_effective + adjusted_input)
        )

        rows85.append({
            "signature": r["signature"],
            "input_offset_lamports": offset,
            "predicted_raw": predicted,
            "actual_raw": actual,
            "signed_error_raw": predicted - actual,
        })

test85 = pd.DataFrame(rows85)

summary85 = (
    test85.groupby("input_offset_lamports")
    .agg(
        transactions=("signature", "count"),
        exact_matches=("signed_error_raw", lambda s: int((s == 0).sum())),
        median_signed_error_raw=("signed_error_raw", "median"),
        median_abs_error_raw=("signed_error_raw", lambda s: s.abs().median()),
        max_abs_error_raw=("signed_error_raw", lambda s: s.abs().max()),
    )
    .reset_index()
    .sort_values(
        ["exact_matches", "median_abs_error_raw"],
        ascending=[False, True],
    )
)

display(summary85)

print("\nPer-transaction errors for the -1 lamport candidate:")
display(
    test85[test85["input_offset_lamports"].eq(-1)][
        ["signature", "actual_raw", "predicted_raw", "signed_error_raw"]
    ].head(10)
)

In [ ]:
import pandas as pd
from IPython.display import display

if "audit73" not in globals() or "formula_results80" not in globals():
    raise NameError("Run the cells that create audit73 and formula_results80 first.")

if "extract_buy_event84b" not in globals():
    raise NameError("Run Step 84B first to define extract_buy_event84b.")

# Transactions already used when discovering the offset.
used_sigs86 = set(
    formula_results80["signature"].dropna().astype(str)
)

# Find buy rows in the original archive dataset.
search_cols86 = [
    c for c in ["action", "ix_name", "archive_ix_name"]
    if c in audit73.columns
]

if not search_cols86 or "signature" not in audit73.columns:
    raise ValueError("audit73 is missing the expected action/name or signature columns.")

buy_mask86 = pd.Series(False, index=audit73.index)
for col in search_cols86:
    buy_mask86 |= audit73[col].astype(str).str.contains(
        "buy", case=False, na=False
    )

candidate_buys86 = (
    audit73.loc[buy_mask86]
    .dropna(subset=["signature"])
    .drop_duplicates("signature")
    .copy()
)

holdout86 = candidate_buys86[
    ~candidate_buys86["signature"].astype(str).isin(used_sigs86)
].copy()

print("Buy transactions in audit73:", len(candidate_buys86))
print("Previously tested signatures:", len(used_sigs86))
print("Unseen buy transactions available:", len(holdout86))

if holdout86.empty:
    print(
        "\nNo unseen buys are available in audit73. "
        "To perform independent validation, add more buy transactions "
        "to the archive dataset before rerunning this step."
    )
else:
    rows86 = []
    failures86 = []

    for sig in holdout86["signature"].astype(str).tolist():
        try:
            ev = extract_buy_event84b(sig)

            B = int(ev["pool_base_token_reserves"])
            R = (
                int(ev["pool_quote_token_reserves"])
                + int(ev["virtual_quote_reserves"])
            )
            x = int(ev["user_quote_amount_in"])
            actual = int(ev["base_amount_out"])

            for offset in [-2, -1, 0, 1]:
                adjusted_x = x + offset
                if adjusted_x < 0 or R + adjusted_x <= 0:
                    continue

                predicted = (B * adjusted_x) // (R + adjusted_x)

                rows86.append({
                    "signature": sig,
                    "offset_lamports": offset,
                    "actual_raw": actual,
                    "predicted_raw": predicted,
                    "error_raw": predicted - actual,
                })

        except Exception as exc:
            failures86.append({
                "signature": sig,
                "error": repr(exc),
            })

    test86 = pd.DataFrame(rows86)

    if not test86.empty:
        summary86 = (
            test86.groupby("offset_lamports")
            .agg(
                transactions=("signature", "nunique"),
                exact_matches=("error_raw", lambda s: int((s == 0).sum())),
                median_signed_error_raw=("error_raw", "median"),
                median_abs_error_raw=("error_raw", lambda s: s.abs().median()),
                max_abs_error_raw=("error_raw", lambda s: s.abs().max()),
            )
            .reset_index()
            .sort_values("offset_lamports")
        )

        print("\nHoldout results:")
        display(summary86)

    if failures86:
        print("\nTransactions that could not be decoded:")
        display(pd.DataFrame(failures86).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

tested_buy_sigs87 = set(
    formula_results80.loc[
        formula_results80["model"].eq("buy_user_input_only"),
        "signature"
    ].dropna().astype(str)
)

reports87 = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame) or "signature" not in obj.columns:
        continue

    df = obj
    sigs = df["signature"].dropna().astype(str)
    unique_sigs = set(sigs)

    action_cols = [
        c for c in ["action", "ix_name", "archive_ix_name", "instruction"]
        if c in df.columns
    ]

    buy_sigs = set()
    for col in action_cols:
        mask = df[col].astype(str).str.contains("buy", case=False, na=False)
        buy_sigs.update(
            df.loc[mask, "signature"].dropna().astype(str).tolist()
        )

    unseen_buys = buy_sigs - tested_buy_sigs87

    reports87.append({
        "dataframe": name,
        "rows": len(df),
        "unique_signatures": len(unique_sigs),
        "buy_signatures": len(buy_sigs),
        "tested_buy_overlap": len(buy_sigs & tested_buy_sigs87),
        "unseen_buys": len(unseen_buys),
        "action_columns": ", ".join(action_cols),
    })

report87 = pd.DataFrame(reports87)

if report87.empty:
    print("No other DataFrames with a signature column were found.")
else:
    display(
        report87.sort_values(
            ["unseen_buys", "unique_signatures"],
            ascending=False
        ).reset_index(drop=True)
    )

In [ ]:
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")
if "extract_buy_event84b" not in globals():
    raise NameError("Run Step 84B first to define extract_buy_event84b.")

# Exclude every signature already included in Step 80.
used_sigs88 = set(
    formula_results80["signature"].dropna().astype(str)
)

buy_pool88 = stratified_trades[
    stratified_trades["action"].astype(str).str.contains(
        "buy", case=False, na=False
    )
].drop_duplicates("signature").copy()

buy_pool88 = buy_pool88[
    ~buy_pool88["signature"].astype(str).isin(used_sigs88)
]

print("Available unseen buys:", len(buy_pool88))

if len(buy_pool88) < 25:
    raise ValueError("Fewer than 25 unseen buys are available.")

# Deterministic sample so the experiment is reproducible.
sample88 = buy_pool88.sample(n=25, random_state=2026)

rows88 = []
failures88 = []

for sig in sample88["signature"].astype(str):
    try:
        ev = extract_buy_event84b(sig)

        B = int(ev["pool_base_token_reserves"])
        R = (
            int(ev["pool_quote_token_reserves"])
            + int(ev["virtual_quote_reserves"])
        )
        x = int(ev["user_quote_amount_in"])
        actual = int(ev["base_amount_out"])

        for offset in [-2, -1, 0, 1]:
            adjusted_x = x + offset
            predicted = (
                B * adjusted_x // (R + adjusted_x)
                if adjusted_x >= 0 and R + adjusted_x > 0
                else None
            )

            rows88.append({
                "signature": sig,
                "offset_lamports": offset,
                "actual_raw": actual,
                "predicted_raw": predicted,
                "error_raw": (
                    predicted - actual if predicted is not None else None
                ),
            })

    except Exception as exc:
        failures88.append({
            "signature": sig,
            "error": repr(exc),
        })

test88 = pd.DataFrame(rows88)

if not test88.empty:
    summary88 = (
        test88.groupby("offset_lamports")
        .agg(
            transactions=("signature", "nunique"),
            exact_matches=("error_raw", lambda s: int((s == 0).sum())),
            median_signed_error_raw=("error_raw", "median"),
            median_abs_error_raw=("error_raw", lambda s: s.abs().median()),
            max_abs_error_raw=("error_raw", lambda s: s.abs().max()),
        )
        .reset_index()
        .sort_values("offset_lamports")
    )

    print("=== Holdout validation results ===")
    display(summary88)

    decoded_sigs88 = test88["signature"].nunique()
    print(f"Successfully decoded distinct buys: {decoded_sigs88} / 25")

if failures88:
    print("=== Fetch/decode failures ===")
    display(pd.DataFrame(failures88).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

# Find signatures whose -1 lamport prediction did not match exactly.
bad89 = test88[
    test88["offset_lamports"].eq(-1)
    & test88["error_raw"].notna()
    & test88["error_raw"].ne(0)
].copy()

bad_sigs89 = bad89["signature"].astype(str).unique().tolist()

print("Mismatching holdout buys:", len(bad_sigs89))

rows89 = []
for sig in bad_sigs89:
    try:
        ev = extract_buy_event84b(sig)

        B = int(ev["pool_base_token_reserves"])
        Q = int(ev["pool_quote_token_reserves"])
        V = int(ev["virtual_quote_reserves"])
        x = int(ev["user_quote_amount_in"])
        actual = int(ev["base_amount_out"])

        predicted = B * (x - 1) // (Q + V + x - 1)

        rows89.append({
            "signature": sig,
            "ix_name": ev.get("ix_name"),
            "actual_output_raw": actual,
            "predicted_minus1_raw": predicted,
            "error_minus1_raw": predicted - actual,
            "user_quote_amount_in_raw": x,
            "quote_amount_in_raw": int(ev["quote_amount_in"]),
            "quote_amount_in_with_lp_fee_raw": int(
                ev["quote_amount_in_with_lp_fee"]
            ),
            "max_quote_amount_in_raw": int(ev["max_quote_amount_in"]),
            "lp_fee_raw": int(ev["lp_fee"]),
            "pool_base_reserves_raw": B,
            "pool_quote_reserves_raw": Q,
            "virtual_quote_reserves_raw": V,
            "pool_pubkey": ev.get("pool"),
        })

    except Exception as exc:
        rows89.append({
            "signature": sig,
            "diagnostic_error": repr(exc),
        })

outliers89 = pd.DataFrame(rows89)
display(outliers89.T)

In [ ]:
import pandas as pd
from IPython.display import display

if "bad_sigs89" not in globals() or not bad_sigs89:
    raise NameError("Run Step 89 first; bad_sigs89 is missing or empty.")

rows90 = []

for sig in bad_sigs89:
    ev = extract_buy_event84b(sig)

    B = int(ev["pool_base_token_reserves"])
    Q = int(ev["pool_quote_token_reserves"])
    V = int(ev["virtual_quote_reserves"])
    y = int(ev["base_amount_out"])

    # Candidate inverse CPMM calculation using the event reserves.
    # Assumes the event reserve values represent the pre-trade state.
    R = Q + V
    denom = B - y

    if denom <= 0:
        raise ValueError(f"Invalid base reserve/output for {sig}")

    numerator = R * y
    curve_floor = numerator // denom
    curve_ceil = (numerator + denom - 1) // denom

    recorded = {
        "quote_amount_in": int(ev["quote_amount_in"]),
        "quote_amount_in_with_lp_fee": int(
            ev["quote_amount_in_with_lp_fee"]
        ),
        "user_quote_amount_in": int(ev["user_quote_amount_in"]),
        "max_quote_amount_in": int(ev["max_quote_amount_in"]),
    }

    row = {
        "signature": sig,
        "ix_name": ev.get("ix_name"),
        "base_amount_out_raw": y,
        "pool_base_reserves_raw": B,
        "effective_quote_reserves_raw": R,
        "curve_quote_floor_raw": curve_floor,
        "curve_quote_ceil_raw": curve_ceil,
    }

    for name, value in recorded.items():
        row[f"{name}_raw"] = value
        row[f"floor_minus_{name}_raw"] = curve_floor - value
        row[f"ceil_minus_{name}_raw"] = curve_ceil - value

    rows90.append(row)

result90 = pd.DataFrame(rows90)

cols90 = [
    "signature",
    "ix_name",
    "base_amount_out_raw",
    "curve_quote_floor_raw",
    "curve_quote_ceil_raw",
    "quote_amount_in_raw",
    "quote_amount_in_with_lp_fee_raw",
    "user_quote_amount_in_raw",
    "max_quote_amount_in_raw",
    "floor_minus_quote_amount_in_raw",
    "ceil_minus_quote_amount_in_raw",
    "floor_minus_quote_amount_in_with_lp_fee_raw",
    "ceil_minus_quote_amount_in_with_lp_fee_raw",
]

display(result90[cols90].T)

In [ ]:
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")
if "extract_buy_event84b" not in globals():
    raise NameError("Run Step 84B first.")

# Exclude signatures already used in the prior formula tests.
used91 = set(formula_results80["signature"].dropna().astype(str))
if "test88" in globals() and not test88.empty:
    used91.update(test88["signature"].dropna().astype(str))

pool91 = (
    stratified_trades[
        stratified_trades["action"].astype(str).str.contains(
            "buy", case=False, na=False
        )
    ]
    .drop_duplicates("signature")
    .copy()
)

pool91 = pool91[
    ~pool91["signature"].astype(str).isin(used91)
]

n91 = min(40, len(pool91))
if n91 == 0:
    raise ValueError("No unseen buy signatures available.")

sample91 = pool91.sample(n=n91, random_state=91026)

rows91, failures91 = [], []

for sig in sample91["signature"].astype(str):
    try:
        ev = extract_buy_event84b(sig)

        ix = str(ev["ix_name"])
        B = int(ev["pool_base_token_reserves"])
        R = (
            int(ev["pool_quote_token_reserves"])
            + int(ev["virtual_quote_reserves"])
        )
        y = int(ev["base_amount_out"])
        quote = int(ev["quote_amount_in"])
        lp_fee = int(ev["lp_fee"])
        quote_with_lp = int(ev["quote_amount_in_with_lp_fee"])

        result = {
            "signature": sig,
            "ix_name": ix,
            "quote_plus_lp_matches": (
                quote_with_lp == quote + lp_fee
            ),
        }

        if ix == "buy_exact_quote_in":
            x = int(ev["user_quote_amount_in"])

            # Test the discovered -1 lamport candidate versus no adjustment.
            pred_minus1 = B * (x - 1) // (R + x - 1)
            pred_zero = B * x // (R + x)

            result.update({
                "formula_test": "exact_quote_input",
                "actual_output_raw": y,
                "pred_minus1_raw": pred_minus1,
                "error_minus1_raw": pred_minus1 - y,
                "pred_zero_raw": pred_zero,
                "error_zero_raw": pred_zero - y,
                "quote_cost_error_raw": None,
            })

        elif ix == "buy":
            # Exact-base-output instruction: calculate the required curve quote.
            denom = B - y
            if denom <= 0:
                raise ValueError("Base output is not below pool base reserve.")

            numerator = R * y
            quote_ceil = (numerator + denom - 1) // denom
            quote_floor = numerator // denom

            result.update({
                "formula_test": "exact_base_output",
                "actual_output_raw": y,
                "pred_minus1_raw": None,
                "error_minus1_raw": None,
                "pred_zero_raw": None,
                "error_zero_raw": None,
                "quote_cost_floor_raw": quote_floor,
                "quote_cost_ceil_raw": quote_ceil,
                "quote_cost_error_raw": quote_ceil - quote,
            })

        else:
            result["formula_test"] = f"unclassified:{ix}"

        rows91.append(result)

    except Exception as exc:
        failures91.append({"signature": sig, "error": repr(exc)})

check91 = pd.DataFrame(rows91)

print(f"Sampled: {n91}; successfully decoded: {len(check91)}")
print(f"Fetch/decode failures: {len(failures91)}")

if not check91.empty:
    print("\nInstruction counts:")
    display(check91["formula_test"].value_counts().to_frame("transactions"))

    exact_in91 = check91[
        check91["formula_test"].eq("exact_quote_input")
    ]
    if not exact_in91.empty:
        print("\nExact-quote-input results:")
        display(pd.DataFrame([{
            "transactions": len(exact_in91),
            "minus1_exact_matches": int(
                (exact_in91["error_minus1_raw"] == 0).sum()
            ),
            "zero_offset_exact_matches": int(
                (exact_in91["error_zero_raw"] == 0).sum()
            ),
            "minus1_max_abs_error_raw": exact_in91[
                "error_minus1_raw"
            ].abs().max(),
        }]))

    exact_out91 = check91[
        check91["formula_test"].eq("exact_base_output")
    ]
    if not exact_out91.empty:
        print("\nExact-base-output results:")
        display(pd.DataFrame([{
            "transactions": len(exact_out91),
            "ceiling_exact_matches": int(
                (exact_out91["quote_cost_error_raw"] == 0).sum()
            ),
            "maximum_absolute_quote_error_raw": exact_out91[
                "quote_cost_error_raw"
            ].abs().max(),
        }]))

    print("\nLP-fee additivity:")
    print(
        int(check91["quote_plus_lp_matches"].sum()),
        "/",
        len(check91),
        "events satisfy quote_amount_in_with_lp_fee = quote_amount_in + lp_fee"
    )

    print("\nAny non-exact formula cases:")
    display(check91[
        (
            check91["formula_test"].eq("exact_quote_input")
            & check91["error_minus1_raw"].ne(0)
        ) | (
            check91["formula_test"].eq("exact_base_output")
            & check91["quote_cost_error_raw"].ne(0)
        )
    ].head(10))

if failures91:
    display(pd.DataFrame(failures91).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

v2_rows92 = check91[
    check91["formula_test"].eq(
        "unclassified:buy_exact_quote_in_v2"
    )
]

if v2_rows92.empty:
    print("No buy_exact_quote_in_v2 event was found in check91.")
else:
    rows92 = []

    for sig in v2_rows92["signature"].astype(str):
        ev = extract_buy_event84b(sig)

        B = int(ev["pool_base_token_reserves"])
        R = (
            int(ev["pool_quote_token_reserves"])
            + int(ev["virtual_quote_reserves"])
        )
        x = int(ev["user_quote_amount_in"])
        actual = int(ev["base_amount_out"])

        for offset in [-2, -1, 0, 1]:
            adjusted_x = x + offset
            predicted = (
                B * adjusted_x // (R + adjusted_x)
                if adjusted_x >= 0 and R + adjusted_x > 0
                else None
            )

            rows92.append({
                "signature": sig,
                "ix_name": ev["ix_name"],
                "input_offset_lamports": offset,
                "user_quote_amount_in_raw": x,
                "actual_output_raw": actual,
                "predicted_output_raw": predicted,
                "error_raw": (
                    predicted - actual if predicted is not None else None
                ),
                "lp_fee_raw": int(ev["lp_fee"]),
                "quote_amount_in_raw": int(ev["quote_amount_in"]),
                "quote_amount_in_with_lp_fee_raw": int(
                    ev["quote_amount_in_with_lp_fee"]
                ),
            })

    result92 = pd.DataFrame(rows92)
    display(result92)

In [ ]:
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")
if "extract_buy_event84b" not in globals():
    raise NameError("Run Step 84B first.")

# Exclude every transaction already used in the earlier tests.
used93 = set()

for name in [
    "formula_results80",
    "test88",
    "sample91",
    "check91",
]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and "signature" in obj.columns:
        used93.update(obj["signature"].dropna().astype(str))

pool93 = stratified_trades[
    stratified_trades["action"].astype(str).str.contains(
        "buy", case=False, na=False
    )
].drop_duplicates("signature").copy()

pool93 = pool93[
    ~pool93["signature"].astype(str).isin(used93)
]

n93 = min(100, len(pool93))
if n93 == 0:
    raise ValueError("No unseen buy transactions available.")

sample93 = pool93.sample(n=n93, random_state=93026)

rows93 = []
failures93 = []

for sig in sample93["signature"].astype(str):
    try:
        ev = extract_buy_event84b(sig)

        ix = str(ev["ix_name"])
        B = int(ev["pool_base_token_reserves"])
        R = (
            int(ev["pool_quote_token_reserves"])
            + int(ev["virtual_quote_reserves"])
        )
        actual = int(ev["base_amount_out"])

        row = {
            "signature": sig,
            "ix_name": ix,
            "actual_output_raw": actual,
            "error_minus1_raw": None,
            "error_zero_raw": None,
            "quote_cost_error_raw": None,
        }

        if ix in ("buy_exact_quote_in", "buy_exact_quote_in_v2"):
            x = int(ev["user_quote_amount_in"])

            for offset, col in [
                (-1, "error_minus1_raw"),
                (0, "error_zero_raw"),
            ]:
                adjusted = x + offset
                predicted = B * adjusted // (R + adjusted)
                row[col] = predicted - actual

            row["formula_test"] = "exact_quote_input"

        elif ix == "buy":
            denom = B - actual
            if denom <= 0:
                raise ValueError("Invalid base reserve/output combination.")

            numerator = R * actual
            quote_ceil = (numerator + denom - 1) // denom
            row["quote_amount_in_raw"] = int(ev["quote_amount_in"])
            row["quote_cost_ceil_raw"] = quote_ceil
            row["quote_cost_error_raw"] = (
                quote_ceil - int(ev["quote_amount_in"])
            )
            row["formula_test"] = "exact_base_output"

        else:
            row["formula_test"] = "other_instruction"

        rows93.append(row)

    except Exception as exc:
        failures93.append({"signature": sig, "error": repr(exc)})

check93 = pd.DataFrame(rows93)

print(f"Sampled: {n93}; decoded: {len(check93)}; failures: {len(failures93)}")

if not check93.empty:
    print("\nDecoded instruction counts:")
    display(
        check93.groupby(["ix_name", "formula_test"])
        .size()
        .reset_index(name="transactions")
    )

    exact_in93 = check93[
        check93["formula_test"].eq("exact_quote_input")
    ]
    if not exact_in93.empty:
        print("\nExact-quote-input formula validation:")
        display(
            exact_in93.groupby("ix_name").agg(
                transactions=("signature", "nunique"),
                minus1_exact_matches=(
                    "error_minus1_raw",
                    lambda s: int((s == 0).sum()),
                ),
                zero_offset_exact_matches=(
                    "error_zero_raw",
                    lambda s: int((s == 0).sum()),
                ),
                minus1_max_abs_error_raw=(
                    "error_minus1_raw",
                    lambda s: s.abs().max(),
                ),
            ).reset_index()
        )

    exact_out93 = check93[
        check93["formula_test"].eq("exact_base_output")
    ]
    if not exact_out93.empty:
        print("\nExact-base-output formula validation:")
        display(pd.DataFrame([{
            "transactions": exact_out93["signature"].nunique(),
            "ceiling_exact_matches": int(
                (exact_out93["quote_cost_error_raw"] == 0).sum()
            ),
            "maximum_absolute_quote_error_raw": (
                exact_out93["quote_cost_error_raw"].abs().max()
            ),
        }]))

if failures93:
    print("\nFirst decode/fetch failures:")
    display(pd.DataFrame(failures93).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

if "check93" not in globals():
    raise NameError("Run Step 93 first.")
if "extract_buy_event84b" not in globals():
    raise NameError("Run Step 84B first.")

v2_sigs94 = (
    check93.loc[
        check93["ix_name"].eq("buy_exact_quote_in_v2"),
        "signature"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if not v2_sigs94:
    print("No new v2 transaction found in check93.")
else:
    rows94 = []

    for sig in v2_sigs94:
        ev = extract_buy_event84b(sig)

        B = int(ev["pool_base_token_reserves"])
        R = (
            int(ev["pool_quote_token_reserves"])
            + int(ev["virtual_quote_reserves"])
        )
        x = int(ev["user_quote_amount_in"])
        actual = int(ev["base_amount_out"])

        for offset in [-1, 0]:
            adjusted_x = x + offset
            predicted = B * adjusted_x // (R + adjusted_x)

            rows94.append({
                "signature": sig,
                "ix_name": ev["ix_name"],
                "input_offset_lamports": offset,
                "actual_output_raw": actual,
                "predicted_output_raw": predicted,
                "error_raw": predicted - actual,
            })

    result94 = pd.DataFrame(rows94)
    display(result94)

In [ ]:
def pumpswap_exact_quote_buy_output_raw(
    ix_name,
    pool_base_reserves_raw,
    pool_quote_reserves_raw,
    virtual_quote_reserves_raw,
    user_quote_amount_in_raw,
):
    """
    Candidate base-token output formula for exact-quote-input buys.
    All arguments and the return value are raw integer units.
    """
    supported = {
        "buy_exact_quote_in",
        "buy_exact_quote_in_v2",
    }
    if ix_name not in supported:
        raise ValueError(f"Unsupported exact-quote-input instruction: {ix_name}")

    B = int(pool_base_reserves_raw)
    Q = int(pool_quote_reserves_raw) + int(virtual_quote_reserves_raw)
    x = int(user_quote_amount_in_raw) - 1

    if B <= 0 or Q <= 0 or x < 0:
        raise ValueError("Invalid reserves or adjusted quote input.")

    return (B * x) // (Q + x)


def pumpswap_exact_base_buy_quote_raw(
    ix_name,
    pool_base_reserves_raw,
    pool_quote_reserves_raw,
    virtual_quote_reserves_raw,
    base_amount_out_raw,
):
    """
    Candidate curve quote cost for an exact-base-output buy.
    Matches the event's quote_amount_in in the validated samples.
    """
    if ix_name != "buy":
        raise ValueError(
            f"Expected exact-base-output instruction 'buy', got {ix_name!r}"
        )

    B = int(pool_base_reserves_raw)
    Q = int(pool_quote_reserves_raw) + int(virtual_quote_reserves_raw)
    y = int(base_amount_out_raw)

    denominator = B - y
    if B <= 0 or Q <= 0 or y <= 0 or denominator <= 0:
        raise ValueError("Invalid reserves or requested base output.")

    numerator = Q * y
    return (numerator + denominator - 1) // denominator

In [ ]:
if "events84b" not in globals() or events84b.empty:
    raise NameError("Run Step 84B first to create events84b.")

regression95 = events84b.copy()

regression95["candidate_output_raw"] = regression95.apply(
    lambda r: pumpswap_exact_quote_buy_output_raw(
        "buy_exact_quote_in",
        r["pool_base_raw"],
        r["pool_quote_raw"],
        r["virtual_quote_raw"],
        r["user_quote_amount_in_raw"],
    ),
    axis=1,
)

regression95["error_raw"] = (
    regression95["candidate_output_raw"]
    - regression95["actual_event_raw"]
)

print("Original buy regression:")
print("Transactions:", len(regression95))
print("Exact matches:", int((regression95["error_raw"] == 0).sum()))
print("Maximum absolute error:", int(regression95["error_raw"].abs().max()))

In [ ]:
import base64
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")
if "fetch_transaction" not in globals():
    raise NameError("fetch_transaction is missing.")

# Exclude signatures used in earlier formula tests.
used96 = set()

for name in [
    "formula_results80",
    "test88",
    "sample91",
    "check91",
    "check93",
]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and "signature" in obj.columns:
        used96.update(obj["signature"].dropna().astype(str))

sell_pool96 = (
    stratified_trades[
        stratified_trades["action"].astype(str).str.contains(
            "sell", case=False, na=False
        )
    ]
    .drop_duplicates("signature")
    .copy()
)

sell_pool96 = sell_pool96[
    ~sell_pool96["signature"].astype(str).isin(used96)
]

n96 = min(40, len(sell_pool96))
print("Unseen sell candidates:", len(sell_pool96))

if n96 == 0:
    raise ValueError("No unseen sell candidates available.")

sample96 = sell_pool96.sample(n=n96, random_state=96026)

def extract_sell_event96(signature):
    tx = fetch_transaction(signature)

    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    logs = ((tx or {}).get("meta") or {}).get("logMessages") or []
    discriminator = event_discriminators["SellEvent"]

    for line in logs:
        marker = "Program data: "
        if marker not in line:
            continue

        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(),
                validate=True,
            )
        except Exception:
            continue

        if raw.startswith(discriminator):
            # decode_all_fields skips the 8-byte discriminator internally.
            return decode_all_fields(raw, event_fields["SellEvent"])

    raise ValueError("SellEvent not found in transaction logs.")

rows96, failures96 = [], []

for sig in sample96["signature"].astype(str):
    try:
        ev = extract_sell_event96(sig)

        B = int(ev["pool_base_token_reserves"])
        Q = (
            int(ev["pool_quote_token_reserves"])
            + int(ev["virtual_quote_reserves"])
        )
        x = int(ev["base_amount_in"])
        actual = int(ev["quote_amount_out"])

        predicted = (Q * x) // (B + x)

        rows96.append({
            "signature": sig,
            "ix_name": ev.get("ix_name"),
            "base_amount_in_raw": x,
            "actual_quote_out_raw": actual,
            "predicted_quote_out_raw": predicted,
            "error_raw": predicted - actual,
            "lp_fee_raw": int(ev["lp_fee"]),
            "quote_out_without_lp_fee_raw": int(
                ev["quote_amount_out_without_lp_fee"]
            ),
        })

    except Exception as exc:
        failures96.append({
            "signature": sig,
            "error": repr(exc),
        })

sell_check96 = pd.DataFrame(rows96)

print(
    f"Sampled: {n96}; successfully decoded: {len(sell_check96)}; "
    f"fetch/decode failures: {len(failures96)}"
)

if not sell_check96.empty:
    display(
        sell_check96.groupby("ix_name", dropna=False).agg(
            transactions=("signature", "nunique"),
            exact_matches=("error_raw", lambda s: int((s == 0).sum())),
            median_abs_error_raw=("error_raw", lambda s: s.abs().median()),
            max_abs_error_raw=("error_raw", lambda s: s.abs().max()),
        ).reset_index()
    )

    print("\nNon-exact cases:")
    display(sell_check96[sell_check96["error_raw"] != 0].head(10))

if failures96:
    print("\nFirst fetch/decode failures:")
    display(pd.DataFrame(failures96).head(10))

In [ ]:
def pumpswap_sell_quote_output_raw(
    pool_base_reserves_raw,
    pool_quote_reserves_raw,
    virtual_quote_reserves_raw,
    base_amount_in_raw,
):
    """
    Candidate PumpSwap gross quote output for a sell.

    All inputs and the result are integer raw units.
    The result corresponds to quote_amount_out before fee deductions.
    """
    B = int(pool_base_reserves_raw)
    Q = (
        int(pool_quote_reserves_raw)
        + int(virtual_quote_reserves_raw)
    )
    x = int(base_amount_in_raw)

    if B <= 0 or Q <= 0 or x <= 0:
        raise ValueError("Reserves and base input must be positive.")

    return (Q * x) // (B + x)

In [ ]:
# Simple synthetic example to verify integer arithmetic.
test_sell97 = pumpswap_sell_quote_output_raw(
    pool_base_reserves_raw=1_000_000,
    pool_quote_reserves_raw=100_000,
    virtual_quote_reserves_raw=10_000,
    base_amount_in_raw=10_000,
)

expected_sell97 = (110_000 * 10_000) // (1_000_000 + 10_000)

assert test_sell97 == expected_sell97
print("Sell helper arithmetic check passed:", test_sell97)

In [ ]:
import pandas as pd
from IPython.display import display

if "sell_check96" not in globals() or sell_check96.empty:
    raise NameError("Run Step 96 first; sell_check96 is missing or empty.")

fee98 = sell_check96.copy()

fee98["gross_minus_lp_fee_raw"] = (
    fee98["actual_quote_out_raw"] - fee98["lp_fee_raw"]
)

fee98["lp_fee_relation_error_raw"] = (
    fee98["gross_minus_lp_fee_raw"]
    - fee98["quote_out_without_lp_fee_raw"]
)

print("Decoded sell events:", len(fee98))
print(
    "Exact LP-fee relationship matches:",
    int((fee98["lp_fee_relation_error_raw"] == 0).sum()),
    "/",
    len(fee98),
)
print(
    "Maximum absolute difference (raw units):",
    int(fee98["lp_fee_relation_error_raw"].abs().max()),
)

print("\nNon-exact cases:")
display(
    fee98.loc[
        fee98["lp_fee_relation_error_raw"] != 0,
        [
            "signature",
            "actual_quote_out_raw",
            "lp_fee_raw",
            "quote_out_without_lp_fee_raw",
            "lp_fee_relation_error_raw",
        ],
    ].head(10)
)

print("\nExample rows:")
display(
    fee98[
        [
            "actual_quote_out_raw",
            "lp_fee_raw",
            "gross_minus_lp_fee_raw",
            "quote_out_without_lp_fee_raw",
            "lp_fee_relation_error_raw",
        ]
    ].head(5)
)

In [ ]:
import pandas as pd
from IPython.display import display

if "sell_check96" not in globals() or sell_check96.empty:
    raise NameError("Run Step 96 first; sell_check96 is missing or empty.")

if "extract_sell_event96" not in globals():
    raise NameError("Run Step 96 first to define extract_sell_event96.")

rows99 = []
failures99 = []

for sig in sell_check96["signature"].dropna().astype(str).unique():
    try:
        ev = extract_sell_event96(sig)

        gross = int(ev["quote_amount_out"])
        lp = int(ev["lp_fee"])
        protocol = int(ev["protocol_fee"])
        creator = int(ev["coin_creator_fee"])
        no_lp = int(ev["quote_amount_out_without_lp_fee"])
        user_net = int(ev["user_quote_amount_out"])

        cashback = int(ev.get("cashback", 0))
        buyback = int(ev.get("buyback_fee", 0))
        holder_rewards = int(ev.get("holder_rewards", 0))

        expected_core = gross - lp - protocol - creator

        rows99.append({
            "signature": sig,
            "gross_quote_out_raw": gross,
            "lp_fee_raw": lp,
            "protocol_fee_raw": protocol,
            "coin_creator_fee_raw": creator,
            "cashback_raw": cashback,
            "buyback_fee_raw": buyback,
            "holder_rewards_raw": holder_rewards,
            "quote_out_without_lp_fee_raw": no_lp,
            "user_quote_amount_out_raw": user_net,

            "gross_minus_lp_error_raw": gross - lp - no_lp,
            "core_net_error_raw": expected_core - user_net,
            "core_minus_cashback_error_raw": (
                expected_core - cashback - user_net
            ),
            "core_plus_cashback_error_raw": (
                expected_core + cashback - user_net
            ),
            "core_minus_buyback_error_raw": (
                expected_core - buyback - user_net
            ),
            "core_minus_holder_rewards_error_raw": (
                expected_core - holder_rewards - user_net
            ),
        })

    except Exception as exc:
        failures99.append({"signature": sig, "error": repr(exc)})

fee_audit99 = pd.DataFrame(rows99)

if not fee_audit99.empty:
    error_cols99 = [
        "gross_minus_lp_error_raw",
        "core_net_error_raw",
        "core_minus_cashback_error_raw",
        "core_plus_cashback_error_raw",
        "core_minus_buyback_error_raw",
        "core_minus_holder_rewards_error_raw",
    ]

    summary99 = pd.DataFrame([
        {
            "identity": col,
            "exact_matches": int((fee_audit99[col] == 0).sum()),
            "transactions": len(fee_audit99),
            "max_absolute_error_raw": int(fee_audit99[col].abs().max()),
        }
        for col in error_cols99
    ])

    print("=== Sell fee identity summary ===")
    display(summary99)

    print("\n=== Sample event fields ===")
    display(
        fee_audit99[
            [
                "gross_quote_out_raw",
                "lp_fee_raw",
                "protocol_fee_raw",
                "coin_creator_fee_raw",
                "cashback_raw",
                "buyback_fee_raw",
                "holder_rewards_raw",
                "quote_out_without_lp_fee_raw",
                "user_quote_amount_out_raw",
                "core_net_error_raw",
            ]
        ].head(10)
    )

if failures99:
    print("\nFetch/decode failures:")
    display(pd.DataFrame(failures99).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

if "fee_audit99" not in globals() or fee_audit99.empty:
    raise NameError("Run Step 99 first; fee_audit99 is missing or empty.")

df100 = fee_audit99.copy()

# Compare buyback against a possible 50% allocation of protocol fees.
df100["expected_buyback_floor_half"] = (
    df100["protocol_fee_raw"] // 2
)
df100["buyback_half_error_raw"] = (
    df100["buyback_fee_raw"] - df100["expected_buyback_floor_half"]
)

# Check whether holder rewards overlap with creator-fee accounting.
df100["holder_minus_creator_raw"] = (
    df100["holder_rewards_raw"] - df100["coin_creator_fee_raw"]
)

checks100 = [
    {
        "check": "Buyback fee equals floor(protocol fee / 2)",
        "matches": int((df100["buyback_half_error_raw"] == 0).sum()),
        "total": len(df100),
        "max_abs_difference_raw": int(
            df100["buyback_half_error_raw"].abs().max()
        ),
    },
    {
        "check": "Holder rewards equal creator fee",
        "matches": int((df100["holder_minus_creator_raw"] == 0).sum()),
        "total": len(df100),
        "max_abs_difference_raw": int(
            df100["holder_minus_creator_raw"].abs().max()
        ),
    },
    {
        "check": "Cashback is zero",
        "matches": int((df100["cashback_raw"] == 0).sum()),
        "total": len(df100),
        "max_abs_difference_raw": int(df100["cashback_raw"].max()),
    },
]

print("=== Fee allocation relationship checks ===")
display(pd.DataFrame(checks100))

print("\n=== Nonzero holder-reward cases ===")
display(
    df100.loc[
        df100["holder_rewards_raw"] != 0,
        [
            "signature",
            "protocol_fee_raw",
            "buyback_fee_raw",
            "coin_creator_fee_raw",
            "holder_rewards_raw",
            "user_quote_amount_out_raw",
            "core_net_error_raw",
        ],
    ]
)

print("\n=== Cases where buyback differs from half the protocol fee ===")
display(
    df100.loc[
        df100["buyback_half_error_raw"] != 0,
        [
            "signature",
            "protocol_fee_raw",
            "buyback_fee_raw",
            "expected_buyback_floor_half",
            "buyback_half_error_raw",
        ],
    ].head(10)
)

In [ ]:
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")
if "extract_sell_event96" not in globals():
    raise NameError("Run Step 96 first.")

# Exclude signatures already used in prior formula and fee audits.
used101 = set()

for name in [
    "formula_results80",
    "test88",
    "sample91",
    "check91",
    "check93",
    "sell_check96",
    "fee_audit99",
]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and "signature" in obj.columns:
        used101.update(obj["signature"].dropna().astype(str))

sell_pool101 = (
    stratified_trades[
        stratified_trades["action"].astype(str).str.contains(
            "sell", case=False, na=False
        )
    ]
    .drop_duplicates("signature")
    .copy()
)

sell_pool101 = sell_pool101[
    ~sell_pool101["signature"].astype(str).isin(used101)
]

n101 = min(100, len(sell_pool101))
print("Available unseen sells:", len(sell_pool101))

if n101 == 0:
    raise ValueError("No unseen sell transactions available.")

sample101 = sell_pool101.sample(n=n101, random_state=101026)

rows101 = []
failures101 = []

for sig in sample101["signature"].astype(str):
    try:
        ev = extract_sell_event96(sig)

        gross = int(ev["quote_amount_out"])
        lp = int(ev["lp_fee"])
        protocol = int(ev["protocol_fee"])
        creator = int(ev["coin_creator_fee"])
        no_lp = int(ev["quote_amount_out_without_lp_fee"])
        user_net = int(ev["user_quote_amount_out"])

        cashback = int(ev.get("cashback", 0))
        buyback = int(ev.get("buyback_fee", 0))
        holder_rewards = int(ev.get("holder_rewards", 0))

        rows101.append({
            "signature": sig,
            "gross": gross,
            "lp_fee": lp,
            "protocol_fee": protocol,
            "creator_fee": creator,
            "cashback": cashback,
            "buyback_fee": buyback,
            "holder_rewards": holder_rewards,
            "no_lp": no_lp,
            "user_net": user_net,
            "lp_relation_error": gross - lp - no_lp,
            "trader_net_error": gross - lp - protocol - creator - user_net,
            "buyback_half_error": buyback - protocol // 2,
            "holder_equals_creator": holder_rewards == creator,
        })

    except Exception as exc:
        failures101.append({"signature": sig, "error": repr(exc)})

audit101 = pd.DataFrame(rows101)

print(
    f"Sampled: {n101}; decoded: {len(audit101)}; "
    f"fetch/decode failures: {len(failures101)}"
)

if not audit101.empty:
    print("\n=== Fee identity validation ===")
    display(pd.DataFrame([
        {
            "check": "Gross minus LP fee equals quote_without_lp_fee",
            "matches": int((audit101["lp_relation_error"] == 0).sum()),
            "transactions": len(audit101),
            "max_abs_error_raw": int(audit101["lp_relation_error"].abs().max()),
        },
        {
            "check": "Trader proceeds equal gross minus three core fees",
            "matches": int((audit101["trader_net_error"] == 0).sum()),
            "transactions": len(audit101),
            "max_abs_error_raw": int(audit101["trader_net_error"].abs().max()),
        },
        {
            "check": "Buyback equals floor(protocol fee / 2)",
            "matches": int((audit101["buyback_half_error"] == 0).sum()),
            "transactions": len(audit101),
            "max_abs_error_raw": int(audit101["buyback_half_error"].abs().max()),
        },
    ]))

    print("\n=== Allocation field prevalence ===")
    display(pd.DataFrame([
        {
            "field": "cashback",
            "nonzero_transactions": int((audit101["cashback"] != 0).sum()),
            "transactions": len(audit101),
        },
        {
            "field": "buyback_fee",
            "nonzero_transactions": int((audit101["buyback_fee"] != 0).sum()),
            "transactions": len(audit101),
        },
        {
            "field": "holder_rewards",
            "nonzero_transactions": int((audit101["holder_rewards"] != 0).sum()),
            "transactions": len(audit101),
        },
        {
            "field": "holder_rewards equals creator fee",
            "nonzero_transactions": int(
                (
                    (audit101["holder_rewards"] != 0)
                    & audit101["holder_equals_creator"]
                ).sum()
            ),
            "transactions": len(audit101),
        },
    ]))

    print("\n=== Sells with nonzero cashback or holder rewards ===")
    display(
        audit101.loc[
            (audit101["cashback"] != 0)
            | (audit101["holder_rewards"] != 0),
            [
                "signature", "protocol_fee", "creator_fee",
                "cashback", "buyback_fee", "holder_rewards",
                "user_net", "trader_net_error",
            ],
        ].head(20)
    )

if failures101:
    print("\nFirst fetch/decode failures:")
    display(pd.DataFrame(failures101).head(10))

In [ ]:
import base64
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")

# Exclude signatures already used in previous buy-formula tests.
used102 = set()

for name in [
    "formula_results80", "test88", "sample88",
    "buy84", "events84b", "sample91", "check91",
    "sample93", "check93",
]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and "signature" in obj.columns:
        used102.update(obj["signature"].dropna().astype(str))

buy_pool102 = (
    stratified_trades[
        stratified_trades["action"].astype(str).str.contains(
            "buy", case=False, na=False
        )
    ]
    .drop_duplicates("signature")
    .copy()
)

buy_pool102 = buy_pool102[
    ~buy_pool102["signature"].astype(str).isin(used102)
]

n102 = min(100, len(buy_pool102))
print("Unseen buy candidates:", len(buy_pool102))

if n102 == 0:
    raise ValueError("No unseen buy transactions available.")

sample102 = buy_pool102.sample(n=n102, random_state=102026)


def extract_buy_event102(signature):
    tx = fetch_transaction(signature)

    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    logs = ((tx or {}).get("meta") or {}).get("logMessages") or []
    discriminator = event_discriminators["BuyEvent"]

    for line in logs:
        marker = "Program data: "
        if marker not in line:
            continue

        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(),
                validate=True,
            )
        except Exception:
            continue

        if raw.startswith(discriminator):
            # The existing decoder already skips the 8-byte discriminator.
            return decode_all_fields(raw, event_fields["BuyEvent"])

    raise ValueError("BuyEvent not found in transaction logs.")


rows102, failures102 = [], []

for sig in sample102["signature"].astype(str):
    try:
        ev = extract_buy_event102(sig)

        user_input = int(ev["user_quote_amount_in"])
        lp_fee = int(ev["lp_fee"])
        protocol_fee = int(ev["protocol_fee"])
        creator_fee = int(ev["coin_creator_fee"])
        cashback = int(ev["cashback"])
        buyback = int(ev["buyback_fee"])

        quote_with_lp = int(ev["quote_amount_in_with_lp_fee"])
        total_quote_in = int(ev["quote_amount_in"])
        max_quote_in = int(ev["max_quote_amount_in"])

        rows102.append({
            "signature": sig,
            "ix_name": ev["ix_name"],
            "user_quote_input_raw": user_input,
            "lp_fee_raw": lp_fee,
            "protocol_fee_raw": protocol_fee,
            "creator_fee_raw": creator_fee,
            "cashback_raw": cashback,
            "buyback_fee_raw": buyback,
            "quote_with_lp_raw": quote_with_lp,
            "quote_amount_in_raw": total_quote_in,
            "max_quote_amount_in_raw": max_quote_in,

            # Check event-field accounting identities.
            "lp_identity_error_raw": (
                quote_with_lp - user_input - lp_fee
            ),
            "total_input_identity_error_raw": (
                total_quote_in
                - quote_with_lp
                - protocol_fee
                - creator_fee
            ),
            "buyback_half_error_raw": buyback - protocol_fee // 2,
            "within_max_quote": total_quote_in <= max_quote_in,
        })

    except Exception as exc:
        failures102.append({"signature": sig, "error": repr(exc)})

buy_fee_audit102 = pd.DataFrame(rows102)

print(
    f"Sampled: {n102}; decoded: {len(buy_fee_audit102)}; "
    f"fetch/decode failures: {len(failures102)}"
)

if not buy_fee_audit102.empty:
    error_cols102 = [
        ("LP fee identity", "lp_identity_error_raw"),
        ("Total quote input identity", "total_input_identity_error_raw"),
        ("Buyback equals floor(protocol fee / 2)", "buyback_half_error_raw"),
    ]

    summary102 = []
    for label, col in error_cols102:
        summary102.append({
            "check": label,
            "exact_matches": int((buy_fee_audit102[col] == 0).sum()),
            "transactions": len(buy_fee_audit102),
            "max_abs_error_raw": int(buy_fee_audit102[col].abs().max()),
        })

    summary102.append({
        "check": "Total quote input does not exceed max",
        "exact_matches": int(buy_fee_audit102["within_max_quote"].sum()),
        "transactions": len(buy_fee_audit102),
        "max_abs_error_raw": None,
    })

    print("\n=== Buy fee identity checks ===")
    display(pd.DataFrame(summary102))

    print("\n=== Results by instruction type ===")
    display(
        buy_fee_audit102.groupby("ix_name", dropna=False).agg(
            transactions=("signature", "nunique"),
            lp_identity_matches=(
                "lp_identity_error_raw",
                lambda s: int((s == 0).sum()),
            ),
            total_input_matches=(
                "total_input_identity_error_raw",
                lambda s: int((s == 0).sum()),
            ),
            buyback_half_matches=(
                "buyback_half_error_raw",
                lambda s: int((s == 0).sum()),
            ),
            nonzero_cashback=("cashback_raw", lambda s: int((s != 0).sum())),
        ).reset_index()
    )

    print("\n=== Any fee-identity mismatches ===")
    display(
        buy_fee_audit102.loc[
            (buy_fee_audit102["lp_identity_error_raw"] != 0)
            | (buy_fee_audit102["total_input_identity_error_raw"] != 0)
            | (buy_fee_audit102["buyback_half_error_raw"] != 0)
            | (~buy_fee_audit102["within_max_quote"]),
        [
            "signature", "ix_name", "user_quote_input_raw",
            "lp_fee_raw", "protocol_fee_raw", "creator_fee_raw",
            "cashback_raw", "buyback_fee_raw", "quote_with_lp_raw",
            "quote_amount_in_raw", "max_quote_amount_in_raw",
            "lp_identity_error_raw", "total_input_identity_error_raw",
            "buyback_half_error_raw", "within_max_quote",
        ],
    ].head(20))

if failures102:
    print("\nFirst fetch/decode failures:")
    display(pd.DataFrame(failures102).head(10))

In [ ]:
import pandas as pd
from IPython.display import display

if "buy_fee_audit102" not in globals() or buy_fee_audit102.empty:
    raise NameError("Run Step 102 first.")

df103 = buy_fee_audit102.copy()

exact_quote_types103 = {
    "buy_exact_quote_in",
    "buy_exact_quote_in_v2",
}

def calculate_checks103(row):
    ix = row["ix_name"]
    quote_with_lp = int(row["quote_with_lp_raw"])
    quote_amount = int(row["quote_amount_in_raw"])
    user_input = int(row["user_quote_input_raw"])
    lp = int(row["lp_fee_raw"])
    protocol = int(row["protocol_fee_raw"])
    creator = int(row["creator_fee_raw"])
    max_quote = int(row["max_quote_amount_in_raw"])

    if ix in exact_quote_types103:
        # Exact quote input: input amount is the basis for LP fee.
        lp_error = quote_with_lp - user_input - lp

        # Event total quote input includes all three fee components.
        total_cost = quote_with_lp + protocol + creator
        total_cost_error = quote_amount - total_cost

        # Compare actual event total input with the maximum.
        budget_ok = quote_amount <= max_quote

    elif ix == "buy":
        # Exact base output: quote_amount is the curve quote cost.
        lp_error = quote_with_lp - quote_amount - lp

        total_cost = quote_with_lp + protocol + creator
        total_cost_error = None  # No separate total-cost field in this event.

        # The maximum is a limit, not the actual curve quote cost.
        budget_ok = total_cost <= max_quote

    else:
        lp_error = None
        total_cost = None
        total_cost_error = None
        budget_ok = None

    return pd.Series({
        "lp_identity_error_corrected": lp_error,
        "implied_total_quote_cost_raw": total_cost,
        "total_cost_field_error_raw": total_cost_error,
        "within_max_quote_corrected": budget_ok,
    })

df103 = pd.concat([df103, df103.apply(calculate_checks103, axis=1)], axis=1)

print("=== Corrected checks by instruction ===")
display(
    df103.groupby("ix_name", dropna=False).agg(
        transactions=("signature", "nunique"),
        lp_identity_matches=(
            "lp_identity_error_corrected",
            lambda s: int((s == 0).sum()),
        ),
        lp_identity_max_abs_error=(
            "lp_identity_error_corrected",
            lambda s: s.abs().max(),
        ),
        total_cost_matches=(
            "total_cost_field_error_raw",
            lambda s: int((s.dropna() == 0).sum()),
        ),
        total_cost_checked=(
            "total_cost_field_error_raw",
            lambda s: int(s.notna().sum()),
        ),
        within_max=("within_max_quote_corrected", lambda s: int((s == True).sum())),
    ).reset_index()
)

print("\n=== Any corrected-check mismatches ===")
display(
    df103.loc[
        (df103["lp_identity_error_corrected"].fillna(0) != 0)
        | (df103["total_cost_field_error_raw"].fillna(0) != 0)
        | (df103["within_max_quote_corrected"] != True),
        [
            "signature",
            "ix_name",
            "user_quote_input_raw",
            "quote_amount_in_raw",
            "quote_with_lp_raw",
            "lp_fee_raw",
            "protocol_fee_raw",
            "creator_fee_raw",
            "max_quote_amount_in_raw",
            "lp_identity_error_corrected",
            "implied_total_quote_cost_raw",
            "total_cost_field_error_raw",
            "within_max_quote_corrected",
        ],
    ].head(20)
)

In [ ]:
import pandas as pd
from IPython.display import display

checks104 = []

def add_zero_check104(name, values):
    """Record a test where every non-null value should equal integer zero."""
    s = pd.Series(values).dropna()
    matches = int((s == 0).sum())

    checks104.append({
        "check": name,
        "matches": matches,
        "tested": len(s),
        "max_abs_error_raw": (
            float(s.abs().max()) if len(s) else None
        ),
        "status": "PASS" if len(s) > 0 and matches == len(s) else "REVIEW",
    })

def add_true_check104(name, values):
    """Record a test where every non-null value should be True."""
    s = pd.Series(values).dropna().astype(bool)
    matches = int(s.sum())

    checks104.append({
        "check": name,
        "matches": matches,
        "tested": len(s),
        "max_abs_error_raw": None,
        "status": "PASS" if len(s) > 0 and matches == len(s) else "REVIEW",
    })


# 1. Exact-quote-input buy formula: original 25-event regression.
add_zero_check104(
    "Buy exact-quote formula — original regression",
    regression95["error_raw"],
)

# 2. Independent exact-quote-input samples.
add_zero_check104(
    "Buy exact-quote formula — Step 91",
    check91.loc[
        check91["formula_test"].eq("exact_quote_input"),
        "error_minus1_raw",
    ],
)

add_zero_check104(
    "Buy exact-quote formula — Step 93",
    check93.loc[
        check93["formula_test"].eq("exact_quote_input"),
        "error_minus1_raw",
    ],
)

# 3. Exact-base-output quote-cost calculation.
add_zero_check104(
    "Exact-base quote cost — Step 90 outliers",
    result90["ceil_minus_quote_amount_in_raw"],
)

add_zero_check104(
    "Exact-base quote cost — Step 91",
    check91.loc[
        check91["formula_test"].eq("exact_base_output"),
        "quote_cost_error_raw",
    ],
)

add_zero_check104(
    "Exact-base quote cost — Step 93",
    check93.loc[
        check93["formula_test"].eq("exact_base_output"),
        "quote_cost_error_raw",
    ],
)

# 4. v2 exact-quote variant.
add_zero_check104(
    "buy_exact_quote_in_v2 — Step 92",
    result92.loc[
        result92["input_offset_lamports"].eq(-1), "error_raw"
    ],
)

add_zero_check104(
    "buy_exact_quote_in_v2 — Step 94",
    result94.loc[
        result94["input_offset_lamports"].eq(-1), "error_raw"
    ],
)

# 5. Sell gross curve output.
sell80 = formula_results80[
    formula_results80["model"].eq("sell_constant_product_gross")
]

add_zero_check104(
    "Sell gross formula — Step 80",
    sell80["predicted_output_raw"] - sell80["actual_output_raw"],
)

add_zero_check104(
    "Sell gross formula — fresh Step 96 sample",
    sell_check96["error_raw"],
)

# 6. Buy fee identities.
add_zero_check104(
    "Buy LP-fee identity",
    df103["lp_identity_error_corrected"],
)

add_zero_check104(
    "Buy total quote-input identity where applicable",
    df103["total_cost_field_error_raw"],
)

add_zero_check104(
    "Buyback allocation — fresh buy sample",
    df103["buyback_half_error_raw"],
)

add_true_check104(
    "Buy maximum quote constraint",
    df103["within_max_quote_corrected"],
)

# 7. Sell fee identities.
add_zero_check104(
    "Sell gross minus LP fee identity — Step 99",
    fee_audit99["gross_minus_lp_error_raw"],
)

add_zero_check104(
    "Sell trader proceeds identity — Step 99",
    fee_audit99["core_net_error_raw"],
)

add_zero_check104(
    "Sell LP-fee identity — Step 101",
    audit101["lp_relation_error"],
)

add_zero_check104(
    "Sell trader proceeds identity — Step 101",
    audit101["trader_net_error"],
)

add_zero_check104(
    "Buyback allocation — fresh sell sample",
    audit101["buyback_half_error"],
)

summary104 = pd.DataFrame(checks104)

print("=== Consolidated regression checklist ===")
display(summary104)

print("\nOverall result:", (
    "PASS"
    if summary104["status"].eq("PASS").all()
    else "REVIEW — inspect any failing checks above"
))

In [ ]:
def pumpswap_event_calculator(event_name, ev):
    """
    Calculate candidate curve values and audit fee identities from
    a decoded PumpSwap BuyEvent or SellEvent.

    All amounts use raw integer units.
    This function does not mutate the pool or wallet state.
    """

    def integer(name):
        if name not in ev or ev[name] is None:
            raise ValueError(f"Missing event field: {name}")
        return int(ev[name])

    def require_positive(name, value):
        if value <= 0:
            raise ValueError(f"{name} must be positive, got {value}")

    result = {
        "event_name": event_name,
        "ix_name": ev.get("ix_name"),
    }

    if event_name == "BuyEvent":
        ix = str(ev["ix_name"])
        B = integer("pool_base_token_reserves")
        Q = (
            integer("pool_quote_token_reserves")
            + integer("virtual_quote_reserves")
        )

        require_positive("effective base reserves", B)
        require_positive("effective quote reserves", Q)

        lp = integer("lp_fee")
        protocol = integer("protocol_fee")
        creator = integer("coin_creator_fee")
        buyback = integer("buyback_fee")

        quote_in = integer("quote_amount_in")
        quote_with_lp = integer("quote_amount_in_with_lp_fee")
        max_quote = integer("max_quote_amount_in")

        if ix in ("buy_exact_quote_in", "buy_exact_quote_in_v2"):
            user_input = integer("user_quote_amount_in")
            adjusted_input = user_input - 1

            if adjusted_input < 0:
                raise ValueError("Adjusted quote input cannot be negative.")

            predicted = (
                B * adjusted_input // (Q + adjusted_input)
            )
            actual = integer("base_amount_out")

            result.update({
                "curve_quantity": "base_output_raw",
                "predicted_curve_raw": predicted,
                "actual_curve_raw": actual,
                "curve_error_raw": predicted - actual,
                "lp_identity_error_raw": (
                    quote_with_lp - user_input - lp
                ),
                "total_quote_identity_error_raw": (
                    quote_in - quote_with_lp - protocol - creator
                ),
                "maximum_quote_check": quote_in <= max_quote,
            })

        elif ix == "buy":
            # Exact-base-output instruction.
            base_out = integer("base_amount_out")
            denominator = B - base_out

            if base_out <= 0 or denominator <= 0:
                raise ValueError(
                    "Base output must be positive and below base reserves."
                )

            numerator = Q * base_out
            predicted = (
                numerator + denominator - 1
            ) // denominator

            total_cost = quote_with_lp + protocol + creator

            result.update({
                "curve_quantity": "quote_cost_raw",
                "predicted_curve_raw": predicted,
                "actual_curve_raw": quote_in,
                "curve_error_raw": predicted - quote_in,
                "lp_identity_error_raw": (
                    quote_with_lp - quote_in - lp
                ),
                "implied_total_quote_cost_raw": total_cost,
                "maximum_quote_check": total_cost <= max_quote,
            })

        else:
            raise ValueError(f"Unsupported buy instruction: {ix}")

        result["buyback_allocation_error_raw"] = (
            buyback - protocol // 2
        )

        # Preserve allocation fields without treating them as
        # additional trader deductions.
        result["cashback_raw"] = int(ev.get("cashback", 0))
        result["holder_rewards_raw"] = int(ev.get("holder_rewards", 0))

    elif event_name == "SellEvent":
        B = integer("pool_base_token_reserves")
        Q = (
            integer("pool_quote_token_reserves")
            + integer("virtual_quote_reserves")
        )
        base_in = integer("base_amount_in")

        require_positive("effective base reserves", B)
        require_positive("effective quote reserves", Q)
        require_positive("base input", base_in)

        gross_predicted = Q * base_in // (B + base_in)
        gross_actual = integer("quote_amount_out")

        lp = integer("lp_fee")
        protocol = integer("protocol_fee")
        creator = integer("coin_creator_fee")
        buyback = integer("buyback_fee")

        quote_without_lp = integer("quote_amount_out_without_lp_fee")
        user_net = integer("user_quote_amount_out")

        predicted_net = gross_predicted - lp - protocol - creator

        result.update({
            "curve_quantity": "gross_quote_output_raw",
            "predicted_curve_raw": gross_predicted,
            "actual_curve_raw": gross_actual,
            "curve_error_raw": gross_predicted - gross_actual,
            "predicted_trader_net_raw": predicted_net,
            "actual_trader_net_raw": user_net,
            "trader_net_error_raw": predicted_net - user_net,
            "lp_identity_error_raw": (
                gross_actual - lp - quote_without_lp
            ),
            "buyback_allocation_error_raw": (
                buyback - protocol // 2
            ),
            "cashback_raw": int(ev.get("cashback", 0)),
            "holder_rewards_raw": int(ev.get("holder_rewards", 0)),
        })

    else:
        raise ValueError(f"Unsupported event type: {event_name}")

    return result

In [ ]:
import pandas as pd
from IPython.display import display

# Use existing transactions from earlier validation steps.
if "events84b" not in globals() or events84b.empty:
    raise NameError("Run Step 84B first: events84b is missing.")

if "sell_check96" not in globals() or sell_check96.empty:
    raise NameError("Run Step 96 first: sell_check96 is missing.")

# Decode one real BuyEvent and one real SellEvent.
buy_sig106 = str(events84b.iloc[0]["signature"])
sell_sig106 = str(sell_check96.iloc[0]["signature"])

buy_event106 = extract_buy_event84b(buy_sig106)
sell_event106 = extract_sell_event96(sell_sig106)

# Run the unified calculator.
buy_result106 = pumpswap_event_calculator("BuyEvent", buy_event106)
sell_result106 = pumpswap_event_calculator("SellEvent", sell_event106)

results106 = pd.DataFrame([
    {"sample": "Buy", "signature": buy_sig106, **buy_result106},
    {"sample": "Sell", "signature": sell_sig106, **sell_result106},
])

display(results106.T)

# Fail loudly if the primary curve/proceeds checks do not match.
assert buy_result106["curve_error_raw"] == 0, "Buy curve calculation mismatch"
assert buy_result106["lp_identity_error_raw"] == 0, "Buy LP-fee identity mismatch"
assert buy_result106["maximum_quote_check"], "Buy maximum quote constraint failed"

assert sell_result106["curve_error_raw"] == 0, "Sell curve calculation mismatch"
assert sell_result106["trader_net_error_raw"] == 0, "Sell trader proceeds mismatch"
assert sell_result106["lp_identity_error_raw"] == 0, "Sell LP-fee identity mismatch"

print("PASS: real buy and sell smoke tests")

In [ ]:
import pandas as pd
from IPython.display import display

if "pumpswap_event_calculator" not in globals():
    raise NameError("Run Step 105 first.")
if "events84b" not in globals() or "sell_check96" not in globals():
    raise NameError("Run Steps 84B and 96 first.")

# Build a regression set from previously tested signatures.
# Keep the exact-base-output and v2 variants represented too.
cases107 = {}

for sig in events84b["signature"].dropna().astype(str):
    cases107[sig] = "BuyEvent"

if "check91" in globals():
    extra = check91[
        check91["formula_test"].eq("exact_base_output")
    ]
    for sig in extra["signature"].dropna().astype(str):
        cases107[sig] = "BuyEvent"

if "check93" in globals():
    extra = check93[
        check93["ix_name"].isin(["buy", "buy_exact_quote_in_v2"])
    ]
    for sig in extra["signature"].dropna().astype(str):
        cases107[sig] = "BuyEvent"

for sig in sell_check96["signature"].dropna().astype(str):
    cases107[sig] = "SellEvent"

print("Unique transactions queued:", len(cases107))

rows107 = []
failures107 = []

for sig, event_name in cases107.items():
    try:
        if event_name == "BuyEvent":
            ev = extract_buy_event84b(sig)
        else:
            ev = extract_sell_event96(sig)

        result = pumpswap_event_calculator(event_name, ev)

        checks = [
            result["curve_error_raw"] == 0,
            result["lp_identity_error_raw"] == 0,
            result["buyback_allocation_error_raw"] == 0,
        ]

        if event_name == "BuyEvent":
            checks.append(result["maximum_quote_check"] is True)

            # Exact-quote-input variants also expose this identity.
            if "total_quote_identity_error_raw" in result:
                checks.append(
                    result["total_quote_identity_error_raw"] == 0
                )
        else:
            checks.append(result["trader_net_error_raw"] == 0)

        rows107.append({
            "signature": sig,
            "event_name": event_name,
            "ix_name": result["ix_name"],
            "curve_error_raw": result["curve_error_raw"],
            "lp_identity_error_raw": result["lp_identity_error_raw"],
            "buyback_allocation_error_raw":
                result["buyback_allocation_error_raw"],
            "trader_net_error_raw": result.get("trader_net_error_raw"),
            "total_quote_identity_error_raw":
                result.get("total_quote_identity_error_raw"),
            "maximum_quote_check": result.get("maximum_quote_check"),
            "pass": all(checks),
        })

    except Exception as exc:
        failures107.append({
            "signature": sig,
            "event_name": event_name,
            "error": repr(exc),
        })

batch107 = pd.DataFrame(rows107)

print(f"Successfully tested: {len(batch107)} / {len(cases107)}")
print(f"Fetch/decode/calculation failures: {len(failures107)}")

if not batch107.empty:
    print("\n=== Results by instruction ===")
    display(
        batch107.groupby(["event_name", "ix_name"], dropna=False).agg(
            transactions=("signature", "nunique"),
            passing=("pass", "sum"),
            max_abs_curve_error=("curve_error_raw",
                                 lambda s: s.abs().max()),
            max_abs_lp_error=("lp_identity_error_raw",
                              lambda s: s.abs().max()),
        ).reset_index()
    )

    print("\nOverall passing:", int(batch107["pass"].sum()), "/", len(batch107))

    failed_checks107 = batch107[~batch107["pass"]]
    if not failed_checks107.empty:
        print("\nTransactions requiring investigation:")
        display(failed_checks107.head(20))

if failures107:
    print("\nFirst fetch/decode/calculation failures:")
    display(pd.DataFrame(failures107).head(10))

In [ ]:
import time
import pandas as pd
from IPython.display import display

if "failures107" not in globals():
    raise NameError("Run Step 107 first; failures107 is missing.")

retry_candidates108 = pd.DataFrame(failures107)
retry_candidates108 = retry_candidates108[
    retry_candidates108["event_name"].eq("SellEvent")
].copy()

rows108 = []
remaining108 = []

for _, item in retry_candidates108.iterrows():
    sig = str(item["signature"])
    last_error = None
    recovered = False

    # Exponential backoff: wait 2, 4, then 8 seconds before retries.
    for attempt in range(1, 5):
        try:
            ev = extract_sell_event96(sig)
            result = pumpswap_event_calculator("SellEvent", ev)

            checks = {
                "curve_match": result["curve_error_raw"] == 0,
                "lp_fee_match": result["lp_identity_error_raw"] == 0,
                "trader_net_match": result["trader_net_error_raw"] == 0,
                "buyback_allocation_match":
                    result["buyback_allocation_error_raw"] == 0,
            }

            rows108.append({
                "signature": sig,
                "attempt": attempt,
                **checks,
                "pass": all(checks.values()),
                "curve_error_raw": result["curve_error_raw"],
                "trader_net_error_raw": result["trader_net_error_raw"],
            })
            recovered = True
            break

        except Exception as exc:
            last_error = repr(exc)
            if attempt < 4:
                time.sleep(2 ** attempt)

    if not recovered:
        remaining108.append({
            "signature": sig,
            "error": last_error,
        })

retry_results108 = pd.DataFrame(rows108)

print("Successfully recovered:", len(retry_results108))
if not retry_results108.empty:
    display(retry_results108)

print("Still unavailable:", len(remaining108))
if remaining108:
    display(pd.DataFrame(remaining108))

print(
    "Retry result:",
    "PASS" if (
        not remaining108
        and not retry_results108.empty
        and retry_results108["pass"].all()
    ) else "INCOMPLETE — inspect remaining failures"
)

In [ ]:
import inspect
import re

keywords109 = re.compile(
    r"sim|env|step|execute|transition|action|buy|sell|swap|pool|reward|trade",
    re.IGNORECASE,
)

found109 = []

for name, obj in list(globals().items()):
    if not inspect.isfunction(obj):
        continue

    if keywords109.search(name):
        try:
            signature = str(inspect.signature(obj))
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        try:
            source = inspect.getsource(obj)
            first_line = next(
                (line.strip() for line in source.splitlines()
                 if line.strip() and not line.strip().startswith("@")),
                "(source unavailable)",
            )
        except (OSError, TypeError):
            first_line = "(source unavailable)"

        found109.append({
            "function": name,
            "signature": signature,
            "definition": first_line[:180],
        })

if found109:
    import pandas as pd
    display(
        pd.DataFrame(found109)
        .sort_values("function")
        .reset_index(drop=True)
    )
else:
    print("No matching functions found. Search the notebook for the environment class or step() method.")

In [ ]:
import inspect

targets110 = [
    "cp_buy_output",
    "cp_buy_raw",
    "cp_sell_output",
    "cp_sell_raw",
    "simulate_round_trip",
    "pumpswap_event_calculator",
]

for name in targets110:
    fn = globals().get(name)

    print("\n" + "=" * 80)
    print(f"FUNCTION: {name}")

    if fn is None:
        print("Not found.")
        continue

    try:
        print(inspect.getsource(fn))
    except (OSError, TypeError) as exc:
        print(f"Source unavailable: {exc}")

# Also search existing classes for an environment transition method.
print("\n" + "=" * 80)
print("POSSIBLE ENVIRONMENT STEP METHODS")

found110 = False

for name, obj in list(globals().items()):
    if not inspect.isclass(obj):
        continue

    for method_name in [
        "step",
        "execute_action",
        "transition",
        "buy",
        "sell",
    ]:
        method = getattr(obj, method_name, None)

        if callable(method):
            try:
                source = inspect.getsource(method)
            except (OSError, TypeError):
                continue

            print(f"\nCLASS: {name}.{method_name}")
            print(source)
            found110 = True

if not found110:
    print("No accessible matching class methods found.")

In [ ]:
import inspect

print("=" * 70)
print("FEE CALCULATOR")
print("=" * 70)

if "calculate_fee_components" not in globals():
    raise NameError("calculate_fee_components() not found.")

print(inspect.getsource(calculate_fee_components))

print("\n" + "=" * 70)
print("FEE RATE CONFIGURATION")
print("=" * 70)
print(repr(fee_rates))

print("\n" + "=" * 70)
print("SAMPLE FEE BREAKDOWN")
print("=" * 70)

sample_input111 = 1_000_000  # 0.001 SOL in lamports

for side in ["buy", "sell"]:
    fees = calculate_fee_components(
        sample_input111,
        fee_rates[side],
    )

    print(f"\n{side.upper()} fees on {sample_input111:,} raw units:")
    print(fees)
    print("Sum of returned components:", sum(fees.values()))
    print("Returned component names:", list(fees.keys()))

print("\n" + "=" * 70)
print("UNIT CONFIGURATION")
print("=" * 70)
print("LAMPORTS_PER_SOL:", globals().get("LAMPORTS_PER_SOL"))
print("token_scale:", globals().get("token_scale"))

In [ ]:
def simulate_round_trip_v2(start_quote, start_base, net_buy_input):
    """
    Instruction-aware candidate round-trip simulator.

    Assumptions validated by the current transaction audit:
      - Exact-quote-input buy output uses input - 1 lamport.
      - Buy LP fee is retained in the effective quote reserves.
      - Protocol and creator fees are included in trader cost
        but excluded from effective pool reserves.
      - Sell gross output uses integer constant-product arithmetic.
      - Sell trader proceeds deduct LP, protocol, and creator fees.
      - Buyback is not charged as an additional fee.

    This function does not modify the original simulator.
    """
    start_quote = int(start_quote)
    start_base = int(start_base)
    net_buy_input = int(net_buy_input)

    if min(start_quote, start_base) <= 0 or net_buy_input <= 1:
        raise ValueError(
            "Reserves must be positive and buy input must exceed 1 lamport."
        )

    # BUY: fee components apply to the specified quote input.
    buy_fees = calculate_fee_components(
        net_buy_input,
        fee_rates["buy"],
    )

    buy_total_fees = sum(buy_fees.values())
    buy_gross_spend = net_buy_input + buy_total_fees

    # The validated exact-quote-input curve calculation uses input - 1.
    curve_buy_input = net_buy_input - 1

    token_out = (
        start_base * curve_buy_input
    ) // (start_quote + curve_buy_input)

    if token_out <= 0 or token_out >= start_base:
        raise ValueError("Buy output is outside valid pool bounds.")

    # The reserve receives the input plus the LP fee.
    quote_after_buy = (
        start_quote + net_buy_input + buy_fees["lp"]
    )
    base_after_buy = start_base - token_out

    # SELL: gross quote output before fees.
    gross_sell_output = (
        quote_after_buy * token_out
    ) // (base_after_buy + token_out)

    sell_fees = calculate_fee_components(
        gross_sell_output,
        fee_rates["sell"],
    )

    sell_total_fees = sum(sell_fees.values())
    net_sell_proceeds = gross_sell_output - sell_total_fees

    if net_sell_proceeds < 0:
        raise ValueError("Fees exceed gross sell proceeds.")

    # The LP fee remains in the quote reserves.
    quote_after_sell = (
        quote_after_buy - gross_sell_output + sell_fees["lp"]
    )
    base_after_sell = base_after_buy + token_out

    pnl_raw = net_sell_proceeds - buy_gross_spend

    # Effective constant-product diagnostics.
    k0 = start_quote * start_base
    k1 = quote_after_buy * base_after_buy
    k2 = quote_after_sell * base_after_sell

    return {
        # Original output fields retained.
        "net_buy_input_lamports": net_buy_input,
        "buy_fees_lamports": buy_total_fees,
        "total_buy_cost_SOL": buy_gross_spend / LAMPORTS_PER_SOL,
        "tokens_bought": token_out / token_scale,
        "gross_sell_output_SOL": (
            gross_sell_output / LAMPORTS_PER_SOL
        ),
        "sell_fees_SOL": sell_total_fees / LAMPORTS_PER_SOL,
        "net_sell_proceeds_SOL": (
            net_sell_proceeds / LAMPORTS_PER_SOL
        ),
        "round_trip_pnl_SOL": pnl_raw / LAMPORTS_PER_SOL,
        "round_trip_return_pct": (
            pnl_raw / buy_gross_spend * 100
        ),
        "k_change_buy_pct": (k1 / k0 - 1) * 100,
        "k_change_sell_pct": (k2 / k1 - 1) * 100,
        "final_quote_reserve_SOL": (
            quote_after_sell / LAMPORTS_PER_SOL
        ),
        "final_base_reserve": base_after_sell / token_scale,

        # Additional raw-unit diagnostics.
        "buy_curve_input_lamports": curve_buy_input,
        "buy_lp_fee_lamports": buy_fees["lp"],
        "buy_protocol_fee_lamports": buy_fees["protocol"],
        "buy_creator_fee_lamports": buy_fees["creator"],
        "total_buy_cost_lamports": buy_gross_spend,
        "tokens_bought_raw": token_out,
        "sell_lp_fee_lamports": sell_fees["lp"],
        "sell_protocol_fee_lamports": sell_fees["protocol"],
        "sell_creator_fee_lamports": sell_fees["creator"],
        "gross_sell_output_lamports": gross_sell_output,
        "net_sell_proceeds_lamports": net_sell_proceeds,
        "round_trip_pnl_lamports": pnl_raw,
    }

In [ ]:
import pandas as pd
from IPython.display import display

if "events84b" not in globals() or events84b.empty:
    raise NameError("Run Step 84B first; events84b is missing.")

sample_sig113 = str(events84b.iloc[0]["signature"])
ev113 = extract_buy_event84b(sample_sig113)

if ev113["ix_name"] not in (
    "buy_exact_quote_in",
    "buy_exact_quote_in_v2",
):
    raise ValueError(
        f"Expected exact-quote-input instruction, got {ev113['ix_name']!r}"
    )

effective_quote113 = (
    int(ev113["pool_quote_token_reserves"])
    + int(ev113["virtual_quote_reserves"])
)
base_reserve113 = int(ev113["pool_base_token_reserves"])
quote_input113 = int(ev113["user_quote_amount_in"])

sim113 = simulate_round_trip_v2(
    effective_quote113,
    base_reserve113,
    quote_input113,
)

checks113 = {
    "buy output matches event": (
        sim113["tokens_bought_raw"] == int(ev113["base_amount_out"])
    ),
    "LP fee matches event": (
        sim113["buy_lp_fee_lamports"] == int(ev113["lp_fee"])
    ),
    "protocol fee matches event": (
        sim113["buy_protocol_fee_lamports"] == int(ev113["protocol_fee"])
    ),
    "creator fee matches event": (
        sim113["buy_creator_fee_lamports"]
        == int(ev113["coin_creator_fee"])
    ),
    "total buy cost matches event": (
        sim113["total_buy_cost_lamports"] == int(ev113["quote_amount_in"])
    ),
}

display(pd.DataFrame([
    {"check": name, "passed": result}
    for name, result in checks113.items()
]))

print("\nRound-trip result:")
display(pd.DataFrame([{
    key: sim113[key]
    for key in [
        "net_buy_input_lamports",
        "buy_curve_input_lamports",
        "tokens_bought_raw",
        "total_buy_cost_lamports",
        "gross_sell_output_lamports",
        "net_sell_proceeds_lamports",
        "round_trip_pnl_lamports",
        "round_trip_pnl_SOL",
        "round_trip_return_pct",
    ]
}]))

assert all(checks113.values()), "Real-event smoke test failed."
print("PASS: round-trip v2 buy-side regression")

In [ ]:
import pandas as pd
from IPython.display import display

# Reuse the Step 113 inputs and results.
if "sim113" not in globals() or "ev113" not in globals():
    raise NameError("Run Step 113 first.")

buy_input114 = int(sim113["net_buy_input_lamports"])
buy_cost114 = int(sim113["total_buy_cost_lamports"])
buy_output114 = int(sim113["tokens_bought_raw"])
gross_sell114 = int(sim113["gross_sell_output_lamports"])
net_sell114 = int(sim113["net_sell_proceeds_lamports"])
pnl114 = int(sim113["round_trip_pnl_lamports"])

buy_fees114 = calculate_fee_components(
    buy_input114, fee_rates["buy"]
)
sell_fees114 = calculate_fee_components(
    gross_sell114, fee_rates["sell"]
)

buy_fee_total114 = sum(buy_fees114.values())
sell_fee_total114 = sum(sell_fees114.values())

checks114 = {
    "Buy cost = input + buy fees":
        buy_cost114 == buy_input114 + buy_fee_total114,

    "Sell proceeds = gross output - sell fees":
        net_sell114 == gross_sell114 - sell_fee_total114,

    "PnL = net sell proceeds - total buy cost":
        pnl114 == net_sell114 - buy_cost114,

    "PnL decomposes into curve difference and fees":
        pnl114 == (
            gross_sell114 - buy_input114
            - buy_fee_total114 - sell_fee_total114
        ),

    "Base reserve restored after selling all tokens":
        int(sim113["final_base_reserve"] * token_scale)
        == int(ev113["pool_base_token_reserves"]),
}

display(pd.DataFrame([
    {"check": name, "passed": passed}
    for name, passed in checks114.items()
]))

display(pd.DataFrame([{
    "buy_input_lamports": buy_input114,
    "buy_fees_lamports": buy_fee_total114,
    "buy_cost_lamports": buy_cost114,
    "gross_sell_lamports": gross_sell114,
    "sell_fees_lamports": sell_fee_total114,
    "net_sell_lamports": net_sell114,
    "round_trip_pnl_lamports": pnl114,
}]))

assert all(checks114.values()), "Round-trip accounting check failed."
print("PASS: round-trip accounting")

In [ ]:
import re
from IPython.display import display
import pandas as pd

history115 = globals().get("In", [])

patterns115 = {
    "environment step": r"\benv\.step\s*\(|\bstep\s*\(\s*action",
    "reward": r"\breward\b|\brewards\b",
    "action selection": r"\bepsilon\b|\bselect_action\b|\bchoose_action\b|\bargmax\b",
    "state transition": r"\bnext_state\b|\bnew_state\b|\bstate\s*=",
    "RL training": r"\bq_table\b|\bq_values\b|\bpolicy\b|\bagent\b|\btrain\b",
    "episode loop": r"\bepisode\b|\bdone\b|\bterminated\b|\btruncated\b",
    "simulator calls": r"\bsimulate_round_trip\b|\bsimulate_round_trip_v2\b",
}

matches115 = []

for cell_num, source in enumerate(history115):
    if not isinstance(source, str) or not source.strip():
        continue

    labels = [
        label
        for label, pattern in patterns115.items()
        if re.search(pattern, source, flags=re.IGNORECASE)
    ]

    if labels:
        lines = source.splitlines()
        relevant = []

        for line_num, line in enumerate(lines):
            if any(
                re.search(pattern, line, flags=re.IGNORECASE)
                for pattern in patterns115.values()
            ):
                relevant.append(
                    f"{line_num + 1}: {line.strip()[:180]}"
                )

        matches115.append({
            "cell": cell_num,
            "categories": ", ".join(labels),
            "source_lines": len(lines),
            "relevant_code": "\n".join(relevant[:12]),
            "full_source": source,
        })

print("Matching notebook cells:", len(matches115))

if matches115:
    display(pd.DataFrame([
        {
            "cell": item["cell"],
            "categories": item["categories"],
            "source_lines": item["source_lines"],
            "relevant_code": item["relevant_code"],
        }
        for item in matches115[-25:]
    ]))
else:
    print(
        "No matching code found in input history. "
        "The RL loop may be in an imported module or a cell whose source "
        "is no longer available."
    )

In [ ]:
import re
from IPython.display import display
import pandas as pd

history116 = globals().get("In", [])

patterns116 = {
    "environment class or step method":
        r"\bclass\s+\w*(?:Env|Environment)\w*|\bdef\s+step\s*\(",
    "environment transition":
        r"\benv\s*\.\s*step\s*\(|\benvironment\s*\.\s*step\s*\(",
    "action selection":
        r"\bselect_action\b|\bchoose_action\b|\bget_action\b|\bepsilon\b",
    "Q-learning":
        r"\bq_table\b|\bq_values\b|\bQ_table\b|\bQ\[[^\]]+\]",
    "reward update":
        r"\breward\s*=|\brewards\.append|\breward_history\b|\btotal_reward\s*\+=",
    "state transition":
        r"\bnext_state\s*=|\bnew_state\s*=|\bstate\s*=\s*next_state",
    "episode control":
        r"\bfor\s+\w+\s+in\s+range\(.*episode|\bwhile\s+not\s+done\b|\bterminated\s*=|\btruncated\s*=",
    "agent training calls":
        r"\bagent\.(?:act|learn|update|train|remember|select_action)\s*\(",
}

hits116 = []

for cell_num, source in enumerate(history116):
    if not isinstance(source, str) or not source.strip():
        continue

    lines = source.splitlines()
    cell_hits = []

    for i, line in enumerate(lines):
        found = [
            label
            for label, pattern in patterns116.items()
            if re.search(pattern, line, flags=re.IGNORECASE)
        ]

        if found:
            lo = max(0, i - 2)
            hi = min(len(lines), i + 3)

            cell_hits.append({
                "line": i + 1,
                "matches": ", ".join(found),
                "context": "\n".join(
                    f"{j + 1}: {lines[j][:220]}"
                    for j in range(lo, hi)
                ),
            })

    if cell_hits:
        hits116.append({
            "cell": cell_num,
            "source": source,
            "hits": cell_hits,
        })

print("Cells with specific RL-control patterns:", len(hits116))

if hits116:
    display(pd.DataFrame([
        {
            "cell": item["cell"],
            "line": hit["line"],
            "matches": hit["matches"],
            "context": hit["context"],
        }
        for item in hits116
        for hit in item["hits"]
    ]))

    print("\nFull source for the matching cells is available in hits116.")
else:
    print(
        "No explicit RL control loop was found in the notebook history. "
        "The RL environment may not have been implemented yet, or it may "
        "be defined in an imported module."
    )

In [ ]:
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")

df117 = stratified_trades

print("=== Dataset overview ===")
print("Rows:", len(df117))
print("Unique signatures:", df117["signature"].nunique()
      if "signature" in df117.columns else "No signature column")
print("Columns:", len(df117.columns))

print("\n=== Full schema ===")
display(pd.DataFrame({
    "column": df117.columns,
    "dtype": [str(df117[c].dtype) for c in df117.columns],
    "non_null": [int(df117[c].notna().sum()) for c in df117.columns],
    "unique_values": [int(df117[c].nunique()) for c in df117.columns],
}))

print("\n=== Action distribution ===")
if "action" in df117.columns:
    display(
        df117["action"]
        .value_counts(dropna=False)
        .rename_axis("action")
        .reset_index(name="rows")
    )

# Identify candidate time, price, reserve, and trade-size fields.
keywords117 = [
    "time", "date", "slot", "block",
    "price", "quote", "token", "reserve",
    "amount", "volume", "pool", "mint",
]

candidates117 = [
    c for c in df117.columns
    if any(k in c.lower() for k in keywords117)
]

print("\n=== Candidate observation and execution fields ===")
display(pd.DataFrame({
    "column": candidates117,
    "dtype": [str(df117[c].dtype) for c in candidates117],
    "examples": [
        df117[c].dropna().head(3).tolist()
        for c in candidates117
    ],
}))

# Check chronological ordering where a plausible time field exists.
time_cols117 = [
    c for c in df117.columns
    if any(k in c.lower() for k in ["timestamp", "datetime", "localtimestamp"])
]

for col in time_cols117:
    parsed = pd.to_datetime(df117[col], errors="coerce", utc=True)
    valid = parsed.dropna()

    print(f"\nTime field: {col}")
    print("Parseable rows:", len(valid), "/", len(df117))

    if not valid.empty:
        print("Minimum:", valid.min())
        print("Maximum:", valid.max())
        print("Already chronological:", bool(parsed.is_monotonic_increasing))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "stratified_trades" not in globals():
    raise NameError("stratified_trades is missing.")

# Preserve the original dataset.
market118 = stratified_trades.copy()

required118 = {
    "mint", "pool", "block", "archive_seq",
    "observed_time", "action", "price",
}
missing118 = required118 - set(market118.columns)

if missing118:
    raise ValueError(f"Missing required columns: {sorted(missing118)}")

# Keep usable price observations.
market118["price"] = pd.to_numeric(market118["price"], errors="coerce")
market118 = market118[
    market118["price"].gt(0)
    & market118["observed_time"].notna()
].copy()

# Within each pool, respect block ordering and archive sequence.
group_cols118 = ["mint", "pool"]
market118 = market118.sort_values(
    group_cols118 + ["block", "archive_seq"],
    kind="mergesort",
).reset_index(drop=True)

grouped118 = market118.groupby(group_cols118, sort=False)

# Only current and previous prices are used to construct these features.
for lag in [1, 3, 10]:
    lag_price = grouped118["price"].shift(lag)
    market118[f"log_return_{lag}"] = np.log(
        market118["price"] / lag_price
    )

# Rolling volatility uses current and prior returns only.
market118["volatility_10"] = (
    market118.groupby(group_cols118, sort=False)["log_return_1"]
    .transform(lambda s: s.rolling(10, min_periods=5).std())
)

# Current price relative to its trailing 20-observation mean.
market118["price_vs_mean_20"] = (
    market118["price"]
    / grouped118["price"].transform(
        lambda s: s.rolling(20, min_periods=5).mean()
    )
    - 1
)

# Previous-trade buy ratio: shifted first, so the current action isn't used.
is_buy118 = market118["action"].eq("buy").astype(float)

market118["prior_buy_ratio_20"] = (
    is_buy118.groupby(
        [market118["mint"], market118["pool"]],
        sort=False,
    )
    .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
)

# Target return from the current observation to the next event
# in the same mint/pool sequence. This is a reward target, not a feature.
market118["next_price"] = grouped118["price"].shift(-1)
market118["next_return"] = (
    market118["next_price"] / market118["price"] - 1
)

feature_cols118 = [
    "log_return_1",
    "log_return_3",
    "log_return_10",
    "volatility_10",
    "price_vs_mean_20",
    "prior_buy_ratio_20",
]

# Keep only rows where every feature and the next-event target are available.
rl_market_df = market118.dropna(
    subset=feature_cols118 + ["next_return"]
).copy()

print("=== RL market dataset ===")
print("Rows before feature filtering:", len(market118))
print("Rows ready for RL:", len(rl_market_df))
print("Distinct mints:", rl_market_df["mint"].nunique())
print("Distinct pools:", rl_market_df["pool"].nunique())
print("Feature columns:", feature_cols118)

print("\n=== Rows per mint/pool episode ===")
episode_summary118 = (
    rl_market_df.groupby(group_cols118)
    .size()
    .rename("steps")
    .reset_index()
)

display(episode_summary118["steps"].describe().to_frame("episode_steps"))

print("\n=== Feature and reward preview ===")
display(
    rl_market_df[
        group_cols118
        + ["block", "observed_time", "price"]
        + feature_cols118
        + ["next_return"]
    ].head(10)
)

# Verify that every target is the next price in its own chronological pool sequence.
expected_next118 = (
    market118.groupby(group_cols118, sort=False)["price"].shift(-1)
)

expected_next118 = expected_next118.loc[rl_market_df.index]
assert np.allclose(
    rl_market_df["next_price"].to_numpy(),
    expected_next118.to_numpy(),
    rtol=1e-12,
    atol=0,
)

assert rl_market_df[feature_cols118].notna().all().all()
print("\nPASS: feature rows contain no missing values, and next-price targets align.")

In [ ]:
import numpy as np
import pandas as pd

class PriceBasedPumpRLEnv:
    """
    Sequential, price-based RL environment.

    Actions:
        0 = target cash / flat position
        1 = target fully invested / long position

    Observation:
        Six historical market features + current position.

    Reward:
        Next-period log return while long, minus log transaction costs.
        Positions still open at episode end are liquidated with a sell fee.

    This is a price-based approximation, not an exact on-chain fill simulator.
    """

    def __init__(
        self,
        market_df,
        feature_cols,
        fee_rates,
        min_episode_steps=20,
    ):
        required = {
            "mint", "pool", "block", "archive_seq",
            "observed_time", "price", "next_return",
            *feature_cols,
        }
        missing = required - set(market_df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        self.feature_cols = list(feature_cols)
        self.min_episode_steps = int(min_episode_steps)

        self.buy_fee_rate = sum(
            int(v) for v in fee_rates["buy"].values()
        ) / 10_000

        self.sell_fee_rate = sum(
            int(v) for v in fee_rates["sell"].values()
        ) / 10_000

        if not (0 <= self.buy_fee_rate < 1):
            raise ValueError("Invalid aggregate buy fee rate.")
        if not (0 <= self.sell_fee_rate < 1):
            raise ValueError("Invalid aggregate sell fee rate.")

        df = market_df.copy()
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df["next_return"] = pd.to_numeric(
            df["next_return"], errors="coerce"
        )

        numeric_cols = self.feature_cols + ["price", "next_return"]
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        finite = np.isfinite(df[numeric_cols].to_numpy(dtype=float)).all(axis=1)
        df = df.loc[finite & df["price"].gt(0)].copy()

        if (df["next_return"] <= -1).any():
            raise ValueError("A next-period return is <= -100%; cannot take log1p.")

        df = df.sort_values(
            ["mint", "pool", "block", "archive_seq"],
            kind="mergesort",
        )

        self.episodes = []
        self.dropped_short_episodes = []

        for key, group in df.groupby(["mint", "pool"], sort=False):
            group = group.reset_index(drop=True)

            if len(group) < self.min_episode_steps:
                self.dropped_short_episodes.append({
                    "mint": key[0],
                    "pool": key[1],
                    "steps": len(group),
                })
                continue

            self.episodes.append({
                "key": key,
                "data": group,
            })

        if not self.episodes:
            raise ValueError(
                "No episodes remain. Lower min_episode_steps or inspect the data."
            )

        self.rng = np.random.default_rng()
        self.current_episode = None
        self.current_index = 0
        self.position = 0

    def _observation(self):
        row = self.current_episode["data"].iloc[self.current_index]
        features = row[self.feature_cols].to_numpy(dtype=np.float32)
        return np.append(features, np.float32(self.position)).astype(np.float32)

    def reset(self, seed=None, episode_index=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        if episode_index is None:
            selected = int(self.rng.integers(len(self.episodes)))
        else:
            selected = int(episode_index)
            if not 0 <= selected < len(self.episodes):
                raise IndexError("episode_index is out of range.")

        self.current_episode = self.episodes[selected]
        self.current_index = 0
        self.position = 0

        return self._observation(), {
            "episode_index": selected,
            "mint": self.current_episode["key"][0],
            "pool": self.current_episode["key"][1],
            "episode_steps": len(self.current_episode["data"]),
        }

    def step(self, action):
        if self.current_episode is None:
            raise RuntimeError("Call reset() before step().")

        if action not in (0, 1):
            raise ValueError("Action must be 0 (cash) or 1 (long).")

        row = self.current_episode["data"].iloc[self.current_index]

        old_position = self.position
        target_position = int(action)
        next_return = float(row["next_return"])

        # Charge a fee only when changing position.
        transaction_log_reward = 0.0
        transaction_fee_rate = 0.0

        if target_position != old_position:
            transaction_fee_rate = (
                self.buy_fee_rate
                if target_position == 1
                else self.sell_fee_rate
            )
            transaction_log_reward = np.log1p(-transaction_fee_rate)

        # The target position earns the next-period return.
        market_log_reward = (
            np.log1p(next_return) if target_position == 1 else 0.0
        )

        next_index = self.current_index + 1
        terminated = next_index >= len(self.current_episode["data"])
        truncated = False

        # Force liquidation if the episode ends while long.
        liquidation_log_reward = 0.0
        if terminated and target_position == 1:
            liquidation_log_reward = np.log1p(-self.sell_fee_rate)

        reward = (
            transaction_log_reward
            + market_log_reward
            + liquidation_log_reward
        )

        self.current_index = next_index

        if terminated:
            self.position = 0
            observation = np.zeros(
                len(self.feature_cols) + 1,
                dtype=np.float32,
            )
        else:
            self.position = target_position
            observation = self._observation()

        info = {
            "mint": self.current_episode["key"][0],
            "pool": self.current_episode["key"][1],
            "price": float(row["price"]),
            "next_return": next_return,
            "position_before": old_position,
            "position_after": self.position,
            "action": target_position,
            "transaction_fee_rate": transaction_fee_rate,
            "market_log_reward": market_log_reward,
            "liquidation_log_reward": liquidation_log_reward,
            "terminated": terminated,
        }

        return observation, float(reward), terminated, truncated, info


# Create the environment from Step 118 data.
env119 = PriceBasedPumpRLEnv(
    market_df=rl_market_df,
    feature_cols=feature_cols118,
    fee_rates=fee_rates,
    min_episode_steps=20,
)

print("Episodes retained:", len(env119.episodes))
print("Short episodes dropped:", len(env119.dropped_short_episodes))
print("Observation dimension:", len(feature_cols118) + 1)
print("Actions: 0 = cash; 1 = long")
print("Buy fee rate:", f"{env119.buy_fee_rate:.2%}")
print("Sell fee rate:", f"{env119.sell_fee_rate:.2%}")


# Deterministic smoke test: enter, hold, exit, stay flat.
obs119, reset_info119 = env119.reset(seed=119, episode_index=0)

print("\nReset info:", reset_info119)
print("Initial observation shape:", obs119.shape)
print("Initial position:", obs119[-1])

demo119 = []
for action in [1, 1, 0, 0, 1]:
    obs119, reward119, terminated119, truncated119, info119 = env119.step(action)

    demo119.append({
        "action": action,
        "next_return": info119["next_return"],
        "position_before": info119["position_before"],
        "position_after": info119["position_after"],
        "reward_log": reward119,
        "terminated": terminated119,
    })

    if terminated119 or truncated119:
        break

display(pd.DataFrame(demo119))

assert obs119.shape == (len(feature_cols118) + 1,)
assert all(np.isfinite(r["reward_log"]) for r in demo119)
print("PASS: environment smoke test")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "rl_market_df" not in globals():
    raise NameError("Run Step 118 first: rl_market_df is missing.")

group_cols120 = ["mint", "pool"]
feature_cols120 = list(feature_cols118)
min_steps120 = 20

# Use global timestamp cutoffs so an event timestamp cannot appear
# in more than one split.
timestamps120 = rl_market_df["observed_time"].dropna()
cut70_120 = timestamps120.quantile(0.70)
cut85_120 = timestamps120.quantile(0.85)

print("Train ends at:", cut70_120)
print("Validation ends at:", cut85_120)

source120 = rl_market_df.copy()
source120["observed_time"] = pd.to_datetime(
    source120["observed_time"], utc=True
)

raw_splits120 = {
    "train": source120[source120["observed_time"] <= cut70_120].copy(),
    "validation": source120[
        (source120["observed_time"] > cut70_120)
        & (source120["observed_time"] <= cut85_120)
    ].copy(),
    "test": source120[source120["observed_time"] > cut85_120].copy(),
}

def prepare_split120(frame):
    """Recalculate next-event returns within this split only."""
    frame = frame.sort_values(
        group_cols120 + ["block", "archive_seq"],
        kind="mergesort",
    ).copy()

    frame["split_next_price"] = (
        frame.groupby(group_cols120, sort=False)["price"].shift(-1)
    )
    frame["split_next_return"] = (
        frame["split_next_price"] / frame["price"] - 1
    )

    frame = frame.dropna(
        subset=feature_cols120 + ["split_next_return"]
    ).copy()

    # Remove very short episodes so the baselines have a consistent minimum
    # episode length with the environment from Step 119.
    sizes = frame.groupby(group_cols120, sort=False).size()
    eligible = sizes[sizes >= min_steps120].index
    mask = pd.MultiIndex.from_frame(
        frame[group_cols120]
    ).isin(eligible)

    return frame.loc[mask].copy()

split_data120 = {
    name: prepare_split120(frame)
    for name, frame in raw_splits120.items()
}

# Verify the split periods do not overlap.
for name, frame in raw_splits120.items():
    print(
        f"{name:10s}: {len(frame):>7,} rows, "
        f"{frame['observed_time'].min()} -> {frame['observed_time'].max()}"
    )

# Fee proportions use the same configured aggregate rates as Step 119.
buy_fee120 = sum(int(x) for x in fee_rates["buy"].values()) / 10_000
sell_fee120 = sum(int(x) for x in fee_rates["sell"].values()) / 10_000

def evaluate_baselines120(frame, split_name):
    rows = []

    for (mint, pool), episode in frame.groupby(group_cols120, sort=False):
        episode = episode.sort_values(
            ["block", "archive_seq"], kind="mergesort"
        )
        returns = episode["split_next_return"].to_numpy(dtype=float)

        # Always cash: no exposure and no trades.
        cash_log_return = 0.0

        # Buy-and-hold: enter once, remain long for every transition,
        # then pay the sell fee to liquidate at episode end.
        market_log_return = np.log1p(returns).sum()
        buy_hold_log_return = (
            np.log1p(-buy_fee120)
            + market_log_return
            + np.log1p(-sell_fee120)
        )

        rows.append({
            "split": split_name,
            "mint": mint,
            "pool": pool,
            "steps": len(episode),
            "policy": "always_cash",
            "episode_return_pct": 100 * np.expm1(cash_log_return),
        })
        rows.append({
            "split": split_name,
            "mint": mint,
            "pool": pool,
            "steps": len(episode),
            "policy": "buy_and_hold",
            "episode_return_pct": 100 * np.expm1(buy_hold_log_return),
        })

    return pd.DataFrame(rows)

baseline_rows120 = [
    evaluate_baselines120(frame, name)
    for name, frame in split_data120.items()
    if not frame.empty
]

if not baseline_rows120:
    raise ValueError(
        "No eligible episodes remain. Recheck timestamp splits and minimum steps."
    )

baseline_results120 = pd.concat(baseline_rows120, ignore_index=True)

baseline_summary120 = (
    baseline_results120
    .groupby(["split", "policy"], sort=False)
    .agg(
        episodes=("episode_return_pct", "size"),
        mean_episode_return_pct=("episode_return_pct", "mean"),
        median_episode_return_pct=("episode_return_pct", "median"),
        win_rate_pct=(
            "episode_return_pct",
            lambda s: 100 * (s > 0).mean(),
        ),
        p10_return_pct=("episode_return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("episode_return_pct", lambda s: s.quantile(0.90)),
        mean_episode_steps=("steps", "mean"),
    )
    .reset_index()
)

print("\n=== Eligible data by chronological split ===")
display(pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "episodes": frame[group_cols120].drop_duplicates().shape[0],
        "minimum_episode_steps": (
            frame.groupby(group_cols120).size().min()
            if len(frame) else None
        ),
    }
    for name, frame in split_data120.items()
]))

print("\n=== Baseline results ===")
display(baseline_summary120.round(4))

print(
    "\nNote: these are price-based baseline results with configured 1% "
    "buy/sell costs, not exact integer PumpSwap fill simulations."
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. Prepare isolated chronological environments
# ------------------------------------------------------------
required121 = [
    "split_data120", "feature_cols120", "fee_rates",
    "PriceBasedPumpRLEnv",
]

for name in required121:
    if name not in globals():
        raise NameError(f"Missing {name}. Run Steps 119-120 first.")

feature_cols121 = list(feature_cols120)
group_cols121 = ["mint", "pool"]

def make_split_env121(split_name):
    frame = split_data120[split_name].copy()

    # Ensure the transition reward stays within the given split.
    if "split_next_return" not in frame:
        raise ValueError(
            f"{split_name} data lacks split_next_return. "
            "Rerun Step 120."
        )

    frame["next_return"] = frame["split_next_return"]

    return PriceBasedPumpRLEnv(
        market_df=frame,
        feature_cols=feature_cols121,
        fee_rates=fee_rates,
        min_episode_steps=20,
    )

env_train121 = make_split_env121("train")
env_val121 = make_split_env121("validation")
env_test121 = make_split_env121("test")

print("Episodes — train / validation / test:",
      len(env_train121.episodes),
      len(env_val121.episodes),
      len(env_test121.episodes))

# ------------------------------------------------------------
# 2. Discretize observations using TRAINING data only
# ------------------------------------------------------------
train_feature_values121 = pd.concat(
    [ep["data"][feature_cols121] for ep in env_train121.episodes],
    ignore_index=True,
)

bin_edges121 = {}

for col in feature_cols121:
    edges = train_feature_values121[col].quantile(
        [1 / 3, 2 / 3]
    ).to_numpy(dtype=float)

    bin_edges121[col] = edges

n_bins121 = 3
n_features121 = len(feature_cols121)
n_states121 = (n_bins121 ** n_features121) * 2  # last factor = position

def state_id121(observation):
    # Each feature maps to one of three training-defined bins.
    code = 0

    for i, col in enumerate(feature_cols121):
        value = float(observation[i])
        edges = bin_edges121[col]

        bucket = int(np.digitize(value, edges))
        bucket = min(max(bucket, 0), n_bins121 - 1)
        code = code * n_bins121 + bucket

    position = int(round(float(observation[-1])))
    position = min(max(position, 0), 1)

    return code * 2 + position

# ------------------------------------------------------------
# 3. Tabular Q-learning
# ------------------------------------------------------------
seed121 = 121026
rng121 = np.random.default_rng(seed121)

q_table121 = np.zeros((n_states121, 2), dtype=np.float64)

epochs121 = 8
alpha121 = 0.10
gamma121 = 0.99
epsilon_start121 = 0.30
epsilon_end121 = 0.05

num_train_episodes121 = len(env_train121.episodes)
total_episode_runs121 = epochs121 * num_train_episodes121
episode_run121 = 0

training_log121 = []

for epoch in range(epochs121):
    episode_order = rng121.permutation(num_train_episodes121)

    for episode_idx in episode_order:
        fraction = (
            episode_run121 / max(total_episode_runs121 - 1, 1)
        )
        epsilon = (
            epsilon_start121
            + fraction * (epsilon_end121 - epsilon_start121)
        )
        episode_run121 += 1

        observation, reset_info = env_train121.reset(
            episode_index=int(episode_idx)
        )

        terminated = False
        episode_reward = 0.0
        steps = 0

        while not terminated:
            s = state_id121(observation)

            if rng121.random() < epsilon:
                action = int(rng121.integers(0, 2))
            else:
                action = int(np.argmax(q_table121[s]))

            next_observation, reward, terminated, truncated, info = (
                env_train121.step(action)
            )

            # Terminal transitions do not bootstrap.
            if terminated or truncated:
                target = float(reward)
            else:
                next_s = state_id121(next_observation)
                target = float(reward) + gamma121 * float(
                    np.max(q_table121[next_s])
                )

            q_table121[s, action] += alpha121 * (
                target - q_table121[s, action]
            )

            episode_reward += float(reward)
            steps += 1
            observation = next_observation

        training_log121.append({
            "epoch": epoch + 1,
            "episode_index": int(episode_idx),
            "steps": steps,
            "training_reward_log": episode_reward,
            "epsilon": epsilon,
        })

print("Q-table shape:", q_table121.shape)
print("Training episode runs:", len(training_log121))
display(
    pd.DataFrame(training_log121)
    .groupby("epoch")
    .agg(
        episodes=("episode_index", "count"),
        mean_episode_reward_log=("training_reward_log", "mean"),
        mean_steps=("steps", "mean"),
        final_epsilon=("epsilon", "last"),
    )
    .reset_index()
    .round(5)
)

# ------------------------------------------------------------
# 4. Evaluate frozen policies on each split
# ------------------------------------------------------------
def evaluate_policy121(env, split_name, policy_name):
    rows = []

    for episode_idx, episode in enumerate(env.episodes):
        observation, reset_info = env.reset(
            episode_index=episode_idx
        )

        terminated = False
        truncated = False
        total_reward_log = 0.0
        steps = 0

        while not (terminated or truncated):
            if policy_name == "always_cash":
                action = 0
            elif policy_name == "buy_and_hold":
                action = 1
            elif policy_name == "q_learning":
                action = int(
                    np.argmax(q_table121[state_id121(observation)])
                )
            else:
                raise ValueError(f"Unknown policy: {policy_name}")

            observation, reward, terminated, truncated, info = (
                env.step(action)
            )

            total_reward_log += float(reward)
            steps += 1

        # Environment rewards are log returns, so compound them back
        # into an episode percentage return.
        episode_return_pct = 100 * np.expm1(total_reward_log)

        rows.append({
            "split": split_name,
            "policy": policy_name,
            "mint": reset_info["mint"],
            "pool": reset_info["pool"],
            "steps": steps,
            "episode_return_pct": episode_return_pct,
        })

    return rows

evaluation_rows121 = []

for split_name, env in [
    ("train", env_train121),
    ("validation", env_val121),
    ("test", env_test121),
]:
    for policy_name in [
        "always_cash", "buy_and_hold", "q_learning"
    ]:
        evaluation_rows121.extend(
            evaluate_policy121(env, split_name, policy_name)
        )

evaluation121 = pd.DataFrame(evaluation_rows121)

summary121 = (
    evaluation121
    .groupby(["split", "policy"], sort=False)
    .agg(
        episodes=("episode_return_pct", "size"),
        mean_return_pct=("episode_return_pct", "mean"),
        median_return_pct=("episode_return_pct", "median"),
        win_rate_pct=(
            "episode_return_pct",
            lambda s: 100 * (s > 0).mean(),
        ),
        p10_return_pct=(
            "episode_return_pct",
            lambda s: s.quantile(0.10),
        ),
        p90_return_pct=(
            "episode_return_pct",
            lambda s: s.quantile(0.90),
        ),
        mean_steps=("steps", "mean"),
    )
    .reset_index()
)

print("\n=== Frozen-policy evaluation ===")
display(summary121.round(4))

print(
    "\nInterpretation: use validation performance for policy selection. "
    "Use the test split only for final comparison. A high training return "
    "alone does not demonstrate generalization."
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

def diagnose_policy122(env, split_name):
    rows = []
    actions_total = {0: 0, 1: 0}
    unique_states = set()
    q_nonzero_states = set()
    entries = 0
    exits = 0
    total_steps = 0
    total_reward = 0.0

    for episode_idx, episode in enumerate(env.episodes):
        obs, reset_info = env.reset(episode_index=episode_idx)
        done = False
        previous_position = 0

        while not done:
            sid = state_id121(obs)
            unique_states.add(sid)

            # A nonzero row means the Q values have moved away from
            # their zero initialization; it is only a proxy for coverage.
            if np.any(np.abs(q_table121[sid]) > 1e-12):
                q_nonzero_states.add(sid)

            action = int(np.argmax(q_table121[sid]))
            actions_total[action] += 1

            if previous_position == 0 and action == 1:
                entries += 1
            elif previous_position == 1 and action == 0:
                exits += 1

            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            total_steps += 1
            previous_position = action
            done = terminated or truncated

    rows.append({
        "split": split_name,
        "steps": total_steps,
        "cash_actions": actions_total[0],
        "long_actions": actions_total[1],
        "long_action_pct": (
            100 * actions_total[1] / total_steps
            if total_steps else 0
        ),
        "entries": entries,
        "exits": exits,
        "distinct_states_seen": len(unique_states),
        "states_with_nonzero_q": len(q_nonzero_states),
        "mean_step_reward_log": (
            total_reward / total_steps if total_steps else 0
        ),
        "total_log_reward": total_reward,
    })

    return rows

diagnostics122 = []
for split_name, env in [
    ("train", env_train121),
    ("validation", env_val121),
    ("test", env_test121),
]:
    diagnostics122.extend(diagnose_policy122(env, split_name))

diagnostics122 = pd.DataFrame(diagnostics122)
display(diagnostics122.round(5))

print("Q-table states:", q_table121.shape[0])
print(
    "Rows with any nonzero Q value:",
    int(np.any(np.abs(q_table121) > 1e-12, axis=1).sum()),
    "/",
    q_table121.shape[0],
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "q_table121" not in globals() or "state_id121" not in globals():
    raise NameError("Run Step 121 first.")

# Q-value preference across states.
q_gap123 = q_table121[:, 1] - q_table121[:, 0]

print("=== Q-table action preference ===")
display(pd.DataFrame([
    {
        "metric": "States preferring long (Q_long > Q_cash)",
        "value": int((q_gap123 > 1e-12).sum()),
    },
    {
        "metric": "States preferring cash (Q_long < Q_cash)",
        "value": int((q_gap123 < -1e-12).sum()),
    },
    {
        "metric": "Tied states",
        "value": int((np.abs(q_gap123) <= 1e-12).sum()),
    },
    {
        "metric": "States with learned Q values",
        "value": int((np.any(np.abs(q_table121) > 1e-12, axis=1)).sum()),
    },
    {
        "metric": "Total states",
        "value": len(q_gap123),
    },
]))

print("\n=== Q_long - Q_cash distribution ===")
display(pd.Series(q_gap123).describe(
    percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
).to_frame("q_gap"))

# Collect policy margin and action frequencies for each split.
def q_policy_margin123(env, split_name):
    rows = []

    for episode_idx, episode in enumerate(env.episodes):
        obs, reset_info = env.reset(episode_index=episode_idx)
        done = False

        while not done:
            sid = state_id121(obs)
            q_cash = float(q_table121[sid, 0])
            q_long = float(q_table121[sid, 1])
            action = int(np.argmax(q_table121[sid]))

            rows.append({
                "split": split_name,
                "state_id": sid,
                "action": action,
                "q_cash": q_cash,
                "q_long": q_long,
                "q_gap": q_long - q_cash,
                "absolute_q_gap": abs(q_long - q_cash),
            })

            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

    return pd.DataFrame(rows)

margin123 = pd.concat([
    q_policy_margin123(env_train121, "train"),
    q_policy_margin123(env_val121, "validation"),
    q_policy_margin123(env_test121, "test"),
], ignore_index=True)

print("\n=== Policy action rate and Q-value margins ===")
display(
    margin123.groupby("split").agg(
        steps=("action", "size"),
        long_actions=("action", "sum"),
        long_action_pct=("action", lambda s: 100 * s.mean()),
        distinct_policy_states=("state_id", "nunique"),
        mean_q_cash=("q_cash", "mean"),
        mean_q_long=("q_long", "mean"),
        median_q_gap=("q_gap", "median"),
        mean_abs_q_gap=("absolute_q_gap", "mean"),
    ).reset_index().round(6)
)

print("\n=== Largest states where long is preferred ===")
display(
    pd.DataFrame({
        "state_id": np.arange(len(q_gap123)),
        "q_cash": q_table121[:, 0],
        "q_long": q_table121[:, 1],
        "q_gap": q_gap123,
    })
    .sort_values("q_gap", ascending=False)
    .head(10)
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "env_val121" not in globals() or "feature_cols121" not in globals():
    raise NameError("Run Steps 118-121 first.")

# Locate features by name so their array order is explicit.
idx_ret3_124 = feature_cols121.index("log_return_3")
idx_ret10_124 = feature_cols121.index("log_return_10")
idx_mean20_124 = feature_cols121.index("price_vs_mean_20")

def policy_action124(policy_name, obs):
    ret3 = float(obs[idx_ret3_124])
    ret10 = float(obs[idx_ret10_124])
    above_mean = float(obs[idx_mean20_124])

    if policy_name == "always_cash":
        return 0

    if policy_name == "buy_and_hold":
        return 1

    if policy_name == "q_learning":
        return int(np.argmax(q_table121[state_id121(obs)]))

    if policy_name == "momentum_3":
        return int(ret3 > 0)

    if policy_name == "momentum_10":
        return int(ret10 > 0)

    if policy_name == "confirmed_trend":
        return int(ret3 > 0 and above_mean > 0)

    if policy_name == "strong_trend":
        # Require a 0.2% positive three-event log return
        # and price above its trailing mean.
        return int(ret3 > 0.002 and above_mean > 0)

    raise ValueError(f"Unknown policy: {policy_name}")


def evaluate_policy124(env, policy_name):
    episode_rows = []

    for episode_idx, episode in enumerate(env.episodes):
        obs, reset_info = env.reset(episode_index=episode_idx)

        total_log_reward = 0.0
        steps = 0
        long_steps = 0
        entries = 0
        exits = 0
        done = False

        while not done:
            action = policy_action124(policy_name, obs)

            obs, reward, terminated, truncated, info = env.step(action)

            total_log_reward += float(reward)
            steps += 1
            long_steps += int(action == 1)

            if info["position_before"] == 0 and action == 1:
                entries += 1
            elif info["position_before"] == 1 and action == 0:
                exits += 1

            # Count the automatic end-of-episode liquidation.
            if terminated and action == 1:
                exits += 1

            done = terminated or truncated

        episode_rows.append({
            "policy": policy_name,
            "mint": reset_info["mint"],
            "pool": reset_info["pool"],
            "steps": steps,
            "entries": entries,
            "exits": exits,
            "long_pct": 100 * long_steps / max(steps, 1),
            "return_pct": 100 * np.expm1(total_log_reward),
        })

    return episode_rows


policies124 = [
    "always_cash",
    "buy_and_hold",
    "q_learning",
    "momentum_3",
    "momentum_10",
    "confirmed_trend",
    "strong_trend",
]

validation_rows124 = []

for policy_name in policies124:
    validation_rows124.extend(
        evaluate_policy124(env_val121, policy_name)
    )

validation_results124 = pd.DataFrame(validation_rows124)

validation_summary124 = (
    validation_results124.groupby("policy", sort=False)
    .agg(
        episodes=("return_pct", "size"),
        mean_return_pct=("return_pct", "mean"),
        median_return_pct=("return_pct", "median"),
        win_rate_pct=("return_pct", lambda s: 100 * (s > 0).mean()),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
        mean_long_exposure_pct=("long_pct", "mean"),
        mean_entries=("entries", "mean"),
        mean_exits=("exits", "mean"),
    )
    .reset_index()
    .round(4)
    .sort_values("mean_return_pct", ascending=False)
)

print("=== Validation policy comparison ===")
display(validation_summary124)

# Choose a validation winner based on mean return.
# The test data is deliberately untouched.
winner124 = validation_summary124.iloc[0]["policy"]

print("\nHighest mean validation return:", winner124)
print(
    "Use validation as a selection signal, not proof of profitability. "
    "Do not tune policy thresholds on the test split."
)

In [ ]:
import pandas as pd
from IPython.display import display

if "validation_summary124" not in globals():
    raise NameError("Run Step 124 first.")
if "evaluation121" not in globals():
    raise NameError("Run Step 121 first.")

# Freeze the policy chosen using validation performance.
validation_winner125 = validation_summary124.sort_values(
    "mean_return_pct", ascending=False
).iloc[0]["policy"]

print("Policy selected from validation:", validation_winner125)
print("This selection is now frozen; do not tune on test results.")

# Reuse the existing held-out evaluation.
test125 = evaluation121[
    evaluation121["split"].eq("test")
].copy()

if test125.empty:
    raise ValueError("No cached test evaluation found in evaluation121.")

print("\n=== Final held-out test summary ===")

test_summary125 = (
    test125.groupby("policy")
    .agg(
        episodes=("episode_return_pct", "size"),
        mean_return_pct=("episode_return_pct", "mean"),
        median_return_pct=("episode_return_pct", "median"),
        win_rate_pct=(
            "episode_return_pct",
            lambda s: 100 * (s > 0).mean(),
        ),
        worst_episode_pct=("episode_return_pct", "min"),
        best_episode_pct=("episode_return_pct", "max"),
        p10_return_pct=("episode_return_pct", lambda s: s.quantile(0.10)),
    )
    .reset_index()
    .sort_values("mean_return_pct", ascending=False)
)

display(test_summary125.round(4))

print("\n=== Per-episode results for the frozen policy ===")

frozen_test125 = test125[
    test125["policy"].eq(validation_winner125)
].sort_values("episode_return_pct")

display(
    frozen_test125[
        ["mint", "pool", "steps", "episode_return_pct"]
    ].round(4)
)

print(
    "\nInterpretation: this is a price-based benchmark with configured "
    "transaction costs, not an exact executable PumpSwap backtest."
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "evaluation121" not in globals() or "split_data120" not in globals():
    raise NameError("Run Steps 120-121 first.")

test_bh126 = evaluation121[
    evaluation121["split"].eq("test")
    & evaluation121["policy"].eq("buy_and_hold")
].sort_values("episode_return_pct").copy()

test_market126 = split_data120["test"].copy()

def episode_diagnostics126(mint, pool):
    ep = test_market126[
        test_market126["mint"].eq(mint)
        & test_market126["pool"].eq(pool)
    ].sort_values(["block", "archive_seq"]).copy()

    if ep.empty:
        return None, None

    prices = ep["price"].to_numpy(dtype=float)

    # Include the next-event price at the end of the episode.
    if "split_next_price" in ep.columns:
        end_price = float(ep["split_next_price"].iloc[-1])
        price_path = np.append(prices, end_price)
    else:
        end_price = float(prices[-1])
        price_path = prices

    peaks = np.maximum.accumulate(price_path)
    drawdowns = price_path / peaks - 1.0

    next_returns = ep["split_next_return"].to_numpy(dtype=float)

    start_price = float(price_path[0])
    gross_price_return = end_price / start_price - 1.0

    max_dd_idx = int(np.argmin(drawdowns))
    worst_step_idx = int(np.argmin(next_returns))

    summary = {
        "mint": mint,
        "pool": pool,
        "steps": len(ep),
        "start_time": ep["observed_time"].iloc[0],
        "end_time": ep["observed_time"].iloc[-1],
        "start_price": start_price,
        "end_price": end_price,
        "gross_price_return_pct": 100 * gross_price_return,
        "worst_one_step_return_pct": 100 * float(np.min(next_returns)),
        "max_drawdown_pct": 100 * float(np.min(drawdowns)),
        "nonpositive_price_rows": int((ep["price"] <= 0).sum()),
        "nonfinite_price_rows": int(
            (~np.isfinite(ep["price"].to_numpy(dtype=float))).sum()
        ),
    }

    # Inspect a small window around the worst price movement.
    lo = max(0, worst_step_idx - 2)
    hi = min(len(ep), worst_step_idx + 4)

    local = ep.iloc[lo:hi][
        ["observed_time", "block", "archive_seq", "action", "price",
         "split_next_return"]
    ].copy()

    local["next_return_pct"] = 100 * local["split_next_return"]
    return summary, local


# Inspect the three worst episodes, starting with the two ~-100% losses.
summaries126 = []
windows126 = []

for _, r in test_bh126.head(3).iterrows():
    diagnostic, local = episode_diagnostics126(r["mint"], r["pool"])

    if diagnostic is not None:
        diagnostic["buy_hold_episode_return_pct"] = r["episode_return_pct"]
        summaries126.append(diagnostic)

        if local is not None:
            local = local.copy()
            local["mint"] = r["mint"]
            local["pool"] = r["pool"]
            windows126.append(local)

print("=== Worst held-out buy-and-hold episodes ===")
display(pd.DataFrame(summaries126).round(6))

if windows126:
    print("\n=== Price movements around each episode's worst one-step return ===")
    display(pd.concat(windows126, ignore_index=True))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "split_data120" not in globals():
    raise NameError("Run Step 120 first.")

df127 = split_data120["test"].copy()

group_cols127 = ["mint", "pool"]
order_cols127 = ["block", "archive_seq"]

df127 = df127.sort_values(
    group_cols127 + order_cols127,
    kind="mergesort",
).reset_index(drop=True)

g127 = df127.groupby(group_cols127, sort=False)

# Reconstruct the next observation explicitly to see exactly which
# row produces each recorded price jump.
for col in [
    "signature", "observed_time", "block", "archive_seq", "action", "price"
]:
    df127[f"next_{col}"] = g127[col].shift(-1)

df127["recomputed_next_return"] = (
    df127["next_price"] / df127["price"] - 1
)

# A large discontinuity means an absolute price move of at least 50%.
jumps127 = df127[
    df127["recomputed_next_return"].abs() >= 0.50
].copy()

print("Extreme transitions (absolute move >= 50%):", len(jumps127))

cols127 = [
    "mint", "pool",
    "signature", "observed_time", "block", "archive_seq", "action", "price",
    "next_signature", "next_observed_time", "next_block",
    "next_archive_seq", "next_action", "next_price",
    "recomputed_next_return",
]

display(
    jumps127[cols127]
    .sort_values("recomputed_next_return")
    .assign(
        current_price=lambda d: d["price"].map(lambda x: f"{x:.12g}"),
        following_price=lambda d: d["next_price"].map(lambda x: f"{x:.12g}"),
        jump_pct=lambda d: 100 * d["recomputed_next_return"],
    )[[
        "mint", "pool",
        "signature", "observed_time", "block", "archive_seq", "action",
        "current_price",
        "next_signature", "next_observed_time", "next_block",
        "next_archive_seq", "next_action", "following_price", "jump_pct",
    ]]
)

# Check timestamp consistency when rows are sorted by block/archive_seq.
df127["previous_time"] = g127["observed_time"].shift(1)
df127["time_delta_ms"] = (
    df127["observed_time"] - df127["previous_time"]
).dt.total_seconds() * 1000

same_pool_previous127 = df127["previous_time"].notna()
backward_time127 = df127.loc[
    same_pool_previous127 & df127["time_delta_ms"].lt(0)
]

print("\n=== Ordering consistency ===")
print("Rows with a previous observation in the same pool:",
      int(same_pool_previous127.sum()))
print("Timestamp decreases in block/archive order:",
      len(backward_time127))

if not backward_time127.empty:
    display(backward_time127[[
        "mint", "pool",
        "signature", "observed_time", "block", "archive_seq",
        "price", "previous_time", "time_delta_ms",
    ]].head(15))

# Show a small local sequence around the largest transitions.
print("\n=== Local context around the three largest downward jumps ===")

for _, jump in jumps127.nsmallest(3, "recomputed_next_return").iterrows():
    mask127 = (
        df127["mint"].eq(jump["mint"])
        & df127["pool"].eq(jump["pool"])
    )
    episode127 = df127.loc[mask127].reset_index(drop=True)

    positions127 = episode127.index[
        episode127["signature"].eq(jump["signature"])
    ].tolist()

    if not positions127:
        continue

    pos127 = positions127[0]
    start127 = max(0, pos127 - 3)
    end127 = min(len(episode127), pos127 + 5)

    print(
        f"\nMint={jump['mint']} | Pool={jump['pool']} | "
        f"Jump={100 * jump['recomputed_next_return']:.2f}%"
    )

    display(episode127.iloc[start127:end127][[
        "observed_time", "block", "archive_seq",
        "signature", "action", "price", "recomputed_next_return",
    ]])

In [ ]:
import base64
import numpy as np
import pandas as pd
from IPython.display import display

if "jumps127" not in globals() or jumps127.empty:
    raise NameError("Run Step 127 first; jumps127 is missing.")
if "fetch_transaction" not in globals():
    raise NameError("fetch_transaction is missing.")

def decode_pool_event128(signature, expected_pool):
    tx = fetch_transaction(str(signature))

    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    logs = ((tx or {}).get("meta") or {}).get("logMessages") or []

    for line in logs:
        marker = "Program data: "
        if marker not in line:
            continue

        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(),
                validate=True,
            )
        except Exception:
            continue

        for event_name, discriminator in event_discriminators.items():
            if not raw.startswith(discriminator):
                continue

            try:
                ev = decode_all_fields(raw, event_fields[event_name])
            except Exception:
                continue

            if ev.get("pool") == expected_pool:
                return event_name, ev

    raise ValueError(
        f"No matching BuyEvent/SellEvent for pool {expected_pool}"
    )


def reserve_spot_price128(quote_raw, base_raw):
    """SOL per whole token, given lamport and token-atom reserves."""
    quote_raw = int(quote_raw)
    base_raw = int(base_raw)

    if quote_raw <= 0 or base_raw <= 0:
        return np.nan

    return (
        quote_raw / LAMPORTS_PER_SOL
    ) / (
        base_raw / token_scale
    )


def raw_trade_price128(quote_raw, base_raw):
    """SOL per whole token implied by raw quote/base trade amounts."""
    return reserve_spot_price128(quote_raw, base_raw)


rows128 = []
errors128 = []

# Inspect the price recorded for both sides of each extreme transition.
for _, jump in jumps127.iterrows():
    expected_pool = str(jump["pool"])
    jump_pct = 100 * float(jump["recomputed_next_return"])

    endpoints = [
        {
            "endpoint": "current",
            "signature": str(jump["signature"]),
            "archive_price": float(jump["price"]),
            "observed_time": jump["observed_time"],
            "block": jump["block"],
            "archive_seq": jump["archive_seq"],
            "action": jump["action"],
        },
        {
            "endpoint": "next",
            "signature": str(jump["next_signature"]),
            "archive_price": float(jump["next_price"]),
            "observed_time": jump["next_observed_time"],
            "block": jump["next_block"],
            "archive_seq": jump["next_archive_seq"],
            "action": jump["next_action"],
        },
    ]

    for endpoint in endpoints:
        try:
            event_name, ev = decode_pool_event128(
                endpoint["signature"], expected_pool
            )

            B = int(ev["pool_base_token_reserves"])
            Q = (
                int(ev["pool_quote_token_reserves"])
                + int(ev["virtual_quote_reserves"])
            )

            pre_spot = reserve_spot_price128(Q, B)

            if event_name == "BuyEvent":
                base_delta = int(ev["base_amount_out"])
                quote_delta = int(ev["quote_amount_in_with_lp_fee"])

                B_after = B - base_delta
                Q_after = Q + quote_delta

                execution_prices = {
                    "curve_execution_price": raw_trade_price128(
                        quote_delta, base_delta
                    ),
                    "trader_cost_price": raw_trade_price128(
                        int(ev["quote_amount_in"]), base_delta
                    ),
                }

            else:
                base_delta = int(ev["base_amount_in"])
                quote_delta = int(ev["quote_amount_out_without_lp_fee"])

                B_after = B + base_delta
                Q_after = Q - quote_delta

                execution_prices = {
                    "curve_execution_price": raw_trade_price128(
                        int(ev["quote_amount_out"]), base_delta
                    ),
                    "trader_cost_price": raw_trade_price128(
                        int(ev["user_quote_amount_out"]), base_delta
                    ),
                }

            post_spot = reserve_spot_price128(Q_after, B_after)

            result = {
                "jump_pct": jump_pct,
                "endpoint": endpoint["endpoint"],
                "signature": endpoint["signature"],
                "action_archive": endpoint["action"],
                "event_name": event_name,
                "ix_name": ev.get("ix_name"),
                "block": endpoint["block"],
                "archive_seq": endpoint["archive_seq"],
                "observed_time": endpoint["observed_time"],
                "archive_price": endpoint["archive_price"],
                "reserve_spot_before": pre_spot,
                "reserve_spot_after_est": post_spot,
                "archive_vs_spot_before": (
                    endpoint["archive_price"] / pre_spot
                    if pre_spot > 0 else np.nan
                ),
                "archive_vs_spot_after": (
                    endpoint["archive_price"] / post_spot
                    if post_spot > 0 else np.nan
                ),
            }

            for name, value in execution_prices.items():
                result[name] = value
                result[f"archive_vs_{name}"] = (
                    endpoint["archive_price"] / value
                    if value > 0 else np.nan
                )

            rows128.append(result)

        except Exception as exc:
            errors128.append({
                "signature": endpoint["signature"],
                "endpoint": endpoint["endpoint"],
                "error": repr(exc),
            })

diagnostics128 = pd.DataFrame(rows128)

if not diagnostics128.empty:
    print("=== Archived vs. on-chain-derived prices ===")
    display(
        diagnostics128[[
            "jump_pct",
            "endpoint",
            "action_archive",
            "event_name",
            "ix_name",
            "block",
            "archive_price",
            "reserve_spot_before",
            "reserve_spot_after_est",
            "archive_vs_spot_before",
            "archive_vs_spot_after",
            "curve_execution_price",
            "trader_cost_price",
        ]].round(12)
    )

if errors128:
    print("\n=== Decode/fetch issues ===")
    display(pd.DataFrame(errors128))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "market118" not in globals():
    raise NameError("Run Step 118 first.")
if "feature_cols118" not in globals():
    raise NameError("feature_cols118 is missing.")

group_cols129 = ["mint", "pool"]
feature_cols129 = list(feature_cols118)
min_steps129 = 20

# Start with the feature-engineered data. Features are historical:
# lags, rolling statistics, and the shifted prior-buy ratio.
source129 = market118.copy()
source129["price"] = pd.to_numeric(source129["price"], errors="coerce")

source129 = source129[
    source129["price"].gt(0)
    & source129["block"].notna()
    & source129["archive_seq"].notna()
].dropna(subset=feature_cols129).copy()

# Canonical transaction order: block first, archive sequence within block.
source129 = source129.sort_values(
    ["block", "archive_seq"],
    kind="mergesort",
).reset_index(drop=True)

n129 = len(source129)
if n129 < 100:
    raise ValueError(f"Unexpectedly small source dataset: {n129} rows.")

# Choose cut points near 70% and 85% of rows, then move each
# boundary to a block boundary to avoid splitting one block.
row_cut_train129 = max(1, int(n129 * 0.70)) - 1
row_cut_val129 = max(2, int(n129 * 0.85)) - 1

train_block_max129 = source129.iloc[row_cut_train129]["block"]
val_block_max129 = source129.iloc[row_cut_val129]["block"]

# Ensure the two boundaries are strictly ordered by block.
if val_block_max129 <= train_block_max129:
    later_blocks129 = source129.loc[
        source129["block"] > train_block_max129, "block"
    ]
    if later_blocks129.empty:
        raise ValueError("Could not create a distinct validation/test boundary.")
    val_block_max129 = later_blocks129.iloc[0]

raw_splits129 = {
    "train": source129[
        source129["block"] <= train_block_max129
    ].copy(),

    "validation": source129[
        (source129["block"] > train_block_max129)
        & (source129["block"] <= val_block_max129)
    ].copy(),

    "test": source129[
        source129["block"] > val_block_max129
    ].copy(),
}


def prepare_split129(frame):
    # Keep canonical chain order within each mint/pool.
    frame = frame.sort_values(
        ["block", "archive_seq"],
        kind="mergesort",
    ).copy()

    # A next-event target is generated only from rows in this split.
    frame["split_next_price"] = (
        frame.groupby(group_cols129, sort=False)["price"].shift(-1)
    )
    frame["split_next_return"] = (
        frame["split_next_price"] / frame["price"] - 1
    )

    frame = frame.dropna(
        subset=feature_cols129 + ["split_next_return"]
    ).copy()

    episode_sizes = frame.groupby(group_cols129, sort=False).size()
    eligible = episode_sizes[episode_sizes >= min_steps129].index

    row_keys = pd.MultiIndex.from_frame(frame[group_cols129])
    eligible_keys = pd.MultiIndex.from_tuples(
        list(eligible),
        names=group_cols129,
    )

    return frame.loc[row_keys.isin(eligible_keys)].copy()


split_data129 = {
    name: prepare_split129(frame)
    for name, frame in raw_splits129.items()
}

# Check that a signature does not occur in more than one split.
signature_sets129 = {
    name: set(frame["signature"].astype(str))
    for name, frame in raw_splits129.items()
}

assert signature_sets129["train"].isdisjoint(signature_sets129["validation"])
assert signature_sets129["train"].isdisjoint(signature_sets129["test"])
assert signature_sets129["validation"].isdisjoint(signature_sets129["test"])

print("=== Chain-order split boundaries ===")
print("Train max block:", raw_splits129["train"]["block"].max())
print("Validation block range:",
      raw_splits129["validation"]["block"].min(),
      "to", raw_splits129["validation"]["block"].max())
print("Test min block:", raw_splits129["test"]["block"].min())

print("\n=== Eligible data after target recomputation ===")
display(pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "episodes": frame[group_cols129].drop_duplicates().shape[0],
        "min_episode_steps": (
            frame.groupby(group_cols129).size().min()
            if len(frame) else None
        ),
    }
    for name, frame in split_data129.items()
]))

# Recompute baseline metrics on the corrected splits.
buy_fee129 = sum(int(v) for v in fee_rates["buy"].values()) / 10_000
sell_fee129 = sum(int(v) for v in fee_rates["sell"].values()) / 10_000

baseline_rows129 = []

for split_name, frame in split_data129.items():
    for (mint, pool), episode in frame.groupby(group_cols129, sort=False):
        returns = episode["split_next_return"].to_numpy(dtype=float)
        log_market_return = np.log1p(returns).sum()

        baseline_rows129.append({
            "split": split_name,
            "mint": mint,
            "pool": pool,
            "steps": len(episode),
            "policy": "always_cash",
            "return_pct": 0.0,
        })

        buy_hold_log_return = (
            np.log1p(-buy_fee129)
            + log_market_return
            + np.log1p(-sell_fee129)
        )

        baseline_rows129.append({
            "split": split_name,
            "mint": mint,
            "pool": pool,
            "steps": len(episode),
            "policy": "buy_and_hold",
            "return_pct": 100 * np.expm1(buy_hold_log_return),
        })

baseline_results129 = pd.DataFrame(baseline_rows129)

baseline_summary129 = (
    baseline_results129.groupby(["split", "policy"], sort=False)
    .agg(
        episodes=("return_pct", "size"),
        mean_return_pct=("return_pct", "mean"),
        median_return_pct=("return_pct", "median"),
        win_rate_pct=("return_pct", lambda s: 100 * (s > 0).mean()),
        worst_episode_pct=("return_pct", "min"),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
    )
    .reset_index()
)

print("\n=== Corrected baseline results ===")
display(baseline_summary129.round(4))

# Timestamp is diagnostic only; block/archive_seq defines the split order.
ordered_time129 = source129["observed_time"]
if not isinstance(ordered_time129.dtype, pd.DatetimeTZDtype):
    ordered_time129 = pd.to_datetime(ordered_time129, utc=True)

time_reversals129 = int(
    ordered_time129.diff().dt.total_seconds().lt(0).sum()
)

print("\nTimestamp reversals in block/archive order:", time_reversals129)
print("Signatures overlap across splits: 0")
print(
    "\nThis replaces the Step 120 split for further experiments. "
    "Treat earlier model evaluation as exploratory, not final."
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. Create fresh environments from the corrected split
# ------------------------------------------------------------
if "split_data129" not in globals():
    raise NameError("Run Step 129 first.")
if "PriceBasedPumpRLEnv" not in globals():
    raise NameError("Run Step 119 first.")

features130 = list(feature_cols129)
groups130 = ["mint", "pool"]
min_steps130 = 20

def make_env130(split_name):
    frame = split_data129[split_name].copy()
    frame["next_return"] = frame["split_next_return"]

    return PriceBasedPumpRLEnv(
        market_df=frame,
        feature_cols=features130,
        fee_rates=fee_rates,
        min_episode_steps=min_steps130,
    )

env_train130 = make_env130("train")
env_val130 = make_env130("validation")
env_test130 = make_env130("test")

print(
    "Episodes train / validation / test:",
    len(env_train130.episodes),
    len(env_val130.episodes),
    len(env_test130.episodes),
)

# ------------------------------------------------------------
# 2. Fit state discretization on TRAINING observations only
# ------------------------------------------------------------
train_features130 = pd.concat(
    [ep["data"][features130] for ep in env_train130.episodes],
    ignore_index=True,
)

bins130 = 3
edges130 = {}

for col in features130:
    edges130[col] = train_features130[col].quantile(
        [1 / 3, 2 / 3]
    ).to_numpy(dtype=float)

def state_id130(observation):
    code = 0

    for i, col in enumerate(features130):
        bucket = int(np.digitize(float(observation[i]), edges130[col]))
        bucket = min(max(bucket, 0), bins130 - 1)
        code = code * bins130 + bucket

    position = int(round(float(observation[-1])))
    position = min(max(position, 0), 1)

    return code * 2 + position

n_states130 = (bins130 ** len(features130)) * 2
q_table130 = np.zeros((n_states130, 2), dtype=np.float64)

# ------------------------------------------------------------
# 3. Train Q-learning exclusively on the training split
# ------------------------------------------------------------
rng130 = np.random.default_rng(130026)

epochs130 = 8
alpha130 = 0.10
gamma130 = 0.99
epsilon_start130 = 0.30
epsilon_end130 = 0.05

num_episodes130 = len(env_train130.episodes)
total_runs130 = epochs130 * num_episodes130
run_num130 = 0

train_log130 = []

for epoch in range(epochs130):
    order = rng130.permutation(num_episodes130)

    for episode_idx in order:
        fraction = run_num130 / max(total_runs130 - 1, 1)
        epsilon = (
            epsilon_start130
            + fraction * (epsilon_end130 - epsilon_start130)
        )
        run_num130 += 1

        obs, reset_info = env_train130.reset(
            episode_index=int(episode_idx)
        )

        done = False
        episode_reward = 0.0
        steps = 0

        while not done:
            s = state_id130(obs)

            if rng130.random() < epsilon:
                action = int(rng130.integers(0, 2))
            else:
                action = int(np.argmax(q_table130[s]))

            next_obs, reward, terminated, truncated, info = (
                env_train130.step(action)
            )
            done = terminated or truncated

            if done:
                target = float(reward)
            else:
                ns = state_id130(next_obs)
                target = float(reward) + gamma130 * float(
                    np.max(q_table130[ns])
                )

            q_table130[s, action] += alpha130 * (
                target - q_table130[s, action]
            )

            episode_reward += float(reward)
            steps += 1
            obs = next_obs

        train_log130.append({
            "epoch": epoch + 1,
            "episode": int(episode_idx),
            "steps": steps,
            "reward_log": episode_reward,
            "epsilon": epsilon,
        })

print("Q-table shape:", q_table130.shape)
display(
    pd.DataFrame(train_log130)
    .groupby("epoch")
    .agg(
        episodes=("episode", "size"),
        mean_episode_reward_log=("reward_log", "mean"),
        final_epsilon=("epsilon", "last"),
    )
    .reset_index()
    .round(5)
)

# ------------------------------------------------------------
# 4. Evaluate fixed policies on each split
# ------------------------------------------------------------
def evaluate_policy130(env, split_name, policy_name):
    rows = []

    for idx, episode in enumerate(env.episodes):
        obs, reset_info = env.reset(episode_index=idx)

        done = False
        total_log_reward = 0.0
        steps = 0
        long_steps = 0
        entries = 0

        while not done:
            if policy_name == "always_cash":
                action = 0
            elif policy_name == "buy_and_hold":
                action = 1
            elif policy_name == "q_learning":
                action = int(np.argmax(q_table130[state_id130(obs)]))
            else:
                raise ValueError(f"Unknown policy: {policy_name}")

            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            total_log_reward += float(reward)
            steps += 1
            long_steps += int(action == 1)
            entries += int(
                info["position_before"] == 0 and action == 1
            )

        rows.append({
            "split": split_name,
            "policy": policy_name,
            "mint": reset_info["mint"],
            "pool": reset_info["pool"],
            "steps": steps,
            "long_exposure_pct": 100 * long_steps / max(steps, 1),
            "entries": entries,
            "return_pct": 100 * np.expm1(total_log_reward),
        })

    return rows

results130 = []

for split_name, env in [
    ("train", env_train130),
    ("validation", env_val130),
    ("test", env_test130),
]:
    for policy in ["always_cash", "buy_and_hold", "q_learning"]:
        results130.extend(evaluate_policy130(env, split_name, policy))

evaluation130 = pd.DataFrame(results130)

summary130 = (
    evaluation130.groupby(["split", "policy"], sort=False)
    .agg(
        episodes=("return_pct", "size"),
        mean_return_pct=("return_pct", "mean"),
        median_return_pct=("return_pct", "median"),
        win_rate_pct=("return_pct", lambda s: 100 * (s > 0).mean()),
        worst_episode_pct=("return_pct", "min"),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
        mean_long_exposure_pct=("long_exposure_pct", "mean"),
        mean_entries=("entries", "mean"),
    )
    .reset_index()
)

print("\n=== Corrected-split evaluation ===")
display(summary130.round(4))

# Select by validation mean return only; don't tune on test.
validation_summary130 = summary130[
    summary130["split"].eq("validation")
].sort_values("mean_return_pct", ascending=False)

selected_policy130 = validation_summary130.iloc[0]["policy"]

print("\nSelected using validation:", selected_policy130)
print("\n=== Test results for the frozen candidates ===")
display(
    summary130[summary130["split"].eq("test")]
    .sort_values("mean_return_pct", ascending=False)
    .round(4)
)

print(
    "\nReminder: this is still a price-based benchmark with configured "
    "percentage fees, not a full integer-execution PumpSwap backtest."
)

In [ ]:
import pandas as pd
from IPython.display import display

if "raw_splits129" not in globals():
    raise NameError("Run Step 129 first.")

# Combine the chronological splits while retaining split labels.
parts131 = []

for split_name, frame in raw_splits129.items():
    tmp = frame.copy()
    tmp["split_name"] = split_name
    parts131.append(tmp)

protocol_data131 = pd.concat(parts131, ignore_index=True)

if "protocol" not in protocol_data131.columns:
    raise ValueError("Dataset has no protocol column.")

print("=== Protocol values ===")
display(
    protocol_data131["protocol"]
    .value_counts(dropna=False)
    .rename_axis("protocol")
    .reset_index(name="transactions")
)

print("\n=== Protocol by split and action ===")
display(
    protocol_data131.groupby(
        ["split_name", "protocol", "action"],
        dropna=False,
    )
    .size()
    .rename("transactions")
    .reset_index()
    .sort_values(["split_name", "protocol", "action"])
)

print("\n=== Pool-to-protocol consistency ===")
pool_protocol131 = (
    protocol_data131.groupby(["mint", "pool"], dropna=False)["protocol"]
    .nunique()
    .rename("distinct_protocols")
    .reset_index()
)

print(
    "Mint/pool groups associated with more than one protocol:",
    int((pool_protocol131["distinct_protocols"] > 1).sum()),
)

if (pool_protocol131["distinct_protocols"] > 1).any():
    inconsistent131 = pool_protocol131[
        pool_protocol131["distinct_protocols"] > 1
    ]
    display(inconsistent131.head(20))

print("\n=== Sample rows per protocol ===")
display(
    protocol_data131.groupby("protocol", dropna=False)
    .head(3)[
        ["mint", "pool", "block", "archive_seq",
         "observed_time", "action", "price", "protocol"]
    ]
)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "raw_splits129" not in globals():
    raise NameError("Run Step 129 first.")
if "feature_cols129" not in globals():
    raise NameError("feature_cols129 is missing.")

groups132 = ["mint", "pool"]
features132 = list(feature_cols129)
min_steps132 = 20

def prepare_pumpswap_split132(frame):
    # Only use records for the protocol whose execution formulas
    # have been validated in the earlier transaction audit.
    df = frame[
        frame["protocol"].astype(str).str.upper().eq("PUMPSWAP")
    ].copy()

    df = df.sort_values(
        ["block", "archive_seq"],
        kind="mergesort",
    )

    # Targets are recomputed inside this split; no transition crosses
    # the train/validation/test boundary.
    df["split_next_price"] = (
        df.groupby(groups132, sort=False)["price"].shift(-1)
    )
    df["split_next_return"] = (
        df["split_next_price"] / df["price"] - 1
    )

    df = df.dropna(
        subset=features132 + ["split_next_return"]
    ).copy()

    sizes = df.groupby(groups132, sort=False).size()
    eligible_groups = sizes[sizes >= min_steps132].index
    row_groups = pd.MultiIndex.from_frame(df[groups132])
    eligible_index = pd.MultiIndex.from_tuples(
        list(eligible_groups),
        names=groups132,
    )

    return df.loc[row_groups.isin(eligible_index)].copy()


pumpswap_splits132 = {
    split_name: prepare_pumpswap_split132(frame)
    for split_name, frame in raw_splits129.items()
}

print("=== PumpSwap-only split sizes ===")
display(pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "episodes": frame[groups132].drop_duplicates().shape[0],
        "min_episode_steps": (
            frame.groupby(groups132).size().min()
            if len(frame) else None
        ),
        "max_episode_steps": (
            frame.groupby(groups132).size().max()
            if len(frame) else None
        ),
    }
    for name, frame in pumpswap_splits132.items()
]))

# Baseline returns with the existing configured 1% buy and 1% sell costs.
buy_fee132 = sum(
    int(v) for v in fee_rates["buy"].values()
) / 10_000

sell_fee132 = sum(
    int(v) for v in fee_rates["sell"].values()
) / 10_000

baseline_rows132 = []

for split_name, frame in pumpswap_splits132.items():
    for (mint, pool), episode in frame.groupby(groups132, sort=False):
        returns = episode["split_next_return"].to_numpy(dtype=float)

        # Always cash: zero exposure and zero trading fees.
        baseline_rows132.append({
            "split": split_name,
            "mint": mint,
            "pool": pool,
            "steps": len(episode),
            "policy": "always_cash",
            "return_pct": 0.0,
        })

        # Fully enter, hold through the episode, then liquidate.
        hold_log_return = (
            np.log1p(-buy_fee132)
            + np.log1p(returns).sum()
            + np.log1p(-sell_fee132)
        )

        baseline_rows132.append({
            "split": split_name,
            "mint": mint,
            "pool": pool,
            "steps": len(episode),
            "policy": "buy_and_hold",
            "return_pct": 100 * np.expm1(hold_log_return),
        })

baseline_results132 = pd.DataFrame(baseline_rows132)

baseline_summary132 = (
    baseline_results132
    .groupby(["split", "policy"], sort=False)
    .agg(
        episodes=("return_pct", "size"),
        mean_return_pct=("return_pct", "mean"),
        median_return_pct=("return_pct", "median"),
        win_rate_pct=("return_pct", lambda s: 100 * (s > 0).mean()),
        worst_episode_pct=("return_pct", "min"),
        p10_return_pct=("return_pct", lambda s: s.quantile(0.10)),
        p90_return_pct=("return_pct", lambda s: s.quantile(0.90)),
    )
    .reset_index()
)

print("\n=== PumpSwap-only baseline results ===")
display(baseline_summary132.round(4))

print("\n=== PumpSwap-only worst test episodes ===")
display(
    baseline_results132[
        baseline_results132["split"].eq("test")
        & baseline_results132["policy"].eq("buy_and_hold")
    ]
    .sort_values("return_pct")
    .head(10)[["mint", "pool", "steps", "return_pct"]]
    .round(4)
)

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

if "jumps127" not in globals() or jumps127.empty:
    raise NameError("Run Step 127 first; jumps127 is missing.")

rows133 = []
errors133 = []

# Check both transactions on either side of each extreme transition.
for _, jump in jumps127.iterrows():
    for endpoint in ["current", "next"]:
        if endpoint == "current":
            sig = str(jump["signature"])
            observed_time = jump["observed_time"]
            dataset_block = int(jump["block"])
            archive_seq = int(jump["archive_seq"])
            price = float(jump["price"])
        else:
            sig = str(jump["next_signature"])
            observed_time = jump["next_observed_time"]
            dataset_block = int(jump["next_block"])
            archive_seq = int(jump["next_archive_seq"])
            price = float(jump["next_price"])

        try:
            tx = fetch_transaction(sig)

            if isinstance(tx, dict) and tx.get("result") is not None:
                tx = tx["result"]

            rpc_slot = tx.get("slot")
            rpc_block_time = tx.get("blockTime")
            rpc_time = (
                pd.to_datetime(rpc_block_time, unit="s", utc=True)
                if rpc_block_time is not None else pd.NaT
            )

            observed_time = pd.to_datetime(observed_time, utc=True)

            rows133.append({
                "mint": jump["mint"],
                "pool": jump["pool"],
                "jump_pct": 100 * float(jump["recomputed_next_return"]),
                "endpoint": endpoint,
                "signature": sig,
                "dataset_block": dataset_block,
                "rpc_slot": rpc_slot,
                "slot_matches": (
                    int(rpc_slot) == dataset_block
                    if rpc_slot is not None else None
                ),
                "archive_seq": archive_seq,
                "observed_time": observed_time,
                "rpc_block_time": rpc_time,
                "time_difference_seconds": (
                    (observed_time - rpc_time).total_seconds()
                    if pd.notna(rpc_time) else None
                ),
                "price": price,
            })

        except Exception as exc:
            errors133.append({
                "signature": sig,
                "endpoint": endpoint,
                "error": repr(exc),
            })

metadata133 = pd.DataFrame(rows133)

print("Transactions checked:", len(metadata133))
if not metadata133.empty:
    display(
        metadata133.sort_values(
            ["mint", "dataset_block", "archive_seq"]
        ).round(6)
    )

    print("\n=== Metadata consistency summary ===")
    display(pd.DataFrame([{
        "transactions_checked": len(metadata133),
        "RPC slots matching dataset block": int(
            metadata133["slot_matches"].fillna(False).sum()
        ),
        "rows with RPC block time": int(
            metadata133["rpc_block_time"].notna().sum()
        ),
        "rows where observed_time matches RPC block time within 1 second": int(
            (
                metadata133["time_difference_seconds"].abs() <= 1
            ).sum()
        ),
    }]))

if errors133:
    print("\nFetch errors:")
    display(pd.DataFrame(errors133))

In [ ]:
import base64
import numpy as np
import pandas as pd
from IPython.display import display

def event_for_signature134(signature, expected_pool):
    tx = fetch_transaction(str(signature))
    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    logs = ((tx or {}).get("meta") or {}).get("logMessages") or []

    for line in logs:
        marker = "Program data: "
        if marker not in line:
            continue
        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(), validate=True
            )
        except Exception:
            continue

        for event_name, discriminator in event_discriminators.items():
            if not raw.startswith(discriminator):
                continue
            try:
                ev = decode_all_fields(raw, event_fields[event_name])
            except Exception:
                continue
            if ev.get("pool") == expected_pool:
                return event_name, ev, tx

    raise ValueError(f"No matching pool event found for {signature}")


def event_reserve_state134(event_name, ev):
    B = int(ev["pool_base_token_reserves"])
    Q = (
        int(ev["pool_quote_token_reserves"])
        + int(ev["virtual_quote_reserves"])
    )

    if event_name == "BuyEvent":
        dB = -int(ev["base_amount_out"])
        dQ = int(ev["quote_amount_in_with_lp_fee"])
    elif event_name == "SellEvent":
        dB = int(ev["base_amount_in"])
        dQ = -int(ev["quote_amount_out_without_lp_fee"])
    else:
        raise ValueError(event_name)

    B_after, Q_after = B + dB, Q + dQ

    return {
        "event_name": event_name,
        "ix_name": ev.get("ix_name"),
        "base_before": B,
        "quote_effective_before": Q,
        "base_after_est": B_after,
        "quote_effective_after_est": Q_after,
        "spot_before": (Q / 1e9) / (B / 1e6),
        "spot_after_est": (
            (Q_after / 1e9) / (B_after / 1e6)
            if B_after > 0 and Q_after > 0 else np.nan
        ),
        "base_trade_raw": (
            int(ev["base_amount_out"])
            if event_name == "BuyEvent"
            else int(ev["base_amount_in"])
        ),
        "quote_curve_raw": (
            int(ev["quote_amount_in_with_lp_fee"])
            if event_name == "BuyEvent"
            else int(ev["quote_amount_out_without_lp_fee"])
        ),
    }


continuity_rows134 = []
archive_rows134 = []
failures134 = []

for _, jump in jumps127.iterrows():
    try:
        cur_name, cur_ev, cur_tx = event_for_signature134(
            jump["signature"], jump["pool"]
        )
        nxt_name, nxt_ev, nxt_tx = event_for_signature134(
            jump["next_signature"], jump["pool"]
        )

        cur = event_reserve_state134(cur_name, cur_ev)
        nxt = event_reserve_state134(nxt_name, nxt_ev)

        continuity_rows134.append({
            "mint": jump["mint"],
            "pool": jump["pool"],
            "jump_pct": 100 * float(jump["recomputed_next_return"]),
            "current_slot": cur_tx.get("slot"),
            "current_tx_index": cur_tx.get("transactionIndex"),
            "next_slot": nxt_tx.get("slot"),
            "next_tx_index": nxt_tx.get("transactionIndex"),
            "current_ix": cur["ix_name"],
            "next_ix": nxt["ix_name"],
            "current_spot_before": cur["spot_before"],
            "current_spot_after_est": cur["spot_after_est"],
            "next_spot_before": nxt["spot_before"],
            "next_to_current_post_spot_ratio": (
                nxt["spot_before"] / cur["spot_after_est"]
            ),
            "current_base_trade_raw": cur["base_trade_raw"],
            "current_quote_curve_raw": cur["quote_curve_raw"],
            "next_base_trade_raw": nxt["base_trade_raw"],
            "next_quote_curve_raw": nxt["quote_curve_raw"],
        })

        # Inspect archive rows between these two slots for the same pool.
        lo = min(int(cur_tx["slot"]), int(nxt_tx["slot"]))
        hi = max(int(cur_tx["slot"]), int(nxt_tx["slot"]))

        pool_rows = stratified_trades[
            stratified_trades["pool"].eq(jump["pool"])
            & stratified_trades["mint"].eq(jump["mint"])
            & stratified_trades["block"].between(lo, hi)
        ].sort_values(["block", "archive_seq"], kind="mergesort")

        archive_rows134.append({
            "mint": jump["mint"],
            "pool": jump["pool"],
            "jump_pct": 100 * float(jump["recomputed_next_return"]),
            "slot_start": lo,
            "slot_end": hi,
            "archive_rows_in_slot_range": len(pool_rows),
            "signatures_present": int(
                pool_rows["signature"].astype(str).isin(
                    [str(jump["signature"]), str(jump["next_signature"])]
                ).sum()
            ),
            "rows": pool_rows[[
                "signature", "block", "archive_seq",
                "observed_time", "action", "price"
            ]].to_dict("records"),
        })

    except Exception as exc:
        failures134.append({
            "mint": jump["mint"],
            "pool": jump["pool"],
            "error": repr(exc),
        })

print("=== Reserve continuity ===")
display(pd.DataFrame(continuity_rows134).round(10))

print("\n=== Archived rows in each slot interval ===")
for item in archive_rows134:
    print(
        f"\nJump {item['jump_pct']:.2f}% | "
        f"slots {item['slot_start']}–{item['slot_end']} | "
        f"archive rows: {item['archive_rows_in_slot_range']} | "
        f"endpoints found: {item['signatures_present']}/2"
    )
    display(pd.DataFrame(item["rows"]))

if failures134:
    print("\n=== RPC or decode issues ===")
    display(pd.DataFrame(failures134))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "jumps127" not in globals() or "event_for_signature134" not in globals():
    raise NameError("Run Steps 127 and 134 first.")

def price_sol_per_token135(quote_raw, base_raw):
    quote_raw = int(quote_raw)
    base_raw = int(base_raw)

    if quote_raw <= 0 or base_raw <= 0:
        return np.nan

    return (quote_raw / LAMPORTS_PER_SOL) / (base_raw / token_scale)


rows135 = []
errors135 = []

for _, jump in jumps127.iterrows():
    for endpoint in ["current", "next"]:
        signature = str(
            jump["signature"] if endpoint == "current"
            else jump["next_signature"]
        )
        archive_price = float(
            jump["price"] if endpoint == "current"
            else jump["next_price"]
        )

        try:
            event_name, ev, tx = event_for_signature134(
                signature, str(jump["pool"])
            )

            B = int(ev["pool_base_token_reserves"])
            Q_real = int(ev["pool_quote_token_reserves"])
            V = int(ev["virtual_quote_reserves"])

            # Event-state prices before this transaction.
            real_spot_before = price_sol_per_token135(Q_real, B)
            effective_spot_before = price_sol_per_token135(Q_real + V, B)

            if event_name == "BuyEvent":
                B_after = B - int(ev["base_amount_out"])
                Q_real_after = Q_real + int(
                    ev["quote_amount_in_with_lp_fee"]
                )
                curve_quote = int(ev["quote_amount_in_with_lp_fee"])
                base_delta = int(ev["base_amount_out"])
            else:
                B_after = B + int(ev["base_amount_in"])
                Q_real_after = Q_real - int(
                    ev["quote_amount_out_without_lp_fee"]
                )
                curve_quote = int(ev["quote_amount_out"])
                base_delta = int(ev["base_amount_in"])

            real_spot_after = (
                price_sol_per_token135(Q_real_after, B_after)
                if Q_real_after > 0 and B_after > 0 else np.nan
            )

            rows135.append({
                "jump_pct": 100 * float(jump["recomputed_next_return"]),
                "endpoint": endpoint,
                "signature": signature,
                "event_name": event_name,
                "slot": tx.get("slot"),
                "transaction_index": tx.get("transactionIndex"),
                "archive_price": archive_price,
                "real_reserve_spot_before": real_spot_before,
                "effective_spot_before": effective_spot_before,
                "real_reserve_spot_after_est": real_spot_after,
                "archive_over_real_spot": (
                    archive_price / real_spot_before
                    if real_spot_before > 0 else np.nan
                ),
                "archive_over_effective_spot": (
                    archive_price / effective_spot_before
                    if effective_spot_before > 0 else np.nan
                ),
                "real_spot_next_over_current_post": np.nan,
            })

        except Exception as exc:
            errors135.append({
                "signature": signature,
                "endpoint": endpoint,
                "error": repr(exc),
            })

prices135 = pd.DataFrame(rows135)

# Compare each next event's pre-state with the prior endpoint's
# reconstructed post-state.
for _, jump in jumps127.iterrows():
    mask = (
        prices135["jump_pct"].eq(
            100 * float(jump["recomputed_next_return"])
        )
    )
    cur_idx = prices135.index[mask & prices135["endpoint"].eq("current")]
    next_idx = prices135.index[mask & prices135["endpoint"].eq("next")]

    if len(cur_idx) == 1 and len(next_idx) == 1:
        cur_row = prices135.loc[cur_idx[0]]
        next_row = prices135.loc[next_idx[0]]

        ratio = (
            next_row["real_reserve_spot_before"]
            / cur_row["real_reserve_spot_after_est"]
        )

        prices135.loc[next_idx[0], "real_spot_next_over_current_post"] = ratio

print("=== Archived price convention ===")
display(
    prices135[[
        "jump_pct",
        "endpoint",
        "event_name",
        "slot",
        "transaction_index",
        "archive_price",
        "real_reserve_spot_before",
        "effective_spot_before",
        "real_reserve_spot_after_est",
        "archive_over_real_spot",
        "archive_over_effective_spot",
        "real_spot_next_over_current_post",
    ]].round(10)
)

if errors135:
    print("\nFetch/decode issues:")
    display(pd.DataFrame(errors135))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

if "pumpswap_splits132" not in globals():
    raise NameError("Run Step 132 first.")

groups136 = ["mint", "pool"]
gap_frames136 = {}

for split_name, frame in pumpswap_splits132.items():
    df = frame.sort_values(
        ["mint", "pool", "block", "archive_seq"],
        kind="mergesort",
    ).copy()

    grouped = df.groupby(groups136, sort=False)

    df["next_block"] = grouped["block"].shift(-1)
    df["next_price"] = grouped["price"].shift(-1)

    # Number of Solana slots between this observation and the next
    # observation in the same mint/pool.
    df["slot_gap"] = df["next_block"] - df["block"]
    df["interval_return"] = df["next_price"] / df["price"] - 1
    df["abs_interval_return"] = df["interval_return"].abs()

    # Rows with no next observation in their pool cannot form a transition.
    transitions = df.dropna(
        subset=["slot_gap", "interval_return"]
    ).copy()

    if (transitions["slot_gap"] < 0).any():
        raise ValueError(
            f"Negative slot gap found in {split_name}; check ordering."
        )

    gap_frames136[split_name] = transitions

# Summarize the distribution of gaps and price changes.
summary_rows136 = []

for split_name, transitions in gap_frames136.items():
    gaps = transitions["slot_gap"].astype(float)
    abs_returns = transitions["abs_interval_return"].astype(float)

    summary_rows136.append({
        "split": split_name,
        "transitions": len(transitions),
        "same_slot_pct": 100 * (gaps == 0).mean(),
        "gap_over_10_slots_pct": 100 * (gaps > 10).mean(),
        "gap_over_100_slots_pct": 100 * (gaps > 100).mean(),
        "gap_over_500_slots_pct": 100 * (gaps > 500).mean(),
        "median_slot_gap": gaps.median(),
        "p90_slot_gap": gaps.quantile(0.90),
        "p99_slot_gap": gaps.quantile(0.99),
        "max_slot_gap": gaps.max(),
        "moves_over_10pct": int((abs_returns > 0.10).sum()),
        "moves_over_50pct": int((abs_returns > 0.50).sum()),
        "max_abs_return_pct": 100 * abs_returns.max(),
    })

print("=== Observation interval diagnostics ===")
display(pd.DataFrame(summary_rows136).round(4))

# Inspect transitions with extreme returns, showing their slot intervals.
extreme_rows136 = []

for split_name, transitions in gap_frames136.items():
    extreme = transitions[
        transitions["abs_interval_return"] >= 0.50
    ]

    for _, row in extreme.iterrows():
        extreme_rows136.append({
            "split": split_name,
            "mint": row["mint"],
            "pool": row["pool"],
            "block": int(row["block"]),
            "next_block": int(row["next_block"]),
            "slot_gap": int(row["slot_gap"]),
            "action": row["action"],
            "price": float(row["price"]),
            "next_price": float(row["next_price"]),
            "interval_return_pct": 100 * float(row["interval_return"]),
            "signature": row["signature"],
        })

print("\n=== Extreme-return transitions (absolute move >= 50%) ===")
extreme_df136 = pd.DataFrame(extreme_rows136)

if extreme_df136.empty:
    print("No extreme transitions found.")
else:
    display(
        extreme_df136.sort_values("interval_return_pct")
        .reset_index(drop=True)
    )

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

if "pumpswap_splits132" not in globals():
    raise NameError("Run Step 132 first.")

# Reconstruct candidate transitions in the current archive order.
pool_frames137 = []

for split_name, frame in pumpswap_splits132.items():
    df = frame.sort_values(
        ["mint", "pool", "block", "archive_seq"],
        kind="mergesort",
    ).copy()

    grp = df.groupby(["mint", "pool"], sort=False)

    df["next_signature"] = grp["signature"].shift(-1)
    df["next_block"] = grp["block"].shift(-1)
    df["next_archive_seq"] = grp["archive_seq"].shift(-1)
    df["next_price"] = grp["price"].shift(-1)

    df["slot_gap"] = df["next_block"] - df["block"]
    df["interval_return"] = df["next_price"] / df["price"] - 1
    df["split_name"] = split_name

    pool_frames137.append(df)

transitions137 = pd.concat(pool_frames137, ignore_index=True)

candidates137 = transitions137[
    transitions137["slot_gap"].eq(0)
    & transitions137["next_signature"].notna()
    & transitions137["interval_return"].abs().ge(0.50)
].copy()

candidates137["abs_return_pct"] = (
    100 * candidates137["interval_return"].abs()
)

print("Same-slot extreme transitions found:", len(candidates137))

# Check the largest cases first, limiting RPC calls for this diagnostic.
candidates137 = candidates137.sort_values(
    "abs_return_pct", ascending=False
).head(15)

def tx_metadata137(signature):
    tx = fetch_transaction(str(signature))

    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    return {
        "slot": tx.get("slot"),
        "transaction_index": tx.get("transactionIndex"),
        "block_time": tx.get("blockTime"),
    }

# Cache each signature to avoid duplicate RPC fetches.
tx_cache137 = {}
rows137, errors137 = [], []

for _, r in candidates137.iterrows():
    sig_a = str(r["signature"])
    sig_b = str(r["next_signature"])

    try:
        if sig_a not in tx_cache137:
            tx_cache137[sig_a] = tx_metadata137(sig_a)
        if sig_b not in tx_cache137:
            tx_cache137[sig_b] = tx_metadata137(sig_b)

        a = tx_cache137[sig_a]
        b = tx_cache137[sig_b]

        same_rpc_slot = (
            a["slot"] is not None
            and b["slot"] is not None
            and int(a["slot"]) == int(b["slot"])
        )

        idx_a = a["transaction_index"]
        idx_b = b["transaction_index"]

        if same_rpc_slot and idx_a is not None and idx_b is not None:
            idx_a, idx_b = int(idx_a), int(idx_b)
            if idx_a < idx_b:
                order_result = "archive order agrees with RPC"
            elif idx_a > idx_b:
                order_result = "ARCHIVE ORDER REVERSED"
            else:
                order_result = "same transaction index — investigate"
        else:
            order_result = "RPC metadata incomplete or slots differ"

        rows137.append({
            "split": r["split_name"],
            "mint": r["mint"],
            "pool": r["pool"],
            "block": int(r["block"]),
            "archive_seq_current": int(r["archive_seq"]),
            "archive_seq_next": int(r["next_archive_seq"]),
            "move_pct": 100 * float(r["interval_return"]),
            "action_current": r["action"],
            "signature_current": sig_a,
            "signature_next": sig_b,
            "rpc_slot_current": a["slot"],
            "rpc_index_current": a["transaction_index"],
            "rpc_slot_next": b["slot"],
            "rpc_index_next": b["transaction_index"],
            "order_check": order_result,
        })

    except Exception as exc:
        errors137.append({
            "signature_current": sig_a,
            "signature_next": sig_b,
            "error": repr(exc),
        })

order_audit137 = pd.DataFrame(rows137)

print("\n=== RPC order validation ===")
if not order_audit137.empty:
    display(
        order_audit137[[
            "split", "block", "move_pct",
            "archive_seq_current", "archive_seq_next",
            "rpc_index_current", "rpc_index_next",
            "action_current", "order_check",
            "signature_current", "signature_next",
        ]].round(4)
    )

    print("\nOrder-check counts:")
    display(
        order_audit137["order_check"]
        .value_counts(dropna=False)
        .rename_axis("order_check")
        .reset_index(name="transitions")
    )

if errors137:
    print("\nRPC or metadata errors:")
    display(pd.DataFrame(errors137))

In [ ]:
import inspect
import re

print("=== fetch_transaction implementation ===")

try:
    src138 = inspect.getsource(fetch_transaction)

    # Redact common inline API-key assignments and query parameters.
    src138 = re.sub(
        r"""(?i)(api[_-]?key\s*["']?\s*[:=]\s*["']?)[^&\s,"']+""",
        r"\1<REDACTED>",
        src138,
    )
    src138 = re.sub(
        r"""(?i)([?&]api[_-]?key=)[^&\s"' ]+""",
        r"\1<REDACTED>",
        src138,
    )

    print(src138)

except (OSError, TypeError) as exc:
    print("Source unavailable:", exc)

print("\n=== Candidate RPC configuration names ===")

for name, value in sorted(globals().items()):
    low = name.lower()

    if any(term in low for term in [
        "rpc_url", "rpc_endpoint", "helius_url",
        "solana_url", "rpc_client", "helius_client",
    ]):
        if isinstance(value, str):
            # Never display the full endpoint or its query parameters.
            from urllib.parse import urlparse
            parsed = urlparse(value)
            host = parsed.hostname or "(not a standard URL)"
            print(f"{name}: string endpoint configured; host={host}")
        else:
            print(f"{name}: {type(value).__name__}")

print("\n=== Dataset block coverage ===")
if "stratified_trades" in globals():
    print("Unique signatures:", stratified_trades["signature"].nunique())
    print("Unique blocks:", stratified_trades["block"].nunique())

In [ ]:
import requests
import pandas as pd
from IPython.display import display

if "order_audit137" not in globals():
    raise NameError("Run Step 137 first; order_audit137 is missing.")

if "RPC_URL" not in globals():
    raise NameError("RPC_URL is not configured.")

reversed139 = order_audit137[
    order_audit137["order_check"].eq("ARCHIVE ORDER REVERSED")
].copy()

if reversed139.empty:
    raise ValueError("No reversed transitions found in order_audit137.")

# Test up to three distinct affected slots.
test_slots139 = (
    reversed139["block"].astype(int).drop_duplicates().head(3).tolist()
)

def rpc_get_block_signatures139(slot):
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "getBlock",
        "params": [
            int(slot),
            {
                "commitment": "finalized",
                "transactionDetails": "signatures",
                "rewards": False,
                "maxSupportedTransactionVersion": 0,
            },
        ],
    }

    response = requests.post(RPC_URL, json=payload, timeout=40)
    response.raise_for_status()
    data = response.json()

    if data.get("error"):
        raise RuntimeError(f"getBlock error for slot {slot}: {data['error']}")

    result = data.get("result")
    if result is None:
        raise RuntimeError(f"No finalized block result for slot {slot}")

    transactions = result.get("transactions")
    if not isinstance(transactions, list):
        raise ValueError(
            f"Unexpected transactions payload for slot {slot}: "
            f"{type(transactions).__name__}"
        )

    def extract_signature(entry):
        # getBlock with transactionDetails='signatures' commonly
        # returns a list of signature strings. Handle object forms too.
        if isinstance(entry, str):
            return entry

        if isinstance(entry, list):
            return entry[0] if entry and isinstance(entry[0], str) else None

        if isinstance(entry, dict):
            sigs = entry.get("signatures")

            if sigs is None:
                tx = entry.get("transaction")
                if isinstance(tx, dict):
                    sigs = tx.get("signatures")
                elif isinstance(tx, list) and tx:
                    sigs = tx

            if isinstance(sigs, list) and sigs:
                return sigs[0]

        return None

    signatures = [
        extract_signature(entry)
        for entry in transactions
    ]

    if any(sig is None for sig in signatures):
        raise ValueError(
            f"Could not extract every transaction signature in slot {slot}"
        )

    return {signature: index for index, signature in enumerate(signatures)}


block_signature_map139 = {}
rows139 = []
errors139 = []

for slot in test_slots139:
    try:
        block_signature_map139[slot] = rpc_get_block_signatures139(slot)
    except Exception as exc:
        errors139.append({"slot": slot, "error": repr(exc)})

for _, row in reversed139[
    reversed139["block"].astype(int).isin(block_signature_map139)
].iterrows():
    slot = int(row["block"])
    index_map = block_signature_map139[slot]

    sig_a = str(row["signature_current"])
    sig_b = str(row["signature_next"])

    index_a = index_map.get(sig_a)
    index_b = index_map.get(sig_b)

    rows139.append({
        "slot": slot,
        "signature_current": sig_a,
        "signature_next": sig_b,
        "getTransaction_index_current": row["rpc_index_current"],
        "getTransaction_index_next": row["rpc_index_next"],
        "getBlock_index_current": index_a,
        "getBlock_index_next": index_b,
        "both_signatures_found": (
            index_a is not None and index_b is not None
        ),
        "canonical_order": (
            "current-before-next" if index_a < index_b
            else "next-before-current" if index_a > index_b
            else "unresolved"
        ) if index_a is not None and index_b is not None else "unresolved",
    })

print("Tested slots:", len(block_signature_map139), "/", len(test_slots139))
print("Transactions in sampled blocks:", {
    slot: len(sig_map)
    for slot, sig_map in block_signature_map139.items()
})

if rows139:
    display(pd.DataFrame(rows139))

if errors139:
    print("\nRPC errors:")
    display(pd.DataFrame(errors139))

if rows139:
    checks = pd.DataFrame(rows139)
    found = checks["both_signatures_found"].all()
    indices_agree = (
        checks["getBlock_index_current"].astype("Int64").equals(
            checks["getTransaction_index_current"].astype("Int64")
        )
        and checks["getBlock_index_next"].astype("Int64").equals(
            checks["getTransaction_index_next"].astype("Int64")
        )
    )
    print("\nBoth signatures found in each tested block:", found)
    print("getBlock positions agree with getTransaction indexes:", indices_agree)

In [ ]:
import requests
import json

slot140 = 453259363  # One of the slots where archive order was reversed.

payload140 = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "getBlock",
    "params": [
        slot140,
        {
            "commitment": "finalized",
            "transactionDetails": "signatures",
            "rewards": False,
            "maxSupportedTransactionVersion": 0,
        },
    ],
}

response140 = requests.post(
    RPC_URL,
    json=payload140,
    timeout=40,
)

print("HTTP status:", response140.status_code)
response140.raise_for_status()

data140 = response140.json()

print("\nTop-level keys:", list(data140.keys()))

if data140.get("error"):
    print("\nRPC error:")
    print(json.dumps(data140["error"], indent=2)[:2000])
else:
    result140 = data140.get("result")

    print("\nResult type:", type(result140).__name__)

    if isinstance(result140, dict):
        print("Result keys:", list(result140.keys()))

        for key in ["parentSlot", "blockTime", "blockHeight", "transactions"]:
            value = result140.get(key)

            print(f"\n{key}:")
            print("  type:", type(value).__name__)

            if isinstance(value, list):
                print("  length:", len(value))
                print("  first item:", repr(value[0])[:500] if value else "(empty)")
            else:
                print("  preview:", repr(value)[:1000])
    else:
        print("Result preview:", repr(result140)[:2000])

In [ ]:
import requests
import pandas as pd

slot140 = 453259363

# Find PumpSwap archive rows in this slot
slot_df140 = stratified_trades.copy()
slot_df140["_slot_num"] = pd.to_numeric(
    slot_df140["block"], errors="coerce"
)

slot_rows140 = slot_df140[
    (slot_df140["_slot_num"] == slot140)
    & (slot_df140["protocol"].astype(str).str.upper() == "PUMPSWAP")
].copy()

print("PumpSwap archive rows in slot:", len(slot_rows140))

if slot_rows140.empty:
    print("No PumpSwap rows found for this slot.")
else:
    display(
        slot_rows140[
            ["signature", "pool", "archive_seq", "action", "price"]
        ].head(20)
    )

    # Ask Helius for the true transaction index of each transaction
    # involving each pool in this slot.
    index_by_signature140 = {}
    errors140 = []

    pools140 = (
        slot_rows140["pool"]
        .dropna()
        .astype(str)
        .unique()
    )

    for pool_address140 in pools140:
        payload140 = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "getTransactionsForAddress",
            "params": [
                pool_address140,
                {
                    "transactionDetails": "signatures",
                    "sortOrder": "asc",
                    "limit": 1000,
                    "commitment": "finalized",
                    "filters": {
                        "slot": {
                            "gte": slot140,
                            "lte": slot140
                        }
                    }
                }
            ]
        }

        try:
            response140 = requests.post(
                RPC_URL, json=payload140, timeout=40
            )
            response140.raise_for_status()
            body140 = response140.json()

            if body140.get("error"):
                errors140.append({
                    "pool": pool_address140,
                    "error": body140["error"]
                })
                continue

            result140 = body140.get("result") or {}
            transactions140 = result140.get("data", [])

            for tx140 in transactions140:
                if int(tx140.get("slot", -1)) != slot140:
                    continue

                sig140 = tx140.get("signature")
                idx140 = tx140.get("transactionIndex")

                if sig140 is not None and idx140 is not None:
                    index_by_signature140[sig140] = int(idx140)

        except Exception as exc:
            errors140.append({
                "pool": pool_address140,
                "error": str(exc)
            })

    slot_rows140["transactionIndex"] = (
        slot_rows140["signature"].map(index_by_signature140)
    )

    matched140 = slot_rows140["transactionIndex"].notna().sum()
    print(f"\nRows matched to transactionIndex: {matched140} "
          f"/ {len(slot_rows140)}")

    if matched140:
        display(
            slot_rows140[
                [
                    "block", "transactionIndex", "archive_seq",
                    "signature", "pool", "action", "price"
                ]
            ]
            .sort_values(
                ["block", "transactionIndex"],
                kind="stable"
            )
        )

    if errors140:
        print("\nRPC errors (first 5):")
        for err140 in errors140[:5]:
            print(err140)

In [ ]:
import pandas as pd

# Start from the full archive, then isolate PumpSwap.
pumpswap_order_check = stratified_trades[
    stratified_trades["protocol"].astype(str).str.upper() == "PUMPSWAP"
].copy()

pumpswap_order_check["block"] = pd.to_numeric(
    pumpswap_order_check["block"], errors="coerce"
)

# Count transactions per pool per slot.
pool_slot_counts = (
    pumpswap_order_check
    .dropna(subset=["pool", "block"])
    .groupby(["pool", "block"])
    .size()
    .rename("trade_count")
    .reset_index()
)

duplicate_pool_slots = pool_slot_counts[
    pool_slot_counts["trade_count"] > 1
].copy()

print("PumpSwap archive rows:", len(pumpswap_order_check))
print("Unique pools:", pumpswap_order_check["pool"].nunique())
print("Unique slots:", pumpswap_order_check["block"].nunique())
print("Pool-slot groups with multiple trades:", len(duplicate_pool_slots))
print(
    "Rows in those groups:",
    int(duplicate_pool_slots["trade_count"].sum())
)
print(
    "Additional rows beyond one per group:",
    int((duplicate_pool_slots["trade_count"] - 1).sum())
)

print("\nLargest same-pool, same-slot groups:")
display(
    duplicate_pool_slots
    .sort_values("trade_count", ascending=False)
    .head(20)
)

In [ ]:
import requests
import pandas as pd

# Prepare PumpSwap archive rows.
pilot_archive = stratified_trades[
    stratified_trades["protocol"].astype(str).str.upper() == "PUMPSWAP"
].copy()

pilot_archive["block"] = pd.to_numeric(
    pilot_archive["block"], errors="coerce"
)

# Select the pool with the largest same-slot group.
pilot_pool = "C6rTTL3RvRV5WBdR2jGTnKPC3jVohb2jbLkRebhQQC15"

pilot_pool_rows = pilot_archive[
    pilot_archive["pool"].astype(str) == pilot_pool
].copy()

pilot_min_slot = int(pilot_pool_rows["block"].min())
pilot_max_slot = int(pilot_pool_rows["block"].max())
pilot_signatures = set(pilot_pool_rows["signature"].astype(str))

print("Pool archive rows:", len(pilot_pool_rows))
print("Slot range:", pilot_min_slot, "to", pilot_max_slot)
print("Unique archive signatures:", len(pilot_signatures))

def fetch_gta_page(pool, min_slot, max_slot, pagination_token=None):
    config = {
        "transactionDetails": "signatures",
        "sortOrder": "asc",
        "limit": 1000,
        "commitment": "finalized",
        "filters": {
            "slot": {
                "gte": min_slot,
                "lte": max_slot
            }
        }
    }

    if pagination_token:
        config["paginationToken"] = pagination_token

    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "getTransactionsForAddress",
        "params": [pool, config]
    }

    response = requests.post(RPC_URL, json=payload, timeout=60)
    response.raise_for_status()
    body = response.json()

    if body.get("error"):
        raise RuntimeError(
            f"Helius RPC error: {body['error']}"
        )

    result = body.get("result")
    if not isinstance(result, dict):
        raise RuntimeError(
            f"Unexpected response shape: {str(body)[:800]}"
        )

    return result

page1 = fetch_gta_page(
    pilot_pool, pilot_min_slot, pilot_max_slot
)

page1_data = page1.get("data", [])
page1_matches = [
    tx for tx in page1_data
    if str(tx.get("signature", "")) in pilot_signatures
]

print("\nPAGE 1")
print("Result keys:", list(page1.keys()))
print("Records returned:", len(page1_data))
print("Archive signature matches:", len(page1_matches))
print("Has next page:", bool(page1.get("paginationToken")))

if page1_data:
    print("First record keys:", list(page1_data[0].keys()))
    display(pd.DataFrame([
        {
            "slot": tx.get("slot"),
            "transactionIndex": tx.get("transactionIndex"),
            "signature": tx.get("signature"),
            "in_archive": str(tx.get("signature", "")) in pilot_signatures
        }
        for tx in page1_data[:5]
    ]))

page2 = None
page2_data = []
page2_matches = []

if page1.get("paginationToken"):
    page2 = fetch_gta_page(
        pilot_pool,
        pilot_min_slot,
        pilot_max_slot,
        page1["paginationToken"]
    )
    page2_data = page2.get("data", [])
    page2_matches = [
        tx for tx in page2_data
        if str(tx.get("signature", "")) in pilot_signatures
    ]

    print("\nPAGE 2")
    print("Records returned:", len(page2_data))
    print("Archive signature matches:", len(page2_matches))
    print("Has next page:", bool(page2.get("paginationToken")))

    if page2_data:
        display(pd.DataFrame([
            {
                "slot": tx.get("slot"),
                "transactionIndex": tx.get("transactionIndex"),
                "signature": tx.get("signature"),
                "in_archive": str(tx.get("signature", "")) in pilot_signatures
            }
            for tx in page2_data[:5]
        ]))

all_pilot_test_data = page1_data + page2_data
pilot_test_matched = {
    str(tx.get("signature"))
    for tx in all_pilot_test_data
    if str(tx.get("signature", "")) in pilot_signatures
}

print("\nPILOT SUMMARY")
print("Pages tested:", 1 + int(page2 is not None))
print("Archive signatures matched:", len(pilot_test_matched))
print("Has transactionIndex on all matched rows:",
      all(
          tx.get("transactionIndex") is not None
          for tx in all_pilot_test_data
          if str(tx.get("signature", "")) in pilot_signatures
      ))

In [ ]:
# Fetch the complete slot range for this one pool.
pilot_all_records = []
pilot_seen_tokens = set()
pilot_token = None
pilot_page_count = 0
pilot_page_limit = 100  # Safety cap; do not run indefinitely.
pilot_page_ranges = []

while True:
    page_result = fetch_gta_page(
        pilot_pool,
        pilot_min_slot,
        pilot_max_slot,
        pilot_token
    )

    page_data = page_result.get("data", [])
    pilot_page_count += 1
    pilot_all_records.extend(page_data)

    page_slots = [
        int(tx["slot"])
        for tx in page_data
        if tx.get("slot") is not None
    ]

    pilot_page_ranges.append({
        "page": pilot_page_count,
        "records": len(page_data),
        "first_slot": min(page_slots) if page_slots else None,
        "last_slot": max(page_slots) if page_slots else None,
    })

    next_token = page_result.get("paginationToken")

    if not next_token:
        print("Reached the final page.")
        break

    if next_token in pilot_seen_tokens:
        raise RuntimeError(
            "Pagination token repeated; stopping to avoid a loop."
        )

    pilot_seen_tokens.add(next_token)
    pilot_token = next_token

    if pilot_page_count >= pilot_page_limit:
        print("Safety cap reached; pagination is incomplete.")
        break

# Summarize pagination progress.
print("Pages fetched:", pilot_page_count)
print("RPC records fetched:", len(pilot_all_records))
display(pd.DataFrame(pilot_page_ranges))

# Check signature uniqueness and archive coverage.
pilot_rpc_df = pd.DataFrame(pilot_all_records)

if pilot_rpc_df.empty:
    raise RuntimeError("The endpoint returned no records.")

pilot_rpc_df["signature"] = pilot_rpc_df["signature"].astype(str)
pilot_archive_signatures = set(
    pilot_pool_rows["signature"].astype(str)
)
pilot_rpc_signatures = set(pilot_rpc_df["signature"])

pilot_matched = pilot_archive_signatures & pilot_rpc_signatures
pilot_missing = pilot_archive_signatures - pilot_rpc_signatures

print("\nCOVERAGE")
print("Archive signatures:", len(pilot_archive_signatures))
print("Unique RPC signatures:", len(pilot_rpc_signatures))
print("Matched archive signatures:", len(pilot_matched))
print("Missing archive signatures:", len(pilot_missing))
print(
    "Archive coverage:",
    f"{len(pilot_matched) / max(1, len(pilot_archive_signatures)):.2%}"
)

# Verify transactionIndex completeness for matched archive trades.
pilot_rpc_archive = pilot_rpc_df[
    pilot_rpc_df["signature"].isin(pilot_archive_signatures)
].copy()

print(
    "Matched rows missing transactionIndex:",
    pilot_rpc_archive["transactionIndex"].isna().sum()
)

# Show examples if archive trades are absent from the RPC result.
if pilot_missing:
    display(
        pilot_pool_rows[
            pilot_pool_rows["signature"].astype(str).isin(pilot_missing)
        ][["block", "archive_seq", "signature", "action", "price"]]
        .head(15)
    )

In [ ]:
import requests
import pandas as pd
import time

# 1. Prepare PumpSwap archive.
pumpswap_archive_full = stratified_trades[
    stratified_trades["protocol"].astype(str).str.upper() == "PUMPSWAP"
].copy()

pumpswap_archive_full["block"] = pd.to_numeric(
    pumpswap_archive_full["block"], errors="coerce"
)

# Use only rows with the fields required for the pool-based query.
queryable_archive = pumpswap_archive_full.dropna(
    subset=["pool", "block", "signature"]
).copy()

queryable_archive["pool"] = queryable_archive["pool"].astype(str)
queryable_archive["signature"] = queryable_archive["signature"].astype(str)
queryable_archive["block"] = queryable_archive["block"].astype(int)

# 2. Reliable request helper with retry handling.
def fetch_gta_page_backfill(
    pool, min_slot, max_slot, pagination_token=None, retries=4
):
    config = {
        "transactionDetails": "signatures",
        "sortOrder": "asc",
        "limit": 1000,
        "commitment": "finalized",
        "filters": {
            "slot": {"gte": int(min_slot), "lte": int(max_slot)}
        }
    }

    if pagination_token:
        config["paginationToken"] = pagination_token

    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "getTransactionsForAddress",
        "params": [pool, config]
    }

    last_error = None

    for attempt in range(retries):
        try:
            response = requests.post(
                RPC_URL, json=payload, timeout=60
            )
            response.raise_for_status()
            body = response.json()

            if body.get("error"):
                raise RuntimeError(str(body["error"]))

            result = body.get("result")
            if not isinstance(result, dict) or not isinstance(
                result.get("data"), list
            ):
                raise RuntimeError(
                    f"Unexpected response structure: {str(body)[:500]}"
                )

            return result

        except Exception as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(1.5 * (2 ** attempt))

    raise RuntimeError(
        f"Request failed after {retries} attempts: {last_error}"
    )


# 3. Query each pool and collect its matching archive signatures.
pool_summaries = []
all_matched_index_records = []
pool_errors = []
total_requests = 0

pool_bounds = (
    queryable_archive.groupby("pool")["block"]
    .agg(["min", "max"])
    .reset_index()
    .sort_values("pool")
)

expected_archive_signatures = set(
    queryable_archive["signature"].unique()
)

print(
    f"Starting backfill: {len(pool_bounds)} pools, "
    f"{len(expected_archive_signatures):,} unique archive signatures."
)

for pool_number, row in enumerate(
    pool_bounds.itertuples(index=False), start=1
):
    pool = row.pool
    min_slot = int(row.min)
    max_slot = int(row.max)

    expected_pool = set(
        queryable_archive.loc[
            queryable_archive["pool"] == pool, "signature"
        ].unique()
    )

    pool_signatures_found = set()
    seen_tokens = set()
    pagination_token = None
    records_fetched = 0
    request_count = 0
    index_records_for_pool = []
    error_message = None

    print(
        f"\n[{pool_number}/{len(pool_bounds)}] "
        f"Pool {pool[:8]}... | {len(expected_pool):,} archive signatures | "
        f"slots {min_slot}–{max_slot}"
    )

    try:
        while True:
            result = fetch_gta_page_backfill(
                pool, min_slot, max_slot, pagination_token
            )
            total_requests += 1
            request_count += 1

            page_data = result["data"]

            # The pilot's final request returned an empty page.
            if not page_data:
                break

            records_fetched += len(page_data)

            for tx in page_data:
                sig = str(tx.get("signature", ""))

                if sig not in expected_pool:
                    continue

                pool_signatures_found.add(sig)

                tx_index = tx.get("transactionIndex")
                tx_slot = tx.get("slot")

                index_records_for_pool.append({
                    "signature": sig,
                    "rpc_slot": (
                        int(tx_slot) if tx_slot is not None else None
                    ),
                    "transactionIndex": (
                        int(tx_index) if tx_index is not None else None
                    ),
                    "pool_source": pool
                })

            next_token = result.get("paginationToken")

            if not next_token:
                break

            if next_token in seen_tokens:
                raise RuntimeError(
                    "Pagination token repeated; stopping this pool."
                )

            seen_tokens.add(next_token)
            pagination_token = next_token

        all_matched_index_records.extend(index_records_for_pool)

    except Exception as exc:
        error_message = str(exc)
        pool_errors.append({
            "pool": pool,
            "error": error_message
        })
        print("  ERROR:", error_message)

    missing_pool = expected_pool - pool_signatures_found
    coverage = (
        len(pool_signatures_found) / len(expected_pool)
        if expected_pool else 1.0
    )

    pool_summaries.append({
        "pool": pool,
        "archive_signatures": len(expected_pool),
        "matched_signatures": len(pool_signatures_found),
        "missing_signatures": len(missing_pool),
        "coverage_pct": round(coverage * 100, 4),
        "rpc_records_fetched": records_fetched,
        "requests": request_count,
        "error": error_message
    })

    print(
        f"  Matched {len(pool_signatures_found):,}/"
        f"{len(expected_pool):,} ({coverage:.2%}); "
        f"RPC records {records_fetched:,}; requests {request_count}"
    )

# 4. Build a signature -> (slot, transactionIndex) mapping.
index_records_df = pd.DataFrame(all_matched_index_records)

if index_records_df.empty:
    raise RuntimeError(
        "No matching transaction indices were collected. "
        "Check the RPC configuration and errors."
    )

# The same transaction may reference multiple pools.
# Its signature should always map to the same slot and transaction index.
mapping_conflicts = (
    index_records_df.groupby("signature")
    .agg(
        slot_versions=("rpc_slot", "nunique"),
        index_versions=("transactionIndex", "nunique")
    )
    .query("slot_versions > 1 or index_versions > 1")
)

index_map_clean = (
    index_records_df[
        ["signature", "rpc_slot", "transactionIndex"]
    ]
    .drop_duplicates()
)

# Retain only unambiguous mappings.
conflicting_signatures = set(mapping_conflicts.index)
index_map_clean = index_map_clean[
    ~index_map_clean["signature"].isin(conflicting_signatures)
].copy()

pumpswap_tx_index_map = index_map_clean.set_index("signature")

# 5. Join the index mapping back onto every PumpSwap archive row.
pumpswap_ordered = pumpswap_archive_full.copy()
pumpswap_ordered["signature"] = (
    pumpswap_ordered["signature"].astype(str)
)

pumpswap_ordered["transactionIndex"] = (
    pumpswap_ordered["signature"].map(
        pumpswap_tx_index_map["transactionIndex"]
    )
)

pumpswap_ordered["rpc_slot"] = (
    pumpswap_ordered["signature"].map(
        pumpswap_tx_index_map["rpc_slot"]
    )
)

pumpswap_ordered["slot_matches_rpc"] = (
    pd.to_numeric(pumpswap_ordered["block"], errors="coerce")
    == pumpswap_ordered["rpc_slot"]
)

# 6. Coverage and consistency diagnostics.
print("\n========== BACKFILL SUMMARY ==========")
print("PumpSwap archive rows:", len(pumpswap_ordered))
print("Unique archive signatures:", pumpswap_ordered["signature"].nunique())
print("Total RPC requests:", total_requests)
print("Unambiguous signature mappings:", len(pumpswap_tx_index_map))
print("Conflicting signature mappings:", len(conflicting_signatures))
print("Archive rows missing transactionIndex:",
      pumpswap_ordered["transactionIndex"].isna().sum())
print("Archive rows with missing/mismatched slot:",
      (~pumpswap_ordered["slot_matches_rpc"].fillna(False)).sum())
print("Pool query errors:", len(pool_errors))

print("\nPer-pool coverage:")
display(
    pd.DataFrame(pool_summaries).sort_values(
        ["missing_signatures", "pool"],
        ascending=[False, True]
    )
)

if not mapping_conflicts.empty:
    print("\nConflicting signature mappings (first 10):")
    display(mapping_conflicts.head(10))

missing_index_rows = pumpswap_ordered[
    pumpswap_ordered["transactionIndex"].isna()
]
if not missing_index_rows.empty:
    print("\nRows missing transactionIndex (first 10):")
    display(
        missing_index_rows[
            ["pool", "block", "signature", "archive_seq", "action"]
        ].head(10)
    )

slot_mismatch_rows = pumpswap_ordered[
    ~pumpswap_ordered["slot_matches_rpc"].fillna(False)
]
if not slot_mismatch_rows.empty:
    print("\nSlot mismatches (first 10):")
    display(
        slot_mismatch_rows[
            ["pool", "block", "rpc_slot", "signature", "archive_seq"]
        ].head(10)
    )

if pool_errors:
    print("\nPool errors (first 10):")
    display(pd.DataFrame(pool_errors).head(10))

In [ ]:
import numpy as np
import pandas as pd

# 1. Validate and prepare the backfilled archive.
pumpswap_fixed = pumpswap_ordered.copy()

pumpswap_fixed["block"] = pd.to_numeric(
    pumpswap_fixed["block"], errors="coerce"
)
pumpswap_fixed["transactionIndex"] = pd.to_numeric(
    pumpswap_fixed["transactionIndex"], errors="coerce"
)
pumpswap_fixed["price"] = pd.to_numeric(
    pumpswap_fixed["price"], errors="coerce"
)
pumpswap_fixed["archive_seq"] = pd.to_numeric(
    pumpswap_fixed["archive_seq"], errors="coerce"
)

assert pumpswap_fixed["transactionIndex"].notna().all(), (
    "Some rows have no transactionIndex."
)
assert pumpswap_fixed["block"].notna().all(), (
    "Some rows have no block/slot."
)

# A pool should normally correspond to a single mint in this dataset.
mints_per_pool = pumpswap_fixed.groupby("pool")["mint"].nunique()
print("Pools mapping to more than one mint:", int((mints_per_pool > 1).sum()))

# 2. Check whether archive_seq and actual transaction order disagree.
order_audit = pumpswap_fixed.dropna(
    subset=["pool", "block", "archive_seq", "transactionIndex"]
).copy()

group_keys = ["pool", "block"]

order_audit["archive_rank"] = (
    order_audit.groupby(group_keys)["archive_seq"]
    .rank(method="first")
)
order_audit["chain_rank"] = (
    order_audit.groupby(group_keys)["transactionIndex"]
    .rank(method="first")
)

order_audit["rank_disagrees"] = (
    order_audit["archive_rank"] != order_audit["chain_rank"]
)

groups_with_order_difference = (
    order_audit.loc[order_audit["rank_disagrees"]]
    .groupby(group_keys)
    .ngroups
)

multi_trade_groups = (
    order_audit.groupby(group_keys)
    .size()
    .gt(1)
    .sum()
)

duplicate_pool_slot_index = pumpswap_fixed.duplicated(
    ["pool", "block", "transactionIndex"], keep=False
).sum()

print("\n========== ORDER AUDIT ==========")
print("Archive rows:", len(pumpswap_fixed))
print("Pool-slot groups with multiple trades:", int(multi_trade_groups))
print(
    "Multi-trade groups where archive order differs:",
    int(groups_with_order_difference)
)
print("Rows whose within-group rank disagrees:", int(order_audit["rank_disagrees"].sum()))
print("Duplicate pool/slot/transactionIndex rows:", int(duplicate_pool_slot_index))

# 3. Sort by the authoritative order: pool, slot, transactionIndex.
pumpswap_fixed = pumpswap_fixed.sort_values(
    ["pool", "block", "transactionIndex"],
    kind="stable"
).reset_index(drop=True)

# Create transition features between consecutive archived trades
# for each mint/pool sequence.
sequence_keys = ["mint", "pool"]
g = pumpswap_fixed.groupby(sequence_keys, sort=False)

pumpswap_fixed["prev_signature"] = g["signature"].shift(1)
pumpswap_fixed["prev_block"] = g["block"].shift(1)
pumpswap_fixed["prev_transactionIndex"] = g["transactionIndex"].shift(1)
pumpswap_fixed["prev_price"] = g["price"].shift(1)

valid_transition = (
    pumpswap_fixed["price"].gt(0)
    & pumpswap_fixed["prev_price"].gt(0)
    & pumpswap_fixed["price"].notna()
    & pumpswap_fixed["prev_price"].notna()
)

pumpswap_fixed["price_ratio_prev"] = np.nan
pumpswap_fixed["pct_return_prev"] = np.nan
pumpswap_fixed["log_return_prev"] = np.nan

pumpswap_fixed.loc[valid_transition, "price_ratio_prev"] = (
    pumpswap_fixed.loc[valid_transition, "price"]
    / pumpswap_fixed.loc[valid_transition, "prev_price"]
)

pumpswap_fixed.loc[valid_transition, "pct_return_prev"] = (
    pumpswap_fixed.loc[valid_transition, "price_ratio_prev"] - 1
) * 100

pumpswap_fixed.loc[valid_transition, "log_return_prev"] = np.log(
    pumpswap_fixed.loc[valid_transition, "price_ratio_prev"]
)

pumpswap_fixed["same_slot_as_prev"] = (
    pumpswap_fixed["block"] == pumpswap_fixed["prev_block"]
)

# 4. Summarize the jumps remaining after correct ordering.
returns = pumpswap_fixed.loc[
    pumpswap_fixed["pct_return_prev"].notna(), "pct_return_prev"
]

print("\n========== PRICE TRANSITION AUDIT ==========")
print("Valid adjacent archived-trade transitions:", len(returns))
print("Median return:", f"{returns.median():.4f}%")
print("1st percentile:", f"{returns.quantile(0.01):.4f}%")
print("99th percentile:", f"{returns.quantile(0.99):.4f}%")
print("Transitions <= -50%:", int((returns <= -50).sum()))
print("Transitions <= -90%:", int((returns <= -90).sum()))
print("Transitions >= +100%:", int((returns >= 100).sum()))
print("Transitions >= +500%:", int((returns >= 500).sum()))
print(
    "Extreme drops <= -50% occurring in the same slot:",
    int(
        (
            (pumpswap_fixed["pct_return_prev"] <= -50)
            & pumpswap_fixed["same_slot_as_prev"]
        ).sum()
    )
)

# 5. Inspect the most extreme transitions after reordering.
display_columns = [
    "mint", "pool",
    "prev_block", "prev_transactionIndex", "prev_signature",
    "block", "transactionIndex", "signature",
    "prev_price", "price", "pct_return_prev", "log_return_prev",
    "same_slot_as_prev"
]

print("\nLargest absolute log-return transitions:")
display(
    pumpswap_fixed.loc[valid_transition]
    .assign(abs_log_return=lambda d: d["log_return_prev"].abs())
    .nlargest(20, "abs_log_return")[display_columns]
)

print("\nWorst 15 percentage-return transitions:")
display(
    pumpswap_fixed.loc[valid_transition]
    .nsmallest(15, "pct_return_prev")[display_columns]
)

print("\nLargest 15 positive percentage-return transitions:")
display(
    pumpswap_fixed.loc[valid_transition]
    .nlargest(15, "pct_return_prev")[display_columns]
)

In [ ]:
import numpy as np
import pandas as pd

audit_df = pumpswap_fixed.sort_values(
    ["pool", "block", "transactionIndex"],
    kind="stable"
).copy().reset_index(drop=True)

pool_keys = ["mint", "pool"]
grouped = audit_df.groupby(pool_keys, sort=False)

# Number the archived trades within each pool.
audit_df["pool_trade_number"] = grouped.cumcount() + 1
audit_df["prev_action"] = grouped["action"].shift(1)

# Mark large negative and positive transitions.
audit_df["extreme_drop"] = audit_df["pct_return_prev"] <= -50
audit_df["extreme_rise"] = audit_df["pct_return_prev"] >= 500

extreme_df = audit_df[
    audit_df["extreme_drop"] | audit_df["extreme_rise"]
].copy()

print("Extreme transitions:", len(extreme_df))
print("Drops <= -50%:", int(extreme_df["extreme_drop"].sum()))
print("Rises >= +500%:", int(extreme_df["extreme_rise"].sum()))

print("\nOutliers by current trade action:")
display(
    extreme_df.groupby(
        ["extreme_drop", "extreme_rise", "action"],
        dropna=False
    ).size().rename("count").reset_index()
)

# Show exact transition metadata.
transition_cols = [
    "mint", "pool", "pool_trade_number",
    "prev_block", "prev_transactionIndex", "prev_signature",
    "block", "transactionIndex", "signature",
    "prev_action", "action", "prev_price", "price",
    "pct_return_prev", "same_slot_as_prev"
]

print("\nAll extreme transitions:")
display(
    extreme_df[transition_cols]
    .sort_values("pct_return_prev")
)

# Show two pool trades before and after each extreme transition.
# This helps establish whether the jumps occur near the start of
# a pool's recorded history or after a longer sequence of trading.
audit_df["pool_position"] = audit_df.groupby(
    pool_keys, sort=False
).cumcount()

wanted = set()

for row in extreme_df.itertuples():
    pos = int(row.pool_position)
    for offset in range(-2, 3):
        if pos + offset >= 0:
            wanted.add((row.mint, row.pool, pos + offset))

context_mask = [
    (row.mint, row.pool, int(row.pool_position)) in wanted
    for row in audit_df.itertuples()
]

context_df = audit_df.loc[context_mask].copy()
context_cols = [
    "mint", "pool", "pool_trade_number",
    "block", "transactionIndex", "archive_seq",
    "signature", "action", "price",
    "pct_return_prev", "same_slot_as_prev"
]

print("\nSurrounding trades for the extreme transitions:")
display(
    context_df[context_cols]
    .sort_values(["pool", "block", "transactionIndex"])
)

In [ ]:
import numpy as np
import pandas as pd

# Sort in verified on-chain order.
audit_df = pumpswap_fixed.sort_values(
    ["pool", "block", "transactionIndex"],
    kind="stable"
).copy().reset_index(drop=True)

pool_keys = ["mint", "pool"]
audit_df["pool_position"] = (
    audit_df.groupby(pool_keys, sort=False).cumcount()
)
audit_df["pool_trade_number"] = audit_df["pool_position"] + 1

# Filter outliers only after pool_position has been created.
audit_df["extreme_drop"] = audit_df["pct_return_prev"] <= -50
audit_df["extreme_rise"] = audit_df["pct_return_prev"] >= 500

extreme_df = audit_df[
    audit_df["extreme_drop"] | audit_df["extreme_rise"]
].copy()

print("Extreme transitions:", len(extreme_df))
print("Drops <= -50%:", int(extreme_df["extreme_drop"].sum()))
print("Rises >= +500%:", int(extreme_df["extreme_rise"].sum()))

# Build a set of pool positions: two trades before through two after
# each extreme transition.
wanted = set()

for _, row in extreme_df.iterrows():
    pos = int(row["pool_position"])
    for offset in range(-2, 3):
        nearby_pos = pos + offset
        if nearby_pos >= 0:
            wanted.add((
                row["mint"],
                row["pool"],
                nearby_pos
            ))

context_df = audit_df[
    [
        (row.mint, row.pool, int(row.pool_position)) in wanted
        for row in audit_df.itertuples()
    ]
].copy()

context_cols = [
    "mint", "pool", "pool_trade_number",
    "block", "transactionIndex", "archive_seq",
    "signature", "action", "price",
    "pct_return_prev", "same_slot_as_prev"
]

print("\nSurrounding trades for extreme transitions:")
display(
    context_df[context_cols].sort_values(
        ["pool", "block", "transactionIndex"],
        kind="stable"
    )
)

# Inspect the first three archived trades for each pool.
first_trades = (
    audit_df.groupby(pool_keys, sort=False, group_keys=False)
    .head(3)
)

print("\nFirst three archived trades per pool:")
display(
    first_trades[
        [
            "mint", "pool", "pool_trade_number",
            "block", "transactionIndex", "signature",
            "action", "price", "pct_return_prev"
        ]
    ].sort_values(["pool", "pool_trade_number"])
)

# Quantify whether nearly identical initial prices recur across pools.
first_rows = audit_df[audit_df["pool_trade_number"] == 1].copy()

print("\nFirst-trade price distribution:")
display(
    first_rows["price"].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    ).to_frame("first_trade_price")
)

print(
    "\nFirst-trade prices rounded to 12 significant digits, "
    "most common:"
)
display(
    first_rows.assign(
        rounded_price=first_rows["price"].map(
            lambda x: f"{x:.11g}" if pd.notna(x) else None
        )
    )["rounded_price"]
    .value_counts()
    .head(15)
    .rename_axis("price")
    .to_frame("pool_count")
)

In [ ]:
import pandas as pd

focus_pools = [
    "61dwue7tnQkNP7VuVRz9HYaXGTLf51DQMNgj3sd5qGMt",
    "D1b5mBrLfjEeSdDM3ToFMNgenyrvDXp4eXq6NGoACRcD",
    "6dGRz1Tw9kCy24Rd6fKtf5vQ42Qa4dDftRQPBERiUW2X",
]

focus_rows = audit_df[
    audit_df["pool"].isin(focus_pools)
    & audit_df["pool_trade_number"].isin([1, 2, 3])
].sort_values(
    ["pool", "pool_trade_number"]
).copy()

def flatten_event_fields(obj, prefix=""):
    """Flatten nested dictionaries for compact event inspection."""
    out = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_name = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, dict):
                out.update(flatten_event_fields(value, key_name))
            elif isinstance(value, (int, float, str, bool)) or value is None:
                out[key_name] = value
            else:
                out[key_name] = repr(value)[:200]

    elif hasattr(obj, "__dict__"):
        out.update(flatten_event_fields(vars(obj), prefix))

    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            out.update(flatten_event_fields(value, f"{prefix}[{i}]"))

    return out


decoded_rows = []

for _, row in focus_rows.iterrows():
    signature = str(row["signature"])
    action = str(row["action"]).lower()

    print(
        f"\nPool {str(row['pool'])[:10]}... | "
        f"trade #{int(row['pool_trade_number'])} | "
        f"{action} | archive price={row['price']:.12g} | "
        f"slot={int(row['block'])}"
    )
    print("Signature:", signature)

    try:
        if action == "buy":
            event = extract_buy_event84b(signature)
        elif action == "sell":
            event = extract_sell_event96(signature)
        else:
            print("Unexpected action; skipping decode.")
            continue

        print("Decoded return type:", type(event).__name__)

        # Show fields relevant to reserve-based price calculation.
        flat = flatten_event_fields(event)

        if not flat:
            print("Could not flatten event; raw result:")
            print(repr(event)[:1500])
            continue

        relevant = {
            key: value
            for key, value in flat.items()
            if any(
                term in key.lower()
                for term in [
                    "reserve", "amount", "fee", "price",
                    "quote", "base", "pool", "mint", "creator"
                ]
            )
        }

        if relevant:
            for key, value in relevant.items():
                print(f"  {key}: {value}")
        else:
            print("Event fields:")
            for key, value in list(flat.items())[:30]:
                print(f"  {key}: {value}")

        decoded_rows.append({
            "pool": row["pool"],
            "trade_number": int(row["pool_trade_number"]),
            "slot": int(row["block"]),
            "transactionIndex": int(row["transactionIndex"]),
            "action": action,
            "archive_price": row["price"],
            "decoded_event": event,
            "decoded_fields": flat,
        })

    except Exception as exc:
        print("Decode error:", repr(exc))

print(
    "\nSuccessfully decoded events:",
    len(decoded_rows),
    "out of",
    len(focus_rows)
)

In [ ]:
import pandas as pd

focus_pool = "61dwue7tnQkNP7VuVRz9HYaXGTLf51DQMNgj3sd5qGMt"

# Bounds taken from the decoded first and second trades.
start_slot, start_index = 453262360, 432
end_slot, end_index = 453262361, 738

# All transactions already present in the PumpSwap archive for this pool.
pool_archive = pumpswap_fixed[
    pumpswap_fixed["pool"].astype(str) == focus_pool
].copy()

archive_lookup = (
    pool_archive.drop_duplicates("signature")
    .set_index("signature")
)

# Fetch all address-history pages for this narrow slot range.
history_records = []
pagination_token = None
seen_tokens = set()
request_count = 0

while True:
    result = fetch_gta_page_backfill(
        focus_pool,
        start_slot,
        end_slot,
        pagination_token
    )
    request_count += 1
    page = result.get("data", [])

    if not page:
        break

    history_records.extend(page)
    next_token = result.get("paginationToken")

    if not next_token:
        break
    if next_token in seen_tokens:
        raise RuntimeError("Repeated pagination token; stopped safely.")

    seen_tokens.add(next_token)
    pagination_token = next_token

history_df = pd.DataFrame(history_records)

if history_df.empty:
    print("No pool-history transactions returned for the selected slots.")
else:
    history_df["slot"] = pd.to_numeric(
        history_df["slot"], errors="coerce"
    )
    history_df["transactionIndex"] = pd.to_numeric(
        history_df["transactionIndex"], errors="coerce"
    )

    # Keep the interval from the first trade through the second trade.
    after_start = (
        (history_df["slot"] > start_slot)
        | (
            (history_df["slot"] == start_slot)
            & (history_df["transactionIndex"] >= start_index)
        )
    )
    before_end = (
        (history_df["slot"] < end_slot)
        | (
            (history_df["slot"] == end_slot)
            & (history_df["transactionIndex"] <= end_index)
        )
    )

    interval = history_df[after_start & before_end].copy()
    interval["in_archive"] = interval["signature"].isin(
        archive_lookup.index
    )

    interval["archive_action"] = interval["signature"].map(
        archive_lookup["action"]
    )
    interval["archive_price"] = interval["signature"].map(
        archive_lookup["price"]
    )

    interval = interval.sort_values(
        ["slot", "transactionIndex"], kind="stable"
    )

    print("RPC requests:", request_count)
    print("All address-history records in the queried slots:",
          len(history_df))
    print("Transactions from first to second sampled observation:",
          len(interval))
    print("Already present in stratified archive:",
          int(interval["in_archive"].sum()))
    print("Not present in stratified archive:",
          int((~interval["in_archive"]).sum()))

    display(
        interval[
            [
                "slot", "transactionIndex", "signature",
                "in_archive", "archive_action", "archive_price",
                "err"
            ]
        ].to_string(index=False)
        if False else interval[
            [
                "slot", "transactionIndex", "signature",
                "in_archive", "archive_action", "archive_price",
                "err"
            ]
        ]
    )

In [ ]:
import pandas as pd

focus_pool = "61dwue7tnQkNP7VuVRz9HYaXGTLf51DQMNgj3sd5qGMt"

sig_first = (
    "2Q5MNdsR6DmB9U5bGtg3AT5KakKCpoDwHNJx6qia7whwkAtVBUmBNfXdNj9xHrmnCQCWPNTiocimY1zZfFdrhDJu"
)
sig_second = (
    "4AG57vDA2T1HGchZaZANjzzxqmvLCmAzUHH56SQD4j4CupkDUWHZZnHVchQZGGAWrZbqNEEA8ZVqSFt7nbKMW6nN"
)

start_slot, start_index = 453262360, 432
end_slot, end_index = 453262361, 738

# 1. Fetch transaction metadata for the two successful buys.
tx_first = fetch_transaction(sig_first)
tx_second = fetch_transaction(sig_second)

def account_pubkey(key):
    if isinstance(key, dict):
        return key.get("pubkey", str(key))
    return str(key)

def inspect_pool_token_balances(tx, label):
    if not tx:
        print(f"{label}: transaction result is empty")
        return pd.DataFrame()

    meta = tx.get("meta") or {}
    message = (tx.get("transaction") or {}).get("message") or {}
    keys = [
        account_pubkey(k)
        for k in message.get("accountKeys", [])
    ]

    rows = []
    for balance_kind in ["preTokenBalances", "postTokenBalances"]:
        for entry in meta.get(balance_kind) or []:
            idx = entry.get("accountIndex")
            address = keys[idx] if idx is not None and idx < len(keys) else None
            amount_info = entry.get("uiTokenAmount") or {}

            rows.append({
                "transaction": label,
                "balance": balance_kind,
                "accountIndex": idx,
                "account": address,
                "mint": entry.get("mint"),
                "owner": entry.get("owner"),
                "amount_raw": amount_info.get("amount"),
                "decimals": amount_info.get("decimals"),
            })

    frame = pd.DataFrame(rows)
    if frame.empty:
        print(f"{label}: no token-balance metadata found")
        return frame

    print(f"\n{label}: token balances controlled by the pool PDA")
    pool_owned = frame[frame["owner"] == focus_pool]
    display(pool_owned)

    return frame

balances_first = inspect_pool_token_balances(tx_first, "First buy")
balances_second = inspect_pool_token_balances(tx_second, "Second buy")

# 2. Derive pool-owned token-vault addresses.
all_balances = pd.concat(
    [balances_first, balances_second],
    ignore_index=True
)

pool_vaults = (
    all_balances.loc[
        (all_balances["owner"] == focus_pool)
        & all_balances["account"].notna(),
        ["account", "mint"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nDerived pool token-vault addresses:")
display(pool_vaults)

if pool_vaults.empty:
    raise RuntimeError(
        "Could not derive vault addresses from token-balance metadata. "
        "Inspect the token balances and account keys above."
    )

# 3. Query each vault directly over the exact slot interval.
vault_history_records = []
vault_query_errors = []

for vault_row in pool_vaults.itertuples(index=False):
    vault = vault_row.account
    token = None
    seen_tokens = set()
    vault_records = []

    try:
        while True:
            result = fetch_gta_page_backfill(
                vault, start_slot, end_slot, token
            )
            page = result.get("data", [])

            if not page:
                break

            vault_records.extend(page)
            next_token = result.get("paginationToken")

            if not next_token:
                break
            if next_token in seen_tokens:
                raise RuntimeError("Repeated pagination token.")

            seen_tokens.add(next_token)
            token = next_token

        for item in vault_records:
            item_copy = dict(item)
            item_copy["queried_vault"] = vault
            item_copy["vault_mint"] = vault_row.mint
            vault_history_records.append(item_copy)

        print(
            f"Vault {vault[:10]}... ({vault_row.mint}): "
            f"{len(vault_records)} history records"
        )

    except Exception as exc:
        vault_query_errors.append({
            "vault": vault,
            "error": str(exc)
        })

vault_history = pd.DataFrame(vault_history_records)

if not vault_history.empty:
    vault_history["slot"] = pd.to_numeric(
        vault_history["slot"], errors="coerce"
    )
    vault_history["transactionIndex"] = pd.to_numeric(
        vault_history["transactionIndex"], errors="coerce"
    )

    # Keep only signatures between the two sampled trades.
    after_first = (
        (vault_history["slot"] > start_slot)
        | (
            (vault_history["slot"] == start_slot)
            & (vault_history["transactionIndex"] >= start_index)
        )
    )
    before_second = (
        (vault_history["slot"] < end_slot)
        | (
            (vault_history["slot"] == end_slot)
            & (vault_history["transactionIndex"] <= end_index)
        )
    )

    vault_interval = vault_history[
        after_first & before_second
    ].copy()

    # Deduplicate transactions found through both vaults.
    vault_interval = (
        vault_interval.sort_values(["slot", "transactionIndex"])
        .drop_duplicates("signature")
        .copy()
    )

    # Compare with pool-PDA history if the earlier cell's variable remains.
    pool_history_signatures = (
        set(interval["signature"].astype(str))
        if "interval" in globals()
        else set()
    )

    vault_interval["in_pool_pda_history"] = (
        vault_interval["signature"].astype(str)
        .isin(pool_history_signatures)
    )

    print("\nTransactions touching either vault between the two buys:")
    display(
        vault_interval[
            [
                "slot", "transactionIndex", "signature",
                "err", "queried_vault", "vault_mint",
                "in_pool_pda_history"
            ]
        ].sort_values(["slot", "transactionIndex"])
    )

    print("\nUnique transactions in vault interval:", len(vault_interval))
    print(
        "Not found in pool-PDA history:",
        int((~vault_interval["in_pool_pda_history"]).sum())
    )
else:
    print("No vault-history transactions were returned.")

if vault_query_errors:
    print("\nVault-query errors:")
    display(pd.DataFrame(vault_query_errors))

In [ ]:
import inspect
import base64
import pandas as pd

sig_debug = sig_first  # First successful buy from the previous cell
tx_debug = fetch_transaction(sig_debug)

print("Top-level transaction keys:", list(tx_debug.keys()))
meta_debug = tx_debug.get("meta") or {}
print("Meta keys:", list(meta_debug.keys()))
print("Transaction error:", meta_debug.get("err"))

# 1. Inspect the current event extractor to see how it chooses an event.
print("\n========== CURRENT EVENT EXTRACTOR ==========")
try:
    print(inspect.getsource(extract_buy_event84b))
except Exception as exc:
    print("Source unavailable:", repr(exc))

# 2. List top-level and inner instructions.
message_debug = (
    tx_debug.get("transaction", {}).get("message", {})
)
account_keys_debug = [
    account_pubkey(k)
    for k in message_debug.get("accountKeys", [])
]

def summarize_instruction(ix, keys, outer_index, inner_index=None):
    pid = ix.get("programId")
    pid_index = ix.get("programIdIndex")

    if pid is None and pid_index is not None and pid_index < len(keys):
        pid = keys[pid_index]

    account_refs = ix.get("accounts", [])
    resolved_accounts = []

    for account in account_refs:
        if isinstance(account, int) and 0 <= account < len(keys):
            resolved_accounts.append(keys[account])
        else:
            resolved_accounts.append(str(account))

    row = {
        "outer_instruction": outer_index,
        "inner_instruction": inner_index,
        "programId": pid,
        "program": ix.get("program"),
        "parsed_type": (
            ix.get("parsed", {}).get("type")
            if isinstance(ix.get("parsed"), dict)
            else None
        ),
        "accounts": ", ".join(resolved_accounts),
        "data_prefix": str(ix.get("data", ""))[:48],
        "parsed_info": (
            repr(ix.get("parsed", {}).get("info", {}))[:300]
            if isinstance(ix.get("parsed"), dict)
            else ""
        )
    }
    return row

instruction_rows_debug = []

for i, ix in enumerate(message_debug.get("instructions", [])):
    instruction_rows_debug.append(
        summarize_instruction(ix, account_keys_debug, i)
    )

for group in meta_debug.get("innerInstructions") or []:
    outer_i = group.get("index")
    for j, ix in enumerate(group.get("instructions", [])):
        instruction_rows_debug.append(
            summarize_instruction(ix, account_keys_debug, outer_i, j)
        )

print("\n========== TRANSACTION INSTRUCTIONS ==========")
display(pd.DataFrame(instruction_rows_debug))

# 3. List every execution log line. These can reveal multiple swaps,
# repeated PumpSwap invocations, and emitted event payloads.
logs_debug = meta_debug.get("logMessages") or []

print("\n========== EXECUTION LOGS ==========")
print("Total log lines:", len(logs_debug))

for i, line in enumerate(logs_debug):
    print(f"{i:03d}: {line}")

# 4. Enumerate every base64 Program data event and its discriminator.
print("\n========== PROGRAM DATA EVENTS ==========")

event_disc_debug = globals().get("event_discriminators", {})
print("Known event discriminators:", repr(event_disc_debug)[:1500])

program_data_rows_debug = []

for i, line in enumerate(logs_debug):
    marker = "Program data: "
    if marker not in line:
        continue

    encoded = line.split(marker, 1)[1].strip()

    try:
        raw = base64.b64decode(encoded, validate=True)
        program_data_rows_debug.append({
            "log_index": i,
            "payload_bytes": len(raw),
            "discriminator_hex": raw[:8].hex(),
            "payload_prefix_hex": raw[:24].hex()
        })
    except Exception:
        program_data_rows_debug.append({
            "log_index": i,
            "payload_bytes": None,
            "discriminator_hex": "base64 decode failed",
            "payload_prefix_hex": encoded[:40]
        })

display(pd.DataFrame(program_data_rows_debug))

# 5. Reconcile the actual vault deltas against the decoded BuyEvent.
decoded_debug = extract_buy_event84b(sig_debug)

quote_pre = 67_405_853_768
quote_post = 226_812_677_950
base_pre = 206_900_000_000_000
base_post = 72_043_900_569_655

print("\n========== RESERVE DELTA RECONCILIATION ==========")
print("Actual quote vault delta (lamports):", quote_post - quote_pre)
print("Actual quote vault delta (SOL):",
      (quote_post - quote_pre) / 1_000_000_000)
print("Actual base vault delta (raw):", base_post - base_pre)
print("Actual base vault delta (tokens):",
      (base_post - base_pre) / 1_000_000)

print("Decoded event quote_amount_in:",
      decoded_debug.get("quote_amount_in"))
print("Decoded event base_amount_out:",
      decoded_debug.get("base_amount_out"))
print("Decoded event pool_quote_token_reserves:",
      decoded_debug.get("pool_quote_token_reserves"))
print("Decoded event pool_base_token_reserves:",
      decoded_debug.get("pool_base_token_reserves"))

In [ ]:
import base64
import pandas as pd

sig_debug = sig_first
tx_debug = fetch_transaction(sig_debug)

logs_debug = (tx_debug.get("meta") or {}).get("logMessages") or []

# Decode every known PumpSwap event in this transaction,
# rather than returning only the first matching event.
event_rows_debug = []

for log_idx, line in enumerate(logs_debug):
    marker = "Program data: "
    if marker not in line:
        continue

    try:
        raw = base64.b64decode(
            line.split(marker, 1)[1].strip(),
            validate=True
        )
    except Exception:
        continue

    for event_name, discriminator in event_discriminators.items():
        if raw.startswith(discriminator):
            decoded = decode_all_fields(
                raw, event_fields[event_name]
            )
            event_rows_debug.append({
                "log_index": log_idx,
                "event_name": event_name,
                **decoded
            })
            break

event_df_debug = pd.DataFrame(event_rows_debug)

print("Decoded event count:", len(event_df_debug))
if not event_df_debug.empty:
    display(
        event_df_debug[
            [
                c for c in [
                    "log_index", "event_name",
                    "quote_amount_in", "user_quote_amount_in",
                    "base_amount_out", "quote_amount_out",
                    "pool_quote_token_reserves",
                    "pool_base_token_reserves",
                    "lp_fee", "protocol_fee",
                    "coin_creator_fee"
                ]
                if c in event_df_debug.columns
            ]
        ]
    )

# Count top-level PumpSwap instructions in the transaction.
message_debug = (
    tx_debug.get("transaction", {}).get("message", {})
)
pump_program_id = "pAMMBay6oceH9fJKBRHGP5D4bD4sWpmSwMn52FMfXEA"

top_level_instructions = message_debug.get("instructions", [])
pump_ixs = [
    ix for ix in top_level_instructions
    if ix.get("programId") == pump_program_id
]

print("\nTop-level PumpSwap instruction count:", len(pump_ixs))
for i, ix in enumerate(pump_ixs):
    print(
        f"PumpSwap instruction {i + 1}: "
        f"data prefix = {str(ix.get('data', ''))[:32]}"
    )

# Compare event reserve snapshots with actual transaction vault balances.
print("\nActual token-vault balance changes:")
display(
    pd.DataFrame([
        {
            "vault": "quote",
            "pre_raw": 67_405_853_768,
            "post_raw": 226_812_677_950,
            "delta_raw": 226_812_677_950 - 67_405_853_768,
        },
        {
            "vault": "base",
            "pre_raw": 206_900_000_000_000,
            "post_raw": 72_043_900_569_655,
            "delta_raw": 72_043_900_569_655 - 206_900_000_000_000,
        }
    ])
)

In [ ]:
import base64
import time
import numpy as np
import pandas as pd

# Collect signatures involved in the 24 extreme transitions.
candidate_signatures = set()

for col in ["signature", "prev_signature"]:
    if col in extreme_df.columns:
        candidate_signatures.update(
            extreme_df[col].dropna().astype(str).tolist()
        )

candidate_signatures.discard("")
print("Unique implicated signatures:", len(candidate_signatures))

archive_lookup = (
    pumpswap_fixed.drop_duplicates("signature")
    .set_index("signature")
)

def decode_swap_events(signature):
    tx = fetch_transaction(signature)

    if isinstance(tx, dict) and tx.get("result") is not None:
        tx = tx["result"]

    if not isinstance(tx, dict):
        raise ValueError("Unexpected transaction response shape")

    meta = tx.get("meta") or {}
    logs = meta.get("logMessages") or []
    events = []

    for log_idx, line in enumerate(logs):
        marker = "Program data: "
        if marker not in line:
            continue

        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(),
                validate=True
            )
        except Exception:
            continue

        for event_name in ["BuyEvent", "SellEvent"]:
            discriminator = event_discriminators.get(event_name)
            if discriminator and raw.startswith(discriminator):
                decoded = decode_all_fields(
                    raw, event_fields[event_name]
                )
                events.append({
                    "log_index": log_idx,
                    "event_name": event_name,
                    **decoded
                })
                break

    events.sort(key=lambda e: e["log_index"])
    return tx, events


event_level_rows = []
failed_fetches = []

for i, signature in enumerate(sorted(candidate_signatures), start=1):
    try:
        tx, events = decode_swap_events(signature)
        meta = tx.get("meta") or {}

        if not events:
            continue

        tx_slot = tx.get("slot")
        tx_index = tx.get("transactionIndex")

        # Build the token-balance lookup for each pool's final
        # post-transaction state.
        post_balances = {}
        decimals_lookup = {}

        for bal in meta.get("postTokenBalances") or []:
            owner = bal.get("owner")
            mint = bal.get("mint")
            info = bal.get("uiTokenAmount") or {}

            if owner and mint and info.get("amount") is not None:
                post_balances[(owner, mint)] = int(info["amount"])

            if mint and info.get("decimals") is not None:
                decimals_lookup[mint] = int(info["decimals"])

        # Work out the resulting state after each event.
        for event_idx, ev in enumerate(events):
            pool = ev.get("pool")
            if not pool:
                continue

            base_mint = ev.get("base_mint")
            quote_mint = ev.get("quote_mint")

            pre_base = ev.get("pool_base_token_reserves")
            pre_quote = ev.get("pool_quote_token_reserves")

            if pre_base is None or pre_quote is None:
                continue

            pre_base = int(pre_base)
            pre_quote = int(pre_quote)

            # Find the next event for this same pool in the transaction.
            next_same_pool = next(
                (
                    later for later in events[event_idx + 1:]
                    if later.get("pool") == pool
                    and later.get("pool_base_token_reserves") is not None
                    and later.get("pool_quote_token_reserves") is not None
                ),
                None
            )

            state_source = None

            if next_same_pool is not None:
                # The next event's pre-state should equal this event's post-state.
                post_base = int(
                    next_same_pool["pool_base_token_reserves"]
                )
                post_quote = int(
                    next_same_pool["pool_quote_token_reserves"]
                )
                state_source = "next event pre-reserves"
            else:
                # Last event for this pool: use actual transaction post-balances.
                post_base = post_balances.get((pool, base_mint))
                post_quote = post_balances.get((pool, quote_mint))
                state_source = "transaction post-balances"

            base_decimals = decimals_lookup.get(base_mint)
            quote_decimals = decimals_lookup.get(quote_mint)

            # Price = quote tokens / base tokens, both normalized by decimals.
            if (
                post_base is not None
                and post_quote is not None
                and base_decimals is not None
                and quote_decimals is not None
                and post_base > 0
                and post_quote > 0
            ):
                price_after = (
                    (post_quote / (10 ** quote_decimals))
                    / (post_base / (10 ** base_decimals))
                )
            else:
                price_after = np.nan

            if (
                pre_base > 0
                and pre_quote > 0
                and base_decimals is not None
                and quote_decimals is not None
            ):
                price_before = (
                    (pre_quote / (10 ** quote_decimals))
                    / (pre_base / (10 ** base_decimals))
                )
            else:
                price_before = np.nan

            archive_price = np.nan
            if (
                signature in archive_lookup.index
                and str(archive_lookup.loc[signature, "pool"]) == str(pool)
            ):
                archive_price = float(
                    archive_lookup.loc[signature, "price"]
                )

            event_level_rows.append({
                "signature": signature,
                "slot": tx_slot,
                "transactionIndex": tx_index,
                "pool": pool,
                "log_index": ev["log_index"],
                "event_number_in_tx": event_idx + 1,
                "event_name": ev["event_name"],
                "base_mint": base_mint,
                "quote_mint": quote_mint,
                "pre_base_raw": pre_base,
                "pre_quote_raw": pre_quote,
                "post_base_raw": post_base,
                "post_quote_raw": post_quote,
                "price_before": price_before,
                "price_after": price_after,
                "state_source": state_source,
                "archive_price": archive_price,
                "event_count_in_tx": sum(
                    1 for other in events
                    if other.get("pool") == pool
                ),
            })

    except Exception as exc:
        failed_fetches.append({
            "signature": signature,
            "error": repr(exc)
        })

    if i % 10 == 0:
        print(f"Processed {i}/{len(candidate_signatures)} signatures")
    time.sleep(0.05)

event_level_df = pd.DataFrame(event_level_rows)

print("\n========== MULTI-EVENT AUDIT ==========")
print("Decoded event rows:", len(event_level_df))
print("Signatures with fetch/decode errors:", len(failed_fetches))

if not event_level_df.empty:
    per_tx_pool = (
        event_level_df
        .sort_values(["signature", "pool", "log_index"])
        .groupby(["signature", "pool"], as_index=False)
        .agg(
            event_count=("event_name", "size"),
            first_event_post_price=("price_after", "first"),
            final_event_post_price=("price_after", "last"),
            archive_price=("archive_price", "first"),
            first_state_source=("state_source", "first"),
            final_state_source=("state_source", "last"),
        )
    )

    per_tx_pool["first_vs_archive_pct"] = (
        100 * (
            per_tx_pool["first_event_post_price"]
            / per_tx_pool["archive_price"] - 1
        )
    )
    per_tx_pool["final_vs_archive_pct"] = (
        100 * (
            per_tx_pool["final_event_post_price"]
            / per_tx_pool["archive_price"] - 1
        )
    )

    print("Transactions with multiple events for the same pool:",
          int((per_tx_pool["event_count"] > 1).sum()))

    print("\nPrice comparison by implicated transaction:")
    display(
        per_tx_pool.sort_values(
            ["event_count", "final_vs_archive_pct"],
            ascending=[False, False]
        )
    )

    print("\nIndividual event states:")
    display(
        event_level_df[
            [
                "signature", "slot", "transactionIndex", "pool",
                "log_index", "event_number_in_tx", "event_name",
                "price_before", "price_after",
                "archive_price", "state_source"
            ]
        ].sort_values(["slot", "transactionIndex", "log_index"])
    )

if failed_fetches:
    print("\nFetch/decode errors:")
    display(pd.DataFrame(failed_fetches))

In [ ]:
import base64
import time
import numpy as np
import pandas as pd

WRAPPED_SOL_MINT = "So11111111111111111111111111111111111111112"

archive_by_sig = (
    pumpswap_fixed.drop_duplicates("signature")
    .set_index("signature")
)

def normalize_tx_response(tx):
    if isinstance(tx, dict) and tx.get("result") is not None:
        return tx["result"]
    return tx

def decode_swap_events_from_tx(tx):
    logs = (tx.get("meta") or {}).get("logMessages") or []
    events = []

    for log_idx, line in enumerate(logs):
        marker = "Program data: "
        if marker not in line:
            continue

        try:
            raw = base64.b64decode(
                line.split(marker, 1)[1].strip(),
                validate=True
            )
        except Exception:
            continue

        for event_name in ["BuyEvent", "SellEvent"]:
            discriminator = event_discriminators.get(event_name)

            if discriminator and raw.startswith(discriminator):
                decoded = decode_all_fields(
                    raw, event_fields[event_name]
                )
                events.append({
                    "log_index": log_idx,
                    "event_name": event_name,
                    **decoded
                })
                break

    return sorted(events, key=lambda e: e["log_index"])

def pool_mint_metadata(tx, pool):
    """Derive the pool's base/quote mints and decimals from token balances."""
    meta = tx.get("meta") or {}
    balances = (
        (meta.get("preTokenBalances") or [])
        + (meta.get("postTokenBalances") or [])
    )

    mint_meta = {}

    for bal in balances:
        if bal.get("owner") != pool:
            continue

        mint = bal.get("mint")
        decimals = (bal.get("uiTokenAmount") or {}).get("decimals")

        if mint is not None and decimals is not None:
            mint_meta[mint] = int(decimals)

    if not mint_meta:
        return None

    # PumpSwap pools in this dataset use wrapped SOL as quote.
    if WRAPPED_SOL_MINT in mint_meta:
        quote_mint = WRAPPED_SOL_MINT
    else:
        quote_candidates = [
            mint for mint, decimals in mint_meta.items()
            if decimals == 9
        ]
        if len(quote_candidates) != 1:
            return None
        quote_mint = quote_candidates[0]

    base_candidates = [
        mint for mint in mint_meta if mint != quote_mint
    ]

    if len(base_candidates) != 1:
        return None

    base_mint = base_candidates[0]

    return {
        "base_mint": base_mint,
        "base_decimals": mint_meta[base_mint],
        "quote_mint": quote_mint,
        "quote_decimals": mint_meta[quote_mint],
    }

def pool_post_amounts(tx, pool, mint_info):
    """Return final token-vault amounts from post-transaction balances."""
    meta = tx.get("meta") or {}
    amounts = {}

    for bal in meta.get("postTokenBalances") or []:
        if bal.get("owner") != pool:
            continue

        mint = bal.get("mint")
        amount = (bal.get("uiTokenAmount") or {}).get("amount")

        if mint is not None and amount is not None:
            amounts[mint] = int(amount)

    base_raw = amounts.get(mint_info["base_mint"])
    quote_raw = amounts.get(mint_info["quote_mint"])

    if base_raw is None or quote_raw is None:
        return None, None

    return base_raw, quote_raw

def spot_price_raw(base_raw, quote_raw, base_decimals, quote_decimals):
    if base_raw is None or quote_raw is None:
        return np.nan
    if int(base_raw) <= 0 or int(quote_raw) <= 0:
        return np.nan

    return (
        (int(quote_raw) / (10 ** quote_decimals))
        / (int(base_raw) / (10 ** base_decimals))
    )

event_price_rows = []
price_audit_errors = []

for i, signature in enumerate(sorted(candidate_signatures), start=1):
    try:
        tx = normalize_tx_response(fetch_transaction(signature))
        if not isinstance(tx, dict):
            raise ValueError("Unexpected transaction response")

        meta = tx.get("meta") or {}
        if meta.get("err") is not None:
            continue

        events = decode_swap_events_from_tx(tx)

        # A transaction may reference multiple pools; audit each separately.
        pools_in_tx = sorted({
            ev.get("pool")
            for ev in events
            if ev.get("pool") is not None
        })

        for pool in pools_in_tx:
            pool_events = [
                ev for ev in events if ev.get("pool") == pool
            ]

            mint_info = pool_mint_metadata(tx, pool)
            if mint_info is None:
                raise ValueError(
                    f"Could not identify base/quote mints for pool {pool}"
                )

            final_base_raw, final_quote_raw = pool_post_amounts(
                tx, pool, mint_info
            )

            archive_price = np.nan
            if signature in archive_by_sig.index:
                archive_row = archive_by_sig.loc[signature]
                if str(archive_row["pool"]) == str(pool):
                    archive_price = float(archive_row["price"])

            for event_idx, ev in enumerate(pool_events):
                pre_base_raw = ev.get("pool_base_token_reserves")
                pre_quote_raw = ev.get("pool_quote_token_reserves")

                pre_price = spot_price_raw(
                    pre_base_raw,
                    pre_quote_raw,
                    mint_info["base_decimals"],
                    mint_info["quote_decimals"]
                )

                # The next event's pre-reserves are the state after this event.
                if event_idx + 1 < len(pool_events):
                    next_ev = pool_events[event_idx + 1]
                    post_base_raw = next_ev.get(
                        "pool_base_token_reserves"
                    )
                    post_quote_raw = next_ev.get(
                        "pool_quote_token_reserves"
                    )
                    state_source = "next same-pool event pre-state"
                else:
                    post_base_raw = final_base_raw
                    post_quote_raw = final_quote_raw
                    state_source = "final transaction token balances"

                post_price = spot_price_raw(
                    post_base_raw,
                    post_quote_raw,
                    mint_info["base_decimals"],
                    mint_info["quote_decimals"]
                )

                event_price_rows.append({
                    "signature": signature,
                    "slot": tx.get("slot"),
                    "transactionIndex": tx.get("transactionIndex"),
                    "pool": pool,
                    "event_number": event_idx + 1,
                    "event_count_in_pool": len(pool_events),
                    "log_index": ev["log_index"],
                    "event_name": ev["event_name"],
                    "pre_price": pre_price,
                    "post_price": post_price,
                    "archive_price": archive_price,
                    "post_state_source": state_source,
                })

    except Exception as exc:
        price_audit_errors.append({
            "signature": signature,
            "error": repr(exc)
        })

    if i % 10 == 0:
        print(f"Audited {i}/{len(candidate_signatures)} signatures")

event_prices = pd.DataFrame(event_price_rows)

print("\n========== CORRECTED EVENT PRICE AUDIT ==========")
print("Event price rows:", len(event_prices))
print("Fetch/decoding errors:", len(price_audit_errors))

if not event_prices.empty:
    event_prices["archive_vs_pre_pct"] = (
        100 * (
            event_prices["archive_price"]
            / event_prices["pre_price"] - 1
        )
    )
    event_prices["archive_vs_post_pct"] = (
        100 * (
            event_prices["archive_price"]
            / event_prices["post_price"] - 1
        )
    )

    per_tx_summary = (
        event_prices
        .sort_values(["signature", "pool", "log_index"])
        .groupby(["signature", "pool"], as_index=False)
        .agg(
            event_count=("event_count_in_pool", "first"),
            first_event_pre_price=("pre_price", "first"),
            first_event_post_price=("post_price", "first"),
            final_event_post_price=("post_price", "last"),
            archive_price=("archive_price", "first"),
        )
    )

    per_tx_summary["archive_vs_first_post_pct"] = (
        100 * (
            per_tx_summary["archive_price"]
            / per_tx_summary["first_event_post_price"] - 1
        )
    )
    per_tx_summary["archive_vs_final_post_pct"] = (
        100 * (
            per_tx_summary["archive_price"]
            / per_tx_summary["final_event_post_price"] - 1
        )
    )

    print("\nTransactions with multiple events:")
    display(
        per_tx_summary[
            per_tx_summary["event_count"] > 1
        ].sort_values("archive_vs_final_post_pct")
    )

    print("\nEvent-level prices:")
    display(
        event_prices[
            [
                "signature", "slot", "transactionIndex", "pool",
                "event_number", "event_count_in_pool", "event_name",
                "pre_price", "post_price", "archive_price",
                "archive_vs_pre_pct", "archive_vs_post_pct",
                "post_state_source"
            ]
        ].sort_values(["slot", "transactionIndex", "log_index"])
    )

if price_audit_errors:
    print("\nErrors (first 10):")
    display(pd.DataFrame(price_audit_errors).head(10))

In [ ]:
import numpy as np
import pandas as pd

# 1. Fix the detailed-table display error.
# event_prices contains "event_number", not "log_index".
if not event_prices.empty:
    display(
        event_prices[
            [
                "signature", "slot", "transactionIndex", "pool",
                "event_number", "event_count_in_pool", "event_name",
                "pre_price", "post_price", "archive_price",
                "post_state_source"
            ]
        ].sort_values(
            ["slot", "transactionIndex", "event_number"],
            kind="stable"
        )
    )

# 2. Extract reliable final transaction prices for the confirmed
# multi-event signatures from the event-level audit.
multi_event_fixes = per_tx_summary[
    (per_tx_summary["event_count"] > 1)
    & per_tx_summary["final_event_post_price"].notna()
    & np.isfinite(per_tx_summary["final_event_post_price"])
    & (per_tx_summary["final_event_post_price"] > 0)
].copy()

# A signature should map to only one pool and one corrected price.
assert not multi_event_fixes["signature"].duplicated().any(), (
    "A signature has multiple candidate correction rows."
)

price_fix_map = multi_event_fixes.set_index(
    "signature"
)["final_event_post_price"].to_dict()

print("Confirmed multi-event signatures corrected:", len(price_fix_map))

# 3. Build a corrected signature-level dataset.
# Keep the original dataset and price intact for comparison.
pumpswap_price_revised = pumpswap_fixed.copy()

pumpswap_price_revised["archive_price_original"] = (
    pumpswap_price_revised["price"]
)

pumpswap_price_revised["price_corrected"] = (
    pumpswap_price_revised["signature"]
    .map(price_fix_map)
    .combine_first(pumpswap_price_revised["price"])
)

pumpswap_price_revised["price_was_corrected"] = (
    pumpswap_price_revised["signature"].isin(price_fix_map)
)

print("Rows corrected:",
      int(pumpswap_price_revised["price_was_corrected"].sum()))

display(
    pumpswap_price_revised.loc[
        pumpswap_price_revised["price_was_corrected"],
        [
            "mint", "pool", "block", "transactionIndex",
            "signature", "archive_price_original", "price_corrected"
        ]
    ].sort_values(["pool", "block", "transactionIndex"])
)

# 4. Recompute adjacent-trade returns with corrected prices.
pumpswap_price_revised = pumpswap_price_revised.sort_values(
    ["mint", "pool", "block", "transactionIndex"],
    kind="stable"
).reset_index(drop=True)

g = pumpswap_price_revised.groupby(
    ["mint", "pool"], sort=False
)

pumpswap_price_revised["prev_price_corrected"] = (
    g["price_corrected"].shift(1)
)

valid = (
    pumpswap_price_revised["price_corrected"].gt(0)
    & pumpswap_price_revised["prev_price_corrected"].gt(0)
    & pumpswap_price_revised["price_corrected"].notna()
    & pumpswap_price_revised["prev_price_corrected"].notna()
)

pumpswap_price_revised["return_pct_corrected"] = np.nan
pumpswap_price_revised["log_return_corrected"] = np.nan

ratio = (
    pumpswap_price_revised.loc[valid, "price_corrected"]
    / pumpswap_price_revised.loc[valid, "prev_price_corrected"]
)

pumpswap_price_revised.loc[valid, "return_pct_corrected"] = (
    ratio - 1
) * 100

pumpswap_price_revised.loc[valid, "log_return_corrected"] = np.log(ratio)

# 5. Compare return statistics before and after the targeted repair.
original_returns = pd.to_numeric(
    pumpswap_price_revised["pct_return_prev"], errors="coerce"
).dropna()

corrected_returns = pumpswap_price_revised[
    "return_pct_corrected"
].dropna()

def return_summary(series):
    return {
        "valid_transitions": len(series),
        "median_pct": series.median(),
        "p01_pct": series.quantile(0.01),
        "p99_pct": series.quantile(0.99),
        "drops_le_minus50_pct": int((series <= -50).sum()),
        "drops_le_minus90_pct": int((series <= -90).sum()),
        "rises_ge_100_pct": int((series >= 100).sum()),
        "rises_ge_500_pct": int((series >= 500).sum()),
    }

summary_comparison = pd.DataFrame([
    {"version": "Original", **return_summary(original_returns)},
    {"version": "Targeted correction", **return_summary(corrected_returns)},
])

print("\n========== RETURN SUMMARY COMPARISON ==========")
display(summary_comparison)

# 6. Display remaining extreme returns after the targeted fix.
remaining_extremes = pumpswap_price_revised[
    (pumpswap_price_revised["return_pct_corrected"] <= -50)
    | (pumpswap_price_revised["return_pct_corrected"] >= 500)
].copy()

print("\nRemaining transitions <= -50% or >= +500%:",
      len(remaining_extremes))

display(
    remaining_extremes[
        [
            "mint", "pool", "block", "transactionIndex",
            "signature", "action", "prev_price_corrected",
            "price_corrected", "return_pct_corrected",
            "price_was_corrected"
        ]
    ].sort_values("return_pct_corrected")
)

In [ ]:
import numpy as np
import pandas as pd

# Use the per-transaction audit to identify the multi-event transactions
# and their first event's pre-swap price.
multi_event_starts = per_tx_summary[
    (per_tx_summary["event_count"] > 1)
    & per_tx_summary["first_event_pre_price"].notna()
    & np.isfinite(per_tx_summary["first_event_pre_price"])
    & (per_tx_summary["first_event_pre_price"] > 0)
].copy()

# Confirm each signature maps to only one pool and one start price.
assert not multi_event_starts["signature"].duplicated().any(), (
    "A signature maps to multiple multi-event price corrections."
)

start_price_map = multi_event_starts.set_index(
    "signature"
)["first_event_pre_price"].to_dict()

print("Multi-event transaction-start prices identified:", len(start_price_map))

# Rebuild from the original price series.
pumpswap_prestate = pumpswap_fixed.copy()
pumpswap_prestate["price_original"] = pumpswap_prestate["price"]

pumpswap_prestate["price_at_tx_start"] = (
    pumpswap_prestate["signature"]
    .map(start_price_map)
    .combine_first(pumpswap_prestate["price"])
)

pumpswap_prestate["start_price_corrected"] = (
    pumpswap_prestate["signature"].isin(start_price_map)
)

# Sort by verified on-chain order.
pumpswap_prestate = pumpswap_prestate.sort_values(
    ["mint", "pool", "block", "transactionIndex"],
    kind="stable"
).reset_index(drop=True)

# Returns from this transaction's starting price to the next
# transaction's starting price, within the same mint/pool.
g = pumpswap_prestate.groupby(["mint", "pool"], sort=False)

pumpswap_prestate["next_price_at_tx_start"] = (
    g["price_at_tx_start"].shift(-1)
)

valid = (
    pumpswap_prestate["price_at_tx_start"].gt(0)
    & pumpswap_prestate["next_price_at_tx_start"].gt(0)
    & pumpswap_prestate["price_at_tx_start"].notna()
    & pumpswap_prestate["next_price_at_tx_start"].notna()
)

pumpswap_prestate["next_return_pct"] = np.nan
pumpswap_prestate.loc[valid, "next_return_pct"] = (
    pumpswap_prestate.loc[valid, "next_price_at_tx_start"]
    / pumpswap_prestate.loc[valid, "price_at_tx_start"] - 1
) * 100

def summarize_returns(series):
    s = series.dropna()
    return {
        "valid_transitions": len(s),
        "median_pct": s.median(),
        "p01_pct": s.quantile(0.01),
        "p99_pct": s.quantile(0.99),
        "drops_le_minus50_pct": int((s <= -50).sum()),
        "drops_le_minus90_pct": int((s <= -90).sum()),
        "rises_ge_100_pct": int((s >= 100).sum()),
        "rises_ge_500_pct": int((s >= 500).sum()),
    }

# Compare the original transition definition against this coherent
# transaction-start series.
comparison = pd.DataFrame([
    {
        "series": "Original archive prices",
        **summarize_returns(
            pumpswap_prestate["pct_return_prev"]
        )
    },
    {
        "series": "Transaction-start prices (targeted correction)",
        **summarize_returns(
            pumpswap_prestate["next_return_pct"]
        )
    }
])

print("\nRETURN COMPARISON")
display(comparison)

print("\nTARGETED TRANSACTION-START PRICE CHANGES")
display(
    pumpswap_prestate[
        pumpswap_prestate["start_price_corrected"]
    ][
        [
            "mint", "pool", "block", "transactionIndex", "signature",
            "price_original", "price_at_tx_start",
            "next_price_at_tx_start", "next_return_pct"
        ]
    ].sort_values(["pool", "block", "transactionIndex"])
)

print("\nLARGEST POSITIVE NEXT RETURNS")
display(
    pumpswap_prestate.loc[valid]
    .nlargest(15, "next_return_pct")[
        [
            "mint", "pool", "block", "transactionIndex",
            "signature", "action", "price_at_tx_start",
            "next_price_at_tx_start", "next_return_pct",
            "start_price_corrected"
        ]
    ]
)

print("\nLARGEST NEGATIVE NEXT RETURNS")
display(
    pumpswap_prestate.loc[valid]
    .nsmallest(15, "next_return_pct")[
        [
            "mint", "pool", "block", "transactionIndex",
            "signature", "action", "price_at_tx_start",
            "next_price_at_tx_start", "next_return_pct",
            "start_price_corrected"
        ]
    ]
)

In [ ]:
import numpy as np
import pandas as pd

# Use final post-transaction prices for every successfully audited
# signature, not just signatures containing multiple swap events.
post_state_map = (
    per_tx_summary.loc[
        per_tx_summary["final_event_post_price"].notna()
        & np.isfinite(per_tx_summary["final_event_post_price"])
        & (per_tx_summary["final_event_post_price"] > 0),
        ["signature", "final_event_post_price"]
    ]
    .drop_duplicates("signature")
    .set_index("signature")["final_event_post_price"]
    .to_dict()
)

print("Audited signatures with valid post-state prices:", len(post_state_map))

# Preserve the original sequence and construct a targeted post-state probe.
post_state_probe = pumpswap_fixed.copy()
post_state_probe["price_original"] = pd.to_numeric(
    post_state_probe["price"], errors="coerce"
)

post_state_probe["price_post_state"] = (
    post_state_probe["signature"].map(post_state_map)
    .combine_first(post_state_probe["price_original"])
)

post_state_probe["post_state_corrected"] = (
    post_state_probe["signature"].isin(post_state_map)
)

post_state_probe = post_state_probe.sort_values(
    ["mint", "pool", "block", "transactionIndex"],
    kind="stable"
).reset_index(drop=True)

# Calculate comparable adjacent-row returns for each series.
g = post_state_probe.groupby(["mint", "pool"], sort=False)

post_state_probe["prev_original"] = g["price_original"].shift(1)
post_state_probe["prev_post_state"] = g["price_post_state"].shift(1)

valid_original = (
    post_state_probe["price_original"].gt(0)
    & post_state_probe["prev_original"].gt(0)
)

valid_post = (
    post_state_probe["price_post_state"].gt(0)
    & post_state_probe["prev_post_state"].gt(0)
)

post_state_probe["return_original_pct"] = np.where(
    valid_original,
    (
        post_state_probe["price_original"]
        / post_state_probe["prev_original"] - 1
    ) * 100,
    np.nan
)

post_state_probe["return_post_state_pct"] = np.where(
    valid_post,
    (
        post_state_probe["price_post_state"]
        / post_state_probe["prev_post_state"] - 1
    ) * 100,
    np.nan
)

def summarize(series):
    s = series.dropna()
    return {
        "valid_transitions": len(s),
        "median_pct": s.median(),
        "p01_pct": s.quantile(0.01),
        "p99_pct": s.quantile(0.99),
        "drops_le_minus50": int((s <= -50).sum()),
        "drops_le_minus90": int((s <= -90).sum()),
        "rises_ge_100": int((s >= 100).sum()),
        "rises_ge_500": int((s >= 500).sum()),
    }

print("\n========== TARGETED PRICE-CONVENTION COMPARISON ==========")
display(pd.DataFrame([
    {"series": "Original archive price", **summarize(
        post_state_probe["return_original_pct"]
    )},
    {"series": "Post-state for audited signatures only", **summarize(
        post_state_probe["return_post_state_pct"]
    )},
]))

# Inspect the remaining extreme transitions and whether their current
# signature has a measured transaction post-state.
remaining = post_state_probe[
    (
        (post_state_probe["return_post_state_pct"] <= -50)
        | (post_state_probe["return_post_state_pct"] >= 500)
    )
].copy()

print("\nExtreme transitions in the targeted post-state probe:", len(remaining))
display(
    remaining[
        [
            "mint", "pool", "block", "transactionIndex", "signature",
            "action", "price_original", "price_post_state",
            "return_original_pct", "return_post_state_pct",
            "post_state_corrected"
        ]
    ].sort_values("return_post_state_pct")
)